# pLM4CPP-XAI — Core Reproducibility Pipeline

This notebook is a cleaned, output-free version of the analysis notebook used for the study.

**Expected input files**
- `pLM4CPPs_dataset_CPP.xlsx`
- `pLM4CPPs_dataset_Non-CPP.xlsx`
- `kelm_dataset_CPP.csv`
- `kelm_dataset_Non-CPP.csv`

The default project directory is `/content/drive/MyDrive/pLM4CPP_XAI_2026`.
Change `PROJECT_DIR` in the first code cell if you want another location.

The notebook covers dataset QC and splitting, PLM embeddings, attention classifiers,
ensemble prediction, residue-level XAI, redundancy-adjusted consensus, residue/motif
analysis, faithfulness, cross-PLM conservation, physicochemical/positional analysis,
hotspot-threshold robustness, composition-preserving motif validation, and
sequence-level alanine-scanning faithfulness.


In [ ]:
# ============================================================
# STEP 1: Mount Google Drive and create permanent project folders
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import json
import datetime

# Permanent project location in Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

# Permanent folder structure
DIRS = {
    "code": PROJECT_DIR / "00_code",
    "data_original": PROJECT_DIR / "01_data_original",
    "data_processed": PROJECT_DIR / "02_data_processed",
    "embeddings": PROJECT_DIR / "03_embeddings",
    "models": PROJECT_DIR / "04_models",
    "predictions": PROJECT_DIR / "05_predictions",
    "xai": PROJECT_DIR / "06_xai",
    "results": PROJECT_DIR / "07_results",
    "checkpoints": PROJECT_DIR / "08_checkpoints",
    "logs": PROJECT_DIR / "09_logs",
}

# Create every folder
for name, folder in DIRS.items():
    folder.mkdir(parents=True, exist_ok=True)

# Subfolders for each protein language model
MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5"
]

for model_name in MODEL_NAMES:
    (DIRS["embeddings"] / model_name).mkdir(parents=True, exist_ok=True)
    (DIRS["models"] / model_name).mkdir(parents=True, exist_ok=True)
    (DIRS["predictions"] / model_name).mkdir(parents=True, exist_ok=True)
    (DIRS["xai"] / model_name).mkdir(parents=True, exist_ok=True)

# Results subfolders
RESULT_SUBDIRS = [
    "figures_main",
    "figures_SI",
    "tables_main",
    "tables_SI",
    "reports"
]

for subdir in RESULT_SUBDIRS:
    (DIRS["results"] / subdir).mkdir(parents=True, exist_ok=True)

# Permanent pipeline status file
STATUS_FILE = DIRS["checkpoints"] / "pipeline_status.json"

if STATUS_FILE.exists():
    with open(STATUS_FILE, "r", encoding="utf-8") as handle:
        PIPELINE_STATUS = json.load(handle)
else:
    PIPELINE_STATUS = {}

def save_pipeline_status():
    """Safely save pipeline progress to Google Drive."""
    temporary_file = STATUS_FILE.with_suffix(".tmp")

    with open(temporary_file, "w", encoding="utf-8") as handle:
        json.dump(PIPELINE_STATUS, handle, indent=2)

    os.replace(temporary_file, STATUS_FILE)

def mark_step_complete(step_name, output_files=None, details=None):
    """Record that a pipeline step has finished."""
    PIPELINE_STATUS[step_name] = {
        "completed": True,
        "completed_at": datetime.datetime.now().isoformat(),
        "output_files": [
            str(path) for path in (output_files or [])
        ],
        "details": details or {}
    }

    save_pipeline_status()
    print(f"[SAVED] {step_name}")

def step_is_complete(step_name, required_files=None):
    """Check whether a step and its expected outputs already exist."""
    record = PIPELINE_STATUS.get(step_name, {})

    if not record.get("completed", False):
        return False

    if required_files:
        return all(Path(path).exists() for path in required_files)

    return True

# Save this setup step
mark_step_complete(
    "01_project_directory_setup",
    output_files=[STATUS_FILE],
    details={
        "project_directory": str(PROJECT_DIR),
        "models": MODEL_NAMES
    }
)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nCreated folders:")
for name, folder in DIRS.items():
    print(f"{name:16s}: {folder}")

print("\nPreviously completed steps:")
for step in PIPELINE_STATUS:
    print(" -", step)

In [ ]:
# ============================================================
# STEP 2: Check GPU, CUDA, Python, and PyTorch
# ============================================================

import sys
import platform
import subprocess
import torch

print("Python version :", sys.version.split()[0])
print("Platform       :", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name       :", torch.cuda.get_device_name(0))
    print("CUDA version   :", torch.version.cuda)

    gpu_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )
    print(f"GPU memory     : {gpu_memory_gb:.2f} GB")
else:
    print("\nWARNING: GPU is not enabled.")
    print("Go to Runtime → Change runtime type → T4 GPU.")

print("\nNVIDIA GPU details:")
subprocess.run(["nvidia-smi"])

In [ ]:
# ============================================================
# STEP 3: Install packages required for the pipeline
# ============================================================

import subprocess
import sys

packages = [
    "fair-esm==2.0.0",
    "transformers",
    "sentencepiece",
    "accelerate",
    "captum",
    "biopython",
    "openpyxl",
    "pandas",
    "scikit-learn",
    "scipy",
    "umap-learn",
    "matplotlib",
    "seaborn",
    "logomaker",
    "tqdm",
    "joblib"
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade"
] + packages

print("Installing required packages...")
subprocess.check_call(command)

print("\nInstallation completed successfully.")

In [ ]:
# ============================================================
# STEP 3B: Test package imports and save environment versions
# ============================================================

import sys
import json
import datetime
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import sklearn
import scipy
import torch
import transformers
import esm
import captum
import Bio
import umap
import matplotlib
import seaborn
import logomaker
import sentencepiece
import accelerate
import openpyxl
import joblib

packages_to_record = [
    "torch",
    "fair-esm",
    "transformers",
    "sentencepiece",
    "accelerate",
    "captum",
    "biopython",
    "openpyxl",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "umap-learn",
    "matplotlib",
    "seaborn",
    "logomaker",
    "tqdm",
    "joblib"
]

environment_info = {
    "recorded_at": datetime.datetime.now().isoformat(),
    "python_version": sys.version,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "packages": {}
}

for package in packages_to_record:
    try:
        environment_info["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        environment_info["packages"][package] = "NOT FOUND"

environment_file = (
    DIRS["checkpoints"] / "environment_versions.json"
)

with open(environment_file, "w", encoding="utf-8") as handle:
    json.dump(environment_info, handle, indent=2)

mark_step_complete(
    "02_environment_setup",
    output_files=[environment_file],
    details={
        "python_version": sys.version.split()[0],
        "pytorch_version": torch.__version__,
        "gpu": environment_info["gpu_name"]
    }
)

print("All required packages imported successfully.\n")

print("Important versions:")
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("fair-esm     :", metadata.version("fair-esm"))
print("Captum       :", captum.__version__)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("GPU          :", environment_info["gpu_name"])

print("\nEnvironment record saved to:")
print(environment_file)

In [ ]:
# ============================================================
# STEP 4D: Preserve four original files and create processed datasets
# ============================================================

from google.colab import files
from pathlib import Path
import pandas as pd
import shutil
import re

ORIGINAL_DIR = DIRS["data_original"]
PROCESSED_DIR = DIRS["data_processed"]
TEMP_DIR = Path("/content")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = {
    "internal_cpp": "pLM4CPPs_dataset_CPP.xlsx",
    "internal_noncpp": "pLM4CPPs_dataset_Non-CPP.xlsx",
    "kelm_cpp": "kelm_dataset_CPP.csv",
    "kelm_noncpp": "kelm_dataset_Non-CPP.csv",
}

# ------------------------------------------------------------
# 1. Remove only previously created incorrect combined files
# ------------------------------------------------------------

incorrect_files = [
    ORIGINAL_DIR / "Final_non_redundant_sequences.xlsx",
    ORIGINAL_DIR / "kelm_dataset.csv",
    ORIGINAL_DIR / "KELM_external_dataset.csv",
]

for file_path in incorrect_files:
    if file_path.exists():
        file_path.unlink()
        print("[DELETED INCORRECT FILE]", file_path.name)

# ------------------------------------------------------------
# 2. Locate the four original files
# ------------------------------------------------------------

located_files = {}

for key, filename in required_files.items():
    drive_path = ORIGINAL_DIR / filename
    content_path = TEMP_DIR / filename

    if drive_path.exists():
        located_files[key] = drive_path
    elif content_path.exists():
        located_files[key] = content_path

missing_keys = [
    key for key in required_files
    if key not in located_files
]

# ------------------------------------------------------------
# 3. Upload missing originals if necessary
# ------------------------------------------------------------

if missing_keys:
    print("\nPlease upload these missing files:")

    for key in missing_keys:
        print(" -", required_files[key])

    uploaded = files.upload()

    for uploaded_name in uploaded:
        uploaded_path = TEMP_DIR / uploaded_name

        for key, expected_name in required_files.items():
            if uploaded_name == expected_name:
                located_files[key] = uploaded_path

still_missing = [
    required_files[key]
    for key in required_files
    if key not in located_files
]

if still_missing:
    raise FileNotFoundError(
        "These files are still missing:\n"
        + "\n".join(still_missing)
    )

# ------------------------------------------------------------
# 4. Save the four untouched originals permanently
# ------------------------------------------------------------

for key, source_path in located_files.items():
    destination_path = ORIGINAL_DIR / required_files[key]

    if source_path.resolve() != destination_path.resolve():
        shutil.copy2(source_path, destination_path)

    located_files[key] = destination_path
    print("[ORIGINAL SAVED]", destination_path.name)

# ------------------------------------------------------------
# 5. Functions for reading and cleaning
# ------------------------------------------------------------

def find_sequence_column(df):
    normalized_columns = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    possible_names = [
        "sequence",
        "seq",
        "peptide",
        "peptide sequence",
        "protein sequence",
    ]

    for name in possible_names:
        if name in normalized_columns:
            return normalized_columns[name]

    raise ValueError(
        "Sequence column not found. Available columns: "
        f"{df.columns.tolist()}"
    )


def clean_sequences(df, label):
    sequence_column = find_sequence_column(df)

    cleaned = pd.DataFrame({
        "sequence": df[sequence_column],
        "label": label,
    })

    cleaned["sequence"] = (
        cleaned["sequence"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
    )

    cleaned = cleaned[
        cleaned["sequence"].notna()
        & cleaned["sequence"].ne("")
        & cleaned["sequence"].ne("NAN")
    ].copy()

    # Retain only the 20 standard amino-acid letters
    standard_pattern = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY]+$")

    cleaned["valid_standard_sequence"] = (
        cleaned["sequence"]
        .apply(lambda sequence: bool(standard_pattern.fullmatch(sequence)))
    )

    invalid = cleaned[
        ~cleaned["valid_standard_sequence"]
    ].copy()

    cleaned = cleaned[
        cleaned["valid_standard_sequence"]
    ].drop(columns="valid_standard_sequence")

    cleaned = cleaned.drop_duplicates(
        subset=["sequence", "label"]
    ).reset_index(drop=True)

    return cleaned, invalid

# ------------------------------------------------------------
# 6. Read the four original files
# ------------------------------------------------------------

internal_cpp_raw = pd.read_excel(
    located_files["internal_cpp"]
)

internal_noncpp_raw = pd.read_excel(
    located_files["internal_noncpp"]
)

kelm_cpp_raw = pd.read_csv(
    located_files["kelm_cpp"]
)

kelm_noncpp_raw = pd.read_csv(
    located_files["kelm_noncpp"]
)

print("\nOriginal columns:")
print("Internal CPP    :", internal_cpp_raw.columns.tolist())
print("Internal non-CPP:", internal_noncpp_raw.columns.tolist())
print("KELM CPP        :", kelm_cpp_raw.columns.tolist())
print("KELM non-CPP    :", kelm_noncpp_raw.columns.tolist())

# ------------------------------------------------------------
# 7. Clean each class separately
# ------------------------------------------------------------

internal_cpp, internal_cpp_invalid = clean_sequences(
    internal_cpp_raw,
    label=1,
)

internal_noncpp, internal_noncpp_invalid = clean_sequences(
    internal_noncpp_raw,
    label=0,
)

kelm_cpp, kelm_cpp_invalid = clean_sequences(
    kelm_cpp_raw,
    label=1,
)

kelm_noncpp, kelm_noncpp_invalid = clean_sequences(
    kelm_noncpp_raw,
    label=0,
)

# ------------------------------------------------------------
# 8. Merge CPP and non-CPP datasets
# ------------------------------------------------------------

internal_processed = pd.concat(
    [internal_cpp, internal_noncpp],
    ignore_index=True,
)

kelm_processed = pd.concat(
    [kelm_cpp, kelm_noncpp],
    ignore_index=True,
)

# Add stable sequence IDs
internal_processed.insert(
    0,
    "sequence_id",
    [
        f"INT_{index:05d}"
        for index in range(1, len(internal_processed) + 1)
    ],
)

kelm_processed.insert(
    0,
    "sequence_id",
    [
        f"KELM_{index:04d}"
        for index in range(1, len(kelm_processed) + 1)
    ],
)

# Add sequence length
internal_processed["length"] = (
    internal_processed["sequence"].str.len()
)

kelm_processed["length"] = (
    kelm_processed["sequence"].str.len()
)

# ------------------------------------------------------------
# 9. Check conflicting labels within each dataset
# ------------------------------------------------------------

internal_conflicts = (
    internal_processed
    .groupby("sequence")["label"]
    .nunique()
)

internal_conflicting_sequences = internal_conflicts[
    internal_conflicts > 1
].index.tolist()

kelm_conflicts = (
    kelm_processed
    .groupby("sequence")["label"]
    .nunique()
)

kelm_conflicting_sequences = kelm_conflicts[
    kelm_conflicts > 1
].index.tolist()

if internal_conflicting_sequences:
    print(
        "\n[WARNING] Internal sequences with conflicting labels:",
        len(internal_conflicting_sequences),
    )

if kelm_conflicting_sequences:
    print(
        "[WARNING] KELM sequences with conflicting labels:",
        len(kelm_conflicting_sequences),
    )

# Do not silently remove conflicting sequences yet.
# Save them for inspection.
internal_conflict_file = (
    PROCESSED_DIR / "internal_conflicting_labels.csv"
)

kelm_conflict_file = (
    PROCESSED_DIR / "kelm_conflicting_labels.csv"
)

internal_processed[
    internal_processed["sequence"].isin(
        internal_conflicting_sequences
    )
].to_csv(internal_conflict_file, index=False)

kelm_processed[
    kelm_processed["sequence"].isin(
        kelm_conflicting_sequences
    )
].to_csv(kelm_conflict_file, index=False)

# ------------------------------------------------------------
# 10. Save processed datasets
# ------------------------------------------------------------

internal_processed_file = (
    PROCESSED_DIR / "internal_dataset_cleaned.csv"
)

kelm_processed_file = (
    PROCESSED_DIR / "kelm_external_dataset_cleaned.csv"
)

internal_processed.to_csv(
    internal_processed_file,
    index=False,
)

kelm_processed.to_csv(
    kelm_processed_file,
    index=False,
)

# Save rejected non-standard sequences
invalid_sequences = pd.concat(
    [
        internal_cpp_invalid.assign(source="internal_CPP"),
        internal_noncpp_invalid.assign(source="internal_nonCPP"),
        kelm_cpp_invalid.assign(source="KELM_CPP"),
        kelm_noncpp_invalid.assign(source="KELM_nonCPP"),
    ],
    ignore_index=True,
)

invalid_file = (
    PROCESSED_DIR / "rejected_nonstandard_sequences.csv"
)

invalid_sequences.to_csv(
    invalid_file,
    index=False,
)

# ------------------------------------------------------------
# 11. Display final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERNAL PROCESSED DATASET")
print("=" * 70)

print("Rows:", len(internal_processed))
print("Class counts:")
print(
    internal_processed["label"]
    .value_counts()
    .sort_index()
)
print(
    "Length range:",
    internal_processed["length"].min(),
    "to",
    internal_processed["length"].max(),
)

display(internal_processed.head())

print("\n" + "=" * 70)
print("KELM PROCESSED DATASET")
print("=" * 70)

print("Rows:", len(kelm_processed))
print("Class counts:")
print(
    kelm_processed["label"]
    .value_counts()
    .sort_index()
)
print(
    "Length range:",
    kelm_processed["length"].min(),
    "to",
    kelm_processed["length"].max(),
)

display(kelm_processed.head())

print("\nRejected non-standard sequences:", len(invalid_sequences))

# ------------------------------------------------------------
# 12. Record completion
# ------------------------------------------------------------

mark_step_complete(
    "03_correct_combined_datasets",
    output_files=[
        internal_processed_file,
        kelm_processed_file,
        invalid_file,
        internal_conflict_file,
        kelm_conflict_file,
    ],
    details={
        "internal_rows": int(len(internal_processed)),
        "internal_class_counts": {
            str(key): int(value)
            for key, value in (
                internal_processed["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "kelm_rows": int(len(kelm_processed)),
        "kelm_class_counts": {
            str(key): int(value)
            for key, value in (
                kelm_processed["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "rejected_nonstandard_sequences": int(
            len(invalid_sequences)
        ),
        "internal_conflicting_sequences": int(
            len(internal_conflicting_sequences)
        ),
        "kelm_conflicting_sequences": int(
            len(kelm_conflicting_sequences)
        ),
    },
)

print("\nSaved processed internal dataset:")
print(internal_processed_file)

print("\nSaved processed KELM dataset:")
print(kelm_processed_file)

print("\nOriginal files preserved in:")
print(ORIGINAL_DIR)

In [ ]:
# ============================================================
# STEP 5: Dataset QC, duplicates, conflicts, and KELM leakage
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

PROCESSED_DIR = DIRS["data_processed"]
CHECKPOINT_DIR = DIRS["checkpoints"]

internal_file = (
    PROCESSED_DIR / "internal_dataset_cleaned.csv"
)

kelm_file = (
    PROCESSED_DIR / "kelm_external_dataset_cleaned.csv"
)

rejected_file = (
    PROCESSED_DIR / "rejected_nonstandard_sequences.csv"
)

# ------------------------------------------------------------
# 1. Load processed datasets
# ------------------------------------------------------------

internal_df = pd.read_csv(internal_file)
kelm_df = pd.read_csv(kelm_file)

if rejected_file.exists():
    rejected_df = pd.read_csv(rejected_file)
else:
    rejected_df = pd.DataFrame()

print("=" * 70)
print("REJECTED NON-STANDARD SEQUENCES")
print("=" * 70)

print("Rejected rows:", len(rejected_df))

if not rejected_df.empty:
    display(rejected_df)

# ------------------------------------------------------------
# 2. Check exact duplicate rows
# ------------------------------------------------------------

internal_exact_duplicates = internal_df.duplicated(
    subset=["sequence", "label"],
    keep=False,
)

kelm_exact_duplicates = kelm_df.duplicated(
    subset=["sequence", "label"],
    keep=False,
)

print("\n" + "=" * 70)
print("EXACT DUPLICATE CHECK")
print("=" * 70)

print(
    "Internal duplicate rows:",
    int(internal_exact_duplicates.sum())
)

print(
    "KELM duplicate rows:",
    int(kelm_exact_duplicates.sum())
)

# ------------------------------------------------------------
# 3. Check conflicting labels
# ------------------------------------------------------------

internal_label_counts = (
    internal_df
    .groupby("sequence")["label"]
    .nunique()
)

kelm_label_counts = (
    kelm_df
    .groupby("sequence")["label"]
    .nunique()
)

internal_conflict_sequences = (
    internal_label_counts[
        internal_label_counts > 1
    ]
    .index
    .tolist()
)

kelm_conflict_sequences = (
    kelm_label_counts[
        kelm_label_counts > 1
    ]
    .index
    .tolist()
)

print("\n" + "=" * 70)
print("CONFLICTING LABEL CHECK")
print("=" * 70)

print(
    "Internal sequences with conflicting labels:",
    len(internal_conflict_sequences)
)

print(
    "KELM sequences with conflicting labels:",
    len(kelm_conflict_sequences)
)

if internal_conflict_sequences:
    display(
        internal_df[
            internal_df["sequence"].isin(
                internal_conflict_sequences
            )
        ].sort_values("sequence")
    )

if kelm_conflict_sequences:
    display(
        kelm_df[
            kelm_df["sequence"].isin(
                kelm_conflict_sequences
            )
        ].sort_values("sequence")
    )

# ------------------------------------------------------------
# 4. Check overlap between internal and KELM datasets
# ------------------------------------------------------------

internal_sequences = set(internal_df["sequence"])
kelm_sequences = set(kelm_df["sequence"])

overlap_sequences = sorted(
    internal_sequences.intersection(kelm_sequences)
)

overlap_df = kelm_df[
    kelm_df["sequence"].isin(overlap_sequences)
].copy()

if len(overlap_df) > 0:
    internal_labels = (
        internal_df[
            internal_df["sequence"].isin(overlap_sequences)
        ][["sequence", "label"]]
        .drop_duplicates()
        .rename(columns={"label": "internal_label"})
    )

    overlap_df = overlap_df.merge(
        internal_labels,
        on="sequence",
        how="left"
    )

    overlap_df = overlap_df.rename(
        columns={"label": "kelm_label"}
    )

    overlap_df["same_label"] = (
        overlap_df["kelm_label"]
        == overlap_df["internal_label"]
    )

print("\n" + "=" * 70)
print("INTERNAL–KELM EXACT OVERLAP")
print("=" * 70)

print("Exact overlapping sequences:", len(overlap_sequences))

if len(overlap_df) > 0:
    print("\nOverlap by KELM label:")
    print(
        overlap_df["kelm_label"]
        .value_counts()
        .sort_index()
    )

    print("\nLabel agreement:")
    print(
        overlap_df["same_label"]
        .value_counts()
    )

    display(overlap_df.head(20))

# Save overlap report
overlap_report_file = (
    PROCESSED_DIR / "internal_kelm_exact_overlap.csv"
)

overlap_df.to_csv(
    overlap_report_file,
    index=False
)

# ------------------------------------------------------------
# 5. Create leakage-free datasets
# ------------------------------------------------------------

# Remove conflicting-label sequences from internal dataset
internal_qc = internal_df[
    ~internal_df["sequence"].isin(
        internal_conflict_sequences
    )
].copy()

# Remove conflicting-label sequences from KELM dataset
kelm_qc = kelm_df[
    ~kelm_df["sequence"].isin(
        kelm_conflict_sequences
    )
].copy()

# Remove from KELM any sequence already found in internal data
kelm_qc = kelm_qc[
    ~kelm_qc["sequence"].isin(
        set(internal_qc["sequence"])
    )
].copy()

# Remove any remaining exact duplicates
internal_qc = internal_qc.drop_duplicates(
    subset=["sequence"],
    keep="first"
).reset_index(drop=True)

kelm_qc = kelm_qc.drop_duplicates(
    subset=["sequence"],
    keep="first"
).reset_index(drop=True)

# Reassign stable IDs after QC
internal_qc["sequence_id"] = [
    f"INTQC_{index:05d}"
    for index in range(1, len(internal_qc) + 1)
]

kelm_qc["sequence_id"] = [
    f"KELMQC_{index:04d}"
    for index in range(1, len(kelm_qc) + 1)
]

# Put columns in consistent order
internal_qc = internal_qc[
    ["sequence_id", "sequence", "label", "length"]
]

kelm_qc = kelm_qc[
    ["sequence_id", "sequence", "label", "length"]
]

# ------------------------------------------------------------
# 6. Save QC datasets
# ------------------------------------------------------------

internal_qc_file = (
    PROCESSED_DIR / "internal_dataset_qc.csv"
)

kelm_qc_file = (
    PROCESSED_DIR / "kelm_external_dataset_qc.csv"
)

internal_qc.to_csv(
    internal_qc_file,
    index=False
)

kelm_qc.to_csv(
    kelm_qc_file,
    index=False
)

# ------------------------------------------------------------
# 7. Create dataset fingerprints
# ------------------------------------------------------------

def dataframe_sha256(df):
    csv_text = df.to_csv(
        index=False,
        lineterminator="\n"
    )

    return hashlib.sha256(
        csv_text.encode("utf-8")
    ).hexdigest()

dataset_manifest = {
    "internal_dataset": {
        "file": str(internal_qc_file),
        "rows": int(len(internal_qc)),
        "class_counts": {
            str(key): int(value)
            for key, value in (
                internal_qc["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "minimum_length": int(
            internal_qc["length"].min()
        ),
        "maximum_length": int(
            internal_qc["length"].max()
        ),
        "sha256": dataframe_sha256(
            internal_qc
        ),
    },
    "kelm_external_dataset": {
        "file": str(kelm_qc_file),
        "rows": int(len(kelm_qc)),
        "class_counts": {
            str(key): int(value)
            for key, value in (
                kelm_qc["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "minimum_length": int(
            kelm_qc["length"].min()
        ),
        "maximum_length": int(
            kelm_qc["length"].max()
        ),
        "sha256": dataframe_sha256(
            kelm_qc
        ),
    },
    "removed": {
        "internal_conflicting_sequences": int(
            len(internal_conflict_sequences)
        ),
        "kelm_conflicting_sequences": int(
            len(kelm_conflict_sequences)
        ),
        "internal_kelm_exact_overlap": int(
            len(overlap_sequences)
        ),
        "rejected_nonstandard_sequences": int(
            len(rejected_df)
        ),
    },
}

manifest_file = (
    CHECKPOINT_DIR / "dataset_manifest.json"
)

with open(
    manifest_file,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        dataset_manifest,
        handle,
        indent=2
    )

# ------------------------------------------------------------
# 8. Display final QC summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL INTERNAL QC DATASET")
print("=" * 70)

print("Rows:", len(internal_qc))
print("Class counts:")
print(
    internal_qc["label"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 70)
print("FINAL LEAKAGE-FREE KELM DATASET")
print("=" * 70)

print("Rows:", len(kelm_qc))
print("Class counts:")
print(
    kelm_qc["label"]
    .value_counts()
    .sort_index()
)

print("\nSaved files:")
print(internal_qc_file)
print(kelm_qc_file)
print(overlap_report_file)
print(manifest_file)

# ------------------------------------------------------------
# 9. Save checkpoint
# ------------------------------------------------------------

mark_step_complete(
    "04_dataset_quality_control",
    output_files=[
        internal_qc_file,
        kelm_qc_file,
        overlap_report_file,
        manifest_file,
    ],
    details={
        "internal_final_rows": int(
            len(internal_qc)
        ),
        "kelm_final_rows": int(
            len(kelm_qc)
        ),
        "exact_internal_kelm_overlap": int(
            len(overlap_sequences)
        ),
        "internal_conflicts": int(
            len(internal_conflict_sequences)
        ),
        "kelm_conflicts": int(
            len(kelm_conflict_sequences)
        ),
    },
)

print("\nDataset quality-control step completed.")

In [ ]:
# ============================================================
# STEP 6: Create permanent stratified train/validation/test splits
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split

PROCESSED_DIR = DIRS["data_processed"]
CHECKPOINT_DIR = DIRS["checkpoints"]

RANDOM_SEED = 42

internal_qc_file = (
    PROCESSED_DIR / "internal_dataset_qc.csv"
)

kelm_qc_file = (
    PROCESSED_DIR / "kelm_external_dataset_qc.csv"
)

# ------------------------------------------------------------
# 1. Load final QC datasets
# ------------------------------------------------------------

internal_df = pd.read_csv(internal_qc_file)
kelm_df = pd.read_csv(kelm_qc_file)

print("Internal dataset shape:", internal_df.shape)
print("KELM dataset shape    :", kelm_df.shape)

# ------------------------------------------------------------
# 2. Create train and temporary split
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    internal_df,
    test_size=0.30,
    stratify=internal_df["label"],
    random_state=RANDOM_SEED,
    shuffle=True,
)

# Split temporary set equally into validation and test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED,
    shuffle=True,
)

# ------------------------------------------------------------
# 3. Reset indices
# ------------------------------------------------------------

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Add split-specific row IDs
train_df.insert(
    0,
    "split_row_id",
    [
        f"TRAIN_{i:05d}"
        for i in range(1, len(train_df) + 1)
    ],
)

val_df.insert(
    0,
    "split_row_id",
    [
        f"VAL_{i:05d}"
        for i in range(1, len(val_df) + 1)
    ],
)

test_df.insert(
    0,
    "split_row_id",
    [
        f"TEST_{i:05d}"
        for i in range(1, len(test_df) + 1)
    ],
)

# ------------------------------------------------------------
# 4. Verify that no sequence appears in multiple splits
# ------------------------------------------------------------

train_sequences = set(train_df["sequence"])
val_sequences = set(val_df["sequence"])
test_sequences = set(test_df["sequence"])

train_val_overlap = train_sequences.intersection(val_sequences)
train_test_overlap = train_sequences.intersection(test_sequences)
val_test_overlap = val_sequences.intersection(test_sequences)

if train_val_overlap:
    raise ValueError(
        f"Train-validation overlap found: {len(train_val_overlap)}"
    )

if train_test_overlap:
    raise ValueError(
        f"Train-test overlap found: {len(train_test_overlap)}"
    )

if val_test_overlap:
    raise ValueError(
        f"Validation-test overlap found: {len(val_test_overlap)}"
    )

# ------------------------------------------------------------
# 5. Save permanent split files
# ------------------------------------------------------------

train_file = PROCESSED_DIR / "train_split.csv"
val_file = PROCESSED_DIR / "validation_split.csv"
test_file = PROCESSED_DIR / "internal_test_split.csv"

train_df.to_csv(train_file, index=False)
val_df.to_csv(val_file, index=False)
test_df.to_csv(test_file, index=False)

# Save one combined split-assignment file
split_assignment_df = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)

split_assignment_file = (
    PROCESSED_DIR / "internal_split_assignments.csv"
)

split_assignment_df.to_csv(
    split_assignment_file,
    index=False,
)

# ------------------------------------------------------------
# 6. Create split summary
# ------------------------------------------------------------

def summarize_split(df, name):
    counts = (
        df["label"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    return {
        "name": name,
        "rows": int(len(df)),
        "class_0": int(counts.get(0, 0)),
        "class_1": int(counts.get(1, 0)),
        "minimum_length": int(df["length"].min()),
        "maximum_length": int(df["length"].max()),
        "mean_length": float(df["length"].mean()),
    }

split_summary = {
    "random_seed": RANDOM_SEED,
    "split_ratio": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "train": summarize_split(train_df, "train"),
    "validation": summarize_split(val_df, "validation"),
    "test": summarize_split(test_df, "test"),
    "kelm_external": summarize_split(kelm_df, "kelm_external"),
    "overlap_checks": {
        "train_validation": len(train_val_overlap),
        "train_test": len(train_test_overlap),
        "validation_test": len(val_test_overlap),
    },
}

split_summary_file = (
    CHECKPOINT_DIR / "split_summary.json"
)

with open(
    split_summary_file,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        split_summary,
        handle,
        indent=2
    )

# ------------------------------------------------------------
# 7. Display summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAIN SPLIT")
print("=" * 70)

print("Rows:", len(train_df))
print(train_df["label"].value_counts().sort_index())
print(
    "Length range:",
    train_df["length"].min(),
    "to",
    train_df["length"].max(),
)

print("\n" + "=" * 70)
print("VALIDATION SPLIT")
print("=" * 70)

print("Rows:", len(val_df))
print(val_df["label"].value_counts().sort_index())
print(
    "Length range:",
    val_df["length"].min(),
    "to",
    val_df["length"].max(),
)

print("\n" + "=" * 70)
print("INTERNAL TEST SPLIT")
print("=" * 70)

print("Rows:", len(test_df))
print(test_df["label"].value_counts().sort_index())
print(
    "Length range:",
    test_df["length"].min(),
    "to",
    test_df["length"].max(),
)

print("\n" + "=" * 70)
print("OVERLAP CHECK")
print("=" * 70)

print("Train-validation overlap:", len(train_val_overlap))
print("Train-test overlap      :", len(train_test_overlap))
print("Validation-test overlap :", len(val_test_overlap))

print("\nSaved files:")
print(train_file)
print(val_file)
print(test_file)
print(split_assignment_file)
print(split_summary_file)

# ------------------------------------------------------------
# 8. Save checkpoint
# ------------------------------------------------------------

mark_step_complete(
    "05_fixed_internal_data_splits",
    output_files=[
        train_file,
        val_file,
        test_file,
        split_assignment_file,
        split_summary_file,
    ],
    details={
        "random_seed": RANDOM_SEED,
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_validation_overlap": int(
            len(train_val_overlap)
        ),
        "train_test_overlap": int(
            len(train_test_overlap)
        ),
        "validation_test_overlap": int(
            len(val_test_overlap)
        ),
    },
)

print("\nPermanent train/validation/test splits created successfully.")

In [ ]:
# ============================================================
# STEP 7: MASTER RESTART-SAFE EMBEDDING PIPELINE
#
# Models:
#   1. ESM2-320
#   2. ESM2-640
#   3. ESM2-1280
#   4. ProtT5
#
# Datasets:
#   1. Train
#   2. Validation
#   3. Internal test
#   4. KELM external
#
# Outputs:
#   Per-residue embeddings: float16 .npy
#   Residue masks: uint8 .npy
#   Metadata: CSV
#   Restart-safe batch chunks: compressed .npz
# ============================================================

from pathlib import Path
import gc
import json
import math
import os
import time
import traceback

import numpy as np
import pandas as pd
import torch
import esm

from transformers import T5Tokenizer, T5EncoderModel


# ============================================================
# 1. GLOBAL CONFIGURATION
# ============================================================

SEED = 42
MAX_LEN = 61

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROCESSED_DIR = DIRS["data_processed"]
EMBEDDING_ROOT = DIRS["embeddings"]
CHECKPOINT_DIR = DIRS["checkpoints"]

EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# Exact PLMs from the original pLM4CPP-XAI pipeline
MODEL_CONFIGS = {
    "ESM2_320": {
        "type": "esm",
        "loader": "esm2_t6_8M_UR50D",
        "layer": 6,
        "embedding_dimension": 320,
        "batch_size": 64,
    },

    "ESM2_640": {
        "type": "esm",
        "loader": "esm2_t30_150M_UR50D",
        "layer": 30,
        "embedding_dimension": 640,
        "batch_size": 24,
    },

    "ESM2_1280": {
        "type": "esm",
        "loader": "esm2_t33_650M_UR50D",
        "layer": 33,
        "embedding_dimension": 1280,
        "batch_size": 8,
    },

    "ProtT5": {
        "type": "prott5",
        "loader": "Rostlab/prot_t5_xl_uniref50",
        "layer": None,
        "embedding_dimension": 1024,
        "batch_size": 8,
    },
}


DATASET_FILES = {
    "train": (
        PROCESSED_DIR / "train_split.csv"
    ),

    "validation": (
        PROCESSED_DIR / "validation_split.csv"
    ),

    "internal_test": (
        PROCESSED_DIR / "internal_test_split.csv"
    ),

    "kelm_external": (
        PROCESSED_DIR / "kelm_external_dataset_qc.csv"
    ),
}


print("=" * 75)
print("MASTER PLM EMBEDDING PIPELINE")
print("=" * 75)

print("Device :", DEVICE)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

print("Models :", list(MODEL_CONFIGS))
print("Datasets:", list(DATASET_FILES))
print("Maximum sequence length:", MAX_LEN)


# ============================================================
# 2. VERIFY DATASET FILES
# ============================================================

for dataset_name, dataset_file in DATASET_FILES.items():

    if not dataset_file.exists():
        raise FileNotFoundError(
            f"Missing dataset file for {dataset_name}:\n"
            f"{dataset_file}"
        )

    dataset_check = pd.read_csv(dataset_file)

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "length",
    }

    missing_columns = (
        required_columns
        - set(dataset_check.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if dataset_check["length"].max() > MAX_LEN:
        raise ValueError(
            f"{dataset_name} contains a sequence longer "
            f"than MAX_LEN={MAX_LEN}."
        )

    print(
        f"[DATASET FOUND] {dataset_name:15s} "
        f"rows={len(dataset_check)}"
    )


# ============================================================
# 3. GENERAL HELPER FUNCTIONS
# ============================================================

def clear_memory():
    """Release CPU and GPU memory."""

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def atomic_save_npz(final_path, **arrays):
    """
    Save an NPZ file using a temporary file first.

    This reduces the possibility of retaining a partially
    written file after a Colab interruption.
    """

    final_path = Path(final_path)

    temporary_path = Path(
        str(final_path) + ".temporary.npz"
    )

    np.savez_compressed(
        temporary_path,
        **arrays,
    )

    temporary_path.replace(final_path)


def validate_saved_chunk(
    chunk_file,
    expected_ids,
    expected_rows,
    embedding_dimension,
):
    """Check whether an existing batch chunk is valid."""

    if not chunk_file.exists():
        return False

    try:
        chunk_data = np.load(
            chunk_file,
            allow_pickle=True,
        )

        saved_ids = (
            chunk_data["sequence_id"]
            .astype(str)
        )

        saved_embeddings = (
            chunk_data["embeddings"]
        )

        saved_masks = (
            chunk_data["masks"]
        )

        valid = (
            np.array_equal(
                saved_ids,
                expected_ids.astype(str),
            )
            and saved_embeddings.shape
            == (
                expected_rows,
                MAX_LEN,
                embedding_dimension,
            )
            and saved_masks.shape
            == (
                expected_rows,
                MAX_LEN,
            )
        )

        chunk_data.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 4. ESM MODEL LOADING
# ============================================================

def load_esm_model(config):
    """Load one ESM2 model."""

    loader_name = config["loader"]

    print(f"\nLoading {loader_name}...")

    loader_function = getattr(
        esm.pretrained,
        loader_name,
    )

    model, alphabet = loader_function()

    model = model.to(DEVICE)
    model.eval()

    batch_converter = (
        alphabet.get_batch_converter()
    )

    print(
        f"Loaded {loader_name} on {DEVICE}"
    )

    return model, alphabet, batch_converter


# ============================================================
# 5. PROTT5 MODEL LOADING
# ============================================================

def load_prott5_model(config):
    """Load ProtT5 tokenizer and encoder."""

    loader_name = config["loader"]

    print(f"\nLoading {loader_name} tokenizer...")

    tokenizer = T5Tokenizer.from_pretrained(
        loader_name,
        do_lower_case=False,
    )

    print(f"Loading {loader_name} encoder...")

    model_dtype = (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )

    model = T5EncoderModel.from_pretrained(
        loader_name,
        torch_dtype=model_dtype,
        low_cpu_mem_usage=True,
    )

    model = model.to(DEVICE)
    model.eval()

    print(
        f"Loaded {loader_name} on {DEVICE}"
    )

    return model, tokenizer


# ============================================================
# 6. ESM PER-RESIDUE EMBEDDINGS
# ============================================================

@torch.no_grad()
def generate_esm_batch(
    batch_df,
    model,
    batch_converter,
    layer,
    embedding_dimension,
):
    """
    Generate padded per-residue ESM2 embeddings.

    Output:
        embeddings:
            shape = batch × MAX_LEN × embedding_dimension

        masks:
            shape = batch × MAX_LEN
    """

    batch_items = [
        (
            str(row.sequence_id),
            str(row.sequence),
        )
        for row in batch_df.itertuples(index=False)
    ]

    _, sequences, tokens = (
        batch_converter(batch_items)
    )

    tokens = tokens.to(DEVICE)

    autocast_enabled = (
        torch.cuda.is_available()
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=autocast_enabled,
    ):
        results = model(
            tokens,
            repr_layers=[layer],
            return_contacts=False,
        )

    token_representations = (
        results["representations"][layer]
    )

    batch_size_actual = len(batch_df)

    padded_embeddings = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )

    masks = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    for row_index, sequence in enumerate(
        sequences
    ):
        sequence_length = min(
            len(sequence),
            MAX_LEN,
        )

        # ESM:
        # token 0 is BOS.
        # Residue tokens begin at position 1.
        residue_embedding = (
            token_representations[
                row_index,
                1:sequence_length + 1,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        padded_embeddings[
            row_index,
            :sequence_length,
            :
        ] = residue_embedding

        masks[
            row_index,
            :sequence_length
        ] = 1

    del tokens
    del results
    del token_representations

    return padded_embeddings, masks


# ============================================================
# 7. PROTT5 PER-RESIDUE EMBEDDINGS
# ============================================================

@torch.no_grad()
def generate_prott5_batch(
    batch_df,
    model,
    tokenizer,
    embedding_dimension,
):
    """
    Generate padded per-residue ProtT5 embeddings.

    ProtT5 requires amino acids separated by spaces.
    """

    raw_sequences = (
        batch_df["sequence"]
        .astype(str)
        .tolist()
    )

    spaced_sequences = [
        " ".join(list(sequence))
        for sequence in raw_sequences
    ]

    encoded = tokenizer(
        spaced_sequences,
        add_special_tokens=True,
        padding=True,
        return_tensors="pt",
    )

    input_ids = (
        encoded["input_ids"]
        .to(DEVICE)
    )

    attention_mask = (
        encoded["attention_mask"]
        .to(DEVICE)
    )

    autocast_enabled = (
        torch.cuda.is_available()
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=autocast_enabled,
    ):
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    hidden_states = output.last_hidden_state

    batch_size_actual = len(batch_df)

    padded_embeddings = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )

    masks = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    for row_index, sequence in enumerate(
        raw_sequences
    ):
        sequence_length = min(
            len(sequence),
            MAX_LEN,
        )

        # ProtT5 places residue tokens first and EOS last.
        residue_embedding = (
            hidden_states[
                row_index,
                :sequence_length,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        padded_embeddings[
            row_index,
            :sequence_length,
            :
        ] = residue_embedding

        masks[
            row_index,
            :sequence_length
        ] = 1

    del input_ids
    del attention_mask
    del encoded
    del output
    del hidden_states

    return padded_embeddings, masks


# ============================================================
# 8. PROCESS ONE DATASET FOR ONE MODEL
# ============================================================

def process_dataset_for_model(
    model_key,
    config,
    dataset_name,
    dataset_file,
    model,
    tokenizer_or_converter,
):
    """
    Generate restart-safe batch chunks and then create
    final memory-mapped NPY files.
    """

    dataset_df = pd.read_csv(
        dataset_file
    )

    dataset_df["sequence_id"] = (
        dataset_df["sequence_id"]
        .astype(str)
    )

    number_of_rows = len(dataset_df)

    embedding_dimension = (
        config["embedding_dimension"]
    )

    batch_size = config["batch_size"]

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    output_directory = (
        EMBEDDING_ROOT
        / model_key
        / dataset_name
    )

    chunk_directory = (
        output_directory
        / "batch_chunks"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_embedding_file = (
        output_directory
        / "X_per_residue.npy"
    )

    final_mask_file = (
        output_directory
        / "M_masks.npy"
    )

    final_metadata_file = (
        output_directory
        / "metadata.csv"
    )

    completion_file = (
        output_directory
        / "COMPLETE.json"
    )

    print("\n" + "=" * 75)
    print(
        f"{model_key} — {dataset_name}"
    )
    print("=" * 75)

    print("Rows              :", number_of_rows)
    print("Embedding dimension:", embedding_dimension)
    print("Batch size         :", batch_size)
    print("Number of batches  :", number_of_batches)

    # --------------------------------------------------------
    # Check whether final output is already complete
    # --------------------------------------------------------

    if (
        completion_file.exists()
        and final_embedding_file.exists()
        and final_mask_file.exists()
        and final_metadata_file.exists()
    ):
        try:
            existing_embeddings = np.load(
                final_embedding_file,
                mmap_mode="r",
            )

            existing_masks = np.load(
                final_mask_file,
                mmap_mode="r",
            )

            valid_final_output = (
                existing_embeddings.shape
                == (
                    number_of_rows,
                    MAX_LEN,
                    embedding_dimension,
                )
                and existing_masks.shape
                == (
                    number_of_rows,
                    MAX_LEN,
                )
            )

            del existing_embeddings
            del existing_masks

            if valid_final_output:
                print(
                    "[SKIP COMPLETE] Final files "
                    "already exist and are valid."
                )

                return {
                    "model": model_key,
                    "dataset": dataset_name,
                    "rows": number_of_rows,
                    "embedding_dimension": (
                        embedding_dimension
                    ),
                    "embedding_file": str(
                        final_embedding_file
                    ),
                    "mask_file": str(
                        final_mask_file
                    ),
                    "metadata_file": str(
                        final_metadata_file
                    ),
                    "status": "previously_complete",
                }

        except Exception:
            print(
                "[REBUILD] Existing final files "
                "failed validation."
            )

    # --------------------------------------------------------
    # Generate batch chunks
    # --------------------------------------------------------

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        batch_df = dataset_df.iloc[
            start_index:end_index
        ].copy()

        expected_ids = (
            batch_df["sequence_id"]
            .astype(str)
            .to_numpy()
        )

        chunk_file = (
            chunk_directory
            / f"batch_{batch_number:05d}.npz"
        )

        valid_chunk = validate_saved_chunk(
            chunk_file=chunk_file,
            expected_ids=expected_ids,
            expected_rows=len(batch_df),
            embedding_dimension=(
                embedding_dimension
            ),
        )

        if valid_chunk:
            print(
                f"[SKIP CHUNK] {model_key} | "
                f"{dataset_name} | "
                f"{batch_number + 1}/"
                f"{number_of_batches}"
            )

            continue

        if chunk_file.exists():
            chunk_file.unlink()

        batch_start_time = time.time()

        try:
            if config["type"] == "esm":
                batch_embeddings, batch_masks = (
                    generate_esm_batch(
                        batch_df=batch_df,
                        model=model,
                        batch_converter=(
                            tokenizer_or_converter
                        ),
                        layer=config["layer"],
                        embedding_dimension=(
                            embedding_dimension
                        ),
                    )
                )

            elif config["type"] == "prott5":
                batch_embeddings, batch_masks = (
                    generate_prott5_batch(
                        batch_df=batch_df,
                        model=model,
                        tokenizer=(
                            tokenizer_or_converter
                        ),
                        embedding_dimension=(
                            embedding_dimension
                        ),
                    )
                )

            else:
                raise ValueError(
                    f"Unknown model type: "
                    f"{config['type']}"
                )

        except torch.cuda.OutOfMemoryError:

            clear_memory()

            raise RuntimeError(
                f"GPU memory error for {model_key}. "
                f"Reduce its batch_size in MODEL_CONFIGS "
                f"and rerun this same cell. Completed "
                f"chunks will be skipped."
            )

        atomic_save_npz(
            chunk_file,
            sequence_id=expected_ids,
            embeddings=batch_embeddings,
            masks=batch_masks,
        )

        elapsed_seconds = (
            time.time() - batch_start_time
        )

        print(
            f"[SAVED CHUNK] {model_key} | "
            f"{dataset_name} | "
            f"{batch_number + 1}/"
            f"{number_of_batches} | "
            f"rows {start_index}:{end_index} | "
            f"{elapsed_seconds:.1f} seconds"
        )

        del batch_embeddings
        del batch_masks

        clear_memory()

    # --------------------------------------------------------
    # Assemble final NPY arrays without loading all data
    # into RAM simultaneously
    # --------------------------------------------------------

    print(
        "\nAssembling final memory-mapped files..."
    )

    temporary_embedding_file = (
        output_directory
        / "X_per_residue.temporary.npy"
    )

    temporary_mask_file = (
        output_directory
        / "M_masks.temporary.npy"
    )

    embedding_memmap = (
        np.lib.format.open_memmap(
            temporary_embedding_file,
            mode="w+",
            dtype=np.float16,
            shape=(
                number_of_rows,
                MAX_LEN,
                embedding_dimension,
            ),
        )
    )

    mask_memmap = (
        np.lib.format.open_memmap(
            temporary_mask_file,
            mode="w+",
            dtype=np.uint8,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        chunk_file = (
            chunk_directory
            / f"batch_{batch_number:05d}.npz"
        )

        if not chunk_file.exists():
            raise FileNotFoundError(
                f"Missing chunk:\n{chunk_file}"
            )

        chunk_data = np.load(
            chunk_file,
            allow_pickle=True,
        )

        embedding_memmap[
            start_index:end_index
        ] = chunk_data["embeddings"]

        mask_memmap[
            start_index:end_index
        ] = chunk_data["masks"]

        chunk_data.close()

    embedding_memmap.flush()
    mask_memmap.flush()

    del embedding_memmap
    del mask_memmap

    os.replace(
        temporary_embedding_file,
        final_embedding_file,
    )

    os.replace(
        temporary_mask_file,
        final_mask_file,
    )

    # Metadata preserves exact row order
    metadata_df = dataset_df[
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    metadata_df.insert(
        0,
        "row_index",
        np.arange(len(metadata_df)),
    )

    metadata_df["dataset"] = (
        dataset_name
    )

    metadata_df["model"] = model_key

    metadata_df.to_csv(
        final_metadata_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validate final output
    # --------------------------------------------------------

    final_embeddings_check = np.load(
        final_embedding_file,
        mmap_mode="r",
    )

    final_masks_check = np.load(
        final_mask_file,
        mmap_mode="r",
    )

    expected_embedding_shape = (
        number_of_rows,
        MAX_LEN,
        embedding_dimension,
    )

    expected_mask_shape = (
        number_of_rows,
        MAX_LEN,
    )

    if (
        final_embeddings_check.shape
        != expected_embedding_shape
    ):
        raise ValueError(
            f"Incorrect final embedding shape for "
            f"{model_key}/{dataset_name}: "
            f"{final_embeddings_check.shape}"
        )

    if (
        final_masks_check.shape
        != expected_mask_shape
    ):
        raise ValueError(
            f"Incorrect mask shape for "
            f"{model_key}/{dataset_name}: "
            f"{final_masks_check.shape}"
        )

    del final_embeddings_check
    del final_masks_check

    completion_data = {
        "model": model_key,
        "loader": config["loader"],
        "model_type": config["type"],
        "dataset": dataset_name,
        "source_dataset_file": str(
            dataset_file
        ),
        "rows": int(number_of_rows),
        "maximum_length": int(MAX_LEN),
        "embedding_dimension": int(
            embedding_dimension
        ),
        "embedding_dtype": "float16",
        "mask_dtype": "uint8",
        "embedding_shape": list(
            expected_embedding_shape
        ),
        "mask_shape": list(
            expected_mask_shape
        ),
        "batch_size": int(batch_size),
        "number_of_batches": int(
            number_of_batches
        ),
        "embedding_file": str(
            final_embedding_file
        ),
        "mask_file": str(
            final_mask_file
        ),
        "metadata_file": str(
            final_metadata_file
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    temporary_completion_file = Path(
        str(completion_file) + ".temporary"
    )

    with open(
        temporary_completion_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            completion_data,
            handle,
            indent=2,
        )

    temporary_completion_file.replace(
        completion_file
    )

    print(
        f"[DATASET COMPLETE] {model_key} | "
        f"{dataset_name}"
    )

    print(
        "Embedding shape:",
        expected_embedding_shape,
    )

    return {
        "model": model_key,
        "dataset": dataset_name,
        "rows": number_of_rows,
        "embedding_dimension": (
            embedding_dimension
        ),
        "embedding_file": str(
            final_embedding_file
        ),
        "mask_file": str(
            final_mask_file
        ),
        "metadata_file": str(
            final_metadata_file
        ),
        "status": "completed_now",
    }


# ============================================================
# 9. PROCESS ONE COMPLETE PLM
# ============================================================

def process_complete_model(
    model_key,
    config,
):
    """Process all four datasets using one PLM."""

    print("\n\n" + "#" * 75)
    print(f"STARTING MODEL: {model_key}")
    print("#" * 75)

    model = None
    tokenizer_or_converter = None
    model_results = []

    try:
        if config["type"] == "esm":

            model, alphabet, batch_converter = (
                load_esm_model(config)
            )

            tokenizer_or_converter = (
                batch_converter
            )

        elif config["type"] == "prott5":

            model, tokenizer = (
                load_prott5_model(config)
            )

            tokenizer_or_converter = tokenizer

        else:
            raise ValueError(
                f"Unsupported model type: "
                f"{config['type']}"
            )

        for dataset_name, dataset_file in (
            DATASET_FILES.items()
        ):
            dataset_result = (
                process_dataset_for_model(
                    model_key=model_key,
                    config=config,
                    dataset_name=dataset_name,
                    dataset_file=dataset_file,
                    model=model,
                    tokenizer_or_converter=(
                        tokenizer_or_converter
                    ),
                )
            )

            model_results.append(
                dataset_result
            )

        model_summary_file = (
            EMBEDDING_ROOT
            / model_key
            / "model_embedding_summary.json"
        )

        with open(
            model_summary_file,
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                {
                    "model": model_key,
                    "configuration": config,
                    "datasets": model_results,
                    "completed_at": (
                        pd.Timestamp.now()
                        .isoformat()
                    ),
                },
                handle,
                indent=2,
            )

        mark_step_complete(
            f"embeddings_{model_key}",
            output_files=[
                model_summary_file
            ],
            details={
                "model": model_key,
                "loader": config["loader"],
                "datasets": [
                    result["dataset"]
                    for result in model_results
                ],
            },
        )

        print("\n" + "#" * 75)
        print(f"MODEL COMPLETE: {model_key}")
        print("#" * 75)

        return model_results

    finally:
        if model is not None:
            del model

        if tokenizer_or_converter is not None:
            del tokenizer_or_converter

        if "alphabet" in locals():
            del alphabet

        clear_memory()


# ============================================================
# 10. RUN ALL FOUR PLMS
# ============================================================

all_model_results = []
failed_models = []

for model_key, config in (
    MODEL_CONFIGS.items()
):
    try:
        model_results = (
            process_complete_model(
                model_key=model_key,
                config=config,
            )
        )

        all_model_results.extend(
            model_results
        )

    except Exception as error:
        print("\n" + "!" * 75)
        print(f"MODEL FAILED: {model_key}")
        print("!" * 75)
        print(type(error).__name__, ":", error)

        traceback.print_exc()

        failed_models.append(
            {
                "model": model_key,
                "error_type": (
                    type(error).__name__
                ),
                "error_message": str(error),
            }
        )

        failure_file = (
            CHECKPOINT_DIR
            / "embedding_failures.json"
        )

        with open(
            failure_file,
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                failed_models,
                handle,
                indent=2,
            )

        clear_memory()

        # Continue to the next model rather than losing
        # all progress.
        continue


# ============================================================
# 11. FINAL VERIFICATION OF ALL OUTPUTS
# ============================================================

print("\n\n" + "=" * 75)
print("FINAL EMBEDDING VERIFICATION")
print("=" * 75)

verification_rows = []

for model_key, config in (
    MODEL_CONFIGS.items()
):

    for dataset_name, dataset_file in (
        DATASET_FILES.items()
    ):
        output_directory = (
            EMBEDDING_ROOT
            / model_key
            / dataset_name
        )

        embedding_file = (
            output_directory
            / "X_per_residue.npy"
        )

        mask_file = (
            output_directory
            / "M_masks.npy"
        )

        metadata_file = (
            output_directory
            / "metadata.csv"
        )

        complete_file = (
            output_directory
            / "COMPLETE.json"
        )

        expected_rows = len(
            pd.read_csv(dataset_file)
        )

        expected_shape = (
            expected_rows,
            MAX_LEN,
            config["embedding_dimension"],
        )

        status = "MISSING"

        actual_shape = None

        if (
            embedding_file.exists()
            and mask_file.exists()
            and metadata_file.exists()
            and complete_file.exists()
        ):
            try:
                embedding_array = np.load(
                    embedding_file,
                    mmap_mode="r",
                )

                mask_array = np.load(
                    mask_file,
                    mmap_mode="r",
                )

                actual_shape = tuple(
                    embedding_array.shape
                )

                if (
                    actual_shape == expected_shape
                    and mask_array.shape
                    == (
                        expected_rows,
                        MAX_LEN,
                    )
                ):
                    status = "COMPLETE"
                else:
                    status = "SHAPE_ERROR"

                del embedding_array
                del mask_array

            except Exception:
                status = "READ_ERROR"

        verification_rows.append(
            {
                "model": model_key,
                "dataset": dataset_name,
                "expected_rows": (
                    expected_rows
                ),
                "embedding_dimension": (
                    config[
                        "embedding_dimension"
                    ]
                ),
                "expected_shape": str(
                    expected_shape
                ),
                "actual_shape": str(
                    actual_shape
                ),
                "status": status,
                "embedding_file": str(
                    embedding_file
                ),
            }
        )

        print(
            f"{model_key:12s} | "
            f"{dataset_name:15s} | "
            f"{status:12s} | "
            f"{actual_shape}"
        )


verification_df = pd.DataFrame(
    verification_rows
)

verification_file = (
    CHECKPOINT_DIR
    / "all_plm_embedding_verification.csv"
)

verification_df.to_csv(
    verification_file,
    index=False,
)


# ============================================================
# 12. FINAL MASTER CHECKPOINT
# ============================================================

all_complete = bool(
    (
        verification_df["status"]
        == "COMPLETE"
    ).all()
)

master_summary = {
    "all_complete": all_complete,
    "models": list(
        MODEL_CONFIGS.keys()
    ),
    "datasets": list(
        DATASET_FILES.keys()
    ),
    "maximum_sequence_length": (
        MAX_LEN
    ),
    "failed_models": failed_models,
    "verification_file": str(
        verification_file
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

master_summary_file = (
    CHECKPOINT_DIR
    / "all_plm_embedding_master_summary.json"
)

with open(
    master_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        master_summary,
        handle,
        indent=2,
    )


if all_complete:
    mark_step_complete(
        "06_all_PLM_embeddings",
        output_files=[
            verification_file,
            master_summary_file,
        ],
        details=master_summary,
    )

    print("\n" + "=" * 75)
    print("ALL FOUR PLM EMBEDDINGS COMPLETED")
    print("=" * 75)

else:
    print("\n" + "=" * 75)
    print("PIPELINE PARTIALLY COMPLETED")
    print("=" * 75)

    print(
        "Rerun this same cell. Valid completed "
        "models and batches will be skipped."
    )


print("\nVerification table saved to:")
print(verification_file)

print("\nMaster summary saved to:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 8A: Check TensorFlow compatibility
# ============================================================

import sys
import subprocess
import importlib.util

# Install TensorFlow only if it is missing
if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow is not installed. Installing it now...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tensorflow"
    ])

import tensorflow as tf
import numpy as np

print("=" * 70)
print("TENSORFLOW ENVIRONMENT")
print("=" * 70)

print("TensorFlow version:", tf.__version__)
print("NumPy version     :", np.__version__)
print("TensorFlow GPUs   :", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("\nTensorFlow can access the GPU.")
else:
    print("\nWARNING: TensorFlow cannot currently access the GPU.")

# Prevent TensorFlow from reserving all GPU memory immediately
gpu_devices = tf.config.list_physical_devices("GPU")

for gpu in gpu_devices:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )
    except RuntimeError as error:
        print("Memory-growth warning:", error)

# Reproducibility
SEED = 42

tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic TensorFlow operations enabled.")
except Exception as error:
    print(
        "Deterministic operations could not be fully enabled:",
        error
    )

# Save the environment check permanently
tensorflow_environment_file = (
    DIRS["checkpoints"]
    / "tensorflow_environment.json"
)

tensorflow_environment = {
    "tensorflow_version": tf.__version__,
    "numpy_version": np.__version__,
    "gpu_available": bool(
        tf.config.list_physical_devices("GPU")
    ),
    "gpu_devices": [
        str(device)
        for device in tf.config.list_physical_devices("GPU")
    ],
    "seed": SEED,
}

import json

with open(
    tensorflow_environment_file,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        tensorflow_environment,
        handle,
        indent=2
    )

mark_step_complete(
    "07_tensorflow_environment",
    output_files=[
        tensorflow_environment_file
    ],
    details=tensorflow_environment
)

print("\nEnvironment record saved:")
print(tensorflow_environment_file)

In [ ]:
# ============================================================
# STEP 8B: RESTART-SAFE ATTENTION CLASSIFIER TRAINING
#
# Trains one masked residue-attention classifier for each PLM:
#   ESM2-320, ESM2-640, ESM2-1280, ProtT5
#
# Uses:
#   Train set       -> model fitting
#   Validation set  -> early stopping + threshold selection
#   Internal test   -> final internal evaluation
#   KELM external   -> independent external evaluation
# ============================================================

from pathlib import Path
import gc
import json
import os
import random
import shutil
import traceback

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
)
from sklearn.utils.class_weight import compute_class_weight


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
MAX_LEN = 61
MAX_EPOCHS = 100
PATIENCE = 12
LEARNING_RATE = 1e-3

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# Prevent TensorFlow from taking all GPU memory immediately
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Mixed precision is efficient on a Tesla T4
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow version:", tf.__version__)
print("Precision policy  :", mixed_precision.global_policy())
print("TensorFlow GPUs   :", tf.config.list_physical_devices("GPU"))


MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 64,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 48,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 32,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 32,
    },
}

DATASET_NAMES = [
    "train",
    "validation",
    "internal_test",
    "kelm_external",
]

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
PREDICTION_ROOT = DIRS["predictions"]
RESULT_ROOT = DIRS["results"]
CHECKPOINT_ROOT = DIRS["checkpoints"]
LOG_ROOT = DIRS["logs"]

for folder in [
    MODEL_ROOT,
    PREDICTION_ROOT,
    RESULT_ROOT,
    CHECKPOINT_ROOT,
    LOG_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. CUSTOM MASKED ATTENTION LAYER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):
    """
    Learn one attention score per residue and calculate a
    mask-aware weighted peptide representation.

    Inputs:
        residue_features: batch × length × features
        residue_mask:     batch × length

    Returns:
        pooled_vector:     batch × features
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def call(self, inputs):
        residue_features, residue_mask = inputs

        # batch × length × 1
        logits = self.attention_dense(
            residue_features
        )

        # batch × length
        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        # Assign extremely negative scores to padded positions
        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        attention_weights = tf.expand_dims(
            attention_weights,
            axis=-1,
        )

        weighted_features = (
            residue_features
            * attention_weights
        )

        pooled_vector = tf.reduce_sum(
            weighted_features,
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. BUILD ATTENTION CLASSIFIER
# ============================================================

def build_attention_classifier(
    embedding_dimension
):
    residue_input = tf.keras.Input(
        shape=(
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=tf.float16,
        name="residue_embeddings",
    )

    mask_input = tf.keras.Input(
        shape=(MAX_LEN,),
        dtype=tf.uint8,
        name="residue_mask",
    )

    # Layer normalization stabilizes PLM embeddings
    x = tf.keras.layers.LayerNormalization(
        epsilon=1e-6,
        name="embedding_layer_norm",
    )(residue_input)

    # Residue-level nonlinear projection
    x = tf.keras.layers.Dense(
        128,
        activation="gelu",
        name="residue_projection",
    )(x)

    x = tf.keras.layers.Dropout(
        0.20,
        name="residue_dropout",
    )(x)

    pooled = MaskedAttentionPooling(
        name="masked_attention_pooling",
    )([x, mask_input])

    x = tf.keras.layers.Dense(
        128,
        activation="gelu",
        name="peptide_dense_1",
    )(pooled)

    x = tf.keras.layers.Dropout(
        0.30,
        name="peptide_dropout_1",
    )(x)

    x = tf.keras.layers.Dense(
        32,
        activation="gelu",
        name="peptide_dense_2",
    )(x)

    x = tf.keras.layers.Dropout(
        0.20,
        name="peptide_dropout_2",
    )(x)

    # Force final probability to float32 under mixed precision
    output = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="cpp_probability",
    )(x)

    model = tf.keras.Model(
        inputs=[
            residue_input,
            mask_input,
        ],
        outputs=output,
        name="pLM4CPP_attention_classifier",
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
    )

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC",
            ),
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR",
            ),
        ],
    )

    return model


# ============================================================
# 4. LOAD ONE MODEL'S SAVED EMBEDDINGS
# ============================================================

def load_embedding_dataset(
    model_key,
    dataset_name
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_key
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n"
                f"{required_file}"
            )

    # Memory mapping avoids unnecessary initial RAM copies
    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    labels = (
        metadata["label"]
        .astype(np.float32)
        .to_numpy()
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Row mismatch for "
            f"{model_key}/{dataset_name}"
        )

    return (
        embeddings,
        masks,
        labels,
        metadata,
    )


# ============================================================
# 5. VALIDATION-BASED THRESHOLD SELECTION
# ============================================================

def choose_validation_threshold(
    y_true,
    probabilities
):
    """
    Choose the probability threshold that maximizes MCC
    on the validation set only.
    """

    thresholds = np.linspace(
        0.05,
        0.95,
        181,
    )

    records = []

    best_threshold = 0.5
    best_mcc = -1.0

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            predictions,
        )

        records.append({
            "threshold": float(threshold),
            "mcc": float(mcc),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    predictions,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    predictions,
                )
            ),
            "f1": float(
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0,
                )
            ),
        })

        if mcc > best_mcc:
            best_mcc = mcc
            best_threshold = float(
                threshold
            )

    return (
        best_threshold,
        pd.DataFrame(records),
    )


# ============================================================
# 6. METRIC CALCULATION
# ============================================================

def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "negative_predictive_value": float(
            npv
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


# ============================================================
# 7. SAVE PREDICTIONS
# ============================================================

def create_prediction_table(
    metadata,
    probabilities,
    threshold,
    dataset_name,
    model_key,
):
    result = metadata.copy()

    result["probability_CPP"] = (
        probabilities.astype(float)
    )

    result["predicted_label"] = (
        probabilities >= threshold
    ).astype(int)

    result["correct_prediction"] = (
        result["predicted_label"]
        == result["label"]
    )

    result["decision_threshold"] = (
        threshold
    )

    result["model"] = model_key
    result["evaluation_dataset"] = (
        dataset_name
    )

    return result


# ============================================================
# 8. TRAIN ONE PLM CLASSIFIER
# ============================================================

def train_one_plm_classifier(
    model_key,
    config,
):
    print("\n\n" + "#" * 76)
    print(f"TRAINING CLASSIFIER: {model_key}")
    print("#" * 76)

    embedding_dimension = (
        config["embedding_dimension"]
    )

    batch_size = config["batch_size"]

    model_dir = (
        MODEL_ROOT / model_key
    )

    prediction_dir = (
        PREDICTION_ROOT / model_key
    )

    log_dir = (
        LOG_ROOT / model_key
    )

    backup_dir = (
        model_dir / "training_backup"
    )

    for folder in [
        model_dir,
        prediction_dir,
        log_dir,
    ]:
        folder.mkdir(
            parents=True,
            exist_ok=True,
        )

    final_model_file = (
        model_dir
        / "final_attention_classifier.keras"
    )

    best_model_file = (
        model_dir
        / "best_attention_classifier.keras"
    )

    completion_file = (
        model_dir
        / "TRAINING_COMPLETE.json"
    )

    metrics_file = (
        model_dir
        / "evaluation_metrics.json"
    )

    threshold_file = (
        model_dir
        / "selected_threshold.json"
    )

    # --------------------------------------------------------
    # Skip a fully completed model
    # --------------------------------------------------------

    if (
        completion_file.exists()
        and final_model_file.exists()
        and metrics_file.exists()
        and threshold_file.exists()
    ):
        print(
            "[SKIP COMPLETE] Model and evaluation "
            "outputs already exist."
        )

        with open(
            metrics_file,
            "r",
            encoding="utf-8",
        ) as handle:
            existing_metrics = json.load(
                handle
            )

        return existing_metrics

    # --------------------------------------------------------
    # Load permanent embedding files
    # --------------------------------------------------------

    (
        X_train,
        M_train,
        y_train,
        meta_train,
    ) = load_embedding_dataset(
        model_key,
        "train",
    )

    (
        X_val,
        M_val,
        y_val,
        meta_val,
    ) = load_embedding_dataset(
        model_key,
        "validation",
    )

    (
        X_test,
        M_test,
        y_test,
        meta_test,
    ) = load_embedding_dataset(
        model_key,
        "internal_test",
    )

    (
        X_kelm,
        M_kelm,
        y_kelm,
        meta_kelm,
    ) = load_embedding_dataset(
        model_key,
        "kelm_external",
    )

    print("Training shape  :", X_train.shape)
    print("Validation shape:", X_val.shape)
    print("Test shape      :", X_test.shape)
    print("KELM shape      :", X_kelm.shape)

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------

    unique_classes = np.unique(
        y_train.astype(int)
    )

    calculated_weights = (
        compute_class_weight(
            class_weight="balanced",
            classes=unique_classes,
            y=y_train.astype(int),
        )
    )

    class_weight = {
        int(class_label): float(weight)
        for class_label, weight in zip(
            unique_classes,
            calculated_weights,
        )
    }

    print("Class weights:", class_weight)

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    model = build_attention_classifier(
        embedding_dimension
    )

    model.summary()

    history_file = (
        log_dir / "training_history.csv"
    )

    callbacks = [
        # Restores optimizer/model state after interruption
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_dir),
            save_freq="epoch",
            delete_checkpoint=False,
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_file),
            monitor="val_roc_auc",
            mode="max",
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_roc_auc",
            mode="max",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),

        tf.keras.callbacks.CSVLogger(
            filename=str(history_file),
            append=True,
        ),

        tf.keras.callbacks.TerminateOnNaN(),
    ]

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(
        x={
            "residue_embeddings": X_train,
            "residue_mask": M_train,
        },
        y=y_train,
        validation_data=(
            {
                "residue_embeddings": X_val,
                "residue_mask": M_val,
            },
            y_val,
        ),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        shuffle=True,
        verbose=1,
    )

    # Save final in-memory best-restored model
    model.save(
        final_model_file
    )

    # --------------------------------------------------------
    # Generate probabilities
    # --------------------------------------------------------

    validation_probabilities = (
        model.predict(
            {
                "residue_embeddings": X_val,
                "residue_mask": M_val,
            },
            batch_size=batch_size,
            verbose=0,
        )
        .reshape(-1)
        .astype(float)
    )

    test_probabilities = (
        model.predict(
            {
                "residue_embeddings": X_test,
                "residue_mask": M_test,
            },
            batch_size=batch_size,
            verbose=0,
        )
        .reshape(-1)
        .astype(float)
    )

    kelm_probabilities = (
        model.predict(
            {
                "residue_embeddings": X_kelm,
                "residue_mask": M_kelm,
            },
            batch_size=batch_size,
            verbose=0,
        )
        .reshape(-1)
        .astype(float)
    )

    # --------------------------------------------------------
    # Select threshold using validation data only
    # --------------------------------------------------------

    (
        selected_threshold,
        threshold_scan_df,
    ) = choose_validation_threshold(
        y_true=y_val.astype(int),
        probabilities=validation_probabilities,
    )

    print(
        "Selected validation threshold:",
        selected_threshold
    )

    threshold_scan_file = (
        model_dir
        / "validation_threshold_scan.csv"
    )

    threshold_scan_df.to_csv(
        threshold_scan_file,
        index=False,
    )

    threshold_information = {
        "selection_dataset": "validation",
        "selection_metric": (
            "Matthews correlation coefficient"
        ),
        "selected_threshold": float(
            selected_threshold
        ),
        "default_threshold_for_comparison": 0.5,
    }

    with open(
        threshold_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            threshold_information,
            handle,
            indent=2,
        )

    # --------------------------------------------------------
    # Evaluate validation, internal test, and KELM
    # --------------------------------------------------------

    validation_metrics = (
        calculate_binary_metrics(
            y_true=y_val.astype(int),
            probabilities=validation_probabilities,
            threshold=selected_threshold,
        )
    )

    internal_test_metrics = (
        calculate_binary_metrics(
            y_true=y_test.astype(int),
            probabilities=test_probabilities,
            threshold=selected_threshold,
        )
    )

    kelm_metrics = (
        calculate_binary_metrics(
            y_true=y_kelm.astype(int),
            probabilities=kelm_probabilities,
            threshold=selected_threshold,
        )
    )

    # Also retain default 0.5 metrics for transparency
    internal_test_default_metrics = (
        calculate_binary_metrics(
            y_true=y_test.astype(int),
            probabilities=test_probabilities,
            threshold=0.5,
        )
    )

    kelm_default_metrics = (
        calculate_binary_metrics(
            y_true=y_kelm.astype(int),
            probabilities=kelm_probabilities,
            threshold=0.5,
        )
    )

    all_metrics = {
        "model": model_key,
        "embedding_dimension": int(
            embedding_dimension
        ),
        "selected_threshold": float(
            selected_threshold
        ),
        "class_weights": {
            str(key): float(value)
            for key, value in class_weight.items()
        },
        "validation_selected_threshold": (
            validation_metrics
        ),
        "internal_test_selected_threshold": (
            internal_test_metrics
        ),
        "kelm_external_selected_threshold": (
            kelm_metrics
        ),
        "internal_test_default_0.5": (
            internal_test_default_metrics
        ),
        "kelm_external_default_0.5": (
            kelm_default_metrics
        ),
    }

    with open(
        metrics_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            all_metrics,
            handle,
            indent=2,
        )

    # --------------------------------------------------------
    # Save prediction tables
    # --------------------------------------------------------

    validation_predictions = (
        create_prediction_table(
            metadata=meta_val,
            probabilities=validation_probabilities,
            threshold=selected_threshold,
            dataset_name="validation",
            model_key=model_key,
        )
    )

    test_predictions = (
        create_prediction_table(
            metadata=meta_test,
            probabilities=test_probabilities,
            threshold=selected_threshold,
            dataset_name="internal_test",
            model_key=model_key,
        )
    )

    kelm_predictions = (
        create_prediction_table(
            metadata=meta_kelm,
            probabilities=kelm_probabilities,
            threshold=selected_threshold,
            dataset_name="kelm_external",
            model_key=model_key,
        )
    )

    validation_prediction_file = (
        prediction_dir
        / "validation_predictions.csv"
    )

    test_prediction_file = (
        prediction_dir
        / "internal_test_predictions.csv"
    )

    kelm_prediction_file = (
        prediction_dir
        / "kelm_external_predictions.csv"
    )

    validation_predictions.to_csv(
        validation_prediction_file,
        index=False,
    )

    test_predictions.to_csv(
        test_prediction_file,
        index=False,
    )

    kelm_predictions.to_csv(
        kelm_prediction_file,
        index=False,
    )

    # --------------------------------------------------------
    # Mark complete atomically
    # --------------------------------------------------------

    completion_information = {
        "model": model_key,
        "embedding_dimension": int(
            embedding_dimension
        ),
        "batch_size": int(batch_size),
        "maximum_epochs": int(MAX_EPOCHS),
        "epochs_run_this_session": int(
            len(history.history.get("loss", []))
        ),
        "selected_threshold": float(
            selected_threshold
        ),
        "final_model_file": str(
            final_model_file
        ),
        "best_model_file": str(
            best_model_file
        ),
        "metrics_file": str(
            metrics_file
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    temporary_completion_file = Path(
        str(completion_file) + ".temporary"
    )

    with open(
        temporary_completion_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            completion_information,
            handle,
            indent=2,
        )

    temporary_completion_file.replace(
        completion_file
    )

    mark_step_complete(
        f"attention_classifier_{model_key}",
        output_files=[
            final_model_file,
            best_model_file,
            metrics_file,
            threshold_file,
            threshold_scan_file,
            validation_prediction_file,
            test_prediction_file,
            kelm_prediction_file,
            history_file,
            completion_file,
        ],
        details=completion_information,
    )

    print("\nInternal test metrics:")
    print(
        json.dumps(
            internal_test_metrics,
            indent=2,
        )
    )

    print("\nKELM external metrics:")
    print(
        json.dumps(
            kelm_metrics,
            indent=2,
        )
    )

    # Release one model before loading the next one
    del model
    del history

    del X_train, M_train, y_train, meta_train
    del X_val, M_val, y_val, meta_val
    del X_test, M_test, y_test, meta_test
    del X_kelm, M_kelm, y_kelm, meta_kelm

    gc.collect()
    tf.keras.backend.clear_session()

    return all_metrics


# ============================================================
# 9. RUN ALL FOUR PLM CLASSIFIERS
# ============================================================

all_results = {}
failed_models = []

for model_key, config in MODEL_CONFIGS.items():
    try:
        model_metrics = (
            train_one_plm_classifier(
                model_key=model_key,
                config=config,
            )
        )

        all_results[model_key] = (
            model_metrics
        )

    except tf.errors.ResourceExhaustedError as error:
        print("\n" + "!" * 76)
        print(f"GPU MEMORY ERROR: {model_key}")
        print("!" * 76)

        print(
            "Reduce the batch size for this model "
            "and rerun the same cell."
        )

        print(error)

        failed_models.append({
            "model": model_key,
            "error": str(error),
        })

        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as error:
        print("\n" + "!" * 76)
        print(f"MODEL FAILED: {model_key}")
        print("!" * 76)

        print(
            type(error).__name__,
            ":",
            error,
        )

        traceback.print_exc()

        failed_models.append({
            "model": model_key,
            "error_type": (
                type(error).__name__
            ),
            "error": str(error),
        })

        tf.keras.backend.clear_session()
        gc.collect()


# ============================================================
# 10. CREATE MASTER PERFORMANCE TABLE
# ============================================================

performance_rows = []

for model_key, model_metrics in (
    all_results.items()
):
    for evaluation_name in [
        "validation_selected_threshold",
        "internal_test_selected_threshold",
        "kelm_external_selected_threshold",
    ]:
        metrics = model_metrics[
            evaluation_name
        ]

        performance_rows.append({
            "model": model_key,
            "evaluation": evaluation_name,
            **metrics,
        })

performance_df = pd.DataFrame(
    performance_rows
)

performance_file = (
    RESULT_ROOT
    / "tables_main"
    / "attention_classifier_performance.csv"
)

performance_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)

performance_df.to_csv(
    performance_file,
    index=False,
)

failure_file = (
    CHECKPOINT_ROOT
    / "attention_classifier_failures.json"
)

with open(
    failure_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        failed_models,
        handle,
        indent=2,
    )

print("\n\n" + "=" * 76)
print("ATTENTION CLASSIFIER TRAINING SUMMARY")
print("=" * 76)

if not performance_df.empty:
    display(
        performance_df[
            [
                "model",
                "evaluation",
                "threshold",
                "accuracy",
                "balanced_accuracy",
                "f1",
                "mcc",
                "roc_auc",
                "pr_auc",
            ]
        ]
    )

print("\nFailed models:", failed_models)
print("\nPerformance table saved to:")
print(performance_file)

if len(failed_models) == 0 and len(all_results) == 4:
    mark_step_complete(
        "08_all_attention_classifiers",
        output_files=[
            performance_file,
            failure_file,
        ],
        details={
            "models_completed": list(
                all_results.keys()
            ),
            "failed_models": [],
        },
    )

    print("\n" + "=" * 76)
    print("ALL FOUR ATTENTION CLASSIFIERS COMPLETED")
    print("=" * 76)

else:
    print("\nSome models are incomplete.")
    print(
        "Rerun this same cell. Completed models "
        "will be skipped and training backups "
        "will restore interrupted models."
    )

In [ ]:
# ============================================================
# STEP 9: FOUR-PLM PROBABILITY ENSEMBLE
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASET_FILES = {
    "validation": "validation_predictions.csv",
    "internal_test": "internal_test_predictions.csv",
    "kelm_external": "kelm_external_predictions.csv",
}

PREDICTION_ROOT = DIRS["predictions"]
RESULT_TABLE_DIR = DIRS["results"] / "tables_main"
ENSEMBLE_DIR = PREDICTION_ROOT / "ensemble"

RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. LOAD AND ALIGN MODEL PREDICTIONS
# ============================================================

def load_aligned_predictions(dataset_name, filename):

    model_tables = {}

    for model_name in MODEL_NAMES:

        prediction_file = (
            PREDICTION_ROOT
            / model_name
            / filename
        )

        if not prediction_file.exists():
            raise FileNotFoundError(
                f"Missing prediction file:\n{prediction_file}"
            )

        table = pd.read_csv(prediction_file)

        required_columns = {
            "sequence_id",
            "sequence",
            "label",
            "probability_CPP",
        }

        missing_columns = (
            required_columns - set(table.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{model_name}/{dataset_name} is missing: "
                f"{sorted(missing_columns)}"
            )

        table["sequence_id"] = (
            table["sequence_id"].astype(str)
        )

        model_tables[model_name] = table.copy()

    reference_model = MODEL_NAMES[0]

    reference = model_tables[
        reference_model
    ][
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    reference = reference.reset_index(drop=True)

    merged = reference.copy()

    for model_name in MODEL_NAMES:

        current = (
            model_tables[model_name]
            .set_index("sequence_id")
        )

        expected_ids = (
            merged["sequence_id"]
            .astype(str)
            .tolist()
        )

        missing_ids = [
            sequence_id
            for sequence_id in expected_ids
            if sequence_id not in current.index
        ]

        if missing_ids:
            raise ValueError(
                f"{model_name}/{dataset_name} is missing "
                f"{len(missing_ids)} sequence IDs."
            )

        current = current.loc[
            expected_ids
        ].reset_index()

        if not np.array_equal(
            current["label"].to_numpy(),
            merged["label"].to_numpy(),
        ):
            raise ValueError(
                f"Label mismatch for {model_name}/{dataset_name}"
            )

        merged[
            f"probability_{model_name}"
        ] = current[
            "probability_CPP"
        ].astype(float).to_numpy()

    return merged


ensemble_tables = {}

for dataset_name, filename in DATASET_FILES.items():

    ensemble_tables[dataset_name] = (
        load_aligned_predictions(
            dataset_name=dataset_name,
            filename=filename,
        )
    )

    print(
        f"[LOADED] {dataset_name}: "
        f"{len(ensemble_tables[dataset_name])} rows"
    )


# ============================================================
# 2. CALCULATE ENSEMBLE PROBABILITIES AND AGREEMENT
# ============================================================

probability_columns = [
    f"probability_{model_name}"
    for model_name in MODEL_NAMES
]

for dataset_name, table in ensemble_tables.items():

    probability_matrix = table[
        probability_columns
    ].to_numpy(dtype=float)

    table[
        "ensemble_probability_mean"
    ] = probability_matrix.mean(axis=1)

    table[
        "ensemble_probability_median"
    ] = np.median(
        probability_matrix,
        axis=1,
    )

    table[
        "model_probability_std"
    ] = probability_matrix.std(
        axis=1,
        ddof=0,
    )

    table[
        "model_probability_range"
    ] = (
        probability_matrix.max(axis=1)
        - probability_matrix.min(axis=1)
    )

    table[
        "minimum_model_probability"
    ] = probability_matrix.min(axis=1)

    table[
        "maximum_model_probability"
    ] = probability_matrix.max(axis=1)


# ============================================================
# 3. THRESHOLD OPTIMIZATION USING VALIDATION ONLY
# ============================================================

def select_threshold_by_mcc(
    y_true,
    probabilities,
):

    thresholds = np.linspace(
        0.01,
        0.99,
        197,
    )

    rows = []

    best_threshold = 0.5
    best_mcc = -np.inf

    for threshold in thresholds:

        predicted = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            predicted,
        )

        rows.append({
            "threshold": float(threshold),
            "mcc": float(mcc),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    predicted,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    predicted,
                )
            ),
            "f1": float(
                f1_score(
                    y_true,
                    predicted,
                    zero_division=0,
                )
            ),
        })

        if mcc > best_mcc:
            best_mcc = mcc
            best_threshold = float(
                threshold
            )

    return (
        best_threshold,
        pd.DataFrame(rows),
    )


validation_table = ensemble_tables[
    "validation"
]

y_validation = (
    validation_table["label"]
    .astype(int)
    .to_numpy()
)

mean_threshold, mean_threshold_scan = (
    select_threshold_by_mcc(
        y_true=y_validation,
        probabilities=(
            validation_table[
                "ensemble_probability_mean"
            ].to_numpy()
        ),
    )
)

median_threshold, median_threshold_scan = (
    select_threshold_by_mcc(
        y_true=y_validation,
        probabilities=(
            validation_table[
                "ensemble_probability_median"
            ].to_numpy()
        ),
    )
)

print("\nSelected mean-ensemble threshold  :", mean_threshold)
print("Selected median-ensemble threshold:", median_threshold)

mean_threshold_scan.to_csv(
    ENSEMBLE_DIR
    / "validation_mean_threshold_scan.csv",
    index=False,
)

median_threshold_scan.to_csv(
    ENSEMBLE_DIR
    / "validation_median_threshold_scan.csv",
    index=False,
)


# ============================================================
# 4. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):

    predicted = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predicted,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "negative_predictive_value": float(
            npv
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predicted,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


# ============================================================
# 5. EVALUATE MEAN AND MEDIAN ENSEMBLES
# ============================================================

all_metrics = {}
performance_rows = []

for dataset_name, table in ensemble_tables.items():

    y_true = (
        table["label"]
        .astype(int)
        .to_numpy()
    )

    mean_probabilities = (
        table[
            "ensemble_probability_mean"
        ].to_numpy()
    )

    median_probabilities = (
        table[
            "ensemble_probability_median"
        ].to_numpy()
    )

    mean_metrics = calculate_metrics(
        y_true=y_true,
        probabilities=mean_probabilities,
        threshold=mean_threshold,
    )

    median_metrics = calculate_metrics(
        y_true=y_true,
        probabilities=median_probabilities,
        threshold=median_threshold,
    )

    all_metrics[dataset_name] = {
        "mean_ensemble": mean_metrics,
        "median_ensemble": median_metrics,
    }

    performance_rows.append({
        "method": "mean_ensemble",
        "dataset": dataset_name,
        **mean_metrics,
    })

    performance_rows.append({
        "method": "median_ensemble",
        "dataset": dataset_name,
        **median_metrics,
    })

    table[
        "mean_ensemble_predicted_label"
    ] = (
        mean_probabilities >= mean_threshold
    ).astype(int)

    table[
        "median_ensemble_predicted_label"
    ] = (
        median_probabilities >= median_threshold
    ).astype(int)

    # Number of individual PLMs voting CPP at threshold 0.5
    model_votes = (
        table[probability_columns]
        .to_numpy()
        >= 0.5
    ).astype(int)

    table[
        "number_of_CPP_votes"
    ] = model_votes.sum(axis=1)

    table[
        "unanimous_model_vote"
    ] = (
        (table["number_of_CPP_votes"] == 0)
        | (table["number_of_CPP_votes"] == 4)
    )

    output_file = (
        ENSEMBLE_DIR
        / f"{dataset_name}_ensemble_predictions.csv"
    )

    table.to_csv(
        output_file,
        index=False,
    )

    print("\n" + "=" * 70)
    print(dataset_name.upper())
    print("=" * 70)

    print("Mean ensemble:")
    print(json.dumps(mean_metrics, indent=2))

    print("\nMedian ensemble:")
    print(json.dumps(median_metrics, indent=2))


# ============================================================
# 6. SAVE MASTER PERFORMANCE TABLE
# ============================================================

ensemble_performance_df = pd.DataFrame(
    performance_rows
)

ensemble_performance_file = (
    RESULT_TABLE_DIR
    / "four_plm_ensemble_performance.csv"
)

ensemble_performance_df.to_csv(
    ensemble_performance_file,
    index=False,
)


# ============================================================
# 7. ADD INDIVIDUAL MODEL PERFORMANCE FOR COMPARISON
# ============================================================

individual_performance_file = (
    RESULT_TABLE_DIR
    / "attention_classifier_performance.csv"
)

individual_performance = pd.read_csv(
    individual_performance_file
)

comparison_rows = []

for _, row in individual_performance.iterrows():

    if row["evaluation"] == (
        "validation_selected_threshold"
    ):
        dataset_name = "validation"

    elif row["evaluation"] == (
        "internal_test_selected_threshold"
    ):
        dataset_name = "internal_test"

    elif row["evaluation"] == (
        "kelm_external_selected_threshold"
    ):
        dataset_name = "kelm_external"

    else:
        continue

    comparison_rows.append({
        "method": row["model"],
        "dataset": dataset_name,
        "threshold": row["threshold"],
        "accuracy": row["accuracy"],
        "balanced_accuracy": row[
            "balanced_accuracy"
        ],
        "f1": row["f1"],
        "mcc": row["mcc"],
        "roc_auc": row["roc_auc"],
        "pr_auc": row["pr_auc"],
    })

for _, row in ensemble_performance_df.iterrows():

    comparison_rows.append({
        "method": row["method"],
        "dataset": row["dataset"],
        "threshold": row["threshold"],
        "accuracy": row["accuracy"],
        "balanced_accuracy": row[
            "balanced_accuracy"
        ],
        "f1": row["f1"],
        "mcc": row["mcc"],
        "roc_auc": row["roc_auc"],
        "pr_auc": row["pr_auc"],
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_file = (
    RESULT_TABLE_DIR
    / "individual_vs_ensemble_comparison.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False,
)


# ============================================================
# 8. SAVE JSON SUMMARY
# ============================================================

summary = {
    "models": MODEL_NAMES,
    "mean_ensemble_threshold": float(
        mean_threshold
    ),
    "median_ensemble_threshold": float(
        median_threshold
    ),
    "metrics": all_metrics,
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    ENSEMBLE_DIR
    / "ensemble_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 9. SAVE CHECKPOINT
# ============================================================

output_files = [
    ensemble_performance_file,
    comparison_file,
    summary_file,
    ENSEMBLE_DIR
    / "validation_ensemble_predictions.csv",
    ENSEMBLE_DIR
    / "internal_test_ensemble_predictions.csv",
    ENSEMBLE_DIR
    / "kelm_external_ensemble_predictions.csv",
]

mark_step_complete(
    "09_four_PLM_ensemble",
    output_files=output_files,
    details={
        "models": MODEL_NAMES,
        "mean_threshold": float(
            mean_threshold
        ),
        "median_threshold": float(
            median_threshold
        ),
        "metrics": all_metrics,
    },
)

print("\n" + "=" * 70)
print("INDIVIDUAL AND ENSEMBLE COMPARISON")
print("=" * 70)

display(
    comparison_df[
        [
            "method",
            "dataset",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "mcc",
            "roc_auc",
            "pr_auc",
        ]
    ].sort_values(
        ["dataset", "mcc"],
        ascending=[True, False],
    )
)

print("\nSaved ensemble performance:")
print(ensemble_performance_file)

print("\nSaved comparison table:")
print(comparison_file)

print("\nSaved ensemble summary:")
print(summary_file)

In [ ]:
# ============================================================
# STEP 10 FIX: Extract attention using eager batch computation
# Compatible with TensorFlow 2.20 / Keras 3
# ============================================================

from pathlib import Path
import gc
import json
import math

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 128,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 96,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 64,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 64,
    },
}

DATASETS_TO_EXPLAIN = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]


# ============================================================
# 2. CUSTOM LAYER FOR LOADING SAVED MODELS
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD EMBEDDING DATASET
# ============================================================

def load_xai_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing file:\n{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    return embeddings, masks, metadata


# ============================================================
# 4. CALCULATE ATTENTION FOR ONE REAL BATCH
# ============================================================

def calculate_attention_batch(
    trained_model,
    embedding_batch,
    mask_batch,
):
    """
    Calculate attention from actual tensors rather than
    symbolic KerasTensor objects.
    """

    embedding_batch = tf.convert_to_tensor(
        embedding_batch,
        dtype=tf.float16,
    )

    mask_batch = tf.convert_to_tensor(
        mask_batch,
    )

    layer_norm = trained_model.get_layer(
        "embedding_layer_norm"
    )

    projection = trained_model.get_layer(
        "residue_projection"
    )

    dropout = trained_model.get_layer(
        "residue_dropout"
    )

    attention_pooling = trained_model.get_layer(
        "masked_attention_pooling"
    )

    # Reproduce the residue features used during inference
    residue_features = layer_norm(
        embedding_batch,
        training=False,
    )

    residue_features = projection(
        residue_features,
        training=False,
    )

    residue_features = dropout(
        residue_features,
        training=False,
    )

    # Learned raw residue-attention scores
    logits = attention_pooling.attention_dense(
        residue_features,
        training=False,
    )

    logits = tf.squeeze(
        logits,
        axis=-1,
    )

    mask_float = tf.cast(
        mask_batch,
        logits.dtype,
    )

    masked_logits = (
        logits
        + (1.0 - mask_float)
        * tf.cast(-1e4, logits.dtype)
    )

    attention_weights = tf.nn.softmax(
        masked_logits,
        axis=1,
    )

    # Ensure padded positions are exactly zero
    attention_weights = (
        attention_weights * mask_float
    )

    row_sums = tf.reduce_sum(
        attention_weights,
        axis=1,
        keepdims=True,
    )

    attention_weights = tf.math.divide_no_nan(
        attention_weights,
        row_sums,
    )

    return (
        attention_weights
        .numpy()
        .astype(np.float32)
    )


# ============================================================
# 5. EXTRACT ONE MODEL / DATASET
# ============================================================

def extract_attention_scores(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        XAI_ROOT
        / model_name
        / "attention"
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    residue_output_file = (
        output_dir
        / "residue_attention_scores.csv"
    )

    matrix_output_file = (
        output_dir
        / "attention_matrix.npy"
    )

    summary_output_file = (
        output_dir
        / "attention_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    # Skip valid completed result
    if (
        complete_file.exists()
        and residue_output_file.exists()
        and matrix_output_file.exists()
        and summary_output_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] {model_name} | "
            f"{dataset_name}"
        )

        return [
            residue_output_file,
            matrix_output_file,
            summary_output_file,
            complete_file,
        ]

    print("\n" + "=" * 75)
    print(f"{model_name} — {dataset_name}")
    print("=" * 75)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found:\n{model_file}"
        )

    trained_model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings, masks, metadata = (
        load_xai_dataset(
            model_name,
            dataset_name,
        )
    )

    number_of_rows = len(metadata)

    temporary_matrix_file = (
        output_dir
        / "attention_matrix.temporary.npy"
    )

    attention_memmap = (
        np.lib.format.open_memmap(
            temporary_matrix_file,
            mode="w+",
            dtype=np.float32,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        embedding_batch = np.asarray(
            embeddings[
                start_index:end_index
            ]
        )

        mask_batch = np.asarray(
            masks[
                start_index:end_index
            ]
        )

        batch_attention = (
            calculate_attention_batch(
                trained_model=trained_model,
                embedding_batch=embedding_batch,
                mask_batch=mask_batch,
            )
        )

        attention_memmap[
            start_index:end_index
        ] = batch_attention

        attention_memmap.flush()

        print(
            f"[ATTENTION] {model_name} | "
            f"{dataset_name} | "
            f"batch {batch_number + 1}/"
            f"{number_of_batches}"
        )

        del embedding_batch
        del mask_batch
        del batch_attention

        gc.collect()

    attention_memmap.flush()
    del attention_memmap

    temporary_matrix_file.replace(
        matrix_output_file
    )

    attention_matrix = np.load(
        matrix_output_file,
        mmap_mode="r",
    )

    # --------------------------------------------------------
    # Convert matrix into one row per residue
    # --------------------------------------------------------

    residue_rows = []

    for row_index, row in metadata.iterrows():
        sequence = str(row["sequence"])
        sequence_length = len(sequence)

        for position_index, residue in enumerate(
            sequence
        ):
            residue_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "row_index": int(row_index),
                "sequence_id": str(
                    row["sequence_id"]
                ),
                "sequence": sequence,
                "label": int(row["label"]),
                "sequence_length": int(
                    sequence_length
                ),
                "position": int(
                    position_index + 1
                ),
                "normalized_position": float(
                    (
                        position_index + 1
                    )
                    / sequence_length
                ),
                "residue": residue,
                "attention_score": float(
                    attention_matrix[
                        row_index,
                        position_index,
                    ]
                ),
            })

    residue_df = pd.DataFrame(
        residue_rows
    )

    residue_df.to_csv(
        residue_output_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    attention_sums = (
        residue_df
        .groupby("sequence_id")[
            "attention_score"
        ]
        .sum()
        .to_numpy()
    )

    maximum_sum_error = float(
        np.max(
            np.abs(
                attention_sums - 1.0
            )
        )
    )

    if maximum_sum_error > 1e-3:
        raise ValueError(
            "Attention normalization failed. "
            f"Maximum error = {maximum_sum_error}"
        )

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            number_of_rows
        ),
        "number_of_residues": int(
            len(residue_df)
        ),
        "attention_matrix_shape": [
            int(number_of_rows),
            int(MAX_LEN),
        ],
        "maximum_sequence_sum_error": (
            maximum_sum_error
        ),
        "mean_attention_score": float(
            residue_df[
                "attention_score"
            ].mean()
        ),
        "maximum_attention_score": float(
            residue_df[
                "attention_score"
            ].max()
        ),
        "minimum_attention_score": float(
            residue_df[
                "attention_score"
            ].min()
        ),
        "completed_at": (
            pd.Timestamp.now()
            .isoformat()
        ),
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    with open(
        complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    print(
        f"[COMPLETE] {model_name} | "
        f"{dataset_name}"
    )

    print(
        "Sequences:",
        number_of_rows,
        "| Residues:",
        len(residue_df),
        "| Maximum normalization error:",
        maximum_sum_error,
    )

    del attention_matrix
    del embeddings
    del masks
    del trained_model

    tf.keras.backend.clear_session()
    gc.collect()

    return [
        residue_output_file,
        matrix_output_file,
        summary_output_file,
        complete_file,
    ]


# ============================================================
# 6. RUN ALL MODELS AND DATASETS
# ============================================================

all_output_files = []
attention_summary_rows = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in (
        DATASETS_TO_EXPLAIN
    ):
        output_files = (
            extract_attention_scores(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=(
                    config["batch_size"]
                ),
            )
        )

        all_output_files.extend(
            output_files
        )

        summary_file = (
            XAI_ROOT
            / model_name
            / "attention"
            / dataset_name
            / "attention_summary.json"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            attention_summary_rows.append(
                json.load(handle)
            )


# ============================================================
# 7. SAVE MASTER SUMMARY
# ============================================================

attention_summary_df = pd.DataFrame(
    attention_summary_rows
)

master_summary_file = (
    CHECKPOINT_ROOT
    / "attention_extraction_summary.csv"
)

attention_summary_df.to_csv(
    master_summary_file,
    index=False,
)

all_output_files.append(
    master_summary_file
)

mark_step_complete(
    "10_all_attention_scores",
    output_files=all_output_files,
    details={
        "models": list(
            MODEL_CONFIGS.keys()
        ),
        "datasets": (
            DATASETS_TO_EXPLAIN
        ),
        "completed_outputs": int(
            len(attention_summary_rows)
        ),
    },
)

print("\n" + "=" * 75)
print("ATTENTION EXTRACTION SUMMARY")
print("=" * 75)

display(
    attention_summary_df[
        [
            "model",
            "dataset",
            "number_of_sequences",
            "number_of_residues",
            "maximum_sequence_sum_error",
            "maximum_attention_score",
        ]
    ]
)

print("\nSaved master summary:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 11: GRADIENT × INPUT ATTRIBUTION
# TensorFlow 2.20 / Keras 3 compatible
# Restart-safe for all four PLMs
# ============================================================

from pathlib import Path
import gc
import json
import math
import os

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 64,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 48,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 24,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 24,
    },
}

DATASETS_TO_EXPLAIN = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]


# ============================================================
# 2. CUSTOM LAYER REQUIRED FOR MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD EMBEDDINGS, MASKS, AND METADATA
# ============================================================

def load_attribution_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Row mismatch for {model_name}/{dataset_name}"
        )

    return embeddings, masks, metadata


# ============================================================
# 4. FORWARD PASS RETURNING THE PRE-SIGMOID CPP LOGIT
# ============================================================

def forward_cpp_logit(
    trained_model,
    embedding_tensor,
    mask_tensor,
):
    """
    Reproduce the trained classifier forward pass and return
    the pre-sigmoid CPP logit.

    Using the logit rather than probability reduces gradient
    saturation for highly confident predictions.
    """

    layer_norm = trained_model.get_layer(
        "embedding_layer_norm"
    )

    residue_projection = trained_model.get_layer(
        "residue_projection"
    )

    residue_dropout = trained_model.get_layer(
        "residue_dropout"
    )

    attention_pooling = trained_model.get_layer(
        "masked_attention_pooling"
    )

    peptide_dense_1 = trained_model.get_layer(
        "peptide_dense_1"
    )

    peptide_dropout_1 = trained_model.get_layer(
        "peptide_dropout_1"
    )

    peptide_dense_2 = trained_model.get_layer(
        "peptide_dense_2"
    )

    peptide_dropout_2 = trained_model.get_layer(
        "peptide_dropout_2"
    )

    output_layer = trained_model.get_layer(
        "cpp_probability"
    )

    x = tf.cast(
        embedding_tensor,
        tf.float16,
    )

    x = layer_norm(
        x,
        training=False,
    )

    x = residue_projection(
        x,
        training=False,
    )

    x = residue_dropout(
        x,
        training=False,
    )

    pooled = attention_pooling(
        [x, mask_tensor],
        training=False,
    )

    x = peptide_dense_1(
        pooled,
        training=False,
    )

    x = peptide_dropout_1(
        x,
        training=False,
    )

    x = peptide_dense_2(
        x,
        training=False,
    )

    x = peptide_dropout_2(
        x,
        training=False,
    )

    # The saved output layer has sigmoid activation.
    # Apply its trained kernel and bias manually to obtain logits.
    kernel = tf.cast(
        output_layer.kernel,
        x.dtype,
    )

    bias = tf.cast(
        output_layer.bias,
        x.dtype,
    )

    logits = tf.linalg.matmul(
        x,
        kernel,
    ) + bias

    return tf.squeeze(
        logits,
        axis=-1,
    )


# ============================================================
# 5. CALCULATE GRADIENT × INPUT FOR ONE BATCH
# ============================================================

def calculate_gradient_input_batch(
    trained_model,
    embedding_batch,
    mask_batch,
):
    """
    Returns normalized residue scores for one batch.

    Raw residue score:
        sum(abs(gradient * embedding), embedding_dimension)
    """

    # Use float32 watched inputs for more stable gradients.
    input_tensor = tf.convert_to_tensor(
        embedding_batch,
        dtype=tf.float32,
    )

    mask_tensor = tf.convert_to_tensor(
        mask_batch,
        dtype=tf.uint8,
    )

    with tf.GradientTape() as tape:
        tape.watch(input_tensor)

        logits = forward_cpp_logit(
            trained_model=trained_model,
            embedding_tensor=input_tensor,
            mask_tensor=mask_tensor,
        )

        # Sum gives an independent gradient for every batch row,
        # because examples do not interact during inference.
        objective = tf.reduce_sum(
            tf.cast(logits, tf.float32)
        )

    gradients = tape.gradient(
        objective,
        input_tensor,
    )

    if gradients is None:
        raise RuntimeError(
            "Gradient calculation returned None."
        )

    gradient_input = tf.abs(
        gradients * input_tensor
    )

    residue_scores = tf.reduce_sum(
        gradient_input,
        axis=-1,
    )

    mask_float = tf.cast(
        mask_tensor,
        tf.float32,
    )

    residue_scores = (
        residue_scores * mask_float
    )

    # Normalize separately within every peptide.
    row_sums = tf.reduce_sum(
        residue_scores,
        axis=1,
        keepdims=True,
    )

    normalized_scores = tf.math.divide_no_nan(
        residue_scores,
        row_sums,
    )

    return (
        normalized_scores
        .numpy()
        .astype(np.float32)
    )


# ============================================================
# 6. VALIDATE A SAVED BATCH CHUNK
# ============================================================

def valid_saved_chunk(
    chunk_file,
    expected_ids,
    expected_rows,
):
    if not chunk_file.exists():
        return False

    try:
        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        saved_ids = (
            chunk["sequence_id"]
            .astype(str)
        )

        saved_scores = chunk[
            "gradient_input_scores"
        ]

        valid = (
            np.array_equal(
                saved_ids,
                expected_ids.astype(str),
            )
            and saved_scores.shape
            == (
                expected_rows,
                MAX_LEN,
            )
        )

        chunk.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 7. PROCESS ONE MODEL / DATASET
# ============================================================

def extract_gradient_input(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        XAI_ROOT
        / model_name
        / "gradient_input"
        / dataset_name
    )

    chunk_dir = (
        output_dir / "batch_chunks"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    matrix_output_file = (
        output_dir
        / "gradient_input_matrix.npy"
    )

    residue_output_file = (
        output_dir
        / "residue_gradient_input_scores.csv"
    )

    summary_output_file = (
        output_dir
        / "gradient_input_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        complete_file.exists()
        and matrix_output_file.exists()
        and residue_output_file.exists()
        and summary_output_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] {model_name} | "
            f"{dataset_name}"
        )

        return [
            matrix_output_file,
            residue_output_file,
            summary_output_file,
            complete_file,
        ]

    print("\n" + "=" * 75)
    print(
        f"GRADIENT × INPUT: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 75)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found:\n{model_file}"
        )

    trained_model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings, masks, metadata = (
        load_attribution_dataset(
            model_name,
            dataset_name,
        )
    )

    number_of_rows = len(metadata)

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    # --------------------------------------------------------
    # Generate restart-safe chunks
    # --------------------------------------------------------

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        batch_metadata = metadata.iloc[
            start_index:end_index
        ]

        expected_ids = (
            batch_metadata[
                "sequence_id"
            ]
            .astype(str)
            .to_numpy()
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        if valid_saved_chunk(
            chunk_file=chunk_file,
            expected_ids=expected_ids,
            expected_rows=len(batch_metadata),
        ):
            print(
                f"[SKIP CHUNK] {model_name} | "
                f"{dataset_name} | "
                f"{batch_number + 1}/"
                f"{number_of_batches}"
            )

            continue

        if chunk_file.exists():
            chunk_file.unlink()

        embedding_batch = np.asarray(
            embeddings[
                start_index:end_index
            ],
            dtype=np.float32,
        )

        mask_batch = np.asarray(
            masks[
                start_index:end_index
            ],
            dtype=np.uint8,
        )

        try:
            batch_scores = (
                calculate_gradient_input_batch(
                    trained_model=trained_model,
                    embedding_batch=embedding_batch,
                    mask_batch=mask_batch,
                )
            )

        except tf.errors.ResourceExhaustedError:
            raise RuntimeError(
                f"GPU memory error for {model_name}. "
                f"Reduce its batch_size and rerun. "
                f"Completed chunks will be skipped."
            )

        temporary_chunk = Path(
            str(chunk_file) + ".temporary.npz"
        )

        np.savez_compressed(
            temporary_chunk,
            sequence_id=expected_ids,
            gradient_input_scores=batch_scores,
        )

        temporary_chunk.replace(
            chunk_file
        )

        print(
            f"[SAVED CHUNK] {model_name} | "
            f"{dataset_name} | "
            f"{batch_number + 1}/"
            f"{number_of_batches}"
        )

        del embedding_batch
        del mask_batch
        del batch_scores

        gc.collect()

    # --------------------------------------------------------
    # Assemble final attribution matrix
    # --------------------------------------------------------

    temporary_matrix_file = (
        output_dir
        / "gradient_input_matrix.temporary.npy"
    )

    matrix_memmap = (
        np.lib.format.open_memmap(
            temporary_matrix_file,
            mode="w+",
            dtype=np.float32,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        matrix_memmap[
            start_index:end_index
        ] = chunk[
            "gradient_input_scores"
        ]

        chunk.close()

    matrix_memmap.flush()
    del matrix_memmap

    os.replace(
        temporary_matrix_file,
        matrix_output_file,
    )

    score_matrix = np.load(
        matrix_output_file,
        mmap_mode="r",
    )

    # --------------------------------------------------------
    # Create one row per real residue
    # --------------------------------------------------------

    residue_rows = []

    for row_index, row in metadata.iterrows():
        sequence = str(row["sequence"])
        sequence_length = len(sequence)

        for position_index, residue in enumerate(
            sequence
        ):
            residue_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "row_index": int(row_index),
                "sequence_id": str(
                    row["sequence_id"]
                ),
                "sequence": sequence,
                "label": int(row["label"]),
                "sequence_length": int(
                    sequence_length
                ),
                "position": int(
                    position_index + 1
                ),
                "normalized_position": float(
                    (
                        position_index + 1
                    )
                    / sequence_length
                ),
                "residue": residue,
                "gradient_input_score": float(
                    score_matrix[
                        row_index,
                        position_index,
                    ]
                ),
            })

    residue_df = pd.DataFrame(
        residue_rows
    )

    residue_df.to_csv(
        residue_output_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    sequence_score_sums = (
        residue_df
        .groupby("sequence_id")[
            "gradient_input_score"
        ]
        .sum()
        .to_numpy()
    )

    zero_sum_sequences = int(
        np.sum(
            np.isclose(
                sequence_score_sums,
                0.0,
            )
        )
    )

    nonzero_sums = sequence_score_sums[
        ~np.isclose(
            sequence_score_sums,
            0.0,
        )
    ]

    if len(nonzero_sums) > 0:
        maximum_sum_error = float(
            np.max(
                np.abs(
                    nonzero_sums - 1.0
                )
            )
        )
    else:
        maximum_sum_error = None

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            number_of_rows
        ),
        "number_of_residues": int(
            len(residue_df)
        ),
        "score_matrix_shape": [
            int(number_of_rows),
            int(MAX_LEN),
        ],
        "zero_sum_sequences": int(
            zero_sum_sequences
        ),
        "maximum_nonzero_sequence_sum_error": (
            maximum_sum_error
        ),
        "mean_score": float(
            residue_df[
                "gradient_input_score"
            ].mean()
        ),
        "maximum_score": float(
            residue_df[
                "gradient_input_score"
            ].max()
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file = Path(
        str(complete_file) + ".temporary"
    )

    with open(
        temporary_complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file.replace(
        complete_file
    )

    print(
        f"[COMPLETE] {model_name} | "
        f"{dataset_name}"
    )

    print(
        "Sequences:",
        number_of_rows,
        "| Residues:",
        len(residue_df),
        "| Zero-sum sequences:",
        zero_sum_sequences,
        "| Max normalization error:",
        maximum_sum_error,
    )

    del score_matrix
    del embeddings
    del masks
    del trained_model

    tf.keras.backend.clear_session()
    gc.collect()

    return [
        matrix_output_file,
        residue_output_file,
        summary_output_file,
        complete_file,
    ]


# ============================================================
# 8. RUN ALL MODELS AND DATASETS
# ============================================================

all_output_files = []
summary_rows = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in (
        DATASETS_TO_EXPLAIN
    ):
        output_files = (
            extract_gradient_input(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=config[
                    "batch_size"
                ],
            )
        )

        all_output_files.extend(
            output_files
        )

        summary_file = (
            XAI_ROOT
            / model_name
            / "gradient_input"
            / dataset_name
            / "gradient_input_summary.json"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            summary_rows.append(
                json.load(handle)
            )


# ============================================================
# 9. SAVE MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

master_summary_file = (
    CHECKPOINT_ROOT
    / "gradient_input_extraction_summary.csv"
)

summary_df.to_csv(
    master_summary_file,
    index=False,
)

all_output_files.append(
    master_summary_file
)

mark_step_complete(
    "11_all_gradient_input_scores",
    output_files=all_output_files,
    details={
        "models": list(
            MODEL_CONFIGS.keys()
        ),
        "datasets": (
            DATASETS_TO_EXPLAIN
        ),
        "completed_outputs": int(
            len(summary_rows)
        ),
    },
)

print("\n" + "=" * 75)
print("GRADIENT × INPUT SUMMARY")
print("=" * 75)

display(
    summary_df[
        [
            "model",
            "dataset",
            "number_of_sequences",
            "number_of_residues",
            "zero_sum_sequences",
            "maximum_nonzero_sequence_sum_error",
            "maximum_score",
        ]
    ]
)

print("\nMaster summary saved to:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 12: INTEGRATED GRADIENTS ATTRIBUTION
# TensorFlow 2.20 / Keras 3 compatible
# Restart-safe for all four PLMs
# ============================================================

from pathlib import Path
import gc
import json
import math
import os

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 16,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 12,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 6,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 6,
    },
}

DATASETS_TO_EXPLAIN = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61
IG_STEPS = 32

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]


# ============================================================
# 2. CUSTOM LAYER FOR MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD EMBEDDINGS AND METADATA
# ============================================================

def load_ig_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Row mismatch for {model_name}/{dataset_name}"
        )

    return embeddings, masks, metadata


# ============================================================
# 4. FORWARD PASS TO PRE-SIGMOID CPP LOGIT
# ============================================================

def forward_cpp_logit(
    trained_model,
    embedding_tensor,
    mask_tensor,
):
    layer_norm = trained_model.get_layer(
        "embedding_layer_norm"
    )

    residue_projection = trained_model.get_layer(
        "residue_projection"
    )

    residue_dropout = trained_model.get_layer(
        "residue_dropout"
    )

    attention_pooling = trained_model.get_layer(
        "masked_attention_pooling"
    )

    peptide_dense_1 = trained_model.get_layer(
        "peptide_dense_1"
    )

    peptide_dropout_1 = trained_model.get_layer(
        "peptide_dropout_1"
    )

    peptide_dense_2 = trained_model.get_layer(
        "peptide_dense_2"
    )

    peptide_dropout_2 = trained_model.get_layer(
        "peptide_dropout_2"
    )

    output_layer = trained_model.get_layer(
        "cpp_probability"
    )

    x = tf.cast(
        embedding_tensor,
        tf.float16,
    )

    x = layer_norm(
        x,
        training=False,
    )

    x = residue_projection(
        x,
        training=False,
    )

    x = residue_dropout(
        x,
        training=False,
    )

    pooled = attention_pooling(
        [x, mask_tensor],
        training=False,
    )

    x = peptide_dense_1(
        pooled,
        training=False,
    )

    x = peptide_dropout_1(
        x,
        training=False,
    )

    x = peptide_dense_2(
        x,
        training=False,
    )

    x = peptide_dropout_2(
        x,
        training=False,
    )

    kernel = tf.cast(
        output_layer.kernel,
        x.dtype,
    )

    bias = tf.cast(
        output_layer.bias,
        x.dtype,
    )

    logits = tf.linalg.matmul(
        x,
        kernel,
    ) + bias

    return tf.squeeze(
        logits,
        axis=-1,
    )


# ============================================================
# 5. CALCULATE INTEGRATED GRADIENTS FOR ONE BATCH
# ============================================================

def calculate_integrated_gradients_batch(
    trained_model,
    embedding_batch,
    mask_batch,
    number_of_steps=IG_STEPS,
):
    """
    Integrated Gradients from a zero-embedding baseline.

    For every interpolation point alpha:
        x_alpha = baseline + alpha * (input - baseline)

    The target is the CPP pre-sigmoid logit.
    """

    input_tensor = tf.convert_to_tensor(
        embedding_batch,
        dtype=tf.float32,
    )

    mask_tensor = tf.convert_to_tensor(
        mask_batch,
        dtype=tf.uint8,
    )

    baseline_tensor = tf.zeros_like(
        input_tensor,
        dtype=tf.float32,
    )

    input_difference = (
        input_tensor - baseline_tensor
    )

    accumulated_gradients = tf.zeros_like(
        input_tensor,
        dtype=tf.float32,
    )

    # Midpoint Riemann approximation is more stable than
    # including only the endpoints.
    alpha_values = (
        (
            tf.range(
                number_of_steps,
                dtype=tf.float32,
            )
            + 0.5
        )
        / float(number_of_steps)
    )

    for alpha in alpha_values:
        interpolated_input = (
            baseline_tensor
            + alpha * input_difference
        )

        with tf.GradientTape() as tape:
            tape.watch(
                interpolated_input
            )

            logits = forward_cpp_logit(
                trained_model=trained_model,
                embedding_tensor=interpolated_input,
                mask_tensor=mask_tensor,
            )

            objective = tf.reduce_sum(
                tf.cast(
                    logits,
                    tf.float32,
                )
            )

        gradients = tape.gradient(
            objective,
            interpolated_input,
        )

        if gradients is None:
            raise RuntimeError(
                "Integrated Gradients returned None."
            )

        accumulated_gradients += gradients

    average_gradients = (
        accumulated_gradients
        / float(number_of_steps)
    )

    integrated_gradients = (
        input_difference
        * average_gradients
    )

    # Absolute attribution summed across embedding dimensions.
    residue_scores = tf.reduce_sum(
        tf.abs(
            integrated_gradients
        ),
        axis=-1,
    )

    mask_float = tf.cast(
        mask_tensor,
        tf.float32,
    )

    residue_scores = (
        residue_scores
        * mask_float
    )

    row_sums = tf.reduce_sum(
        residue_scores,
        axis=1,
        keepdims=True,
    )

    normalized_scores = tf.math.divide_no_nan(
        residue_scores,
        row_sums,
    )

    return (
        normalized_scores
        .numpy()
        .astype(np.float32)
    )


# ============================================================
# 6. VALIDATE EXISTING CHUNKS
# ============================================================

def valid_saved_ig_chunk(
    chunk_file,
    expected_ids,
    expected_rows,
):
    if not chunk_file.exists():
        return False

    try:
        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        saved_ids = (
            chunk["sequence_id"]
            .astype(str)
        )

        saved_scores = (
            chunk[
                "integrated_gradients_scores"
            ]
        )

        saved_steps = int(
            chunk["ig_steps"][0]
        )

        valid = (
            np.array_equal(
                saved_ids,
                expected_ids.astype(str),
            )
            and saved_scores.shape
            == (
                expected_rows,
                MAX_LEN,
            )
            and saved_steps == IG_STEPS
        )

        chunk.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 7. PROCESS ONE MODEL / DATASET
# ============================================================

def extract_integrated_gradients(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        XAI_ROOT
        / model_name
        / "integrated_gradients"
        / dataset_name
    )

    chunk_dir = (
        output_dir / "batch_chunks"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    matrix_output_file = (
        output_dir
        / "integrated_gradients_matrix.npy"
    )

    residue_output_file = (
        output_dir
        / "residue_integrated_gradients_scores.csv"
    )

    summary_output_file = (
        output_dir
        / "integrated_gradients_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        complete_file.exists()
        and matrix_output_file.exists()
        and residue_output_file.exists()
        and summary_output_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] {model_name} | "
            f"{dataset_name}"
        )

        return [
            matrix_output_file,
            residue_output_file,
            summary_output_file,
            complete_file,
        ]

    print("\n" + "=" * 75)
    print(
        f"INTEGRATED GRADIENTS: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 75)

    print("Integration steps:", IG_STEPS)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found:\n{model_file}"
        )

    trained_model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings, masks, metadata = (
        load_ig_dataset(
            model_name,
            dataset_name,
        )
    )

    number_of_rows = len(metadata)

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    # --------------------------------------------------------
    # Generate restart-safe batch chunks
    # --------------------------------------------------------

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        batch_metadata = metadata.iloc[
            start_index:end_index
        ]

        expected_ids = (
            batch_metadata[
                "sequence_id"
            ]
            .astype(str)
            .to_numpy()
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        if valid_saved_ig_chunk(
            chunk_file=chunk_file,
            expected_ids=expected_ids,
            expected_rows=len(batch_metadata),
        ):
            print(
                f"[SKIP CHUNK] {model_name} | "
                f"{dataset_name} | "
                f"{batch_number + 1}/"
                f"{number_of_batches}"
            )

            continue

        if chunk_file.exists():
            chunk_file.unlink()

        embedding_batch = np.asarray(
            embeddings[
                start_index:end_index
            ],
            dtype=np.float32,
        )

        mask_batch = np.asarray(
            masks[
                start_index:end_index
            ],
            dtype=np.uint8,
        )

        try:
            batch_scores = (
                calculate_integrated_gradients_batch(
                    trained_model=trained_model,
                    embedding_batch=embedding_batch,
                    mask_batch=mask_batch,
                    number_of_steps=IG_STEPS,
                )
            )

        except tf.errors.ResourceExhaustedError:
            raise RuntimeError(
                f"GPU memory error for {model_name}. "
                f"Reduce batch_size and rerun. "
                f"Completed chunks will be skipped."
            )

        temporary_chunk = Path(
            str(chunk_file) + ".temporary.npz"
        )

        np.savez_compressed(
            temporary_chunk,
            sequence_id=expected_ids,
            integrated_gradients_scores=(
                batch_scores
            ),
            ig_steps=np.array(
                [IG_STEPS],
                dtype=np.int32,
            ),
        )

        temporary_chunk.replace(
            chunk_file
        )

        print(
            f"[SAVED CHUNK] {model_name} | "
            f"{dataset_name} | "
            f"{batch_number + 1}/"
            f"{number_of_batches}"
        )

        del embedding_batch
        del mask_batch
        del batch_scores

        gc.collect()

    # --------------------------------------------------------
    # Assemble final matrix
    # --------------------------------------------------------

    temporary_matrix_file = (
        output_dir
        / "integrated_gradients_matrix.temporary.npy"
    )

    matrix_memmap = (
        np.lib.format.open_memmap(
            temporary_matrix_file,
            mode="w+",
            dtype=np.float32,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        matrix_memmap[
            start_index:end_index
        ] = chunk[
            "integrated_gradients_scores"
        ]

        chunk.close()

    matrix_memmap.flush()
    del matrix_memmap

    os.replace(
        temporary_matrix_file,
        matrix_output_file,
    )

    score_matrix = np.load(
        matrix_output_file,
        mmap_mode="r",
    )

    # --------------------------------------------------------
    # Create residue-level table
    # --------------------------------------------------------

    residue_rows = []

    for row_index, row in metadata.iterrows():
        sequence = str(row["sequence"])
        sequence_length = len(sequence)

        for position_index, residue in enumerate(
            sequence
        ):
            residue_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "row_index": int(row_index),
                "sequence_id": str(
                    row["sequence_id"]
                ),
                "sequence": sequence,
                "label": int(row["label"]),
                "sequence_length": int(
                    sequence_length
                ),
                "position": int(
                    position_index + 1
                ),
                "normalized_position": float(
                    (
                        position_index + 1
                    )
                    / sequence_length
                ),
                "residue": residue,
                "integrated_gradients_score": float(
                    score_matrix[
                        row_index,
                        position_index,
                    ]
                ),
            })

    residue_df = pd.DataFrame(
        residue_rows
    )

    residue_df.to_csv(
        residue_output_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    sequence_score_sums = (
        residue_df
        .groupby("sequence_id")[
            "integrated_gradients_score"
        ]
        .sum()
        .to_numpy()
    )

    zero_sum_sequences = int(
        np.sum(
            np.isclose(
                sequence_score_sums,
                0.0,
            )
        )
    )

    nonzero_sums = sequence_score_sums[
        ~np.isclose(
            sequence_score_sums,
            0.0,
        )
    ]

    if len(nonzero_sums) > 0:
        maximum_sum_error = float(
            np.max(
                np.abs(
                    nonzero_sums - 1.0
                )
            )
        )
    else:
        maximum_sum_error = None

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            number_of_rows
        ),
        "number_of_residues": int(
            len(residue_df)
        ),
        "ig_steps": int(
            IG_STEPS
        ),
        "baseline": (
            "all-zero residue embedding tensor"
        ),
        "target": (
            "pre-sigmoid CPP logit"
        ),
        "score_matrix_shape": [
            int(number_of_rows),
            int(MAX_LEN),
        ],
        "zero_sum_sequences": int(
            zero_sum_sequences
        ),
        "maximum_nonzero_sequence_sum_error": (
            maximum_sum_error
        ),
        "mean_score": float(
            residue_df[
                "integrated_gradients_score"
            ].mean()
        ),
        "maximum_score": float(
            residue_df[
                "integrated_gradients_score"
            ].max()
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file = Path(
        str(complete_file) + ".temporary"
    )

    with open(
        temporary_complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file.replace(
        complete_file
    )

    print(
        f"[COMPLETE] {model_name} | "
        f"{dataset_name}"
    )

    print(
        "Sequences:",
        number_of_rows,
        "| Residues:",
        len(residue_df),
        "| Zero-sum sequences:",
        zero_sum_sequences,
        "| Maximum normalization error:",
        maximum_sum_error,
    )

    del score_matrix
    del embeddings
    del masks
    del trained_model

    tf.keras.backend.clear_session()
    gc.collect()

    return [
        matrix_output_file,
        residue_output_file,
        summary_output_file,
        complete_file,
    ]


# ============================================================
# 8. RUN ALL MODELS AND DATASETS
# ============================================================

all_output_files = []
summary_rows = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in (
        DATASETS_TO_EXPLAIN
    ):
        output_files = (
            extract_integrated_gradients(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=config[
                    "batch_size"
                ],
            )
        )

        all_output_files.extend(
            output_files
        )

        summary_file = (
            XAI_ROOT
            / model_name
            / "integrated_gradients"
            / dataset_name
            / "integrated_gradients_summary.json"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            summary_rows.append(
                json.load(handle)
            )


# ============================================================
# 9. SAVE MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

master_summary_file = (
    CHECKPOINT_ROOT
    / "integrated_gradients_extraction_summary.csv"
)

summary_df.to_csv(
    master_summary_file,
    index=False,
)

all_output_files.append(
    master_summary_file
)

mark_step_complete(
    "12_all_integrated_gradients_scores",
    output_files=all_output_files,
    details={
        "models": list(
            MODEL_CONFIGS.keys()
        ),
        "datasets": (
            DATASETS_TO_EXPLAIN
        ),
        "integration_steps": int(
            IG_STEPS
        ),
        "baseline": (
            "all-zero residue embeddings"
        ),
        "completed_outputs": int(
            len(summary_rows)
        ),
    },
)

print("\n" + "=" * 75)
print("INTEGRATED GRADIENTS SUMMARY")
print("=" * 75)

display(
    summary_df[
        [
            "model",
            "dataset",
            "number_of_sequences",
            "number_of_residues",
            "ig_steps",
            "zero_sum_sequences",
            "maximum_nonzero_sequence_sum_error",
            "maximum_score",
        ]
    ]
)

print("\nMaster summary saved to:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 13: CONSENSUS XAI ACROSS METHODS AND PLMS
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]
RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)
CHECKPOINT_DIR = DIRS["checkpoints"]

CONSENSUS_ROOT = (
    XAI_ROOT / "consensus"
)

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONSENSUS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. RANK NORMALIZATION WITHIN EACH PEPTIDE
# ============================================================

def rank_normalize_within_sequence(
    dataframe,
    score_column,
):
    """
    Convert residue scores to percentile ranks within
    each peptide.

    Highest-scoring residue approaches 1.0.
    """

    ranked = (
        dataframe
        .groupby("sequence_id")[
            score_column
        ]
        .rank(
            method="average",
            pct=True,
        )
    )

    return ranked.astype(float)


# ============================================================
# 3. LOAD THREE XAI METHODS FOR ONE MODEL/DATASET
# ============================================================

def load_model_xai(
    model_name,
    dataset_name,
):

    attention_file = (
        XAI_ROOT
        / model_name
        / "attention"
        / dataset_name
        / "residue_attention_scores.csv"
    )

    gradient_file = (
        XAI_ROOT
        / model_name
        / "gradient_input"
        / dataset_name
        / "residue_gradient_input_scores.csv"
    )

    ig_file = (
        XAI_ROOT
        / model_name
        / "integrated_gradients"
        / dataset_name
        / "residue_integrated_gradients_scores.csv"
    )

    for required_file in [
        attention_file,
        gradient_file,
        ig_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing XAI file:\n{required_file}"
            )

    attention = pd.read_csv(
        attention_file
    )

    gradient = pd.read_csv(
        gradient_file
    )

    integrated = pd.read_csv(
        ig_file
    )

    identity_columns = [
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
    ]

    merged = attention[
        identity_columns
        + ["attention_score"]
    ].copy()

    merged = merged.merge(
        gradient[
            identity_columns
            + ["gradient_input_score"]
        ],
        on=identity_columns,
        how="inner",
        validate="one_to_one",
    )

    merged = merged.merge(
        integrated[
            identity_columns
            + ["integrated_gradients_score"]
        ],
        on=identity_columns,
        how="inner",
        validate="one_to_one",
    )

    expected_rows = len(attention)

    if len(merged) != expected_rows:
        raise ValueError(
            f"Merge row mismatch for "
            f"{model_name}/{dataset_name}: "
            f"{len(merged)} versus {expected_rows}"
        )

    merged["model"] = model_name
    merged["dataset"] = dataset_name

    return merged


# ============================================================
# 4. CREATE PER-MODEL CONSENSUS
# ============================================================

def calculate_model_consensus(
    model_name,
    dataset_name,
):

    merged = load_model_xai(
        model_name,
        dataset_name,
    )

    merged["attention_rank"] = (
        rank_normalize_within_sequence(
            merged,
            "attention_score",
        )
    )

    merged["gradient_input_rank"] = (
        rank_normalize_within_sequence(
            merged,
            "gradient_input_score",
        )
    )

    merged["integrated_gradients_rank"] = (
        rank_normalize_within_sequence(
            merged,
            "integrated_gradients_score",
        )
    )

    method_rank_columns = [
        "attention_rank",
        "gradient_input_rank",
        "integrated_gradients_rank",
    ]

    merged[
        "model_consensus_score"
    ] = merged[
        method_rank_columns
    ].mean(axis=1)

    merged[
        "method_rank_std"
    ] = merged[
        method_rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    # Number of methods placing residue in top 20%
    merged[
        "methods_top20_count"
    ] = (
        merged[
            method_rank_columns
        ] >= 0.80
    ).sum(axis=1)

    merged[
        "methods_top15_count"
    ] = (
        merged[
            method_rank_columns
        ] >= 0.85
    ).sum(axis=1)

    merged[
        "methods_top10_count"
    ] = (
        merged[
            method_rank_columns
        ] >= 0.90
    ).sum(axis=1)

    merged[
        "model_consensus_rank"
    ] = (
        rank_normalize_within_sequence(
            merged,
            "model_consensus_score",
        )
    )

    return merged


# ============================================================
# 5. METHOD AGREEMENT
# ============================================================

def calculate_method_agreement(
    dataframe,
    model_name,
    dataset_name,
):

    rows = []

    method_pairs = [
        (
            "attention_rank",
            "gradient_input_rank",
            "Attention vs Gradient×Input",
        ),
        (
            "attention_rank",
            "integrated_gradients_rank",
            "Attention vs Integrated Gradients",
        ),
        (
            "gradient_input_rank",
            "integrated_gradients_rank",
            "Gradient×Input vs Integrated Gradients",
        ),
    ]

    for (
        method_a,
        method_b,
        pair_name,
    ) in method_pairs:

        sequence_correlations = []

        for _, group in dataframe.groupby(
            "sequence_id"
        ):
            if len(group) < 3:
                continue

            correlation, _ = spearmanr(
                group[method_a],
                group[method_b],
            )

            if np.isfinite(correlation):
                sequence_correlations.append(
                    correlation
                )

        rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "method_pair": pair_name,
            "number_of_sequences": int(
                len(sequence_correlations)
            ),
            "mean_sequence_spearman": float(
                np.mean(
                    sequence_correlations
                )
            ),
            "median_sequence_spearman": float(
                np.median(
                    sequence_correlations
                )
            ),
            "standard_deviation": float(
                np.std(
                    sequence_correlations,
                    ddof=0,
                )
            ),
        })

    return rows


# ============================================================
# 6. RUN PER-MODEL CONSENSUS
# ============================================================

all_model_consensus = {}
method_agreement_rows = []
all_output_files = []

for dataset_name in DATASETS:

    all_model_consensus[
        dataset_name
    ] = {}

    for model_name in MODEL_NAMES:

        print(
            f"[CONSENSUS] {model_name} | "
            f"{dataset_name}"
        )

        model_consensus = (
            calculate_model_consensus(
                model_name,
                dataset_name,
            )
        )

        model_output_dir = (
            CONSENSUS_ROOT
            / model_name
            / dataset_name
        )

        model_output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model_output_file = (
            model_output_dir
            / "model_method_consensus.csv"
        )

        model_consensus.to_csv(
            model_output_file,
            index=False,
        )

        all_output_files.append(
            model_output_file
        )

        all_model_consensus[
            dataset_name
        ][model_name] = (
            model_consensus
        )

        method_agreement_rows.extend(
            calculate_method_agreement(
                dataframe=model_consensus,
                model_name=model_name,
                dataset_name=dataset_name,
            )
        )


# ============================================================
# 7. FOUR-PLM GLOBAL CONSENSUS
# ============================================================

global_consensus_tables = {}

identity_columns = [
    "sequence_id",
    "sequence",
    "label",
    "sequence_length",
    "position",
    "normalized_position",
    "residue",
]

for dataset_name in DATASETS:

    reference = (
        all_model_consensus[
            dataset_name
        ][MODEL_NAMES[0]][
            identity_columns
        ]
        .copy()
    )

    global_table = reference.copy()

    model_score_columns = []
    model_rank_columns = []

    for model_name in MODEL_NAMES:

        current = (
            all_model_consensus[
                dataset_name
            ][model_name][
                identity_columns
                + [
                    "model_consensus_score",
                    "model_consensus_rank",
                    "method_rank_std",
                    "methods_top20_count",
                ]
            ]
            .copy()
        )

        rename_map = {
            "model_consensus_score":
                f"consensus_score_{model_name}",
            "model_consensus_rank":
                f"consensus_rank_{model_name}",
            "method_rank_std":
                f"method_rank_std_{model_name}",
            "methods_top20_count":
                f"methods_top20_count_{model_name}",
        }

        current = current.rename(
            columns=rename_map
        )

        global_table = global_table.merge(
            current,
            on=identity_columns,
            how="inner",
            validate="one_to_one",
        )

        model_score_columns.append(
            f"consensus_score_{model_name}"
        )

        model_rank_columns.append(
            f"consensus_rank_{model_name}"
        )

    global_table[
        "global_consensus_score"
    ] = global_table[
        model_rank_columns
    ].mean(axis=1)

    global_table[
        "cross_model_rank_std"
    ] = global_table[
        model_rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    global_table[
        "models_top20_count"
    ] = (
        global_table[
            model_rank_columns
        ] >= 0.80
    ).sum(axis=1)

    global_table[
        "models_top15_count"
    ] = (
        global_table[
            model_rank_columns
        ] >= 0.85
    ).sum(axis=1)

    global_table[
        "models_top10_count"
    ] = (
        global_table[
            model_rank_columns
        ] >= 0.90
    ).sum(axis=1)

    global_table[
        "global_consensus_rank"
    ] = (
        rank_normalize_within_sequence(
            global_table,
            "global_consensus_score",
        )
    )

    global_table[
        "hotspot_top20"
    ] = (
        global_table[
            "global_consensus_rank"
        ] >= 0.80
    )

    global_table[
        "hotspot_top15"
    ] = (
        global_table[
            "global_consensus_rank"
        ] >= 0.85
    )

    global_table[
        "hotspot_top10"
    ] = (
        global_table[
            "global_consensus_rank"
        ] >= 0.90
    )

    # Strict consensus: top 20% in at least 3/4 models
    global_table[
        "strict_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ] >= 3
    )

    # Very strict: top 20% in all four models
    global_table[
        "unanimous_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ] == 4
    )

    output_dir = (
        CONSENSUS_ROOT
        / "global"
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    global_output_file = (
        output_dir
        / "global_consensus_residue_scores.csv"
    )

    global_table.to_csv(
        global_output_file,
        index=False,
    )

    all_output_files.append(
        global_output_file
    )

    global_consensus_tables[
        dataset_name
    ] = global_table


# ============================================================
# 8. GLOBAL CONSENSUS SUMMARY
# ============================================================

summary_rows = []

for dataset_name, table in (
    global_consensus_tables.items()
):

    total_residues = len(table)

    for label_value, label_name in [
        (0, "non_CPP"),
        (1, "CPP"),
    ]:

        subset = table[
            table["label"] == label_value
        ]

        summary_rows.append({
            "dataset": dataset_name,
            "class": label_name,
            "number_of_sequences": int(
                subset[
                    "sequence_id"
                ].nunique()
            ),
            "number_of_residues": int(
                len(subset)
            ),
            "top20_hotspots": int(
                subset[
                    "hotspot_top20"
                ].sum()
            ),
            "strict_cross_model_hotspots": int(
                subset[
                    "strict_cross_model_hotspot"
                ].sum()
            ),
            "unanimous_cross_model_hotspots": int(
                subset[
                    "unanimous_cross_model_hotspot"
                ].sum()
            ),
            "mean_cross_model_rank_std": float(
                subset[
                    "cross_model_rank_std"
                ].mean()
            ),
            "mean_global_consensus_score": float(
                subset[
                    "global_consensus_score"
                ].mean()
            ),
        })


summary_df = pd.DataFrame(
    summary_rows
)

summary_file = (
    RESULT_TABLE_DIR
    / "consensus_xai_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)

all_output_files.append(
    summary_file
)


# ============================================================
# 9. METHOD AGREEMENT TABLE
# ============================================================

method_agreement_df = pd.DataFrame(
    method_agreement_rows
)

method_agreement_file = (
    RESULT_TABLE_DIR
    / "xai_method_agreement.csv"
)

method_agreement_df.to_csv(
    method_agreement_file,
    index=False,
)

all_output_files.append(
    method_agreement_file
)


# ============================================================
# 10. CROSS-MODEL AGREEMENT
# ============================================================

cross_model_rows = []

for dataset_name, table in (
    global_consensus_tables.items()
):

    rank_columns = {
        model_name:
            f"consensus_rank_{model_name}"
        for model_name in MODEL_NAMES
    }

    for index_a in range(
        len(MODEL_NAMES)
    ):
        for index_b in range(
            index_a + 1,
            len(MODEL_NAMES),
        ):

            model_a = MODEL_NAMES[index_a]
            model_b = MODEL_NAMES[index_b]

            sequence_correlations = []

            for _, group in table.groupby(
                "sequence_id"
            ):
                if len(group) < 3:
                    continue

                correlation, _ = spearmanr(
                    group[
                        rank_columns[model_a]
                    ],
                    group[
                        rank_columns[model_b]
                    ],
                )

                if np.isfinite(correlation):
                    sequence_correlations.append(
                        correlation
                    )

            cross_model_rows.append({
                "dataset": dataset_name,
                "model_a": model_a,
                "model_b": model_b,
                "number_of_sequences": int(
                    len(sequence_correlations)
                ),
                "mean_sequence_spearman": float(
                    np.mean(
                        sequence_correlations
                    )
                ),
                "median_sequence_spearman": float(
                    np.median(
                        sequence_correlations
                    )
                ),
                "standard_deviation": float(
                    np.std(
                        sequence_correlations,
                        ddof=0,
                    )
                ),
            })


cross_model_df = pd.DataFrame(
    cross_model_rows
)

cross_model_file = (
    RESULT_TABLE_DIR
    / "cross_model_xai_agreement.csv"
)

cross_model_df.to_csv(
    cross_model_file,
    index=False,
)

all_output_files.append(
    cross_model_file
)


# ============================================================
# 11. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "13_consensus_XAI",
    output_files=all_output_files,
    details={
        "models": MODEL_NAMES,
        "methods": [
            "attention",
            "gradient_input",
            "integrated_gradients",
        ],
        "datasets": DATASETS,
        "hotspot_thresholds": [
            0.80,
            0.85,
            0.90,
        ],
    },
)


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("CONSENSUS XAI SUMMARY")
print("=" * 75)

display(summary_df)

print("\n" + "=" * 75)
print("METHOD AGREEMENT")
print("=" * 75)

display(
    method_agreement_df[
        [
            "model",
            "dataset",
            "method_pair",
            "mean_sequence_spearman",
            "median_sequence_spearman",
        ]
    ]
)

print("\n" + "=" * 75)
print("CROSS-MODEL AGREEMENT")
print("=" * 75)

display(
    cross_model_df[
        [
            "dataset",
            "model_a",
            "model_b",
            "mean_sequence_spearman",
            "median_sequence_spearman",
        ]
    ]
)

print("\nSaved consensus summary:")
print(summary_file)

print("\nSaved method agreement:")
print(method_agreement_file)

print("\nSaved cross-model agreement:")
print(cross_model_file)

In [ ]:
# ============================================================
# STEP 13B: REDUNDANCY-ADJUSTED CONSENSUS XAI
#
# Rationale:
# Gradient × Input and Integrated Gradients were almost
# perfectly correlated. They are therefore combined into one
# gradient-family score before integration with Attention.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]
RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)
CHECKPOINT_DIR = DIRS["checkpoints"]

ADJUSTED_ROOT = (
    XAI_ROOT / "consensus_adjusted"
)

ADJUSTED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. HELPER: RANK WITHIN EACH PEPTIDE
# ============================================================

def rank_within_sequence(
    dataframe,
    score_column,
):
    return (
        dataframe
        .groupby("sequence_id")[
            score_column
        ]
        .rank(
            method="average",
            pct=True,
        )
        .astype(float)
    )


# ============================================================
# 3. LOAD ORIGINAL METHOD-LEVEL CONSENSUS FILE
# ============================================================

def load_original_model_consensus(
    model_name,
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus"
        / model_name
        / dataset_name
        / "model_method_consensus.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Missing original consensus file:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "attention_rank",
        "gradient_input_rank",
        "integrated_gradients_rank",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"{model_name}/{dataset_name} "
            f"is missing columns: {sorted(missing)}"
        )

    return dataframe


# ============================================================
# 4. CREATE ADJUSTED PER-MODEL CONSENSUS
# ============================================================

adjusted_model_tables = {}
output_files = []

for dataset_name in DATASETS:

    adjusted_model_tables[
        dataset_name
    ] = {}

    for model_name in MODEL_NAMES:

        print(
            f"[ADJUSTED CONSENSUS] "
            f"{model_name} | {dataset_name}"
        )

        dataframe = (
            load_original_model_consensus(
                model_name,
                dataset_name,
            )
        )

        # Combine the highly redundant gradient methods.
        dataframe[
            "gradient_family_rank"
        ] = dataframe[
            [
                "gradient_input_rank",
                "integrated_gradients_rank",
            ]
        ].mean(axis=1)

        # Give equal weight to Attention and gradient family.
        dataframe[
            "adjusted_model_consensus_score"
        ] = (
            dataframe["attention_rank"]
            + dataframe[
                "gradient_family_rank"
            ]
        ) / 2.0

        dataframe[
            "adjusted_model_consensus_rank"
        ] = rank_within_sequence(
            dataframe,
            "adjusted_model_consensus_score",
        )

        # Difference between the two attribution families.
        dataframe[
            "attention_gradient_disagreement"
        ] = np.abs(
            dataframe["attention_rank"]
            - dataframe[
                "gradient_family_rank"
            ]
        )

        dataframe[
            "attention_top20"
        ] = (
            dataframe["attention_rank"]
            >= 0.80
        )

        dataframe[
            "gradient_family_top20"
        ] = (
            dataframe[
                "gradient_family_rank"
            ]
            >= 0.80
        )

        dataframe[
            "both_families_top20"
        ] = (
            dataframe["attention_top20"]
            & dataframe[
                "gradient_family_top20"
            ]
        )

        dataframe[
            "adjusted_hotspot_top20"
        ] = (
            dataframe[
                "adjusted_model_consensus_rank"
            ]
            >= 0.80
        )

        dataframe[
            "adjusted_hotspot_top15"
        ] = (
            dataframe[
                "adjusted_model_consensus_rank"
            ]
            >= 0.85
        )

        dataframe[
            "adjusted_hotspot_top10"
        ] = (
            dataframe[
                "adjusted_model_consensus_rank"
            ]
            >= 0.90
        )

        output_dir = (
            ADJUSTED_ROOT
            / model_name
            / dataset_name
        )

        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        output_file = (
            output_dir
            / "adjusted_model_consensus.csv"
        )

        dataframe.to_csv(
            output_file,
            index=False,
        )

        adjusted_model_tables[
            dataset_name
        ][model_name] = dataframe

        output_files.append(
            output_file
        )


# ============================================================
# 5. CREATE ADJUSTED GLOBAL CROSS-MODEL CONSENSUS
# ============================================================

identity_columns = [
    "sequence_id",
    "sequence",
    "label",
    "sequence_length",
    "position",
    "normalized_position",
    "residue",
]

global_tables = {}

for dataset_name in DATASETS:

    reference = (
        adjusted_model_tables[
            dataset_name
        ][MODEL_NAMES[0]][
            identity_columns
        ]
        .copy()
    )

    global_table = reference.copy()

    model_rank_columns = []
    model_score_columns = []

    for model_name in MODEL_NAMES:

        current = (
            adjusted_model_tables[
                dataset_name
            ][model_name][
                identity_columns
                + [
                    "adjusted_model_consensus_score",
                    "adjusted_model_consensus_rank",
                    "attention_gradient_disagreement",
                    "both_families_top20",
                ]
            ]
            .copy()
        )

        current = current.rename(
            columns={
                "adjusted_model_consensus_score":
                    f"adjusted_score_{model_name}",
                "adjusted_model_consensus_rank":
                    f"adjusted_rank_{model_name}",
                "attention_gradient_disagreement":
                    f"family_disagreement_{model_name}",
                "both_families_top20":
                    f"both_families_top20_{model_name}",
            }
        )

        global_table = global_table.merge(
            current,
            on=identity_columns,
            how="inner",
            validate="one_to_one",
        )

        model_rank_columns.append(
            f"adjusted_rank_{model_name}"
        )

        model_score_columns.append(
            f"adjusted_score_{model_name}"
        )

    # Equal weight to each PLM.
    global_table[
        "adjusted_global_consensus_score"
    ] = global_table[
        model_rank_columns
    ].mean(axis=1)

    global_table[
        "adjusted_global_consensus_rank"
    ] = rank_within_sequence(
        global_table,
        "adjusted_global_consensus_score",
    )

    global_table[
        "cross_model_rank_std"
    ] = global_table[
        model_rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    global_table[
        "models_top20_count"
    ] = (
        global_table[
            model_rank_columns
        ]
        >= 0.80
    ).sum(axis=1)

    global_table[
        "models_top15_count"
    ] = (
        global_table[
            model_rank_columns
        ]
        >= 0.85
    ).sum(axis=1)

    global_table[
        "models_top10_count"
    ] = (
        global_table[
            model_rank_columns
        ]
        >= 0.90
    ).sum(axis=1)

    global_table[
        "adjusted_hotspot_top20"
    ] = (
        global_table[
            "adjusted_global_consensus_rank"
        ]
        >= 0.80
    )

    global_table[
        "adjusted_hotspot_top15"
    ] = (
        global_table[
            "adjusted_global_consensus_rank"
        ]
        >= 0.85
    )

    global_table[
        "adjusted_hotspot_top10"
    ] = (
        global_table[
            "adjusted_global_consensus_rank"
        ]
        >= 0.90
    )

    global_table[
        "strict_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ]
        >= 3
    )

    global_table[
        "unanimous_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ]
        == 4
    )

    global_table[
        "mean_family_disagreement"
    ] = global_table[
        [
            f"family_disagreement_{model}"
            for model in MODEL_NAMES
        ]
    ].mean(axis=1)

    global_output_dir = (
        ADJUSTED_ROOT
        / "global"
        / dataset_name
    )

    global_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    global_output_file = (
        global_output_dir
        / "adjusted_global_consensus_residue_scores.csv"
    )

    global_table.to_csv(
        global_output_file,
        index=False,
    )

    output_files.append(
        global_output_file
    )

    global_tables[
        dataset_name
    ] = global_table


# ============================================================
# 6. COMPARE ORIGINAL AND ADJUSTED CONSENSUS
# ============================================================

comparison_rows = []

for dataset_name in DATASETS:

    original_file = (
        XAI_ROOT
        / "consensus"
        / "global"
        / dataset_name
        / "global_consensus_residue_scores.csv"
    )

    original = pd.read_csv(
        original_file
    )

    adjusted = global_tables[
        dataset_name
    ]

    comparison = original[
        identity_columns
        + [
            "global_consensus_rank",
            "hotspot_top20",
            "strict_cross_model_hotspot",
        ]
    ].merge(
        adjusted[
            identity_columns
            + [
                "adjusted_global_consensus_rank",
                "adjusted_hotspot_top20",
                "strict_cross_model_hotspot",
            ]
        ],
        on=identity_columns,
        suffixes=(
            "_original",
            "_adjusted",
        ),
        validate="one_to_one",
    )

    overall_correlation, _ = spearmanr(
        comparison[
            "global_consensus_rank"
        ],
        comparison[
            "adjusted_global_consensus_rank"
        ],
    )

    original_hotspots = (
        comparison[
            "hotspot_top20"
        ].astype(bool)
    )

    adjusted_hotspots = (
        comparison[
            "adjusted_hotspot_top20"
        ].astype(bool)
    )

    intersection = int(
        (
            original_hotspots
            & adjusted_hotspots
        ).sum()
    )

    union = int(
        (
            original_hotspots
            | adjusted_hotspots
        ).sum()
    )

    jaccard = (
        intersection / union
        if union > 0
        else np.nan
    )

    sequence_correlations = []

    for _, group in comparison.groupby(
        "sequence_id"
    ):
        if len(group) < 3:
            continue

        correlation, _ = spearmanr(
            group[
                "global_consensus_rank"
            ],
            group[
                "adjusted_global_consensus_rank"
            ],
        )

        if np.isfinite(correlation):
            sequence_correlations.append(
                correlation
            )

    comparison_rows.append({
        "dataset": dataset_name,
        "number_of_residues": int(
            len(comparison)
        ),
        "overall_rank_spearman": float(
            overall_correlation
        ),
        "mean_within_sequence_spearman": float(
            np.mean(
                sequence_correlations
            )
        ),
        "median_within_sequence_spearman": float(
            np.median(
                sequence_correlations
            )
        ),
        "original_top20_residues": int(
            original_hotspots.sum()
        ),
        "adjusted_top20_residues": int(
            adjusted_hotspots.sum()
        ),
        "top20_intersection": int(
            intersection
        ),
        "top20_jaccard": float(
            jaccard
        ),
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_file = (
    RESULT_TABLE_DIR
    / "original_vs_adjusted_consensus.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False,
)

output_files.append(
    comparison_file
)


# ============================================================
# 7. ADJUSTED CONSENSUS SUMMARY
# ============================================================

summary_rows = []

for dataset_name, table in (
    global_tables.items()
):

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
    ]:

        subset = table[
            table["label"]
            == label_value
        ]

        summary_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "number_of_sequences": int(
                subset[
                    "sequence_id"
                ].nunique()
            ),
            "number_of_residues": int(
                len(subset)
            ),
            "adjusted_top20_hotspots": int(
                subset[
                    "adjusted_hotspot_top20"
                ].sum()
            ),
            "strict_cross_model_hotspots": int(
                subset[
                    "strict_cross_model_hotspot"
                ].sum()
            ),
            "unanimous_cross_model_hotspots": int(
                subset[
                    "unanimous_cross_model_hotspot"
                ].sum()
            ),
            "mean_cross_model_rank_std": float(
                subset[
                    "cross_model_rank_std"
                ].mean()
            ),
            "mean_family_disagreement": float(
                subset[
                    "mean_family_disagreement"
                ].mean()
            ),
        })


summary_df = pd.DataFrame(
    summary_rows
)

summary_file = (
    RESULT_TABLE_DIR
    / "adjusted_consensus_xai_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)

output_files.append(
    summary_file
)


# ============================================================
# 8. SAVE METHODOLOGICAL NOTE
# ============================================================

method_note = {
    "reason_for_adjustment": (
        "Gradient × Input and Integrated Gradients "
        "showed near-perfect residue-rank agreement and were "
        "therefore treated as one gradient-attribution family."
    ),
    "gradient_family_definition": (
        "mean of Gradient × Input percentile rank and "
        "Integrated Gradients percentile rank"
    ),
    "per_model_consensus_definition": (
        "equal-weight mean of Attention rank and "
        "gradient-family rank"
    ),
    "global_consensus_definition": (
        "equal-weight mean of adjusted per-model ranks "
        "across four PLMs"
    ),
    "models": MODEL_NAMES,
    "datasets": DATASETS,
}

method_note_file = (
    CHECKPOINT_DIR
    / "adjusted_consensus_method.json"
)

with open(
    method_note_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        method_note,
        handle,
        indent=2,
    )

output_files.append(
    method_note_file
)


# ============================================================
# 9. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "13B_redundancy_adjusted_consensus",
    output_files=output_files,
    details=method_note,
)


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("ORIGINAL VS ADJUSTED CONSENSUS")
print("=" * 75)

display(comparison_df)

print("\n" + "=" * 75)
print("ADJUSTED CONSENSUS SUMMARY")
print("=" * 75)

display(summary_df)

print("\nAdjusted global consensus files:")

for dataset_name in DATASETS:
    print(
        ADJUSTED_ROOT
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

print("\nComparison table:")
print(comparison_file)

print("\nAdjusted summary:")
print(summary_file)

In [ ]:
# ============================================================
# STEP 14: RESIDUE ENRICHMENT IN ADJUSTED CONSENSUS HOTSPOTS
#
# Analyses:
# 1. Within CPPs:
#    hotspot residues vs non-hotspot CPP residues
#
# 2. Between classes:
#    CPP hotspot residues vs non-CPP hotspot residues
#
# Thresholds:
#    Top 10%, 15%, and 20%
#
# Statistics:
#    Fisher exact test
#    Odds ratio
#    Benjamini-Hochberg FDR correction
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from scipy.stats import fisher_exact


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

AMINO_ACIDS = list(
    "ACDEFGHIKLMNPQRSTVWY"
)

HOTSPOT_DEFINITIONS = {
    "top10": "adjusted_hotspot_top10",
    "top15": "adjusted_hotspot_top15",
    "top20": "adjusted_hotspot_top20",
    "strict_cross_model": (
        "strict_cross_model_hotspot"
    ),
    "unanimous_cross_model": (
        "unanimous_cross_model_hotspot"
    ),
}

XAI_ROOT = DIRS["xai"]
RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)
RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)
CHECKPOINT_DIR = DIRS["checkpoints"]

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_SI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. BENJAMINI-HOCHBERG FDR CORRECTION
# ============================================================

def benjamini_hochberg(
    p_values
):
    """
    Return Benjamini-Hochberg adjusted p-values.
    """

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(p_values)

    order = np.argsort(
        p_values
    )

    ranked_p_values = (
        p_values[order]
    )

    adjusted_ranked = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ranked_p_values[
                reverse_index
            ]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ranked[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


# ============================================================
# 3. SAFE ODDS RATIO WITH HALDANE-ANSCOMBE CORRECTION
# ============================================================

def corrected_odds_ratio(
    a,
    b,
    c,
    d,
):
    """
    Calculate an odds ratio using a 0.5 continuity correction.

    Table:
                 Residue AA   Other residues
        Group 1      a             b
        Group 2      c             d
    """

    return (
        (a + 0.5)
        * (d + 0.5)
        / (
            (b + 0.5)
            * (c + 0.5)
        )
    )


# ============================================================
# 4. LOAD ADJUSTED GLOBAL CONSENSUS
# ============================================================

def load_adjusted_consensus(
    dataset_name
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file not found:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "position",
        "residue",
        "adjusted_global_consensus_rank",
        *HOTSPOT_DEFINITIONS.values(),
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe["residue"] = (
        dataframe["residue"]
        .astype(str)
        .str.upper()
    )

    invalid_residues = sorted(
        set(dataframe["residue"])
        - set(AMINO_ACIDS)
    )

    if invalid_residues:
        raise ValueError(
            "Unexpected amino-acid symbols: "
            f"{invalid_residues}"
        )

    return dataframe, input_file


# ============================================================
# 5. WITHIN-CPP HOTSPOT ENRICHMENT
# ============================================================

def calculate_within_cpp_enrichment(
    dataframe,
    dataset_name,
    hotspot_name,
    hotspot_column,
):
    """
    Compare CPP hotspot residues against non-hotspot
    residues from the same CPP sequences.
    """

    cpp_data = dataframe[
        dataframe["label"] == 1
    ].copy()

    hotspot_data = cpp_data[
        cpp_data[hotspot_column].astype(bool)
    ]

    non_hotspot_data = cpp_data[
        ~cpp_data[hotspot_column].astype(bool)
    ]

    hotspot_total = len(
        hotspot_data
    )

    non_hotspot_total = len(
        non_hotspot_data
    )

    rows = []

    for amino_acid in AMINO_ACIDS:

        hotspot_aa = int(
            (
                hotspot_data["residue"]
                == amino_acid
            ).sum()
        )

        hotspot_other = int(
            hotspot_total
            - hotspot_aa
        )

        non_hotspot_aa = int(
            (
                non_hotspot_data[
                    "residue"
                ]
                == amino_acid
            ).sum()
        )

        non_hotspot_other = int(
            non_hotspot_total
            - non_hotspot_aa
        )

        contingency_table = [
            [
                hotspot_aa,
                hotspot_other,
            ],
            [
                non_hotspot_aa,
                non_hotspot_other,
            ],
        ]

        scipy_odds_ratio, p_value = (
            fisher_exact(
                contingency_table,
                alternative="two-sided",
            )
        )

        corrected_or = (
            corrected_odds_ratio(
                hotspot_aa,
                hotspot_other,
                non_hotspot_aa,
                non_hotspot_other,
            )
        )

        hotspot_frequency = (
            hotspot_aa
            / hotspot_total
            if hotspot_total > 0
            else np.nan
        )

        non_hotspot_frequency = (
            non_hotspot_aa
            / non_hotspot_total
            if non_hotspot_total > 0
            else np.nan
        )

        log2_enrichment = np.log2(
            (
                hotspot_frequency
                + 1e-12
            )
            / (
                non_hotspot_frequency
                + 1e-12
            )
        )

        rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_CPP_nonhotspot"
            ),
            "hotspot_definition": (
                hotspot_name
            ),
            "residue": amino_acid,
            "group1_count": hotspot_aa,
            "group1_total": hotspot_total,
            "group1_frequency": float(
                hotspot_frequency
            ),
            "group2_count": (
                non_hotspot_aa
            ),
            "group2_total": (
                non_hotspot_total
            ),
            "group2_frequency": float(
                non_hotspot_frequency
            ),
            "odds_ratio_scipy": float(
                scipy_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_or
            ),
            "log2_enrichment": float(
                log2_enrichment
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    result["fdr_bh"] = (
        benjamini_hochberg(
            result["p_value"]
            .to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "enrichment_direction"
    ] = np.where(
        result[
            "odds_ratio_corrected"
        ] > 1.0,
        "enriched_in_hotspots",
        "depleted_in_hotspots",
    )

    return result


# ============================================================
# 6. CPP VS NON-CPP HOTSPOT ENRICHMENT
# ============================================================

def calculate_between_class_enrichment(
    dataframe,
    dataset_name,
    hotspot_name,
    hotspot_column,
):
    """
    Compare residue composition of CPP hotspots against
    residue composition of non-CPP hotspots.
    """

    hotspot_data = dataframe[
        dataframe[
            hotspot_column
        ].astype(bool)
    ].copy()

    cpp_hotspots = hotspot_data[
        hotspot_data["label"] == 1
    ]

    noncpp_hotspots = hotspot_data[
        hotspot_data["label"] == 0
    ]

    cpp_total = len(
        cpp_hotspots
    )

    noncpp_total = len(
        noncpp_hotspots
    )

    rows = []

    for amino_acid in AMINO_ACIDS:

        cpp_aa = int(
            (
                cpp_hotspots["residue"]
                == amino_acid
            ).sum()
        )

        cpp_other = int(
            cpp_total
            - cpp_aa
        )

        noncpp_aa = int(
            (
                noncpp_hotspots[
                    "residue"
                ]
                == amino_acid
            ).sum()
        )

        noncpp_other = int(
            noncpp_total
            - noncpp_aa
        )

        contingency_table = [
            [
                cpp_aa,
                cpp_other,
            ],
            [
                noncpp_aa,
                noncpp_other,
            ],
        ]

        scipy_odds_ratio, p_value = (
            fisher_exact(
                contingency_table,
                alternative="two-sided",
            )
        )

        corrected_or = (
            corrected_odds_ratio(
                cpp_aa,
                cpp_other,
                noncpp_aa,
                noncpp_other,
            )
        )

        cpp_frequency = (
            cpp_aa / cpp_total
            if cpp_total > 0
            else np.nan
        )

        noncpp_frequency = (
            noncpp_aa
            / noncpp_total
            if noncpp_total > 0
            else np.nan
        )

        log2_enrichment = np.log2(
            (
                cpp_frequency
                + 1e-12
            )
            / (
                noncpp_frequency
                + 1e-12
            )
        )

        rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            "hotspot_definition": (
                hotspot_name
            ),
            "residue": amino_acid,
            "group1_count": cpp_aa,
            "group1_total": cpp_total,
            "group1_frequency": float(
                cpp_frequency
            ),
            "group2_count": (
                noncpp_aa
            ),
            "group2_total": (
                noncpp_total
            ),
            "group2_frequency": float(
                noncpp_frequency
            ),
            "odds_ratio_scipy": float(
                scipy_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_or
            ),
            "log2_enrichment": float(
                log2_enrichment
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    result["fdr_bh"] = (
        benjamini_hochberg(
            result["p_value"]
            .to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "enrichment_direction"
    ] = np.where(
        result[
            "odds_ratio_corrected"
        ] > 1.0,
        "enriched_in_CPP_hotspots",
        "enriched_in_nonCPP_hotspots",
    )

    return result


# ============================================================
# 7. RUN ALL ENRICHMENT ANALYSES
# ============================================================

all_results = []
source_files = []

for dataset_name in DATASETS:

    consensus_df, source_file = (
        load_adjusted_consensus(
            dataset_name
        )
    )

    source_files.append(
        source_file
    )

    print("\n" + "=" * 75)
    print(
        f"RESIDUE ENRICHMENT: "
        f"{dataset_name}"
    )
    print("=" * 75)

    for (
        hotspot_name,
        hotspot_column,
    ) in HOTSPOT_DEFINITIONS.items():

        print(
            f"[ANALYSIS] {dataset_name} | "
            f"{hotspot_name}"
        )

        within_cpp = (
            calculate_within_cpp_enrichment(
                dataframe=consensus_df,
                dataset_name=dataset_name,
                hotspot_name=hotspot_name,
                hotspot_column=(
                    hotspot_column
                ),
            )
        )

        between_classes = (
            calculate_between_class_enrichment(
                dataframe=consensus_df,
                dataset_name=dataset_name,
                hotspot_name=hotspot_name,
                hotspot_column=(
                    hotspot_column
                ),
            )
        )

        all_results.append(
            within_cpp
        )

        all_results.append(
            between_classes
        )


enrichment_df = pd.concat(
    all_results,
    ignore_index=True,
)


# ============================================================
# 8. SAVE COMPLETE TABLE
# ============================================================

complete_output_file = (
    RESULT_SI_DIR
    / "residue_enrichment_all_thresholds.csv"
)

enrichment_df.to_csv(
    complete_output_file,
    index=False,
)


# ============================================================
# 9. CREATE MAIN TOP-20 SUMMARY
# ============================================================

main_summary = enrichment_df[
    enrichment_df[
        "hotspot_definition"
    ] == "top20"
].copy()

main_summary = main_summary.sort_values(
    [
        "dataset",
        "analysis",
        "fdr_bh",
        "odds_ratio_corrected",
    ],
    ascending=[
        True,
        True,
        True,
        False,
    ],
)

main_summary_file = (
    RESULT_TABLE_DIR
    / "residue_enrichment_top20.csv"
)

main_summary.to_csv(
    main_summary_file,
    index=False,
)


# ============================================================
# 10. SIGNIFICANT RESULTS ONLY
# ============================================================

significant_df = enrichment_df[
    enrichment_df[
        "significant_fdr_0_05"
    ]
].copy()

significant_df = significant_df.sort_values(
    [
        "dataset",
        "analysis",
        "hotspot_definition",
        "fdr_bh",
    ]
)

significant_file = (
    RESULT_TABLE_DIR
    / "significant_residue_enrichment.csv"
)

significant_df.to_csv(
    significant_file,
    index=False,
)


# ============================================================
# 11. CROSS-DATASET REPLICATION
# ============================================================

replication_source = enrichment_df[
    (
        enrichment_df[
            "hotspot_definition"
        ] == "top20"
    )
    & (
        enrichment_df[
            "analysis"
        ] == (
            "CPP_hotspot_vs_CPP_nonhotspot"
        )
    )
][
    [
        "dataset",
        "residue",
        "odds_ratio_corrected",
        "log2_enrichment",
        "fdr_bh",
        "significant_fdr_0_05",
    ]
].copy()

internal_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "internal_test"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)

kelm_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "kelm_external"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)

replication_df = (
    internal_replication.merge(
        kelm_replication,
        on="residue",
        how="outer",
        validate="one_to_one",
    )
)

replication_df[
    "same_enrichment_direction"
] = (
    np.sign(
        replication_df[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replication_df[
            "kelm_log2_enrichment"
        ]
    )
)

replication_df[
    "significant_in_both"
] = (
    replication_df[
        "internal_significant"
    ].fillna(False)
    & replication_df[
        "kelm_significant"
    ].fillna(False)
)

replication_df = (
    replication_df.sort_values(
        [
            "significant_in_both",
            "same_enrichment_direction",
            "internal_log2_enrichment",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)

replication_file = (
    RESULT_TABLE_DIR
    / "residue_enrichment_replication.csv"
)

replication_df.to_csv(
    replication_file,
    index=False,
)


# ============================================================
# 12. SUMMARY JSON
# ============================================================

summary = {
    "datasets": DATASETS,
    "amino_acids": AMINO_ACIDS,
    "hotspot_definitions": (
        HOTSPOT_DEFINITIONS
    ),
    "number_of_tests": int(
        len(enrichment_df)
    ),
    "number_significant_fdr_0_05": int(
        enrichment_df[
            "significant_fdr_0_05"
        ].sum()
    ),
    "top20_number_significant": int(
        main_summary[
            "significant_fdr_0_05"
        ].sum()
    ),
    "replicated_significant_residues": (
        replication_df[
            replication_df[
                "significant_in_both"
            ]
        ]["residue"]
        .tolist()
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "residue_enrichment_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 13. SAVE CHECKPOINT
# ============================================================

output_files = [
    complete_output_file,
    main_summary_file,
    significant_file,
    replication_file,
    summary_file,
]

mark_step_complete(
    "14_residue_enrichment",
    output_files=output_files,
    details=summary,
)


# ============================================================
# 14. DISPLAY IMPORTANT RESULTS
# ============================================================

print("\n" + "=" * 75)
print("TOP-20% CPP HOTSPOT ENRICHMENT")
print("=" * 75)

display(
    main_summary[
        (
            main_summary[
                "analysis"
            ]
            == (
                "CPP_hotspot_vs_CPP_nonhotspot"
            )
        )
    ][
        [
            "dataset",
            "residue",
            "group1_frequency",
            "group2_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "p_value",
            "fdr_bh",
            "significant_fdr_0_05",
            "enrichment_direction",
        ]
    ].sort_values(
        [
            "dataset",
            "fdr_bh",
        ]
    )
)

print("\n" + "=" * 75)
print("INTERNAL–KELM REPLICATION")
print("=" * 75)

display(
    replication_df[
        [
            "residue",
            "internal_odds_ratio",
            "internal_log2_enrichment",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_log2_enrichment",
            "kelm_fdr",
            "same_enrichment_direction",
            "significant_in_both",
        ]
    ]
)

print("\nSignificant replicated residues:")
print(
    replication_df[
        replication_df[
            "significant_in_both"
        ]
    ]["residue"].tolist()
)

print("\nComplete SI table:")
print(complete_output_file)

print("\nMain top-20 table:")
print(main_summary_file)

print("\nReplication table:")
print(replication_file)

In [ ]:
# ============================================================
# STEP 15: HOTSPOT-CENTERED MOTIF DISCOVERY
#
# Extracts motifs around adjusted consensus hotspots and tests
# whether motifs are enriched in CPP hotspot neighborhoods.
#
# Motif lengths: 2 to 5 residues
# Datasets: internal test and KELM external
# ============================================================

from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd

from scipy.stats import fisher_exact


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

MIN_MOTIF_LENGTH = 2
MAX_MOTIF_LENGTH = 5

# Require a motif to occur in at least this many sequences
MIN_CPP_SEQUENCE_SUPPORT = {
    "internal_test": 5,
    "kelm_external": 3,
}

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_SI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. BENJAMINI-HOCHBERG CORRECTION
# ============================================================

def benjamini_hochberg(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(p_values)

    if number_of_tests == 0:
        return np.array([])

    order = np.argsort(p_values)

    ranked = p_values[order]

    adjusted_ranked = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ranked[reverse_index]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ranked[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


# ============================================================
# 3. LOAD ADJUSTED CONSENSUS
# ============================================================

def load_consensus(dataset_name):

    consensus_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not consensus_file.exists():
        raise FileNotFoundError(
            f"Consensus file missing:\n{consensus_file}"
        )

    dataframe = pd.read_csv(
        consensus_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "position",
        "residue",
        "adjusted_hotspot_top20",
        "adjusted_global_consensus_rank",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    return dataframe


# ============================================================
# 4. EXTRACT MOTIFS THAT CONTAIN AT LEAST ONE HOTSPOT
# ============================================================

def extract_sequence_motifs(
    sequence,
    hotspot_positions,
    motif_length,
):
    """
    Return unique motifs of a given length that overlap at
    least one top-20% hotspot.

    hotspot_positions are 1-based.
    """

    sequence = str(sequence)

    zero_based_hotspots = {
        position - 1
        for position in hotspot_positions
    }

    motifs = set()

    maximum_start = (
        len(sequence) - motif_length
    )

    for start in range(
        maximum_start + 1
    ):
        end = start + motif_length

        motif_positions = set(
            range(start, end)
        )

        if motif_positions.intersection(
            zero_based_hotspots
        ):
            motif = sequence[
                start:end
            ]

            motifs.add(motif)

    return motifs


# ============================================================
# 5. BUILD SEQUENCE-LEVEL MOTIF PRESENCE TABLE
# ============================================================

def build_motif_presence(
    consensus_df,
    motif_length,
):

    sequence_records = []

    for sequence_id, group in (
        consensus_df.groupby(
            "sequence_id",
            sort=False,
        )
    ):
        sequence = str(
            group["sequence"].iloc[0]
        )

        label = int(
            group["label"].iloc[0]
        )

        hotspot_positions = (
            group.loc[
                group[
                    "adjusted_hotspot_top20"
                ].astype(bool),
                "position",
            ]
            .astype(int)
            .tolist()
        )

        motifs = extract_sequence_motifs(
            sequence=sequence,
            hotspot_positions=hotspot_positions,
            motif_length=motif_length,
        )

        sequence_records.append({
            "sequence_id": sequence_id,
            "sequence": sequence,
            "label": label,
            "motifs": motifs,
        })

    return sequence_records


# ============================================================
# 6. TEST MOTIF ENRICHMENT BETWEEN CPP AND NON-CPP SEQUENCES
# ============================================================

def calculate_motif_enrichment(
    consensus_df,
    dataset_name,
    motif_length,
):

    sequence_records = build_motif_presence(
        consensus_df=consensus_df,
        motif_length=motif_length,
    )

    cpp_records = [
        record
        for record in sequence_records
        if record["label"] == 1
    ]

    noncpp_records = [
        record
        for record in sequence_records
        if record["label"] == 0
    ]

    cpp_total = len(cpp_records)
    noncpp_total = len(noncpp_records)

    cpp_motif_counter = Counter()

    noncpp_motif_counter = Counter()

    for record in cpp_records:
        cpp_motif_counter.update(
            record["motifs"]
        )

    for record in noncpp_records:
        noncpp_motif_counter.update(
            record["motifs"]
        )

    all_motifs = sorted(
        set(cpp_motif_counter)
        | set(noncpp_motif_counter)
    )

    rows = []

    minimum_support = (
        MIN_CPP_SEQUENCE_SUPPORT[
            dataset_name
        ]
    )

    for motif in all_motifs:

        cpp_present = int(
            cpp_motif_counter.get(
                motif,
                0,
            )
        )

        if cpp_present < minimum_support:
            continue

        noncpp_present = int(
            noncpp_motif_counter.get(
                motif,
                0,
            )
        )

        cpp_absent = (
            cpp_total - cpp_present
        )

        noncpp_absent = (
            noncpp_total
            - noncpp_present
        )

        contingency = [
            [
                cpp_present,
                cpp_absent,
            ],
            [
                noncpp_present,
                noncpp_absent,
            ],
        ]

        scipy_odds_ratio, p_value = (
            fisher_exact(
                contingency,
                alternative="two-sided",
            )
        )

        corrected_odds_ratio = (
            (cpp_present + 0.5)
            * (noncpp_absent + 0.5)
            / (
                (cpp_absent + 0.5)
                * (noncpp_present + 0.5)
            )
        )

        cpp_frequency = (
            cpp_present / cpp_total
        )

        noncpp_frequency = (
            noncpp_present / noncpp_total
        )

        rows.append({
            "dataset": dataset_name,
            "motif_length": int(
                motif_length
            ),
            "motif": motif,
            "cpp_sequence_count": int(
                cpp_present
            ),
            "cpp_sequence_total": int(
                cpp_total
            ),
            "cpp_sequence_frequency": float(
                cpp_frequency
            ),
            "noncpp_sequence_count": int(
                noncpp_present
            ),
            "noncpp_sequence_total": int(
                noncpp_total
            ),
            "noncpp_sequence_frequency": float(
                noncpp_frequency
            ),
            "odds_ratio_scipy": float(
                scipy_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_odds_ratio
            ),
            "log2_enrichment": float(
                np.log2(
                    (
                        cpp_frequency
                        + 1e-12
                    )
                    / (
                        noncpp_frequency
                        + 1e-12
                    )
                )
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    if len(result) == 0:
        return result

    result["fdr_bh"] = (
        benjamini_hochberg(
            result["p_value"]
            .to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "contains_K_or_R"
    ] = result["motif"].apply(
        lambda motif: (
            "K" in motif
            or "R" in motif
        )
    )

    result[
        "basic_residue_fraction"
    ] = result["motif"].apply(
        lambda motif: (
            sum(
                residue in {"K", "R"}
                for residue in motif
            )
            / len(motif)
        )
    )

    return result


# ============================================================
# 7. RUN ALL MOTIF LENGTHS AND DATASETS
# ============================================================

all_motif_results = []

for dataset_name in DATASETS:

    consensus_df = load_consensus(
        dataset_name
    )

    print("\n" + "=" * 75)
    print(
        f"MOTIF DISCOVERY: {dataset_name}"
    )
    print("=" * 75)

    for motif_length in range(
        MIN_MOTIF_LENGTH,
        MAX_MOTIF_LENGTH + 1,
    ):
        print(
            f"[MOTIFS] length={motif_length}"
        )

        motif_result = (
            calculate_motif_enrichment(
                consensus_df=consensus_df,
                dataset_name=dataset_name,
                motif_length=motif_length,
            )
        )

        if len(motif_result) > 0:
            all_motif_results.append(
                motif_result
            )


if not all_motif_results:
    raise RuntimeError(
        "No motifs passed the minimum-support criteria."
    )

motif_df = pd.concat(
    all_motif_results,
    ignore_index=True,
)


# ============================================================
# 8. SAVE COMPLETE MOTIF TABLE
# ============================================================

complete_motif_file = (
    RESULT_SI_DIR
    / "hotspot_motif_enrichment_all.csv"
)

motif_df.to_csv(
    complete_motif_file,
    index=False,
)


# ============================================================
# 9. SIGNIFICANT MOTIFS
# ============================================================

significant_motifs = motif_df[
    motif_df[
        "significant_fdr_0_05"
    ]
].copy()

significant_motifs = (
    significant_motifs.sort_values(
        [
            "dataset",
            "fdr_bh",
            "odds_ratio_corrected",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

significant_motif_file = (
    RESULT_TABLE_DIR
    / "significant_hotspot_motifs.csv"
)

significant_motifs.to_csv(
    significant_motif_file,
    index=False,
)


# ============================================================
# 10. INTERNAL–KELM MOTIF REPLICATION
# ============================================================

internal_motifs = (
    motif_df[
        motif_df["dataset"]
        == "internal_test"
    ][
        [
            "motif",
            "motif_length",
            "cpp_sequence_count",
            "cpp_sequence_frequency",
            "noncpp_sequence_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
            "contains_K_or_R",
            "basic_residue_fraction",
        ]
    ]
    .rename(
        columns={
            "cpp_sequence_count":
                "internal_cpp_count",
            "cpp_sequence_frequency":
                "internal_cpp_frequency",
            "noncpp_sequence_frequency":
                "internal_noncpp_frequency",
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)

kelm_motifs = (
    motif_df[
        motif_df["dataset"]
        == "kelm_external"
    ][
        [
            "motif",
            "motif_length",
            "cpp_sequence_count",
            "cpp_sequence_frequency",
            "noncpp_sequence_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
    .rename(
        columns={
            "cpp_sequence_count":
                "kelm_cpp_count",
            "cpp_sequence_frequency":
                "kelm_cpp_frequency",
            "noncpp_sequence_frequency":
                "kelm_noncpp_frequency",
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)

replicated_motifs = internal_motifs.merge(
    kelm_motifs,
    on=[
        "motif",
        "motif_length",
    ],
    how="inner",
    validate="one_to_one",
)

replicated_motifs[
    "same_enrichment_direction"
] = (
    np.sign(
        replicated_motifs[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replicated_motifs[
            "kelm_log2_enrichment"
        ]
    )
)

replicated_motifs[
    "significant_in_both"
] = (
    replicated_motifs[
        "internal_significant"
    ]
    & replicated_motifs[
        "kelm_significant"
    ]
)

replicated_motifs[
    "mean_log2_enrichment"
] = (
    replicated_motifs[
        [
            "internal_log2_enrichment",
            "kelm_log2_enrichment",
        ]
    ].mean(axis=1)
)

replicated_motifs = (
    replicated_motifs.sort_values(
        [
            "significant_in_both",
            "same_enrichment_direction",
            "mean_log2_enrichment",
            "internal_cpp_count",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
)

replication_file = (
    RESULT_TABLE_DIR
    / "hotspot_motif_replication.csv"
)

replicated_motifs.to_csv(
    replication_file,
    index=False,
)


# ============================================================
# 11. STRICT REPLICATED MOTIFS
# ============================================================

strict_replicated = replicated_motifs[
    replicated_motifs[
        "significant_in_both"
    ]
    & replicated_motifs[
        "same_enrichment_direction"
    ]
    & (
        replicated_motifs[
            "internal_log2_enrichment"
        ] > 0
    )
    & (
        replicated_motifs[
            "kelm_log2_enrichment"
        ] > 0
    )
].copy()

strict_replicated_file = (
    RESULT_TABLE_DIR
    / "strict_replicated_CPP_hotspot_motifs.csv"
)

strict_replicated.to_csv(
    strict_replicated_file,
    index=False,
)


# ============================================================
# 12. SUMMARY
# ============================================================

summary = {
    "motif_lengths": list(
        range(
            MIN_MOTIF_LENGTH,
            MAX_MOTIF_LENGTH + 1,
        )
    ),
    "total_tested_motifs": int(
        len(motif_df)
    ),
    "significant_motifs": int(
        len(significant_motifs)
    ),
    "strict_replicated_enriched_motifs": (
        strict_replicated[
            "motif"
        ].tolist()
    ),
    "number_strict_replicated": int(
        len(strict_replicated)
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "hotspot_motif_discovery_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 13. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "15_hotspot_motif_discovery",
    output_files=[
        complete_motif_file,
        significant_motif_file,
        replication_file,
        strict_replicated_file,
        summary_file,
    ],
    details=summary,
)


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("STRICT REPLICATED ENRICHED MOTIFS")
print("=" * 75)

if len(strict_replicated) > 0:

    display(
        strict_replicated[
            [
                "motif",
                "motif_length",
                "internal_cpp_count",
                "internal_odds_ratio",
                "internal_fdr",
                "kelm_cpp_count",
                "kelm_odds_ratio",
                "kelm_fdr",
                "contains_K_or_R",
                "basic_residue_fraction",
            ]
        ]
    )

else:
    print(
        "No motif was significant in both datasets "
        "under the current criteria."
    )

print("\nTop replicated motifs regardless of significance:")

display(
    replicated_motifs[
        [
            "motif",
            "motif_length",
            "internal_cpp_count",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_cpp_count",
            "kelm_odds_ratio",
            "kelm_fdr",
            "same_enrichment_direction",
            "significant_in_both",
        ]
    ].head(30)
)

print("\nComplete motif table:")
print(complete_motif_file)

print("\nReplication table:")
print(replication_file)

print("\nStrict replicated motifs:")
print(strict_replicated_file)

In [ ]:
# ============================================================
# STEP 16: HOTSPOT FAITHFULNESS AND PERTURBATION VALIDATION
#
# Tests:
# 1. Comprehensiveness:
#    Remove top-20% adjusted consensus hotspots.
#
# 2. Random matched control:
#    Remove the same number of randomly selected residues.
#
# 3. Sufficiency:
#    Retain hotspot residues and remove all other residues.
#
# Uses saved residue embeddings; no PLM recomputation required.
# ============================================================

from pathlib import Path
import gc
import json
import math

import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.stats import (
    wilcoxon,
    mannwhitneyu,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
RANDOM_REPEATS = 20

MODEL_CONFIGS = {
    "ESM2_320": {
        "batch_size": 128,
    },
    "ESM2_640": {
        "batch_size": 96,
    },
    "ESM2_1280": {
        "batch_size": 64,
    },
    "ProtT5": {
        "batch_size": 64,
    },
}

DATASETS = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61

rng_master = np.random.default_rng(
    SEED
)

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

CHECKPOINT_ROOT = DIRS["checkpoints"]

FAITHFULNESS_ROOT = (
    XAI_ROOT / "faithfulness"
)

for folder in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FAITHFULNESS_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. CUSTOM LAYER FOR MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = (
            tf.keras.layers.Dense(
                1,
                use_bias=True,
                name="residue_attention_score",
            )
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        return tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD CONSENSUS HOTSPOT MASKS
# ============================================================

def load_consensus_hotspot_masks(
    dataset_name,
    metadata,
):
    consensus_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not consensus_file.exists():
        raise FileNotFoundError(
            f"Consensus file missing:\n"
            f"{consensus_file}"
        )

    consensus = pd.read_csv(
        consensus_file
    )

    hotspot_masks = np.zeros(
        (
            len(metadata),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    row_lookup = {
        str(sequence_id): row_index
        for row_index, sequence_id in enumerate(
            metadata["sequence_id"]
            .astype(str)
        )
    }

    hotspot_rows = consensus[
        consensus[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    for row in hotspot_rows.itertuples(
        index=False
    ):
        sequence_id = str(
            row.sequence_id
        )

        position_index = int(
            row.position
        ) - 1

        if sequence_id not in row_lookup:
            raise ValueError(
                f"Sequence ID absent from metadata: "
                f"{sequence_id}"
            )

        if not (
            0 <= position_index < MAX_LEN
        ):
            raise ValueError(
                f"Invalid position for {sequence_id}: "
                f"{position_index + 1}"
            )

        hotspot_masks[
            row_lookup[sequence_id],
            position_index,
        ] = 1

    hotspot_counts = (
        hotspot_masks.sum(axis=1)
    )

    if np.any(hotspot_counts == 0):
        raise ValueError(
            "At least one sequence has no top-20% "
            "consensus hotspot."
        )

    return hotspot_masks


# ============================================================
# 4. LOAD ONE MODEL/DATASET
# ============================================================

def load_model_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if (
        len(embeddings) != len(metadata)
        or len(masks) != len(metadata)
    ):
        raise ValueError(
            f"Row mismatch for "
            f"{model_name}/{dataset_name}"
        )

    return embeddings, masks, metadata


# ============================================================
# 5. PREDICT PROBABILITIES
# ============================================================

def predict_probabilities(
    model,
    embeddings,
    masks,
    batch_size,
):
    probabilities = model.predict(
        {
            "residue_embeddings": embeddings,
            "residue_mask": masks,
        },
        batch_size=batch_size,
        verbose=0,
    )

    return (
        probabilities
        .reshape(-1)
        .astype(np.float32)
    )


# ============================================================
# 6. CREATE RANDOM MATCHED ABLATION MASK
# ============================================================

def create_random_ablation_mask(
    valid_mask,
    hotspot_mask,
    seed,
):
    """
    Select the same number of random valid positions as
    top-20% hotspots for every sequence.

    Hotspot positions are excluded from the random pool where
    sufficient non-hotspot positions are available.
    """

    rng = np.random.default_rng(
        seed
    )

    random_mask = np.zeros_like(
        valid_mask,
        dtype=np.uint8,
    )

    for row_index in range(
        len(valid_mask)
    ):
        valid_positions = np.flatnonzero(
            valid_mask[row_index]
        )

        hotspot_positions = np.flatnonzero(
            hotspot_mask[row_index]
        )

        number_to_select = len(
            hotspot_positions
        )

        non_hotspot_positions = (
            np.setdiff1d(
                valid_positions,
                hotspot_positions,
                assume_unique=True,
            )
        )

        if (
            len(non_hotspot_positions)
            >= number_to_select
        ):
            candidate_positions = (
                non_hotspot_positions
            )
        else:
            candidate_positions = (
                valid_positions
            )

        selected = rng.choice(
            candidate_positions,
            size=number_to_select,
            replace=False,
        )

        random_mask[
            row_index,
            selected,
        ] = 1

    return random_mask


# ============================================================
# 7. PERTURB EMBEDDINGS
# ============================================================

def ablate_positions(
    embeddings,
    position_mask,
):
    """
    Set selected residue embeddings to zero.
    """

    return (
        embeddings
        * (
            1.0
            - position_mask[
                :,
                :,
                None,
            ].astype(
                embeddings.dtype
            )
        )
    )


def retain_positions_only(
    embeddings,
    position_mask,
):
    """
    Retain only selected residue embeddings.
    """

    return (
        embeddings
        * position_mask[
            :,
            :,
            None,
        ].astype(
            embeddings.dtype
        )
    )


# ============================================================
# 8. SAFE STATISTICAL TESTS
# ============================================================

def safe_wilcoxon(
    values_a,
    values_b,
):
    difference = (
        np.asarray(values_a)
        - np.asarray(values_b)
    )

    if np.allclose(
        difference,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


# ============================================================
# 9. PROCESS ONE MODEL/DATASET
# ============================================================

def run_faithfulness_analysis(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        FAITHFULNESS_ROOT
        / model_name
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    per_sequence_file = (
        output_dir
        / "per_sequence_faithfulness.csv"
    )

    random_repeat_file = (
        output_dir
        / "random_ablation_repeats.csv"
    )

    summary_file = (
        output_dir
        / "faithfulness_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        per_sequence_file.exists()
        and random_repeat_file.exists()
        and summary_file.exists()
        and complete_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] "
            f"{model_name} | {dataset_name}"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            return json.load(handle)

    print("\n" + "=" * 75)
    print(
        f"FAITHFULNESS: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 75)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings_memmap, masks, metadata = (
        load_model_dataset(
            model_name,
            dataset_name,
        )
    )

    # Copy one dataset into RAM for efficient repeated testing.
    embeddings = np.asarray(
        embeddings_memmap,
        dtype=np.float16,
    )

    valid_masks = np.asarray(
        masks,
        dtype=np.uint8,
    )

    hotspot_masks = (
        load_consensus_hotspot_masks(
            dataset_name=dataset_name,
            metadata=metadata,
        )
    )

    # --------------------------------------------------------
    # Original predictions
    # --------------------------------------------------------

    original_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    # --------------------------------------------------------
    # Consensus hotspot ablation
    # --------------------------------------------------------

    hotspot_ablated_embeddings = (
        ablate_positions(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_ablated_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=(
                hotspot_ablated_embeddings
            ),
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    hotspot_probability_drop = (
        original_probabilities
        - hotspot_ablated_probabilities
    )

    del hotspot_ablated_embeddings

    # --------------------------------------------------------
    # Hotspot-only sufficiency
    # --------------------------------------------------------

    hotspot_only_embeddings = (
        retain_positions_only(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_only_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=(
                hotspot_only_embeddings
            ),
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    del hotspot_only_embeddings

    # --------------------------------------------------------
    # Random matched controls
    # --------------------------------------------------------

    random_repeat_rows = []

    random_drop_matrix = np.zeros(
        (
            len(metadata),
            RANDOM_REPEATS,
        ),
        dtype=np.float32,
    )

    for repeat_index in range(
        RANDOM_REPEATS
    ):
        random_mask = (
            create_random_ablation_mask(
                valid_mask=valid_masks,
                hotspot_mask=hotspot_masks,
                seed=(
                    SEED
                    + repeat_index
                    + 1000
                ),
            )
        )

        random_ablated_embeddings = (
            ablate_positions(
                embeddings,
                random_mask,
            )
        )

        random_probabilities = (
            predict_probabilities(
                model=model,
                embeddings=(
                    random_ablated_embeddings
                ),
                masks=valid_masks,
                batch_size=batch_size,
            )
        )

        random_drops = (
            original_probabilities
            - random_probabilities
        )

        random_drop_matrix[
            :,
            repeat_index
        ] = random_drops

        random_repeat_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "repeat": int(
                repeat_index + 1
            ),
            "mean_probability_drop_all": float(
                random_drops.mean()
            ),
            "mean_probability_drop_CPP": float(
                random_drops[
                    metadata["label"]
                    .to_numpy()
                    == 1
                ].mean()
            ),
            "mean_probability_drop_nonCPP": float(
                random_drops[
                    metadata["label"]
                    .to_numpy()
                    == 0
                ].mean()
            ),
        })

        print(
            f"[RANDOM CONTROL] "
            f"{repeat_index + 1}/"
            f"{RANDOM_REPEATS}"
        )

        del random_mask
        del random_ablated_embeddings
        del random_probabilities
        del random_drops

        gc.collect()

    random_mean_drop = (
        random_drop_matrix.mean(
            axis=1
        )
    )

    random_drop_std = (
        random_drop_matrix.std(
            axis=1,
            ddof=0,
        )
    )

    # --------------------------------------------------------
    # Per-sequence results
    # --------------------------------------------------------

    per_sequence = metadata[
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    per_sequence.insert(
        0,
        "model",
        model_name,
    )

    per_sequence.insert(
        1,
        "dataset",
        dataset_name,
    )

    per_sequence[
        "number_of_hotspots"
    ] = hotspot_masks.sum(
        axis=1
    )

    per_sequence[
        "hotspot_fraction"
    ] = (
        per_sequence[
            "number_of_hotspots"
        ]
        / per_sequence["length"]
    )

    per_sequence[
        "original_probability"
    ] = original_probabilities

    per_sequence[
        "hotspot_ablated_probability"
    ] = hotspot_ablated_probabilities

    per_sequence[
        "hotspot_probability_drop"
    ] = hotspot_probability_drop

    per_sequence[
        "random_mean_probability_drop"
    ] = random_mean_drop

    per_sequence[
        "random_drop_std"
    ] = random_drop_std

    per_sequence[
        "hotspot_minus_random_drop"
    ] = (
        hotspot_probability_drop
        - random_mean_drop
    )

    per_sequence[
        "hotspot_only_probability"
    ] = hotspot_only_probabilities

    per_sequence[
        "sufficiency_probability_loss"
    ] = (
        original_probabilities
        - hotspot_only_probabilities
    )

    per_sequence[
        "hotspot_ablation_stronger_than_random"
    ] = (
        per_sequence[
            "hotspot_probability_drop"
        ]
        > per_sequence[
            "random_mean_probability_drop"
        ]
    )

    per_sequence.to_csv(
        per_sequence_file,
        index=False,
    )

    random_repeat_df = pd.DataFrame(
        random_repeat_rows
    )

    random_repeat_df.to_csv(
        random_repeat_file,
        index=False,
    )

    # --------------------------------------------------------
    # Statistical summaries
    # --------------------------------------------------------

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            len(per_sequence)
        ),
        "number_of_CPPs": int(
            (per_sequence["label"] == 1).sum()
        ),
        "number_of_nonCPPs": int(
            (per_sequence["label"] == 0).sum()
        ),
        "random_repeats": int(
            RANDOM_REPEATS
        ),
        "classes": {},
    }

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
        ("all", "all"),
    ]:
        if label_value == "all":
            subset = per_sequence
        else:
            subset = per_sequence[
                per_sequence["label"]
                == label_value
            ]

        wilcoxon_statistic, wilcoxon_p = (
            safe_wilcoxon(
                subset[
                    "hotspot_probability_drop"
                ],
                subset[
                    "random_mean_probability_drop"
                ],
            )
        )

        summary["classes"][
            class_name
        ] = {
            "n": int(
                len(subset)
            ),
            "mean_original_probability": float(
                subset[
                    "original_probability"
                ].mean()
            ),
            "mean_hotspot_ablated_probability": float(
                subset[
                    "hotspot_ablated_probability"
                ].mean()
            ),
            "mean_hotspot_probability_drop": float(
                subset[
                    "hotspot_probability_drop"
                ].mean()
            ),
            "median_hotspot_probability_drop": float(
                subset[
                    "hotspot_probability_drop"
                ].median()
            ),
            "mean_random_probability_drop": float(
                subset[
                    "random_mean_probability_drop"
                ].mean()
            ),
            "median_random_probability_drop": float(
                subset[
                    "random_mean_probability_drop"
                ].median()
            ),
            "mean_hotspot_minus_random_drop": float(
                subset[
                    "hotspot_minus_random_drop"
                ].mean()
            ),
            "fraction_hotspot_stronger_than_random": float(
                subset[
                    "hotspot_ablation_stronger_than_random"
                ].mean()
            ),
            "mean_hotspot_only_probability": float(
                subset[
                    "hotspot_only_probability"
                ].mean()
            ),
            "wilcoxon_hotspot_vs_random_statistic": (
                wilcoxon_statistic
            ),
            "wilcoxon_hotspot_vs_random_p": (
                wilcoxon_p
            ),
        }

    with open(
        summary_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    with open(
        complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    print(
        "[COMPLETE]",
        model_name,
        dataset_name,
    )

    cpp_summary = summary[
        "classes"
    ]["CPP"]

    print(
        "CPP hotspot drop:",
        cpp_summary[
            "mean_hotspot_probability_drop"
        ],
    )

    print(
        "CPP random drop:",
        cpp_summary[
            "mean_random_probability_drop"
        ],
    )

    print(
        "CPP hotspot-minus-random:",
        cpp_summary[
            "mean_hotspot_minus_random_drop"
        ],
    )

    del model
    del embeddings_memmap
    del embeddings
    del masks
    del valid_masks
    del hotspot_masks
    del random_drop_matrix

    tf.keras.backend.clear_session()
    gc.collect()

    return summary


# ============================================================
# 10. RUN ALL MODELS AND DATASETS
# ============================================================

summary_records = []
all_output_files = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in DATASETS:
        summary = (
            run_faithfulness_analysis(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=config[
                    "batch_size"
                ],
            )
        )

        for class_name, values in (
            summary["classes"].items()
        ):
            summary_records.append({
                "model": model_name,
                "dataset": dataset_name,
                "class": class_name,
                **values,
            })

        output_dir = (
            FAITHFULNESS_ROOT
            / model_name
            / dataset_name
        )

        all_output_files.extend([
            output_dir
            / "per_sequence_faithfulness.csv",
            output_dir
            / "random_ablation_repeats.csv",
            output_dir
            / "faithfulness_summary.json",
            output_dir
            / "COMPLETE.json",
        ])


# ============================================================
# 11. MASTER SUMMARY TABLE
# ============================================================

faithfulness_summary_df = (
    pd.DataFrame(
        summary_records
    )
)

summary_output_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_summary.csv"
)

faithfulness_summary_df.to_csv(
    summary_output_file,
    index=False,
)

all_output_files.append(
    summary_output_file
)


# ============================================================
# 12. CROSS-MODEL AVERAGE
# ============================================================

numeric_columns = [
    "mean_original_probability",
    "mean_hotspot_ablated_probability",
    "mean_hotspot_probability_drop",
    "median_hotspot_probability_drop",
    "mean_random_probability_drop",
    "median_random_probability_drop",
    "mean_hotspot_minus_random_drop",
    "fraction_hotspot_stronger_than_random",
    "mean_hotspot_only_probability",
]

cross_model_summary = (
    faithfulness_summary_df
    .groupby(
        [
            "dataset",
            "class",
        ],
        as_index=False,
    )[numeric_columns]
    .mean()
)

cross_model_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_cross_model_summary.csv"
)

cross_model_summary.to_csv(
    cross_model_file,
    index=False,
)

all_output_files.append(
    cross_model_file
)


# ============================================================
# 13. SAVE CHECKPOINT
# ============================================================

checkpoint_details = {
    "models": list(
        MODEL_CONFIGS.keys()
    ),
    "datasets": DATASETS,
    "hotspot_definition": (
        "adjusted global consensus top 20%"
    ),
    "random_control_repeats": int(
        RANDOM_REPEATS
    ),
    "ablation_method": (
        "set selected fixed PLM residue embeddings to zero"
    ),
    "sufficiency_method": (
        "retain only selected hotspot residue embeddings"
    ),
}

mark_step_complete(
    "16_hotspot_faithfulness",
    output_files=all_output_files,
    details=checkpoint_details,
)


# ============================================================
# 14. DISPLAY CPP RESULTS
# ============================================================

print("\n" + "=" * 75)
print("CPP HOTSPOT FAITHFULNESS BY MODEL")
print("=" * 75)

display(
    faithfulness_summary_df[
        faithfulness_summary_df[
            "class"
        ] == "CPP"
    ][
        [
            "model",
            "dataset",
            "n",
            "mean_original_probability",
            "mean_hotspot_ablated_probability",
            "mean_hotspot_probability_drop",
            "mean_random_probability_drop",
            "mean_hotspot_minus_random_drop",
            "fraction_hotspot_stronger_than_random",
            "mean_hotspot_only_probability",
            "wilcoxon_hotspot_vs_random_p",
        ]
    ]
)

print("\n" + "=" * 75)
print("CROSS-MODEL FAITHFULNESS SUMMARY")
print("=" * 75)

display(
    cross_model_summary[
        cross_model_summary[
            "class"
        ] == "CPP"
    ]
)

print("\nMaster faithfulness table:")
print(summary_output_file)

print("\nCross-model summary:")
print(cross_model_file)

In [ ]:
# ============================================================
# STEP 16: PUBLICATION-READY HOTSPOT FAITHFULNESS ANALYSIS
#
# Tests:
# 1. Comprehensiveness:
#    Set consensus-hotspot residue embeddings to zero.
#
# 2. Matched random controls:
#    Remove the same number of non-hotspot residues,
#    repeated RANDOM_REPEATS times.
#
# 3. Sufficiency:
#    Retain only consensus-hotspot residue embeddings.
#
# Outputs:
# - Per-sequence results
# - Restart-safe random repeat checkpoints
# - Statistical tests and effect sizes
# - Bootstrap confidence intervals
# - Manuscript-ready summary tables
# - PNG, PDF, and SVG figures
# ============================================================

from pathlib import Path
import gc
import json
import math
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy.stats import wilcoxon


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
RANDOM_REPEATS = 20
BOOTSTRAP_REPEATS = 2000
CONFIDENCE_LEVEL = 0.95
MAX_LEN = 61

MODEL_CONFIGS = {
    "ESM2_320": {
        "batch_size": 128,
    },
    "ESM2_640": {
        "batch_size": 96,
    },
    "ESM2_1280": {
        "batch_size": 64,
    },
    "ProtT5": {
        "batch_size": 64,
    },
}

DATASETS = [
    "internal_test",
    "kelm_external",
]

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass


EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

FAITHFULNESS_ROOT = (
    XAI_ROOT / "faithfulness"
)

for folder in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
    FAITHFULNESS_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("=" * 78)
print("PUBLICATION-READY HOTSPOT FAITHFULNESS ANALYSIS")
print("=" * 78)
print("TensorFlow version :", tf.__version__)
print("TensorFlow GPUs    :", tf.config.list_physical_devices("GPU"))
print("Random repeats     :", RANDOM_REPEATS)
print("Bootstrap repeats  :", BOOTSTRAP_REPEATS)


# ============================================================
# 2. CUSTOM LAYER FOR SAVED MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. GENERAL HELPERS
# ============================================================

def clear_memory():
    gc.collect()

    tf.keras.backend.clear_session()

    if tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.reset_memory_stats(
                "GPU:0"
            )
        except Exception:
            pass


def atomic_save_json(
    output_file,
    data,
):
    output_file = Path(output_file)

    temporary_file = Path(
        str(output_file) + ".temporary"
    )

    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            data,
            handle,
            indent=2,
        )

    temporary_file.replace(
        output_file
    )


def atomic_save_csv(
    dataframe,
    output_file,
):
    output_file = Path(output_file)

    temporary_file = Path(
        str(output_file) + ".temporary"
    )

    dataframe.to_csv(
        temporary_file,
        index=False,
    )

    temporary_file.replace(
        output_file
    )


# ============================================================
# 4. LOAD MODEL DATASET
# ============================================================

def load_model_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n"
                f"{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Embedding/metadata row mismatch for "
            f"{model_name}/{dataset_name}"
        )

    if len(masks) != len(metadata):
        raise ValueError(
            f"Mask/metadata row mismatch for "
            f"{model_name}/{dataset_name}"
        )

    return (
        embeddings,
        masks,
        metadata,
    )


# ============================================================
# 5. LOAD GLOBAL CONSENSUS HOTSPOTS
# ============================================================

def load_consensus_hotspot_masks(
    dataset_name,
    metadata,
):
    consensus_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not consensus_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file missing:\n"
            f"{consensus_file}"
        )

    consensus = pd.read_csv(
        consensus_file
    )

    required_columns = {
        "sequence_id",
        "position",
        "adjusted_hotspot_top20",
    }

    missing_columns = (
        required_columns
        - set(consensus.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Consensus file is missing columns: "
            f"{sorted(missing_columns)}"
        )

    hotspot_masks = np.zeros(
        (
            len(metadata),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    row_lookup = {
        str(sequence_id): row_index
        for row_index, sequence_id in enumerate(
            metadata["sequence_id"].astype(str)
        )
    }

    hotspot_rows = consensus[
        consensus[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    for row in hotspot_rows.itertuples(
        index=False
    ):
        sequence_id = str(
            row.sequence_id
        )

        position_index = int(
            row.position
        ) - 1

        if sequence_id not in row_lookup:
            raise ValueError(
                f"Consensus sequence not found in metadata: "
                f"{sequence_id}"
            )

        if not (
            0 <= position_index < MAX_LEN
        ):
            raise ValueError(
                f"Invalid hotspot position "
                f"{position_index + 1} for {sequence_id}"
            )

        hotspot_masks[
            row_lookup[sequence_id],
            position_index,
        ] = 1

    hotspot_counts = (
        hotspot_masks.sum(axis=1)
    )

    zero_hotspot_sequences = int(
        np.sum(hotspot_counts == 0)
    )

    if zero_hotspot_sequences > 0:
        raise ValueError(
            f"{zero_hotspot_sequences} sequences have no "
            f"top-20% consensus hotspot."
        )

    return (
        hotspot_masks,
        consensus_file,
    )


# ============================================================
# 6. PREDICTION
# ============================================================

def predict_probabilities(
    model,
    embeddings,
    masks,
    batch_size,
):
    probabilities = model.predict(
        {
            "residue_embeddings": embeddings,
            "residue_mask": masks,
        },
        batch_size=batch_size,
        verbose=0,
    )

    return (
        probabilities
        .reshape(-1)
        .astype(np.float32)
    )


# ============================================================
# 7. PERTURBATIONS
# ============================================================

def ablate_positions(
    embeddings,
    position_mask,
):
    keep_mask = (
        1.0
        - position_mask[
            :,
            :,
            None,
        ].astype(
            embeddings.dtype
        )
    )

    return embeddings * keep_mask


def retain_positions_only(
    embeddings,
    position_mask,
):
    retain_mask = (
        position_mask[
            :,
            :,
            None,
        ].astype(
            embeddings.dtype
        )
    )

    return embeddings * retain_mask


def create_random_ablation_mask(
    valid_mask,
    hotspot_mask,
    seed,
):
    """
    Select the same number of valid non-hotspot positions
    as consensus hotspots for each sequence.

    For very short sequences where insufficient non-hotspot
    positions exist, sample from all valid positions.
    """

    rng = np.random.default_rng(
        seed
    )

    random_mask = np.zeros_like(
        valid_mask,
        dtype=np.uint8,
    )

    for row_index in range(
        len(valid_mask)
    ):
        valid_positions = np.flatnonzero(
            valid_mask[row_index]
        )

        hotspot_positions = np.flatnonzero(
            hotspot_mask[row_index]
        )

        number_to_select = len(
            hotspot_positions
        )

        non_hotspot_positions = np.setdiff1d(
            valid_positions,
            hotspot_positions,
            assume_unique=True,
        )

        if (
            len(non_hotspot_positions)
            >= number_to_select
        ):
            candidate_positions = (
                non_hotspot_positions
            )
        else:
            candidate_positions = (
                valid_positions
            )

        selected_positions = rng.choice(
            candidate_positions,
            size=number_to_select,
            replace=False,
        )

        random_mask[
            row_index,
            selected_positions,
        ] = 1

    return random_mask


# ============================================================
# 8. STATISTICAL HELPERS
# ============================================================

def safe_wilcoxon(
    values_a,
    values_b,
    alternative="two-sided",
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    differences = (
        values_a - values_b
    )

    finite_mask = np.isfinite(
        differences
    )

    differences = differences[
        finite_mask
    ]

    if len(differences) == 0:
        return np.nan, np.nan

    if np.allclose(
        differences,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        differences,
        alternative=alternative,
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


def paired_cohens_dz(
    values_a,
    values_b,
):
    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    differences = differences[
        np.isfinite(differences)
    ]

    if len(differences) < 2:
        return np.nan

    standard_deviation = (
        differences.std(ddof=1)
    )

    if np.isclose(
        standard_deviation,
        0.0,
    ):
        return np.nan

    return float(
        differences.mean()
        / standard_deviation
    )


def rank_biserial_effect(
    values_a,
    values_b,
):
    """
    Matched-pairs rank-biserial effect size.

    Positive values indicate values_a > values_b.
    """

    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    differences = differences[
        np.isfinite(differences)
        & ~np.isclose(differences, 0.0)
    ]

    if len(differences) == 0:
        return 0.0

    absolute_differences = np.abs(
        differences
    )

    ranks = pd.Series(
        absolute_differences
    ).rank(
        method="average"
    ).to_numpy()

    positive_rank_sum = ranks[
        differences > 0
    ].sum()

    negative_rank_sum = ranks[
        differences < 0
    ].sum()

    denominator = (
        positive_rank_sum
        + negative_rank_sum
    )

    if denominator == 0:
        return 0.0

    return float(
        (
            positive_rank_sum
            - negative_rank_sum
        )
        / denominator
    )


def bootstrap_mean_confidence_interval(
    values,
    repeats=BOOTSTRAP_REPEATS,
    confidence_level=CONFIDENCE_LEVEL,
    seed=SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    rng = np.random.default_rng(
        seed
    )

    sample_indices = rng.integers(
        low=0,
        high=len(values),
        size=(
            repeats,
            len(values),
        ),
    )

    bootstrap_means = values[
        sample_indices
    ].mean(axis=1)

    alpha = (
        1.0 - confidence_level
    ) / 2.0

    lower = np.quantile(
        bootstrap_means,
        alpha,
    )

    upper = np.quantile(
        bootstrap_means,
        1.0 - alpha,
    )

    return (
        float(values.mean()),
        float(lower),
        float(upper),
    )


def bootstrap_paired_difference_ci(
    values_a,
    values_b,
    repeats=BOOTSTRAP_REPEATS,
    confidence_level=CONFIDENCE_LEVEL,
    seed=SEED,
):
    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    return bootstrap_mean_confidence_interval(
        values=differences,
        repeats=repeats,
        confidence_level=confidence_level,
        seed=seed,
    )


# ============================================================
# 9. RANDOM REPEAT CHECKPOINT HELPERS
# ============================================================

def random_repeat_file_path(
    repeat_directory,
    repeat_index,
):
    return (
        repeat_directory
        / f"random_repeat_{repeat_index:03d}.npz"
    )


def validate_random_repeat_file(
    repeat_file,
    expected_sequence_ids,
):
    if not repeat_file.exists():
        return False

    try:
        data = np.load(
            repeat_file,
            allow_pickle=True,
        )

        saved_ids = (
            data["sequence_id"]
            .astype(str)
        )

        saved_probabilities = data[
            "random_ablated_probability"
        ]

        valid = (
            np.array_equal(
                saved_ids,
                expected_sequence_ids.astype(str),
            )
            and len(saved_probabilities)
            == len(expected_sequence_ids)
        )

        data.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 10. ANALYZE ONE MODEL/DATASET
# ============================================================

def run_faithfulness_analysis(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        FAITHFULNESS_ROOT
        / model_name
        / dataset_name
    )

    repeat_directory = (
        output_dir
        / "random_repeat_checkpoints"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    repeat_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    per_sequence_file = (
        output_dir
        / "per_sequence_faithfulness.csv"
    )

    random_repeat_summary_file = (
        output_dir
        / "random_ablation_repeat_summary.csv"
    )

    summary_file = (
        output_dir
        / "faithfulness_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        per_sequence_file.exists()
        and random_repeat_summary_file.exists()
        and summary_file.exists()
        and complete_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] "
            f"{model_name} | {dataset_name}"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            return json.load(handle)

    print("\n" + "=" * 78)
    print(
        f"FAITHFULNESS: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 78)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Trained model missing:\n"
            f"{model_file}"
        )

    model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    (
        embeddings_memmap,
        masks_memmap,
        metadata,
    ) = load_model_dataset(
        model_name,
        dataset_name,
    )

    embeddings = np.asarray(
        embeddings_memmap,
        dtype=np.float16,
    )

    valid_masks = np.asarray(
        masks_memmap,
        dtype=np.uint8,
    )

    (
        hotspot_masks,
        consensus_file,
    ) = load_consensus_hotspot_masks(
        dataset_name=dataset_name,
        metadata=metadata,
    )

    sequence_ids = (
        metadata["sequence_id"]
        .astype(str)
        .to_numpy()
    )

    labels = (
        metadata["label"]
        .astype(int)
        .to_numpy()
    )

    # --------------------------------------------------------
    # A. Original predictions
    # --------------------------------------------------------

    print("[PREDICT] Original embeddings")

    original_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    # --------------------------------------------------------
    # B. Hotspot ablation
    # --------------------------------------------------------

    print("[PREDICT] Consensus hotspot ablation")

    hotspot_ablated_embeddings = (
        ablate_positions(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_ablated_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=hotspot_ablated_embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    hotspot_probability_drop = (
        original_probabilities
        - hotspot_ablated_probabilities
    )

    del hotspot_ablated_embeddings
    gc.collect()

    # --------------------------------------------------------
    # C. Hotspot-only sufficiency
    # --------------------------------------------------------

    print("[PREDICT] Hotspot-only embeddings")

    hotspot_only_embeddings = (
        retain_positions_only(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_only_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=hotspot_only_embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    sufficiency_probability_loss = (
        original_probabilities
        - hotspot_only_probabilities
    )

    del hotspot_only_embeddings
    gc.collect()

    # --------------------------------------------------------
    # D. Restart-safe matched random controls
    # --------------------------------------------------------

    random_probability_matrix = np.zeros(
        (
            len(metadata),
            RANDOM_REPEATS,
        ),
        dtype=np.float32,
    )

    random_repeat_rows = []

    for repeat_index in range(
        1,
        RANDOM_REPEATS + 1,
    ):
        repeat_file = random_repeat_file_path(
            repeat_directory,
            repeat_index,
        )

        if validate_random_repeat_file(
            repeat_file=repeat_file,
            expected_sequence_ids=sequence_ids,
        ):
            repeat_data = np.load(
                repeat_file,
                allow_pickle=True,
            )

            random_probabilities = (
                repeat_data[
                    "random_ablated_probability"
                ].astype(np.float32)
            )

            repeat_data.close()

            print(
                f"[SKIP RANDOM] "
                f"{repeat_index}/{RANDOM_REPEATS}"
            )

        else:
            random_mask = (
                create_random_ablation_mask(
                    valid_mask=valid_masks,
                    hotspot_mask=hotspot_masks,
                    seed=(
                        SEED
                        + 1000
                        + repeat_index
                    ),
                )
            )

            random_ablated_embeddings = (
                ablate_positions(
                    embeddings,
                    random_mask,
                )
            )

            random_probabilities = (
                predict_probabilities(
                    model=model,
                    embeddings=random_ablated_embeddings,
                    masks=valid_masks,
                    batch_size=batch_size,
                )
            )

            temporary_repeat_file = Path(
                str(repeat_file)
                + ".temporary.npz"
            )

            np.savez_compressed(
                temporary_repeat_file,
                sequence_id=sequence_ids,
                random_ablated_probability=(
                    random_probabilities
                ),
                repeat=np.array(
                    [repeat_index],
                    dtype=np.int32,
                ),
                seed=np.array(
                    [
                        SEED
                        + 1000
                        + repeat_index
                    ],
                    dtype=np.int32,
                ),
            )

            temporary_repeat_file.replace(
                repeat_file
            )

            print(
                f"[SAVED RANDOM] "
                f"{repeat_index}/{RANDOM_REPEATS}"
            )

            del random_mask
            del random_ablated_embeddings
            gc.collect()

        random_probability_matrix[
            :,
            repeat_index - 1
        ] = random_probabilities

        random_drops = (
            original_probabilities
            - random_probabilities
        )

        cpp_mask = (
            labels == 1
        )

        noncpp_mask = (
            labels == 0
        )

        random_repeat_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "repeat": int(
                repeat_index
            ),
            "mean_random_drop_all": float(
                random_drops.mean()
            ),
            "mean_random_drop_CPP": float(
                random_drops[
                    cpp_mask
                ].mean()
            ),
            "mean_random_drop_nonCPP": float(
                random_drops[
                    noncpp_mask
                ].mean()
            ),
            "median_random_drop_CPP": float(
                np.median(
                    random_drops[
                        cpp_mask
                    ]
                )
            ),
            "median_random_drop_nonCPP": float(
                np.median(
                    random_drops[
                        noncpp_mask
                    ]
                )
            ),
        })

        del random_probabilities
        del random_drops
        gc.collect()

    random_repeat_df = pd.DataFrame(
        random_repeat_rows
    )

    atomic_save_csv(
        random_repeat_df,
        random_repeat_summary_file,
    )

    random_mean_probabilities = (
        random_probability_matrix.mean(
            axis=1
        )
    )

    random_probability_std = (
        random_probability_matrix.std(
            axis=1,
            ddof=0,
        )
    )

    random_mean_drop = (
        original_probabilities
        - random_mean_probabilities
    )

    # --------------------------------------------------------
    # E. Per-sequence table
    # --------------------------------------------------------

    per_sequence = metadata[
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    per_sequence.insert(
        0,
        "model",
        model_name,
    )

    per_sequence.insert(
        1,
        "dataset",
        dataset_name,
    )

    per_sequence[
        "number_of_hotspots"
    ] = hotspot_masks.sum(
        axis=1
    )

    per_sequence[
        "hotspot_fraction"
    ] = (
        per_sequence[
            "number_of_hotspots"
        ]
        / per_sequence["length"]
    )

    per_sequence[
        "original_probability"
    ] = original_probabilities

    per_sequence[
        "hotspot_ablated_probability"
    ] = hotspot_ablated_probabilities

    per_sequence[
        "hotspot_probability_drop"
    ] = hotspot_probability_drop

    per_sequence[
        "random_mean_ablated_probability"
    ] = random_mean_probabilities

    per_sequence[
        "random_mean_probability_drop"
    ] = random_mean_drop

    per_sequence[
        "random_probability_std"
    ] = random_probability_std

    per_sequence[
        "hotspot_minus_random_drop"
    ] = (
        hotspot_probability_drop
        - random_mean_drop
    )

    per_sequence[
        "hotspot_ablation_stronger_than_random"
    ] = (
        per_sequence[
            "hotspot_probability_drop"
        ]
        > per_sequence[
            "random_mean_probability_drop"
        ]
    )

    per_sequence[
        "hotspot_only_probability"
    ] = hotspot_only_probabilities

    per_sequence[
        "sufficiency_probability_loss"
    ] = sufficiency_probability_loss

    per_sequence[
        "hotspot_only_probability_fraction"
    ] = np.divide(
        hotspot_only_probabilities,
        original_probabilities,
        out=np.full_like(
            hotspot_only_probabilities,
            np.nan,
            dtype=np.float32,
        ),
        where=(
            original_probabilities > 1e-8
        ),
    )

    atomic_save_csv(
        per_sequence,
        per_sequence_file,
    )

    # --------------------------------------------------------
    # F. Class-specific statistical summaries
    # --------------------------------------------------------

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            len(metadata)
        ),
        "number_of_CPPs": int(
            np.sum(labels == 1)
        ),
        "number_of_nonCPPs": int(
            np.sum(labels == 0)
        ),
        "hotspot_definition": (
            "adjusted global consensus top 20%"
        ),
        "random_repeats": int(
            RANDOM_REPEATS
        ),
        "bootstrap_repeats": int(
            BOOTSTRAP_REPEATS
        ),
        "consensus_file": str(
            consensus_file
        ),
        "classes": {},
    }

    class_definitions = [
        ("all", np.ones(
            len(metadata),
            dtype=bool,
        )),
        ("CPP", labels == 1),
        ("non_CPP", labels == 0),
    ]

    for class_name, class_mask in (
        class_definitions
    ):
        subset = per_sequence.loc[
            class_mask
        ].copy()

        hotspot_drops = subset[
            "hotspot_probability_drop"
        ].to_numpy()

        random_drops = subset[
            "random_mean_probability_drop"
        ].to_numpy()

        hotspot_minus_random = subset[
            "hotspot_minus_random_drop"
        ].to_numpy()

        hotspot_only_values = subset[
            "hotspot_only_probability"
        ].to_numpy()

        original_values = subset[
            "original_probability"
        ].to_numpy()

        wilcoxon_statistic, wilcoxon_p = (
            safe_wilcoxon(
                hotspot_drops,
                random_drops,
                alternative="greater",
            )
        )

        (
            hotspot_drop_mean,
            hotspot_drop_ci_low,
            hotspot_drop_ci_high,
        ) = bootstrap_mean_confidence_interval(
            hotspot_drops,
            seed=(
                SEED
                + len(class_name)
            ),
        )

        (
            random_drop_mean,
            random_drop_ci_low,
            random_drop_ci_high,
        ) = bootstrap_mean_confidence_interval(
            random_drops,
            seed=(
                SEED
                + 100
                + len(class_name)
            ),
        )

        (
            difference_mean,
            difference_ci_low,
            difference_ci_high,
        ) = bootstrap_paired_difference_ci(
            hotspot_drops,
            random_drops,
            seed=(
                SEED
                + 200
                + len(class_name)
            ),
        )

        summary[
            "classes"
        ][class_name] = {
            "n": int(
                len(subset)
            ),
            "mean_original_probability": float(
                original_values.mean()
            ),
            "median_original_probability": float(
                np.median(
                    original_values
                )
            ),
            "mean_hotspot_ablated_probability": float(
                subset[
                    "hotspot_ablated_probability"
                ].mean()
            ),
            "mean_hotspot_probability_drop": float(
                hotspot_drop_mean
            ),
            "hotspot_drop_ci95_low": float(
                hotspot_drop_ci_low
            ),
            "hotspot_drop_ci95_high": float(
                hotspot_drop_ci_high
            ),
            "median_hotspot_probability_drop": float(
                np.median(
                    hotspot_drops
                )
            ),
            "mean_random_probability_drop": float(
                random_drop_mean
            ),
            "random_drop_ci95_low": float(
                random_drop_ci_low
            ),
            "random_drop_ci95_high": float(
                random_drop_ci_high
            ),
            "median_random_probability_drop": float(
                np.median(
                    random_drops
                )
            ),
            "mean_hotspot_minus_random_drop": float(
                difference_mean
            ),
            "difference_ci95_low": float(
                difference_ci_low
            ),
            "difference_ci95_high": float(
                difference_ci_high
            ),
            "fraction_hotspot_stronger_than_random": float(
                subset[
                    "hotspot_ablation_stronger_than_random"
                ].mean()
            ),
            "paired_cohens_dz": float(
                paired_cohens_dz(
                    hotspot_drops,
                    random_drops,
                )
            ),
            "rank_biserial_effect_size": float(
                rank_biserial_effect(
                    hotspot_drops,
                    random_drops,
                )
            ),
            "wilcoxon_alternative": (
                "hotspot drop > random drop"
            ),
            "wilcoxon_statistic": float(
                wilcoxon_statistic
            ),
            "wilcoxon_p_value": float(
                wilcoxon_p
            ),
            "mean_hotspot_only_probability": float(
                hotspot_only_values.mean()
            ),
            "median_hotspot_only_probability": float(
                np.median(
                    hotspot_only_values
                )
            ),
            "mean_sufficiency_probability_loss": float(
                subset[
                    "sufficiency_probability_loss"
                ].mean()
            ),
            "mean_hotspot_only_probability_fraction": float(
                np.nanmean(
                    subset[
                        "hotspot_only_probability_fraction"
                    ]
                )
            ),
        }

    atomic_save_json(
        summary_file,
        summary,
    )

    atomic_save_json(
        complete_file,
        summary,
    )

    cpp_summary = (
        summary["classes"]["CPP"]
    )

    print(
        "[COMPLETE]",
        model_name,
        dataset_name,
    )

    print(
        "CPP hotspot drop       :",
        cpp_summary[
            "mean_hotspot_probability_drop"
        ],
    )

    print(
        "CPP random drop        :",
        cpp_summary[
            "mean_random_probability_drop"
        ],
    )

    print(
        "CPP hotspot - random   :",
        cpp_summary[
            "mean_hotspot_minus_random_drop"
        ],
    )

    print(
        "CPP Wilcoxon p-value   :",
        cpp_summary[
            "wilcoxon_p_value"
        ],
    )

    print(
        "CPP paired Cohen's dz  :",
        cpp_summary[
            "paired_cohens_dz"
        ],
    )

    del model
    del embeddings_memmap
    del masks_memmap
    del embeddings
    del valid_masks
    del hotspot_masks
    del random_probability_matrix

    clear_memory()

    return summary


# ============================================================
# 11. RUN ALL MODEL/DATASET COMBINATIONS
# ============================================================

summary_records = []
all_output_files = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in DATASETS:
        summary = run_faithfulness_analysis(
            model_name=model_name,
            dataset_name=dataset_name,
            batch_size=config[
                "batch_size"
            ],
        )

        for class_name, values in (
            summary["classes"].items()
        ):
            summary_records.append({
                "model": model_name,
                "dataset": dataset_name,
                "class": class_name,
                **values,
            })

        output_dir = (
            FAITHFULNESS_ROOT
            / model_name
            / dataset_name
        )

        all_output_files.extend([
            output_dir
            / "per_sequence_faithfulness.csv",
            output_dir
            / "random_ablation_repeat_summary.csv",
            output_dir
            / "faithfulness_summary.json",
            output_dir
            / "COMPLETE.json",
        ])


# ============================================================
# 12. MASTER SUMMARY TABLE
# ============================================================

faithfulness_summary_df = pd.DataFrame(
    summary_records
)

master_summary_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_summary.csv"
)

atomic_save_csv(
    faithfulness_summary_df,
    master_summary_file,
)

all_output_files.append(
    master_summary_file
)


# ============================================================
# 13. CPP-ONLY MANUSCRIPT TABLE
# ============================================================

cpp_summary_df = (
    faithfulness_summary_df[
        faithfulness_summary_df[
            "class"
        ] == "CPP"
    ]
    .copy()
)

cpp_manuscript_columns = [
    "model",
    "dataset",
    "n",
    "mean_original_probability",
    "mean_hotspot_probability_drop",
    "hotspot_drop_ci95_low",
    "hotspot_drop_ci95_high",
    "mean_random_probability_drop",
    "random_drop_ci95_low",
    "random_drop_ci95_high",
    "mean_hotspot_minus_random_drop",
    "difference_ci95_low",
    "difference_ci95_high",
    "fraction_hotspot_stronger_than_random",
    "paired_cohens_dz",
    "rank_biserial_effect_size",
    "wilcoxon_p_value",
    "mean_hotspot_only_probability",
    "mean_hotspot_only_probability_fraction",
]

cpp_manuscript_table = (
    cpp_summary_df[
        cpp_manuscript_columns
    ]
    .copy()
)

cpp_manuscript_file = (
    RESULT_TABLE_DIR
    / "CPP_hotspot_faithfulness_manuscript_table.csv"
)

atomic_save_csv(
    cpp_manuscript_table,
    cpp_manuscript_file,
)

all_output_files.append(
    cpp_manuscript_file
)


# ============================================================
# 14. CROSS-MODEL SUMMARY
# ============================================================

cross_model_numeric_columns = [
    "mean_original_probability",
    "mean_hotspot_probability_drop",
    "mean_random_probability_drop",
    "mean_hotspot_minus_random_drop",
    "fraction_hotspot_stronger_than_random",
    "paired_cohens_dz",
    "rank_biserial_effect_size",
    "mean_hotspot_only_probability",
    "mean_hotspot_only_probability_fraction",
]

cross_model_summary = (
    faithfulness_summary_df
    .groupby(
        [
            "dataset",
            "class",
        ],
        as_index=False,
    )[cross_model_numeric_columns]
    .mean()
)

cross_model_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_cross_model_summary.csv"
)

atomic_save_csv(
    cross_model_summary,
    cross_model_file,
)

all_output_files.append(
    cross_model_file
)


# ============================================================
# 15. COMBINE ALL PER-SEQUENCE OUTPUTS
# ============================================================

all_per_sequence_tables = []

for model_name in MODEL_CONFIGS:
    for dataset_name in DATASETS:
        per_sequence_file = (
            FAITHFULNESS_ROOT
            / model_name
            / dataset_name
            / "per_sequence_faithfulness.csv"
        )

        all_per_sequence_tables.append(
            pd.read_csv(
                per_sequence_file
            )
        )

combined_per_sequence_df = pd.concat(
    all_per_sequence_tables,
    ignore_index=True,
)

combined_per_sequence_file = (
    RESULT_SI_DIR
    / "faithfulness_all_sequences_all_models.csv"
)

atomic_save_csv(
    combined_per_sequence_df,
    combined_per_sequence_file,
)

all_output_files.append(
    combined_per_sequence_file
)


# ============================================================
# 16. FIGURE 1 — HOTSPOT VS RANDOM DROP BY MODEL
# ============================================================

cpp_plot_data = (
    combined_per_sequence_df[
        combined_per_sequence_df[
            "label"
        ] == 1
    ]
    .copy()
)

models = list(
    MODEL_CONFIGS.keys()
)

datasets_display = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

for dataset_name in DATASETS:
    dataset_subset = cpp_plot_data[
        cpp_plot_data["dataset"]
        == dataset_name
    ]

    hotspot_values = [
        dataset_subset[
            dataset_subset["model"]
            == model_name
        ][
            "hotspot_probability_drop"
        ].to_numpy()
        for model_name in models
    ]

    random_values = [
        dataset_subset[
            dataset_subset["model"]
            == model_name
        ][
            "random_mean_probability_drop"
        ].to_numpy()
        for model_name in models
    ]

    positions_hotspot = (
        np.arange(
            len(models)
        ) * 3.0
    )

    positions_random = (
        positions_hotspot + 1.0
    )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    hotspot_box = ax.boxplot(
        hotspot_values,
        positions=positions_hotspot,
        widths=0.75,
        patch_artist=False,
        showfliers=False,
        medianprops={
            "linewidth": 1.5,
        },
    )

    random_box = ax.boxplot(
        random_values,
        positions=positions_random,
        widths=0.75,
        patch_artist=False,
        showfliers=False,
        medianprops={
            "linewidth": 1.5,
        },
    )

    ax.axhline(
        0.0,
        linewidth=1.0,
        linestyle="--",
    )

    ax.set_xticks(
        positions_hotspot + 0.5
    )

    ax.set_xticklabels(
        models,
        rotation=20,
        ha="right",
    )

    ax.set_ylabel(
        "Decrease in predicted CPP probability"
    )

    ax.set_title(
        f"Consensus-hotspot ablation versus "
        f"matched random ablation\n"
        f"{datasets_display[dataset_name]} CPPs"
    )

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            linewidth=2,
            label="Consensus hotspots",
        ),
        plt.Line2D(
            [0],
            [0],
            linewidth=2,
            linestyle="--",
            label="Matched random residues",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        frameon=False,
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / f"faithfulness_hotspot_vs_random_{dataset_name}"
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    all_output_files.extend([
        figure_base.with_suffix(".png"),
        figure_base.with_suffix(".pdf"),
        figure_base.with_suffix(".svg"),
    ])


# ============================================================
# 17. FIGURE 2 — MEAN DROPS WITH 95% CI
# ============================================================

for dataset_name in DATASETS:
    table_subset = cpp_manuscript_table[
        cpp_manuscript_table[
            "dataset"
        ] == dataset_name
    ].copy()

    table_subset["model"] = pd.Categorical(
        table_subset["model"],
        categories=models,
        ordered=True,
    )

    table_subset = table_subset.sort_values(
        "model"
    )

    x_positions = np.arange(
        len(table_subset)
    )

    hotspot_means = table_subset[
        "mean_hotspot_probability_drop"
    ].to_numpy()

    hotspot_lower = (
        hotspot_means
        - table_subset[
            "hotspot_drop_ci95_low"
        ].to_numpy()
    )

    hotspot_upper = (
        table_subset[
            "hotspot_drop_ci95_high"
        ].to_numpy()
        - hotspot_means
    )

    random_means = table_subset[
        "mean_random_probability_drop"
    ].to_numpy()

    random_lower = (
        random_means
        - table_subset[
            "random_drop_ci95_low"
        ].to_numpy()
    )

    random_upper = (
        table_subset[
            "random_drop_ci95_high"
        ].to_numpy()
        - random_means
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.errorbar(
        x_positions - 0.12,
        hotspot_means,
        yerr=np.vstack([
            hotspot_lower,
            hotspot_upper,
        ]),
        fmt="o",
        capsize=4,
        label="Consensus hotspot ablation",
    )

    ax.errorbar(
        x_positions + 0.12,
        random_means,
        yerr=np.vstack([
            random_lower,
            random_upper,
        ]),
        fmt="s",
        capsize=4,
        label="Matched random ablation",
    )

    ax.axhline(
        0.0,
        linewidth=1.0,
        linestyle="--",
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        table_subset[
            "model"
        ].astype(str),
        rotation=20,
        ha="right",
    )

    ax.set_ylabel(
        "Mean decrease in CPP probability"
    )

    ax.set_title(
        f"Faithfulness of consensus hotspots "
        f"across PLMs\n"
        f"{datasets_display[dataset_name]}"
    )

    ax.legend(
        frameon=False,
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / f"faithfulness_mean_CI_{dataset_name}"
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    all_output_files.extend([
        figure_base.with_suffix(".png"),
        figure_base.with_suffix(".pdf"),
        figure_base.with_suffix(".svg"),
    ])


# ============================================================
# 18. FIGURE 3 — ORIGINAL VS HOTSPOT-ONLY SUFFICIENCY
# ============================================================

for dataset_name in DATASETS:
    dataset_subset = cpp_plot_data[
        cpp_plot_data[
            "dataset"
        ] == dataset_name
    ]

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    for model_name in models:
        model_subset = dataset_subset[
            dataset_subset[
                "model"
            ] == model_name
        ]

        ax.scatter(
            model_subset[
                "original_probability"
            ],
            model_subset[
                "hotspot_only_probability"
            ],
            s=16,
            alpha=0.45,
            label=model_name,
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_xlim(
        0,
        1.02,
    )

    ax.set_ylim(
        0,
        1.02,
    )

    ax.set_xlabel(
        "Original predicted CPP probability"
    )

    ax.set_ylabel(
        "Hotspot-only predicted CPP probability"
    )

    ax.set_title(
        f"Predictive sufficiency of consensus hotspots\n"
        f"{datasets_display[dataset_name]} CPPs"
    )

    ax.legend(
        frameon=False,
        fontsize=8,
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_SI_DIR
        / f"faithfulness_sufficiency_scatter_{dataset_name}"
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    all_output_files.extend([
        figure_base.with_suffix(".png"),
        figure_base.with_suffix(".pdf"),
        figure_base.with_suffix(".svg"),
    ])


# ============================================================
# 19. FIGURE 4 — EFFECT SIZE BY MODEL
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

for dataset_index, dataset_name in enumerate(
    DATASETS
):
    subset = cpp_manuscript_table[
        cpp_manuscript_table[
            "dataset"
        ] == dataset_name
    ].copy()

    subset["model"] = pd.Categorical(
        subset["model"],
        categories=models,
        ordered=True,
    )

    subset = subset.sort_values(
        "model"
    )

    x_positions = (
        np.arange(
            len(models)
        )
        + dataset_index * 0.16
        - 0.08
    )

    ax.scatter(
        x_positions,
        subset[
            "paired_cohens_dz"
        ],
        s=55,
        label=datasets_display[
            dataset_name
        ],
    )

ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

ax.axhline(
    0.8,
    linestyle=":",
    linewidth=1.0,
)

ax.set_xticks(
    np.arange(
        len(models)
    )
)

ax.set_xticklabels(
    models,
    rotation=20,
    ha="right",
)

ax.set_ylabel(
    "Paired Cohen's $d_z$\n"
    "(hotspot drop − random drop)"
)

ax.set_title(
    "Effect size of consensus-hotspot ablation"
)

ax.legend(
    frameon=False,
)

fig.tight_layout()

effect_figure_base = (
    FIGURE_MAIN_DIR
    / "faithfulness_effect_sizes"
)

fig.savefig(
    effect_figure_base.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    effect_figure_base.with_suffix(".pdf"),
    bbox_inches="tight",
)

fig.savefig(
    effect_figure_base.with_suffix(".svg"),
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

all_output_files.extend([
    effect_figure_base.with_suffix(".png"),
    effect_figure_base.with_suffix(".pdf"),
    effect_figure_base.with_suffix(".svg"),
])


# ============================================================
# 20. SAVE ANALYSIS DESCRIPTION
# ============================================================

analysis_description = {
    "analysis": (
        "Consensus-hotspot faithfulness and "
        "matched random perturbation"
    ),
    "hotspot_definition": (
        "Top 20% redundancy-adjusted global "
        "consensus residue ranks"
    ),
    "ablation_operation": (
        "Selected fixed PLM residue embeddings "
        "were replaced by zero vectors while "
        "sequence masks remained unchanged."
    ),
    "random_control": (
        "The same number of valid non-hotspot "
        "residues was randomly selected per sequence."
    ),
    "random_repeats": int(
        RANDOM_REPEATS
    ),
    "sufficiency_operation": (
        "Only hotspot residue embeddings were "
        "retained; all other residue embeddings "
        "were replaced by zero vectors."
    ),
    "statistical_test": (
        "One-sided paired Wilcoxon signed-rank test: "
        "hotspot probability drop > mean matched "
        "random probability drop."
    ),
    "effect_sizes": [
        "paired Cohen's dz",
        "matched-pairs rank-biserial effect size",
    ],
    "confidence_intervals": (
        f"{int(CONFIDENCE_LEVEL * 100)}% percentile "
        f"bootstrap intervals with "
        f"{BOOTSTRAP_REPEATS} resamples"
    ),
    "important_limitation": (
        "This analysis tests downstream-classifier "
        "faithfulness using fixed PLM residue embeddings. "
        "It is not equivalent to amino-acid mutation followed "
        "by re-embedding with the original PLM."
    ),
    "models": list(
        MODEL_CONFIGS.keys()
    ),
    "datasets": DATASETS,
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

analysis_description_file = (
    CHECKPOINT_ROOT
    / "faithfulness_analysis_method.json"
)

atomic_save_json(
    analysis_description_file,
    analysis_description,
)

all_output_files.append(
    analysis_description_file
)


# ============================================================
# 21. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "16_publication_ready_hotspot_faithfulness",
    output_files=all_output_files,
    details=analysis_description,
)


# ============================================================
# 22. DISPLAY FINAL TABLES
# ============================================================

print("\n" + "=" * 78)
print("CPP HOTSPOT FAITHFULNESS — MANUSCRIPT TABLE")
print("=" * 78)

display(
    cpp_manuscript_table[
        [
            "model",
            "dataset",
            "n",
            "mean_original_probability",
            "mean_hotspot_probability_drop",
            "hotspot_drop_ci95_low",
            "hotspot_drop_ci95_high",
            "mean_random_probability_drop",
            "random_drop_ci95_low",
            "random_drop_ci95_high",
            "mean_hotspot_minus_random_drop",
            "difference_ci95_low",
            "difference_ci95_high",
            "fraction_hotspot_stronger_than_random",
            "paired_cohens_dz",
            "rank_biserial_effect_size",
            "wilcoxon_p_value",
            "mean_hotspot_only_probability",
        ]
    ]
)

print("\n" + "=" * 78)
print("CROSS-MODEL CPP SUMMARY")
print("=" * 78)

display(
    cross_model_summary[
        cross_model_summary[
            "class"
        ] == "CPP"
    ]
)

print("\nSaved main summary:")
print(master_summary_file)

print("\nSaved manuscript table:")
print(cpp_manuscript_file)

print("\nSaved combined per-sequence SI table:")
print(combined_per_sequence_file)

print("\nSaved cross-model summary:")
print(cross_model_file)

print("\n" + "=" * 78)
print("STEP 16 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 17: CROSS-PLM HOTSPOT CONSERVATION
#
# Uses adjusted per-model top-20% hotspots.
#
# Analyses:
# 1. Pairwise Jaccard similarity between PLM hotspot sets
# 2. Residue support by 1, 2, 3, or all 4 PLMs
# 3. Per-sequence cross-model conservation
# 4. Unique versus shared hotspots
# 5. CPP versus non-CPP comparison
# 6. Internal versus KELM replication
# ============================================================

from pathlib import Path
from itertools import combinations
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    mannwhitneyu,
    fisher_exact,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

HOTSPOT_RANK_THRESHOLD = 0.80

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

CONSERVATION_ROOT = (
    XAI_ROOT / "hotspot_conservation"
)

for folder in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
    CONSERVATION_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. LOAD ADJUSTED PER-MODEL CONSENSUS
# ============================================================

def load_model_hotspots(
    model_name,
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / model_name
        / dataset_name
        / "adjusted_model_consensus.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Missing adjusted model consensus:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "adjusted_model_consensus_rank",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"{model_name}/{dataset_name} "
            f"is missing columns: {sorted(missing)}"
        )

    dataframe[
        f"hotspot_{model_name}"
    ] = (
        dataframe[
            "adjusted_model_consensus_rank"
        ]
        >= HOTSPOT_RANK_THRESHOLD
    )

    return dataframe


# ============================================================
# 3. MERGE ALL FOUR PLMS
# ============================================================

identity_columns = [
    "sequence_id",
    "sequence",
    "label",
    "sequence_length",
    "position",
    "normalized_position",
    "residue",
]

merged_dataset_tables = {}

for dataset_name in DATASETS:

    merged = None

    for model_name in MODEL_NAMES:

        current = load_model_hotspots(
            model_name=model_name,
            dataset_name=dataset_name,
        )

        current = current[
            identity_columns
            + [
                "adjusted_model_consensus_rank",
                f"hotspot_{model_name}",
            ]
        ].copy()

        current = current.rename(
            columns={
                "adjusted_model_consensus_rank":
                    f"rank_{model_name}"
            }
        )

        if merged is None:
            merged = current
        else:
            merged = merged.merge(
                current,
                on=identity_columns,
                how="inner",
                validate="one_to_one",
            )

    hotspot_columns = [
        f"hotspot_{model_name}"
        for model_name in MODEL_NAMES
    ]

    merged[
        "model_support_count"
    ] = (
        merged[
            hotspot_columns
        ]
        .astype(int)
        .sum(axis=1)
    )

    merged[
        "supported_by_at_least_2"
    ] = (
        merged["model_support_count"] >= 2
    )

    merged[
        "supported_by_at_least_3"
    ] = (
        merged["model_support_count"] >= 3
    )

    merged[
        "supported_by_all_4"
    ] = (
        merged["model_support_count"] == 4
    )

    merged[
        "model_unique_hotspot"
    ] = (
        merged["model_support_count"] == 1
    )

    merged[
        "any_model_hotspot"
    ] = (
        merged["model_support_count"] >= 1
    )

    rank_columns = [
        f"rank_{model_name}"
        for model_name in MODEL_NAMES
    ]

    merged[
        "mean_model_rank"
    ] = merged[
        rank_columns
    ].mean(axis=1)

    merged[
        "model_rank_std"
    ] = merged[
        rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    output_dir = (
        CONSERVATION_ROOT
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    residue_output_file = (
        output_dir
        / "cross_plm_residue_support.csv"
    )

    merged.to_csv(
        residue_output_file,
        index=False,
    )

    merged_dataset_tables[
        dataset_name
    ] = merged

    print(
        f"[MERGED] {dataset_name}: "
        f"{len(merged)} residues"
    )


# ============================================================
# 4. PAIRWISE JACCARD PER SEQUENCE
# ============================================================

pairwise_rows = []
per_sequence_pairwise_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for model_a, model_b in combinations(
        MODEL_NAMES,
        2,
    ):

        column_a = (
            f"hotspot_{model_a}"
        )

        column_b = (
            f"hotspot_{model_b}"
        )

        sequence_jaccards = []

        cpp_jaccards = []
        noncpp_jaccards = []

        for sequence_id, group in (
            dataframe.groupby(
                "sequence_id",
                sort=False,
            )
        ):

            set_a = set(
                group.loc[
                    group[column_a].astype(bool),
                    "position",
                ].astype(int)
            )

            set_b = set(
                group.loc[
                    group[column_b].astype(bool),
                    "position",
                ].astype(int)
            )

            union = (
                set_a | set_b
            )

            intersection = (
                set_a & set_b
            )

            if len(union) == 0:
                jaccard = 1.0
            else:
                jaccard = (
                    len(intersection)
                    / len(union)
                )

            label = int(
                group["label"].iloc[0]
            )

            sequence_jaccards.append(
                jaccard
            )

            if label == 1:
                cpp_jaccards.append(
                    jaccard
                )
            else:
                noncpp_jaccards.append(
                    jaccard
                )

            per_sequence_pairwise_rows.append({
                "dataset": dataset_name,
                "sequence_id": sequence_id,
                "label": label,
                "model_a": model_a,
                "model_b": model_b,
                "intersection_size": int(
                    len(intersection)
                ),
                "union_size": int(
                    len(union)
                ),
                "jaccard_similarity": float(
                    jaccard
                ),
            })

        pairwise_rows.append({
            "dataset": dataset_name,
            "model_a": model_a,
            "model_b": model_b,
            "number_of_sequences": int(
                len(sequence_jaccards)
            ),
            "mean_jaccard_all": float(
                np.mean(
                    sequence_jaccards
                )
            ),
            "median_jaccard_all": float(
                np.median(
                    sequence_jaccards
                )
            ),
            "mean_jaccard_CPP": float(
                np.mean(
                    cpp_jaccards
                )
            ),
            "median_jaccard_CPP": float(
                np.median(
                    cpp_jaccards
                )
            ),
            "mean_jaccard_nonCPP": float(
                np.mean(
                    noncpp_jaccards
                )
            ),
            "median_jaccard_nonCPP": float(
                np.median(
                    noncpp_jaccards
                )
            ),
        })


pairwise_df = pd.DataFrame(
    pairwise_rows
)

per_sequence_pairwise_df = (
    pd.DataFrame(
        per_sequence_pairwise_rows
    )
)

pairwise_file = (
    RESULT_TABLE_DIR
    / "cross_plm_pairwise_hotspot_jaccard.csv"
)

pairwise_sequence_file = (
    RESULT_SI_DIR
    / "cross_plm_pairwise_jaccard_per_sequence.csv"
)

pairwise_df.to_csv(
    pairwise_file,
    index=False,
)

per_sequence_pairwise_df.to_csv(
    pairwise_sequence_file,
    index=False,
)


# ============================================================
# 5. SUPPORT DISTRIBUTION
# ============================================================

support_distribution_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
        ("all", "all"),
    ]:

        if label_value == "all":
            subset = dataframe
        else:
            subset = dataframe[
                dataframe["label"]
                == label_value
            ]

        total_residues = len(
            subset
        )

        hotspot_union = subset[
            subset[
                "model_support_count"
            ] >= 1
        ]

        union_total = len(
            hotspot_union
        )

        for support_count in [
            0,
            1,
            2,
            3,
            4,
        ]:

            count = int(
                (
                    subset[
                        "model_support_count"
                    ]
                    == support_count
                ).sum()
            )

            support_distribution_rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "model_support_count": (
                    support_count
                ),
                "residue_count": count,
                "fraction_all_residues": float(
                    count / total_residues
                ),
                "fraction_hotspot_union": (
                    float(
                        count / union_total
                    )
                    if (
                        support_count > 0
                        and union_total > 0
                    )
                    else 0.0
                ),
            })


support_distribution_df = pd.DataFrame(
    support_distribution_rows
)

support_distribution_file = (
    RESULT_TABLE_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)

support_distribution_df.to_csv(
    support_distribution_file,
    index=False,
)


# ============================================================
# 6. PER-SEQUENCE CONSERVATION SUMMARY
# ============================================================

per_sequence_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for sequence_id, group in (
        dataframe.groupby(
            "sequence_id",
            sort=False,
        )
    ):

        sequence_length = int(
            group[
                "sequence_length"
            ].iloc[0]
        )

        support_counts = group[
            "model_support_count"
        ].to_numpy()

        any_hotspot_count = int(
            np.sum(
                support_counts >= 1
            )
        )

        at_least_2_count = int(
            np.sum(
                support_counts >= 2
            )
        )

        at_least_3_count = int(
            np.sum(
                support_counts >= 3
            )
        )

        unanimous_count = int(
            np.sum(
                support_counts == 4
            )
        )

        unique_count = int(
            np.sum(
                support_counts == 1
            )
        )

        mean_pairwise_jaccard = (
            per_sequence_pairwise_df[
                (
                    per_sequence_pairwise_df[
                        "dataset"
                    ] == dataset_name
                )
                & (
                    per_sequence_pairwise_df[
                        "sequence_id"
                    ] == sequence_id
                )
            ][
                "jaccard_similarity"
            ].mean()
        )

        per_sequence_rows.append({
            "dataset": dataset_name,
            "sequence_id": sequence_id,
            "sequence": str(
                group["sequence"].iloc[0]
            ),
            "label": int(
                group["label"].iloc[0]
            ),
            "sequence_length": (
                sequence_length
            ),
            "any_model_hotspot_count": (
                any_hotspot_count
            ),
            "at_least_2_models_count": (
                at_least_2_count
            ),
            "at_least_3_models_count": (
                at_least_3_count
            ),
            "all_4_models_count": (
                unanimous_count
            ),
            "model_unique_count": (
                unique_count
            ),
            "fraction_supported_by_at_least_2": float(
                at_least_2_count
                / sequence_length
            ),
            "fraction_supported_by_at_least_3": float(
                at_least_3_count
                / sequence_length
            ),
            "fraction_supported_by_all_4": float(
                unanimous_count
                / sequence_length
            ),
            "fraction_unique_to_one_model": float(
                unique_count
                / sequence_length
            ),
            "mean_pairwise_jaccard": float(
                mean_pairwise_jaccard
            ),
        })


per_sequence_df = pd.DataFrame(
    per_sequence_rows
)

per_sequence_file = (
    RESULT_SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)

per_sequence_df.to_csv(
    per_sequence_file,
    index=False,
)


# ============================================================
# 7. CPP VS NON-CPP CONSERVATION STATISTICS
# ============================================================

class_comparison_rows = []

metrics_to_compare = [
    "fraction_supported_by_at_least_2",
    "fraction_supported_by_at_least_3",
    "fraction_supported_by_all_4",
    "fraction_unique_to_one_model",
    "mean_pairwise_jaccard",
]

for dataset_name in DATASETS:

    dataset_table = (
        per_sequence_df[
            per_sequence_df["dataset"]
            == dataset_name
        ]
    )

    cpp = dataset_table[
        dataset_table["label"] == 1
    ]

    noncpp = dataset_table[
        dataset_table["label"] == 0
    ]

    for metric in metrics_to_compare:

        statistic, p_value = (
            mannwhitneyu(
                cpp[metric],
                noncpp[metric],
                alternative="two-sided",
            )
        )

        class_comparison_rows.append({
            "dataset": dataset_name,
            "metric": metric,
            "CPP_mean": float(
                cpp[metric].mean()
            ),
            "CPP_median": float(
                cpp[metric].median()
            ),
            "nonCPP_mean": float(
                noncpp[metric].mean()
            ),
            "nonCPP_median": float(
                noncpp[metric].median()
            ),
            "mannwhitney_statistic": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })


class_comparison_df = pd.DataFrame(
    class_comparison_rows
)

class_comparison_file = (
    RESULT_TABLE_DIR
    / "CPP_vs_nonCPP_hotspot_conservation.csv"
)

class_comparison_df.to_csv(
    class_comparison_file,
    index=False,
)


# ============================================================
# 8. RESIDUE-LEVEL CPP VS NON-CPP CONSERVATION
# ============================================================

residue_class_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for minimum_support in [
        2,
        3,
        4,
    ]:

        cpp = dataframe[
            dataframe["label"] == 1
        ]

        noncpp = dataframe[
            dataframe["label"] == 0
        ]

        cpp_supported = int(
            (
                cpp[
                    "model_support_count"
                ] >= minimum_support
            ).sum()
        )

        cpp_not_supported = int(
            len(cpp) - cpp_supported
        )

        noncpp_supported = int(
            (
                noncpp[
                    "model_support_count"
                ] >= minimum_support
            ).sum()
        )

        noncpp_not_supported = int(
            len(noncpp)
            - noncpp_supported
        )

        odds_ratio, p_value = (
            fisher_exact(
                [
                    [
                        cpp_supported,
                        cpp_not_supported,
                    ],
                    [
                        noncpp_supported,
                        noncpp_not_supported,
                    ],
                ],
                alternative="two-sided",
            )
        )

        corrected_odds_ratio = (
            (cpp_supported + 0.5)
            * (
                noncpp_not_supported
                + 0.5
            )
            / (
                (cpp_not_supported + 0.5)
                * (
                    noncpp_supported
                    + 0.5
                )
            )
        )

        residue_class_rows.append({
            "dataset": dataset_name,
            "minimum_model_support": (
                minimum_support
            ),
            "CPP_supported_count": (
                cpp_supported
            ),
            "CPP_total_residues": int(
                len(cpp)
            ),
            "CPP_supported_fraction": float(
                cpp_supported / len(cpp)
            ),
            "nonCPP_supported_count": (
                noncpp_supported
            ),
            "nonCPP_total_residues": int(
                len(noncpp)
            ),
            "nonCPP_supported_fraction": float(
                noncpp_supported
                / len(noncpp)
            ),
            "odds_ratio_scipy": float(
                odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_odds_ratio
            ),
            "p_value": float(
                p_value
            ),
        })


residue_class_df = pd.DataFrame(
    residue_class_rows
)

residue_class_file = (
    RESULT_TABLE_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

residue_class_df.to_csv(
    residue_class_file,
    index=False,
)


# ============================================================
# 9. JACCARD MATRICES
# ============================================================

jaccard_matrix_files = []

for dataset_name in DATASETS:

    for class_column, class_name in [
        ("mean_jaccard_all", "all"),
        ("mean_jaccard_CPP", "CPP"),
        ("mean_jaccard_nonCPP", "nonCPP"),
    ]:

        matrix = pd.DataFrame(
            np.eye(
                len(MODEL_NAMES)
            ),
            index=MODEL_NAMES,
            columns=MODEL_NAMES,
        )

        subset = pairwise_df[
            pairwise_df["dataset"]
            == dataset_name
        ]

        for row in subset.itertuples(
            index=False
        ):

            matrix.loc[
                row.model_a,
                row.model_b,
            ] = getattr(
                row,
                class_column
            )

            matrix.loc[
                row.model_b,
                row.model_a,
            ] = getattr(
                row,
                class_column
            )

        matrix_file = (
            RESULT_TABLE_DIR
            / (
                f"hotspot_jaccard_matrix_"
                f"{dataset_name}_{class_name}.csv"
            )
        )

        matrix.to_csv(
            matrix_file
        )

        jaccard_matrix_files.append(
            matrix_file
        )

        # Plot matrix
        fig, ax = plt.subplots(
            figsize=(6.5, 5.5)
        )

        image = ax.imshow(
            matrix.to_numpy(),
            vmin=0,
            vmax=1,
            aspect="equal",
        )

        ax.set_xticks(
            np.arange(
                len(MODEL_NAMES)
            )
        )

        ax.set_yticks(
            np.arange(
                len(MODEL_NAMES)
            )
        )

        ax.set_xticklabels(
            MODEL_NAMES,
            rotation=35,
            ha="right",
        )

        ax.set_yticklabels(
            MODEL_NAMES
        )

        for row_index in range(
            len(MODEL_NAMES)
        ):
            for column_index in range(
                len(MODEL_NAMES)
            ):

                ax.text(
                    column_index,
                    row_index,
                    f"{matrix.iloc[row_index, column_index]:.2f}",
                    ha="center",
                    va="center",
                )

        ax.set_title(
            f"Cross-PLM hotspot Jaccard similarity\n"
            f"{dataset_name} — {class_name}"
        )

        figure_colorbar = fig.colorbar(
            image,
            ax=ax,
        )

        figure_colorbar.set_label(
            "Mean per-sequence Jaccard similarity"
        )

        fig.tight_layout()

        figure_base = (
            FIGURE_MAIN_DIR
            / (
                f"hotspot_jaccard_"
                f"{dataset_name}_{class_name}"
            )
        )

        fig.savefig(
            figure_base.with_suffix(".png"),
            dpi=600,
            bbox_inches="tight",
        )

        fig.savefig(
            figure_base.with_suffix(".pdf"),
            bbox_inches="tight",
        )

        fig.savefig(
            figure_base.with_suffix(".svg"),
            bbox_inches="tight",
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 10. SUPPORT-COUNT BAR PLOTS
# ============================================================

for dataset_name in DATASETS:

    plot_subset = (
        support_distribution_df[
            (
                support_distribution_df[
                    "dataset"
                ] == dataset_name
            )
            & (
                support_distribution_df[
                    "class"
                ].isin(
                    [
                        "CPP",
                        "non_CPP",
                    ]
                )
            )
            & (
                support_distribution_df[
                    "model_support_count"
                ] > 0
            )
        ]
    )

    support_counts = [
        1,
        2,
        3,
        4,
    ]

    cpp_values = [
        float(
            plot_subset[
                (
                    plot_subset["class"]
                    == "CPP"
                )
                & (
                    plot_subset[
                        "model_support_count"
                    ] == support
                )
            ][
                "fraction_hotspot_union"
            ].iloc[0]
        )
        for support in support_counts
    ]

    noncpp_values = [
        float(
            plot_subset[
                (
                    plot_subset["class"]
                    == "non_CPP"
                )
                & (
                    plot_subset[
                        "model_support_count"
                    ] == support
                )
            ][
                "fraction_hotspot_union"
            ].iloc[0]
        )
        for support in support_counts
    ]

    x = np.arange(
        len(support_counts)
    )

    width = 0.36

    fig, ax = plt.subplots(
        figsize=(8, 5.5)
    )

    ax.bar(
        x - width / 2,
        cpp_values,
        width=width,
        label="CPP",
    )

    ax.bar(
        x + width / 2,
        noncpp_values,
        width=width,
        label="Non-CPP",
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            "1 PLM",
            "2 PLMs",
            "3 PLMs",
            "4 PLMs",
        ]
    )

    ax.set_ylabel(
        "Fraction of hotspot-union residues"
    )

    ax.set_title(
        f"Cross-PLM support of hotspot residues\n"
        f"{dataset_name}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            f"hotspot_model_support_"
            f"{dataset_name}"
        )
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 11. DATASET-LEVEL SUMMARY
# ============================================================

dataset_summary_rows = []

for dataset_name in DATASETS:

    residue_table = (
        merged_dataset_tables[
            dataset_name
        ]
    )

    sequence_table = (
        per_sequence_df[
            per_sequence_df["dataset"]
            == dataset_name
        ]
    )

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
    ]:

        residue_subset = (
            residue_table[
                residue_table["label"]
                == label_value
            ]
        )

        sequence_subset = (
            sequence_table[
                sequence_table["label"]
                == label_value
            ]
        )

        dataset_summary_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "number_of_sequences": int(
                len(sequence_subset)
            ),
            "number_of_residues": int(
                len(residue_subset)
            ),
            "fraction_any_model_hotspot": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] >= 1
                ).mean()
            ),
            "fraction_supported_by_at_least_2": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] >= 2
                ).mean()
            ),
            "fraction_supported_by_at_least_3": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] >= 3
                ).mean()
            ),
            "fraction_supported_by_all_4": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] == 4
                ).mean()
            ),
            "mean_sequence_pairwise_jaccard": float(
                sequence_subset[
                    "mean_pairwise_jaccard"
                ].mean()
            ),
            "median_sequence_pairwise_jaccard": float(
                sequence_subset[
                    "mean_pairwise_jaccard"
                ].median()
            ),
        })


dataset_summary_df = pd.DataFrame(
    dataset_summary_rows
)

dataset_summary_file = (
    RESULT_TABLE_DIR
    / "cross_plm_hotspot_conservation_summary.csv"
)

dataset_summary_df.to_csv(
    dataset_summary_file,
    index=False,
)


# ============================================================
# 12. SAVE CHECKPOINT
# ============================================================

analysis_summary = {
    "models": MODEL_NAMES,
    "datasets": DATASETS,
    "hotspot_threshold": float(
        HOTSPOT_RANK_THRESHOLD
    ),
    "hotspot_definition": (
        "Adjusted per-model consensus percentile rank "
        "greater than or equal to 0.80"
    ),
    "pairwise_similarity": (
        "Per-sequence Jaccard similarity between "
        "model-specific hotspot position sets"
    ),
    "support_definition": (
        "Number of PLMs independently classifying a "
        "residue as a top-20% hotspot"
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

analysis_summary_file = (
    CHECKPOINT_DIR
    / "cross_plm_hotspot_conservation_method.json"
)

with open(
    analysis_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        analysis_summary,
        handle,
        indent=2,
    )

output_files = [
    pairwise_file,
    pairwise_sequence_file,
    support_distribution_file,
    per_sequence_file,
    class_comparison_file,
    residue_class_file,
    dataset_summary_file,
    analysis_summary_file,
    *jaccard_matrix_files,
]

for dataset_name in DATASETS:
    output_files.append(
        CONSERVATION_ROOT
        / dataset_name
        / "cross_plm_residue_support.csv"
    )

mark_step_complete(
    "17_cross_PLM_hotspot_conservation",
    output_files=output_files,
    details=analysis_summary,
)


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 78)
print("PAIRWISE CROSS-PLM HOTSPOT JACCARD")
print("=" * 78)

display(
    pairwise_df[
        [
            "dataset",
            "model_a",
            "model_b",
            "mean_jaccard_CPP",
            "median_jaccard_CPP",
            "mean_jaccard_nonCPP",
            "median_jaccard_nonCPP",
        ]
    ]
)

print("\n" + "=" * 78)
print("CROSS-PLM CONSERVATION SUMMARY")
print("=" * 78)

display(
    dataset_summary_df
)

print("\n" + "=" * 78)
print("CPP VS NON-CPP CONSERVATION")
print("=" * 78)

display(
    class_comparison_df
)

print("\n" + "=" * 78)
print("RESIDUE-LEVEL CONSERVATION ENRICHMENT")
print("=" * 78)

display(
    residue_class_df
)

print("\nSaved pairwise Jaccard table:")
print(pairwise_file)

print("\nSaved conservation summary:")
print(dataset_summary_file)

print("\nSaved per-sequence SI table:")
print(per_sequence_file)

print("\n" + "=" * 78)
print("STEP 17 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 18: PHYSICOCHEMICAL ENRICHMENT OF CONSENSUS HOTSPOTS
#
# Analyses:
# 1. Categorical residue-property enrichment
#    - CPP hotspots vs CPP non-hotspots
#    - CPP hotspots vs non-CPP hotspots
#
# 2. Sequence-level continuous-property comparison
#    - paired hotspot vs non-hotspot comparison within CPPs
#    - CPP hotspot vs non-CPP hotspot comparison
#
# 3. Internal-test vs KELM replication
#
# Primary hotspot definition:
#    redundancy-adjusted global top-20% residues
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    fisher_exact,
    wilcoxon,
    mannwhitneyu,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

for directory in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. NON-OVERLAPPING PHYSICOCHEMICAL CLASSES
#
# Every standard amino acid belongs to exactly one class.
# Histidine is assigned to the basic class.
# ============================================================

PHYSICOCHEMICAL_CLASSES = {
    "basic_positive": set("KRH"),
    "acidic_negative": set("DE"),
    "polar_uncharged": set("STNQ"),
    "aromatic": set("FWY"),
    "aliphatic_hydrophobic": set("AILMV"),
    "structure_special": set("GPC"),
}

AMINO_ACIDS = set(
    "ACDEFGHIKLMNPQRSTVWY"
)

all_classified_residues = set().union(
    *PHYSICOCHEMICAL_CLASSES.values()
)

if all_classified_residues != AMINO_ACIDS:
    missing = (
        AMINO_ACIDS
        - all_classified_residues
    )

    duplicated = []

    for residue in AMINO_ACIDS:
        number_of_classes = sum(
            residue in residues
            for residues
            in PHYSICOCHEMICAL_CLASSES.values()
        )

        if number_of_classes != 1:
            duplicated.append(residue)

    raise ValueError(
        f"Physicochemical classes are invalid. "
        f"Missing={missing}, non-unique={duplicated}"
    )


def residue_to_class(residue):
    for class_name, residue_set in (
        PHYSICOCHEMICAL_CLASSES.items()
    ):
        if residue in residue_set:
            return class_name

    raise ValueError(
        f"Unknown residue: {residue}"
    )


# ============================================================
# 3. RESIDUE-LEVEL PROPERTY LOOKUPS
#
# Charge is a simplified physiological-pH proxy.
# Hydropathy uses the Kyte-Doolittle scale.
# ============================================================

CHARGE_SCORE = {
    "A": 0.0,
    "C": 0.0,
    "D": -1.0,
    "E": -1.0,
    "F": 0.0,
    "G": 0.0,
    "H": 0.1,
    "I": 0.0,
    "K": 1.0,
    "L": 0.0,
    "M": 0.0,
    "N": 0.0,
    "P": 0.0,
    "Q": 0.0,
    "R": 1.0,
    "S": 0.0,
    "T": 0.0,
    "V": 0.0,
    "W": 0.0,
    "Y": 0.0,
}

KYTE_DOOLITTLE = {
    "A": 1.8,
    "C": 2.5,
    "D": -3.5,
    "E": -3.5,
    "F": 2.8,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "N": -3.5,
    "P": -1.6,
    "Q": -3.5,
    "R": -4.5,
    "S": -0.8,
    "T": -0.7,
    "V": 4.2,
    "W": -0.9,
    "Y": -1.3,
}

AROMATIC_SCORE = {
    residue: float(
        residue in set("FWY")
    )
    for residue in AMINO_ACIDS
}

BASIC_SCORE = {
    residue: float(
        residue in set("KRH")
    )
    for residue in AMINO_ACIDS
}

ACIDIC_SCORE = {
    residue: float(
        residue in set("DE")
    )
    for residue in AMINO_ACIDS
}

POLAR_SCORE = {
    residue: float(
        residue in set("STNQ")
    )
    for residue in AMINO_ACIDS
}


CONTINUOUS_PROPERTIES = {
    "charge_score": CHARGE_SCORE,
    "hydropathy_KD": KYTE_DOOLITTLE,
    "aromatic_fraction": AROMATIC_SCORE,
    "basic_fraction": BASIC_SCORE,
    "acidic_fraction": ACIDIC_SCORE,
    "polar_fraction": POLAR_SCORE,
}


# ============================================================
# 4. BENJAMINI-HOCHBERG FDR
# ============================================================

def benjamini_hochberg(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    if number_of_tests == 0:
        return np.array(
            [],
            dtype=float,
        )

    order = np.argsort(
        p_values
    )

    ordered_p_values = (
        p_values[order]
    )

    adjusted_ordered = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ordered_p_values[
                reverse_index
            ]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ordered[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ordered,
        1.0,
    )

    return adjusted


# ============================================================
# 5. ODDS RATIO WITH CONTINUITY CORRECTION
# ============================================================

def corrected_odds_ratio(
    a,
    b,
    c,
    d,
):
    return float(
        (
            (a + 0.5)
            * (d + 0.5)
        )
        / (
            (b + 0.5)
            * (c + 0.5)
        )
    )


# ============================================================
# 6. SAFE STATISTICAL TESTS
# ============================================================

def safe_wilcoxon(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    valid = (
        np.isfinite(values_a)
        & np.isfinite(values_b)
    )

    values_a = values_a[
        valid
    ]

    values_b = values_b[
        valid
    ]

    differences = (
        values_a - values_b
    )

    if len(differences) == 0:
        return np.nan, np.nan

    if np.allclose(
        differences,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


def safe_mannwhitney(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    values_a = values_a[
        np.isfinite(values_a)
    ]

    values_b = values_b[
        np.isfinite(values_b)
    ]

    if (
        len(values_a) == 0
        or len(values_b) == 0
    ):
        return np.nan, np.nan

    statistic, p_value = mannwhitneyu(
        values_a,
        values_b,
        alternative="two-sided",
    )

    return (
        float(statistic),
        float(p_value),
    )


# ============================================================
# 7. LOAD ADJUSTED GLOBAL CONSENSUS
# ============================================================

def load_consensus(
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file not found:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "adjusted_hotspot_top20",
        "strict_cross_model_hotspot",
        "unanimous_cross_model_hotspot",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe["residue"] = (
        dataframe["residue"]
        .astype(str)
        .str.upper()
    )

    invalid_residues = (
        set(dataframe["residue"])
        - AMINO_ACIDS
    )

    if invalid_residues:
        raise ValueError(
            f"Invalid residues detected: "
            f"{sorted(invalid_residues)}"
        )

    dataframe[
        "physicochemical_class"
    ] = dataframe[
        "residue"
    ].map(
        residue_to_class
    )

    for property_name, lookup in (
        CONTINUOUS_PROPERTIES.items()
    ):
        dataframe[
            property_name
        ] = dataframe[
            "residue"
        ].map(
            lookup
        ).astype(float)

    return (
        dataframe,
        input_file,
    )


# ============================================================
# 8. CATEGORICAL ENRICHMENT
# ============================================================

def categorical_enrichment(
    group_1,
    group_2,
    dataset_name,
    analysis_name,
    group_1_name,
    group_2_name,
):
    rows = []

    group_1_total = len(
        group_1
    )

    group_2_total = len(
        group_2
    )

    if (
        group_1_total == 0
        or group_2_total == 0
    ):
        raise ValueError(
            f"Empty comparison group for "
            f"{dataset_name}/{analysis_name}"
        )

    for category_name in (
        PHYSICOCHEMICAL_CLASSES
    ):
        group_1_count = int(
            (
                group_1[
                    "physicochemical_class"
                ]
                == category_name
            ).sum()
        )

        group_2_count = int(
            (
                group_2[
                    "physicochemical_class"
                ]
                == category_name
            ).sum()
        )

        group_1_other = (
            group_1_total
            - group_1_count
        )

        group_2_other = (
            group_2_total
            - group_2_count
        )

        contingency_table = [
            [
                group_1_count,
                group_1_other,
            ],
            [
                group_2_count,
                group_2_other,
            ],
        ]

        raw_odds_ratio, p_value = (
            fisher_exact(
                contingency_table,
                alternative="two-sided",
            )
        )

        corrected_or = (
            corrected_odds_ratio(
                group_1_count,
                group_1_other,
                group_2_count,
                group_2_other,
            )
        )

        group_1_frequency = (
            group_1_count
            / group_1_total
        )

        group_2_frequency = (
            group_2_count
            / group_2_total
        )

        log2_enrichment = np.log2(
            (
                group_1_frequency
                + 1e-12
            )
            / (
                group_2_frequency
                + 1e-12
            )
        )

        rows.append({
            "dataset": dataset_name,
            "analysis": analysis_name,
            "group_1": group_1_name,
            "group_2": group_2_name,
            "property_class": (
                category_name
            ),
            "group_1_count": (
                group_1_count
            ),
            "group_1_total": (
                group_1_total
            ),
            "group_1_frequency": float(
                group_1_frequency
            ),
            "group_2_count": (
                group_2_count
            ),
            "group_2_total": (
                group_2_total
            ),
            "group_2_frequency": float(
                group_2_frequency
            ),
            "odds_ratio_raw": float(
                raw_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_or
            ),
            "log2_enrichment": float(
                log2_enrichment
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    result["fdr_bh"] = (
        benjamini_hochberg(
            result[
                "p_value"
            ].to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "direction"
    ] = np.where(
        result[
            "odds_ratio_corrected"
        ] > 1.0,
        f"enriched_in_{group_1_name}",
        f"enriched_in_{group_2_name}",
    )

    return result


# ============================================================
# 9. CREATE SEQUENCE-LEVEL PROPERTY TABLE
# ============================================================

def build_sequence_property_table(
    dataframe,
    dataset_name,
):
    rows = []

    for sequence_id, group in (
        dataframe.groupby(
            "sequence_id",
            sort=False,
        )
    ):
        hotspot_group = group[
            group[
                "adjusted_hotspot_top20"
            ].astype(bool)
        ]

        non_hotspot_group = group[
            ~group[
                "adjusted_hotspot_top20"
            ].astype(bool)
        ]

        record = {
            "dataset": dataset_name,
            "sequence_id": (
                sequence_id
            ),
            "sequence": str(
                group[
                    "sequence"
                ].iloc[0]
            ),
            "label": int(
                group[
                    "label"
                ].iloc[0]
            ),
            "sequence_length": int(
                group[
                    "sequence_length"
                ].iloc[0]
            ),
            "number_of_hotspots": int(
                len(hotspot_group)
            ),
            "number_of_nonhotspots": int(
                len(non_hotspot_group)
            ),
        }

        for property_name in (
            CONTINUOUS_PROPERTIES
        ):
            record[
                f"hotspot_{property_name}"
            ] = float(
                hotspot_group[
                    property_name
                ].mean()
            )

            record[
                f"nonhotspot_{property_name}"
            ] = float(
                non_hotspot_group[
                    property_name
                ].mean()
            )

            record[
                f"hotspot_minus_nonhotspot_{property_name}"
            ] = (
                record[
                    f"hotspot_{property_name}"
                ]
                - record[
                    f"nonhotspot_{property_name}"
                ]
            )

        rows.append(
            record
        )

    return pd.DataFrame(
        rows
    )


# ============================================================
# 10. RUN DATASET ANALYSES
# ============================================================

categorical_results = []
sequence_property_tables = []
source_files = []

for dataset_name in DATASETS:
    dataframe, source_file = (
        load_consensus(
            dataset_name
        )
    )

    source_files.append(
        source_file
    )

    print("\n" + "=" * 78)
    print(
        f"PHYSICOCHEMICAL ANALYSIS: "
        f"{dataset_name}"
    )
    print("=" * 78)

    cpp = dataframe[
        dataframe["label"] == 1
    ]

    noncpp = dataframe[
        dataframe["label"] == 0
    ]

    cpp_hotspots = cpp[
        cpp[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    cpp_nonhotspots = cpp[
        ~cpp[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    noncpp_hotspots = noncpp[
        noncpp[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    # Analysis A:
    # CPP hotspots vs non-hotspots from CPPs
    within_cpp_result = (
        categorical_enrichment(
            group_1=cpp_hotspots,
            group_2=cpp_nonhotspots,
            dataset_name=dataset_name,
            analysis_name=(
                "CPP_hotspot_vs_CPP_nonhotspot"
            ),
            group_1_name="CPP_hotspot",
            group_2_name="CPP_nonhotspot",
        )
    )

    categorical_results.append(
        within_cpp_result
    )

    # Analysis B:
    # CPP hotspots vs non-CPP hotspots
    between_class_result = (
        categorical_enrichment(
            group_1=cpp_hotspots,
            group_2=noncpp_hotspots,
            dataset_name=dataset_name,
            analysis_name=(
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            group_1_name="CPP_hotspot",
            group_2_name="nonCPP_hotspot",
        )
    )

    categorical_results.append(
        between_class_result
    )

    sequence_property_table = (
        build_sequence_property_table(
            dataframe=dataframe,
            dataset_name=dataset_name,
        )
    )

    sequence_property_tables.append(
        sequence_property_table
    )


categorical_df = pd.concat(
    categorical_results,
    ignore_index=True,
)

sequence_property_df = pd.concat(
    sequence_property_tables,
    ignore_index=True,
)


# ============================================================
# 11. SEQUENCE-LEVEL STATISTICAL TESTS
# ============================================================

sequence_test_rows = []

for dataset_name in DATASETS:
    dataset_table = (
        sequence_property_df[
            sequence_property_df[
                "dataset"
            ] == dataset_name
        ]
    )

    cpp_table = dataset_table[
        dataset_table["label"] == 1
    ]

    noncpp_table = dataset_table[
        dataset_table["label"] == 0
    ]

    for property_name in (
        CONTINUOUS_PROPERTIES
    ):
        # Paired within-CPP comparison
        hotspot_column = (
            f"hotspot_{property_name}"
        )

        nonhotspot_column = (
            f"nonhotspot_{property_name}"
        )

        wilcoxon_statistic, wilcoxon_p = (
            safe_wilcoxon(
                cpp_table[
                    hotspot_column
                ],
                cpp_table[
                    nonhotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "paired_CPP_hotspot_vs_CPP_nonhotspot"
            ),
            "property": property_name,
            "group_1": "CPP_hotspot",
            "group_2": "CPP_nonhotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(cpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_1_median": float(
                cpp_table[
                    hotspot_column
                ].median()
            ),
            "group_2_mean": float(
                cpp_table[
                    nonhotspot_column
                ].mean()
            ),
            "group_2_median": float(
                cpp_table[
                    nonhotspot_column
                ].median()
            ),
            "mean_difference": float(
                (
                    cpp_table[
                        hotspot_column
                    ]
                    - cpp_table[
                        nonhotspot_column
                    ]
                ).mean()
            ),
            "test": (
                "paired Wilcoxon signed-rank"
            ),
            "statistic": float(
                wilcoxon_statistic
            ),
            "p_value": float(
                wilcoxon_p
            ),
        })

        # CPP hotspot vs non-CPP hotspot
        mannwhitney_statistic, mannwhitney_p = (
            safe_mannwhitney(
                cpp_table[
                    hotspot_column
                ],
                noncpp_table[
                    hotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            "property": property_name,
            "group_1": "CPP_hotspot",
            "group_2": "nonCPP_hotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(noncpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_1_median": float(
                cpp_table[
                    hotspot_column
                ].median()
            ),
            "group_2_mean": float(
                noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_2_median": float(
                noncpp_table[
                    hotspot_column
                ].median()
            ),
            "mean_difference": float(
                cpp_table[
                    hotspot_column
                ].mean()
                - noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "test": (
                "Mann-Whitney U"
            ),
            "statistic": float(
                mannwhitney_statistic
            ),
            "p_value": float(
                mannwhitney_p
            ),
        })


sequence_tests_df = pd.DataFrame(
    sequence_test_rows
)

sequence_tests_df["fdr_bh"] = (
    sequence_tests_df.groupby(
        [
            "dataset",
            "analysis",
        ]
    )[
        "p_value"
    ].transform(
        lambda values:
            benjamini_hochberg(
                values.to_numpy()
            )
    )
)

sequence_tests_df[
    "significant_fdr_0_05"
] = (
    sequence_tests_df[
        "fdr_bh"
    ] < 0.05
)


# ============================================================
# 12. INTERNAL–KELM CATEGORY REPLICATION
# ============================================================

replication_source = categorical_df[
    categorical_df[
        "analysis"
    ] == (
        "CPP_hotspot_vs_CPP_nonhotspot"
    )
][
    [
        "dataset",
        "property_class",
        "odds_ratio_corrected",
        "log2_enrichment",
        "fdr_bh",
        "significant_fdr_0_05",
    ]
].copy()


internal_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "internal_test"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)


kelm_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "kelm_external"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)


replication_df = (
    internal_replication.merge(
        kelm_replication,
        on="property_class",
        how="outer",
        validate="one_to_one",
    )
)

replication_df[
    "same_direction"
] = (
    np.sign(
        replication_df[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replication_df[
            "kelm_log2_enrichment"
        ]
    )
)

replication_df[
    "significant_in_both"
] = (
    replication_df[
        "internal_significant"
    ].fillna(False)
    & replication_df[
        "kelm_significant"
    ].fillna(False)
)

replication_df = (
    replication_df.sort_values(
        [
            "significant_in_both",
            "same_direction",
            "internal_log2_enrichment",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)


# ============================================================
# 13. SAVE TABLES
# ============================================================

categorical_file = (
    RESULT_TABLE_DIR
    / "physicochemical_category_enrichment.csv"
)

sequence_property_file = (
    RESULT_SI_DIR
    / "physicochemical_properties_per_sequence.csv"
)

sequence_tests_file = (
    RESULT_TABLE_DIR
    / "physicochemical_sequence_level_tests.csv"
)

replication_file = (
    RESULT_TABLE_DIR
    / "physicochemical_enrichment_replication.csv"
)

significant_categorical_file = (
    RESULT_TABLE_DIR
    / "significant_physicochemical_enrichment.csv"
)

categorical_df.to_csv(
    categorical_file,
    index=False,
)

sequence_property_df.to_csv(
    sequence_property_file,
    index=False,
)

sequence_tests_df.to_csv(
    sequence_tests_file,
    index=False,
)

replication_df.to_csv(
    replication_file,
    index=False,
)

categorical_df[
    categorical_df[
        "significant_fdr_0_05"
    ]
].to_csv(
    significant_categorical_file,
    index=False,
)


# ============================================================
# 14. FIGURE — CATEGORY LOG2 ENRICHMENT
# ============================================================

category_order = list(
    PHYSICOCHEMICAL_CLASSES.keys()
)

for analysis_name, analysis_title in [
    (
        "CPP_hotspot_vs_CPP_nonhotspot",
        "CPP hotspots versus CPP non-hotspots",
    ),
    (
        "CPP_hotspot_vs_nonCPP_hotspot",
        "CPP hotspots versus non-CPP hotspots",
    ),
]:
    plot_table = categorical_df[
        categorical_df[
            "analysis"
        ] == analysis_name
    ].copy()

    plot_table[
        "property_class"
    ] = pd.Categorical(
        plot_table[
            "property_class"
        ],
        categories=category_order,
        ordered=True,
    )

    plot_table = plot_table.sort_values(
        [
            "property_class",
            "dataset",
        ]
    )

    x = np.arange(
        len(category_order)
    )

    width = 0.36

    internal_values = []

    kelm_values = []

    internal_significance = []

    kelm_significance = []

    for category_name in category_order:
        internal_row = plot_table[
            (
                plot_table["dataset"]
                == "internal_test"
            )
            & (
                plot_table[
                    "property_class"
                ] == category_name
            )
        ].iloc[0]

        kelm_row = plot_table[
            (
                plot_table["dataset"]
                == "kelm_external"
            )
            & (
                plot_table[
                    "property_class"
                ] == category_name
            )
        ].iloc[0]

        internal_values.append(
            internal_row[
                "log2_enrichment"
            ]
        )

        kelm_values.append(
            kelm_row[
                "log2_enrichment"
            ]
        )

        internal_significance.append(
            bool(
                internal_row[
                    "significant_fdr_0_05"
                ]
            )
        )

        kelm_significance.append(
            bool(
                kelm_row[
                    "significant_fdr_0_05"
                ]
            )
        )

    fig, ax = plt.subplots(
        figsize=(10, 5.8)
    )

    internal_bars = ax.bar(
        x - width / 2,
        internal_values,
        width=width,
        label="Internal test",
    )

    kelm_bars = ax.bar(
        x + width / 2,
        kelm_values,
        width=width,
        label="KELM external",
    )

    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            category.replace(
                "_",
                "\n",
            )
            for category
            in category_order
        ],
        rotation=0,
    )

    ax.set_ylabel(
        "log$_2$ enrichment"
    )

    ax.set_title(
        analysis_title
    )

    ax.legend(
        frameon=False,
    )

    for bar, significant in zip(
        internal_bars,
        internal_significance,
    ):
        if significant:
            height = bar.get_height()

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height
                + (
                    0.08
                    if height >= 0
                    else -0.18
                ),
                "*",
                ha="center",
                va="center",
                fontsize=13,
            )

    for bar, significant in zip(
        kelm_bars,
        kelm_significance,
    ):
        if significant:
            height = bar.get_height()

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height
                + (
                    0.08
                    if height >= 0
                    else -0.18
                ),
                "*",
                ha="center",
                va="center",
                fontsize=13,
            )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            "physicochemical_enrichment_"
            + analysis_name
        )
    )

    fig.savefig(
        figure_base.with_suffix(
            ".png"
        ),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".svg"
        ),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 15. FIGURE — SEQUENCE-LEVEL CONTINUOUS PROPERTIES
# ============================================================

property_display_names = {
    "charge_score": "Mean charge score",
    "hydropathy_KD": "Mean hydropathy",
    "aromatic_fraction": "Aromatic fraction",
    "basic_fraction": "Basic-residue fraction",
    "acidic_fraction": "Acidic-residue fraction",
    "polar_fraction": "Polar-residue fraction",
}

cpp_property_table = (
    sequence_property_df[
        sequence_property_df[
            "label"
        ] == 1
    ]
)

for property_name in (
    CONTINUOUS_PROPERTIES
):
    internal_cpp = cpp_property_table[
        cpp_property_table[
            "dataset"
        ] == "internal_test"
    ]

    kelm_cpp = cpp_property_table[
        cpp_property_table[
            "dataset"
        ] == "kelm_external"
    ]

    plot_groups = [
        internal_cpp[
            f"hotspot_{property_name}"
        ].to_numpy(),
        internal_cpp[
            f"nonhotspot_{property_name}"
        ].to_numpy(),
        kelm_cpp[
            f"hotspot_{property_name}"
        ].to_numpy(),
        kelm_cpp[
            f"nonhotspot_{property_name}"
        ].to_numpy(),
    ]

    fig, ax = plt.subplots(
        figsize=(8.5, 5.5)
    )

    ax.boxplot(
        plot_groups,
        showfliers=False,
    )

    ax.set_xticks(
        [
            1,
            2,
            3,
            4,
        ]
    )

    ax.set_xticklabels(
        [
            "Internal\nhotspots",
            "Internal\nnon-hotspots",
            "KELM\nhotspots",
            "KELM\nnon-hotspots",
        ]
    )

    ax.set_ylabel(
        property_display_names[
            property_name
        ]
    )

    ax.set_title(
        f"CPP hotspot physicochemical property:\n"
        f"{property_display_names[property_name]}"
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_SI_DIR
        / (
            "physicochemical_sequence_"
            + property_name
        )
    )

    fig.savefig(
        figure_base.with_suffix(
            ".png"
        ),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".svg"
        ),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 16. SUMMARY JSON
# ============================================================

summary = {
    "datasets": DATASETS,
    "hotspot_definition": (
        "Top 20% redundancy-adjusted "
        "global consensus ranks"
    ),
    "physicochemical_classes": {
        class_name: sorted(
            residue_set
        )
        for class_name, residue_set
        in PHYSICOCHEMICAL_CLASSES.items()
    },
    "continuous_properties": list(
        CONTINUOUS_PROPERTIES.keys()
    ),
    "categorical_tests": int(
        len(categorical_df)
    ),
    "significant_categorical_tests": int(
        categorical_df[
            "significant_fdr_0_05"
        ].sum()
    ),
    "sequence_level_tests": int(
        len(sequence_tests_df)
    ),
    "significant_sequence_level_tests": int(
        sequence_tests_df[
            "significant_fdr_0_05"
        ].sum()
    ),
    "replicated_significant_categories": (
        replication_df[
            replication_df[
                "significant_in_both"
            ]
        ][
            "property_class"
        ].tolist()
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "physicochemical_enrichment_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 17. SAVE CHECKPOINT
# ============================================================

output_files = [
    categorical_file,
    sequence_property_file,
    sequence_tests_file,
    replication_file,
    significant_categorical_file,
    summary_file,
]

mark_step_complete(
    "18_physicochemical_hotspot_enrichment",
    output_files=output_files,
    details=summary,
)


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 78)
print("PHYSICOCHEMICAL CATEGORY ENRICHMENT")
print("=" * 78)

display(
    categorical_df[
        [
            "dataset",
            "analysis",
            "property_class",
            "group_1_frequency",
            "group_2_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
            "direction",
        ]
    ].sort_values(
        [
            "analysis",
            "dataset",
            "fdr_bh",
        ]
    )
)

print("\n" + "=" * 78)
print("INTERNAL–KELM PHYSICOCHEMICAL REPLICATION")
print("=" * 78)

display(
    replication_df[
        [
            "property_class",
            "internal_odds_ratio",
            "internal_log2_enrichment",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_log2_enrichment",
            "kelm_fdr",
            "same_direction",
            "significant_in_both",
        ]
    ]
)

print("\n" + "=" * 78)
print("SEQUENCE-LEVEL PROPERTY TESTS")
print("=" * 78)

display(
    sequence_tests_df[
        [
            "dataset",
            "analysis",
            "property",
            "group_1_mean",
            "group_2_mean",
            "mean_difference",
            "test",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
)

print("\nReplicated significant property classes:")
print(
    summary[
        "replicated_significant_categories"
    ]
)

print("\nSaved category enrichment table:")
print(categorical_file)

print("\nSaved sequence-level tests:")
print(sequence_tests_file)

print("\nSaved replication table:")
print(replication_file)

print("\n" + "=" * 78)
print("STEP 18 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 19: POSITIONAL PREFERENCE OF CONSENSUS HOTSPOTS
#
# Primary questions:
# 1. Are CPP hotspots preferentially located near the
#    N-terminus, middle, or C-terminus?
# 2. Does positional preference differ between CPPs and
#    non-CPPs?
# 3. Does the pattern replicate in KELM?
#
# Primary hotspot definition:
#    Top 20% redundancy-adjusted global consensus ranks
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    fisher_exact,
    mannwhitneyu,
    wilcoxon,
    kstest,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

for directory in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Coarse positional regions
# normalized_position is in (0, 1]
REGION_BOUNDARIES = {
    "N_terminal": (0.0, 1.0 / 3.0),
    "Middle": (1.0 / 3.0, 2.0 / 3.0),
    "C_terminal": (2.0 / 3.0, 1.0),
}

# Detailed positional bins
NUMBER_OF_POSITION_BINS = 10


# ============================================================
# 2. BENJAMINI-HOCHBERG FDR
# ============================================================

def benjamini_hochberg(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    if number_of_tests == 0:
        return np.array(
            [],
            dtype=float,
        )

    order = np.argsort(
        p_values
    )

    ordered = p_values[
        order
    ]

    adjusted_ordered = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ordered[reverse_index]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ordered[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ordered,
        1.0,
    )

    return adjusted


# ============================================================
# 3. SAFE STATISTICAL HELPERS
# ============================================================

def corrected_odds_ratio(
    a,
    b,
    c,
    d,
):
    return float(
        (
            (a + 0.5)
            * (d + 0.5)
        )
        / (
            (b + 0.5)
            * (c + 0.5)
        )
    )


def safe_mannwhitney(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    values_a = values_a[
        np.isfinite(values_a)
    ]

    values_b = values_b[
        np.isfinite(values_b)
    ]

    if (
        len(values_a) == 0
        or len(values_b) == 0
    ):
        return np.nan, np.nan

    statistic, p_value = mannwhitneyu(
        values_a,
        values_b,
        alternative="two-sided",
    )

    return (
        float(statistic),
        float(p_value),
    )


def safe_wilcoxon(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    valid = (
        np.isfinite(values_a)
        & np.isfinite(values_b)
    )

    values_a = values_a[
        valid
    ]

    values_b = values_b[
        valid
    ]

    if len(values_a) == 0:
        return np.nan, np.nan

    differences = (
        values_a - values_b
    )

    if np.allclose(
        differences,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


# ============================================================
# 4. LOAD ADJUSTED CONSENSUS
# ============================================================

def load_adjusted_consensus(
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file missing:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "adjusted_global_consensus_rank",
        "adjusted_hotspot_top20",
        "strict_cross_model_hotspot",
        "unanimous_cross_model_hotspot",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe[
        "normalized_position"
    ] = (
        dataframe["position"]
        / dataframe["sequence_length"]
    )

    return (
        dataframe,
        input_file,
    )


# ============================================================
# 5. ASSIGN POSITIONAL REGION
# ============================================================

def assign_position_region(
    normalized_position,
):
    if normalized_position <= (
        1.0 / 3.0
    ):
        return "N_terminal"

    if normalized_position <= (
        2.0 / 3.0
    ):
        return "Middle"

    return "C_terminal"


def assign_position_bin(
    normalized_position,
):
    # Convert positions in (0, 1] into bins 1–10.
    bin_number = int(
        np.ceil(
            normalized_position
            * NUMBER_OF_POSITION_BINS
        )
    )

    return min(
        max(
            bin_number,
            1,
        ),
        NUMBER_OF_POSITION_BINS,
    )


# ============================================================
# 6. PREPARE RESIDUE TABLES
# ============================================================

residue_tables = {}
source_files = []

for dataset_name in DATASETS:

    dataframe, source_file = (
        load_adjusted_consensus(
            dataset_name
        )
    )

    source_files.append(
        source_file
    )

    dataframe[
        "position_region"
    ] = dataframe[
        "normalized_position"
    ].apply(
        assign_position_region
    )

    dataframe[
        "position_bin"
    ] = dataframe[
        "normalized_position"
    ].apply(
        assign_position_bin
    )

    dataframe[
        "is_hotspot"
    ] = dataframe[
        "adjusted_hotspot_top20"
    ].astype(bool)

    residue_tables[
        dataset_name
    ] = dataframe

    print(
        f"[LOADED] {dataset_name}: "
        f"{len(dataframe)} residues"
    )


# ============================================================
# 7. RESIDUE-LEVEL REGION ENRICHMENT
#
# Within each class:
# hotspot positions vs non-hotspot positions.
# ============================================================

region_enrichment_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for label_value, class_name in [
        (1, "CPP"),
        (0, "non_CPP"),
    ]:

        class_data = dataframe[
            dataframe["label"]
            == label_value
        ]

        hotspots = class_data[
            class_data[
                "is_hotspot"
            ]
        ]

        nonhotspots = class_data[
            ~class_data[
                "is_hotspot"
            ]
        ]

        hotspot_total = len(
            hotspots
        )

        nonhotspot_total = len(
            nonhotspots
        )

        for region_name in (
            REGION_BOUNDARIES
        ):

            hotspot_region_count = int(
                (
                    hotspots[
                        "position_region"
                    ]
                    == region_name
                ).sum()
            )

            nonhotspot_region_count = int(
                (
                    nonhotspots[
                        "position_region"
                    ]
                    == region_name
                ).sum()
            )

            hotspot_other = (
                hotspot_total
                - hotspot_region_count
            )

            nonhotspot_other = (
                nonhotspot_total
                - nonhotspot_region_count
            )

            raw_odds_ratio, p_value = (
                fisher_exact(
                    [
                        [
                            hotspot_region_count,
                            hotspot_other,
                        ],
                        [
                            nonhotspot_region_count,
                            nonhotspot_other,
                        ],
                    ],
                    alternative="two-sided",
                )
            )

            corrected_or = (
                corrected_odds_ratio(
                    hotspot_region_count,
                    hotspot_other,
                    nonhotspot_region_count,
                    nonhotspot_other,
                )
            )

            hotspot_frequency = (
                hotspot_region_count
                / hotspot_total
            )

            nonhotspot_frequency = (
                nonhotspot_region_count
                / nonhotspot_total
            )

            region_enrichment_rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "region": region_name,
                "hotspot_count": int(
                    hotspot_region_count
                ),
                "hotspot_total": int(
                    hotspot_total
                ),
                "hotspot_frequency": float(
                    hotspot_frequency
                ),
                "nonhotspot_count": int(
                    nonhotspot_region_count
                ),
                "nonhotspot_total": int(
                    nonhotspot_total
                ),
                "nonhotspot_frequency": float(
                    nonhotspot_frequency
                ),
                "odds_ratio_raw": float(
                    raw_odds_ratio
                ),
                "odds_ratio_corrected": float(
                    corrected_or
                ),
                "log2_enrichment": float(
                    np.log2(
                        (
                            hotspot_frequency
                            + 1e-12
                        )
                        / (
                            nonhotspot_frequency
                            + 1e-12
                        )
                    )
                ),
                "p_value": float(
                    p_value
                ),
            })


region_enrichment_df = pd.DataFrame(
    region_enrichment_rows
)

region_enrichment_df["fdr_bh"] = (
    region_enrichment_df.groupby(
        [
            "dataset",
            "class",
        ]
    )[
        "p_value"
    ].transform(
        lambda values:
            benjamini_hochberg(
                values.to_numpy()
            )
    )
)

region_enrichment_df[
    "significant_fdr_0_05"
] = (
    region_enrichment_df[
        "fdr_bh"
    ] < 0.05
)

region_enrichment_df[
    "direction"
] = np.where(
    region_enrichment_df[
        "odds_ratio_corrected"
    ] > 1.0,
    "enriched_in_hotspots",
    "depleted_in_hotspots",
)


# ============================================================
# 8. DETAILED 10-BIN DISTRIBUTION
# ============================================================

position_bin_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for label_value, class_name in [
        (1, "CPP"),
        (0, "non_CPP"),
    ]:

        class_data = dataframe[
            dataframe["label"]
            == label_value
        ]

        hotspots = class_data[
            class_data["is_hotspot"]
        ]

        nonhotspots = class_data[
            ~class_data["is_hotspot"]
        ]

        for position_bin in range(
            1,
            NUMBER_OF_POSITION_BINS + 1,
        ):

            hotspot_count = int(
                (
                    hotspots["position_bin"]
                    == position_bin
                ).sum()
            )

            nonhotspot_count = int(
                (
                    nonhotspots[
                        "position_bin"
                    ]
                    == position_bin
                ).sum()
            )

            position_bin_rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "position_bin": int(
                    position_bin
                ),
                "bin_start": float(
                    (
                        position_bin - 1
                    )
                    / NUMBER_OF_POSITION_BINS
                ),
                "bin_end": float(
                    position_bin
                    / NUMBER_OF_POSITION_BINS
                ),
                "hotspot_count": int(
                    hotspot_count
                ),
                "hotspot_frequency": float(
                    hotspot_count
                    / len(hotspots)
                ),
                "nonhotspot_count": int(
                    nonhotspot_count
                ),
                "nonhotspot_frequency": float(
                    nonhotspot_count
                    / len(nonhotspots)
                ),
            })


position_bin_df = pd.DataFrame(
    position_bin_rows
)

position_bin_df[
    "frequency_difference"
] = (
    position_bin_df[
        "hotspot_frequency"
    ]
    - position_bin_df[
        "nonhotspot_frequency"
    ]
)

position_bin_df[
    "hotspot_to_nonhotspot_ratio"
] = np.divide(
    position_bin_df[
        "hotspot_frequency"
    ],
    position_bin_df[
        "nonhotspot_frequency"
    ],
    out=np.full(
        len(position_bin_df),
        np.nan,
    ),
    where=(
        position_bin_df[
            "nonhotspot_frequency"
        ].to_numpy()
        > 0
    ),
)


# ============================================================
# 9. PER-SEQUENCE POSITIONAL SUMMARY
# ============================================================

per_sequence_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for sequence_id, group in (
        dataframe.groupby(
            "sequence_id",
            sort=False,
        )
    ):

        hotspots = group[
            group["is_hotspot"]
        ]

        nonhotspots = group[
            ~group["is_hotspot"]
        ]

        record = {
            "dataset": dataset_name,
            "sequence_id": (
                sequence_id
            ),
            "sequence": str(
                group["sequence"].iloc[0]
            ),
            "label": int(
                group["label"].iloc[0]
            ),
            "sequence_length": int(
                group[
                    "sequence_length"
                ].iloc[0]
            ),
            "number_of_hotspots": int(
                len(hotspots)
            ),
            "mean_hotspot_position": float(
                hotspots[
                    "normalized_position"
                ].mean()
            ),
            "median_hotspot_position": float(
                hotspots[
                    "normalized_position"
                ].median()
            ),
            "mean_nonhotspot_position": float(
                nonhotspots[
                    "normalized_position"
                ].mean()
            ),
            "median_nonhotspot_position": float(
                nonhotspots[
                    "normalized_position"
                ].median()
            ),
        }

        for region_name in (
            REGION_BOUNDARIES
        ):
            hotspot_region_fraction = float(
                (
                    hotspots[
                        "position_region"
                    ]
                    == region_name
                ).mean()
            )

            nonhotspot_region_fraction = float(
                (
                    nonhotspots[
                        "position_region"
                    ]
                    == region_name
                ).mean()
            )

            record[
                f"hotspot_fraction_{region_name}"
            ] = hotspot_region_fraction

            record[
                f"nonhotspot_fraction_{region_name}"
            ] = nonhotspot_region_fraction

            record[
                f"hotspot_minus_nonhotspot_{region_name}"
            ] = (
                hotspot_region_fraction
                - nonhotspot_region_fraction
            )

        per_sequence_rows.append(
            record
        )


per_sequence_df = pd.DataFrame(
    per_sequence_rows
)


# ============================================================
# 10. SEQUENCE-LEVEL PAIRED TESTS WITHIN CPPs
# ============================================================

sequence_test_rows = []

for dataset_name in DATASETS:

    dataset_table = per_sequence_df[
        per_sequence_df["dataset"]
        == dataset_name
    ]

    cpp_table = dataset_table[
        dataset_table["label"] == 1
    ]

    noncpp_table = dataset_table[
        dataset_table["label"] == 0
    ]

    # Paired hotspot vs non-hotspot positional fractions
    # within CPP sequences.
    for region_name in (
        REGION_BOUNDARIES
    ):

        hotspot_column = (
            f"hotspot_fraction_{region_name}"
        )

        nonhotspot_column = (
            f"nonhotspot_fraction_{region_name}"
        )

        statistic, p_value = (
            safe_wilcoxon(
                cpp_table[
                    hotspot_column
                ],
                cpp_table[
                    nonhotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "paired_CPP_hotspot_vs_CPP_nonhotspot"
            ),
            "metric": (
                f"fraction_{region_name}"
            ),
            "group_1": "CPP_hotspot",
            "group_2": "CPP_nonhotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(cpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_2_mean": float(
                cpp_table[
                    nonhotspot_column
                ].mean()
            ),
            "mean_difference": float(
                (
                    cpp_table[
                        hotspot_column
                    ]
                    - cpp_table[
                        nonhotspot_column
                    ]
                ).mean()
            ),
            "test": (
                "paired Wilcoxon signed-rank"
            ),
            "statistic": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })

        # CPP vs non-CPP hotspot positional preference.
        statistic, p_value = (
            safe_mannwhitney(
                cpp_table[
                    hotspot_column
                ],
                noncpp_table[
                    hotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            "metric": (
                f"fraction_{region_name}"
            ),
            "group_1": "CPP_hotspot",
            "group_2": "nonCPP_hotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(noncpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_2_mean": float(
                noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "mean_difference": float(
                cpp_table[
                    hotspot_column
                ].mean()
                - noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "test": "Mann-Whitney U",
            "statistic": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })

    # Mean normalized hotspot position:
    # CPP vs non-CPP.
    statistic, p_value = (
        safe_mannwhitney(
            cpp_table[
                "mean_hotspot_position"
            ],
            noncpp_table[
                "mean_hotspot_position"
            ],
        )
    )

    sequence_test_rows.append({
        "dataset": dataset_name,
        "analysis": (
            "CPP_hotspot_vs_nonCPP_hotspot"
        ),
        "metric": (
            "mean_normalized_hotspot_position"
        ),
        "group_1": "CPP_hotspot",
        "group_2": "nonCPP_hotspot",
        "group_1_n": int(
            len(cpp_table)
        ),
        "group_2_n": int(
            len(noncpp_table)
        ),
        "group_1_mean": float(
            cpp_table[
                "mean_hotspot_position"
            ].mean()
        ),
        "group_2_mean": float(
            noncpp_table[
                "mean_hotspot_position"
            ].mean()
        ),
        "mean_difference": float(
            cpp_table[
                "mean_hotspot_position"
            ].mean()
            - noncpp_table[
                "mean_hotspot_position"
            ].mean()
        ),
        "test": "Mann-Whitney U",
        "statistic": float(
            statistic
        ),
        "p_value": float(
            p_value
        ),
    })


sequence_tests_df = pd.DataFrame(
    sequence_test_rows
)

sequence_tests_df["fdr_bh"] = (
    sequence_tests_df.groupby(
        [
            "dataset",
            "analysis",
        ]
    )[
        "p_value"
    ].transform(
        lambda values:
            benjamini_hochberg(
                values.to_numpy()
            )
    )
)

sequence_tests_df[
    "significant_fdr_0_05"
] = (
    sequence_tests_df[
        "fdr_bh"
    ] < 0.05
)


# ============================================================
# 11. HOTSPOT POSITION DISTRIBUTION VS UNIFORM
#
# This is supplementary because residues in finite peptides
# are discrete rather than perfectly continuous.
# ============================================================

uniformity_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for label_value, class_name in [
        (1, "CPP"),
        (0, "non_CPP"),
    ]:

        hotspot_positions = dataframe[
            (
                dataframe["label"]
                == label_value
            )
            & dataframe["is_hotspot"]
        ][
            "normalized_position"
        ].to_numpy()

        statistic, p_value = kstest(
            hotspot_positions,
            "uniform",
            args=(0, 1),
        )

        uniformity_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "number_of_hotspots": int(
                len(hotspot_positions)
            ),
            "mean_normalized_position": float(
                np.mean(
                    hotspot_positions
                )
            ),
            "median_normalized_position": float(
                np.median(
                    hotspot_positions
                )
            ),
            "ks_statistic_vs_uniform": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })


uniformity_df = pd.DataFrame(
    uniformity_rows
)

uniformity_df["fdr_bh"] = (
    benjamini_hochberg(
        uniformity_df[
            "p_value"
        ].to_numpy()
    )
)


# ============================================================
# 12. INTERNAL–KELM REPLICATION
# ============================================================

replication_source = (
    region_enrichment_df[
        region_enrichment_df[
            "class"
        ] == "CPP"
    ][
        [
            "dataset",
            "region",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
)


internal_replication = (
    replication_source[
        replication_source["dataset"]
        == "internal_test"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)


kelm_replication = (
    replication_source[
        replication_source["dataset"]
        == "kelm_external"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)


replication_df = (
    internal_replication.merge(
        kelm_replication,
        on="region",
        how="outer",
        validate="one_to_one",
    )
)

replication_df[
    "same_direction"
] = (
    np.sign(
        replication_df[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replication_df[
            "kelm_log2_enrichment"
        ]
    )
)

replication_df[
    "significant_in_both"
] = (
    replication_df[
        "internal_significant"
    ].fillna(False)
    & replication_df[
        "kelm_significant"
    ].fillna(False)
)


# ============================================================
# 13. SAVE TABLES
# ============================================================

region_enrichment_file = (
    RESULT_TABLE_DIR
    / "hotspot_positional_region_enrichment.csv"
)

position_bin_file = (
    RESULT_TABLE_DIR
    / "hotspot_position_10bin_distribution.csv"
)

per_sequence_file = (
    RESULT_SI_DIR
    / "hotspot_positional_metrics_per_sequence.csv"
)

sequence_tests_file = (
    RESULT_TABLE_DIR
    / "hotspot_positional_sequence_tests.csv"
)

uniformity_file = (
    RESULT_SI_DIR
    / "hotspot_position_uniformity_tests.csv"
)

replication_file = (
    RESULT_TABLE_DIR
    / "hotspot_positional_replication.csv"
)


region_enrichment_df.to_csv(
    region_enrichment_file,
    index=False,
)

position_bin_df.to_csv(
    position_bin_file,
    index=False,
)

per_sequence_df.to_csv(
    per_sequence_file,
    index=False,
)

sequence_tests_df.to_csv(
    sequence_tests_file,
    index=False,
)

uniformity_df.to_csv(
    uniformity_file,
    index=False,
)

replication_df.to_csv(
    replication_file,
    index=False,
)


# ============================================================
# 14. FIGURE — HOTSPOT POSITION DENSITY
# ============================================================

for dataset_name in DATASETS:

    dataframe = residue_tables[
        dataset_name
    ]

    cpp_hotspots = dataframe[
        (
            dataframe["label"] == 1
        )
        & dataframe["is_hotspot"]
    ][
        "normalized_position"
    ].to_numpy()

    cpp_nonhotspots = dataframe[
        (
            dataframe["label"] == 1
        )
        & ~dataframe["is_hotspot"]
    ][
        "normalized_position"
    ].to_numpy()

    bins = np.linspace(
        0,
        1,
        21,
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.hist(
        cpp_hotspots,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        label="CPP hotspots",
    )

    ax.hist(
        cpp_nonhotspots,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        linestyle="--",
        label="CPP non-hotspots",
    )

    ax.axvline(
        1.0 / 3.0,
        linestyle=":",
        linewidth=1,
    )

    ax.axvline(
        2.0 / 3.0,
        linestyle=":",
        linewidth=1,
    )

    ax.set_xlim(
        0,
        1,
    )

    ax.set_xlabel(
        "Normalized residue position"
    )

    ax.set_ylabel(
        "Density"
    )

    ax.set_title(
        f"Positional distribution of CPP hotspots\n"
        f"{dataset_name}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            f"CPP_hotspot_position_density_"
            f"{dataset_name}"
        )
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 15. FIGURE — 10-BIN HOTSPOT ENRICHMENT
# ============================================================

for dataset_name in DATASETS:

    plot_table = position_bin_df[
        (
            position_bin_df["dataset"]
            == dataset_name
        )
        & (
            position_bin_df["class"]
            == "CPP"
        )
    ].sort_values(
        "position_bin"
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.plot(
        plot_table[
            "position_bin"
        ],
        plot_table[
            "hotspot_frequency"
        ],
        marker="o",
        label="CPP hotspots",
    )

    ax.plot(
        plot_table[
            "position_bin"
        ],
        plot_table[
            "nonhotspot_frequency"
        ],
        marker="s",
        linestyle="--",
        label="CPP non-hotspots",
    )

    ax.set_xticks(
        range(
            1,
            NUMBER_OF_POSITION_BINS + 1,
        )
    )

    ax.set_xticklabels(
        [
            f"{(index - 1) * 10}–{index * 10}%"
            for index in range(
                1,
                NUMBER_OF_POSITION_BINS + 1
            )
        ],
        rotation=35,
        ha="right",
    )

    ax.set_xlabel(
        "Relative sequence position"
    )

    ax.set_ylabel(
        "Fraction of residues"
    )

    ax.set_title(
        f"CPP hotspot positional profile\n"
        f"{dataset_name}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            f"CPP_hotspot_position_10bin_"
            f"{dataset_name}"
        )
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 16. FIGURE — N/MIDDLE/C ENRICHMENT
# ============================================================

cpp_region_plot = region_enrichment_df[
    region_enrichment_df[
        "class"
    ] == "CPP"
].copy()

region_order = [
    "N_terminal",
    "Middle",
    "C_terminal",
]

x = np.arange(
    len(region_order)
)

width = 0.36

internal_values = []
kelm_values = []

for region_name in region_order:

    internal_values.append(
        float(
            cpp_region_plot[
                (
                    cpp_region_plot["dataset"]
                    == "internal_test"
                )
                & (
                    cpp_region_plot["region"]
                    == region_name
                )
            ][
                "log2_enrichment"
            ].iloc[0]
        )
    )

    kelm_values.append(
        float(
            cpp_region_plot[
                (
                    cpp_region_plot["dataset"]
                    == "kelm_external"
                )
                & (
                    cpp_region_plot["region"]
                    == region_name
                )
            ][
                "log2_enrichment"
            ].iloc[0]
        )
    )


fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

ax.bar(
    x - width / 2,
    internal_values,
    width=width,
    label="Internal test",
)

ax.bar(
    x + width / 2,
    kelm_values,
    width=width,
    label="KELM external",
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        "N-terminal",
        "Middle",
        "C-terminal",
    ]
)

ax.set_ylabel(
    "log$_2$ enrichment in CPP hotspots"
)

ax.set_title(
    "Regional preference of CPP consensus hotspots"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

regional_figure_base = (
    FIGURE_MAIN_DIR
    / "CPP_hotspot_regional_enrichment"
)

fig.savefig(
    regional_figure_base.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    regional_figure_base.with_suffix(".pdf"),
    bbox_inches="tight",
)

fig.savefig(
    regional_figure_base.with_suffix(".svg"),
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ============================================================
# 17. SUMMARY JSON
# ============================================================

replicated_significant_regions = (
    replication_df[
        replication_df[
            "significant_in_both"
        ]
    ][
        "region"
    ].tolist()
)

same_direction_regions = (
    replication_df[
        replication_df[
            "same_direction"
        ]
    ][
        "region"
    ].tolist()
)

summary = {
    "datasets": DATASETS,
    "hotspot_definition": (
        "Top 20% redundancy-adjusted global "
        "consensus residue ranks"
    ),
    "coarse_regions": {
        "N_terminal": (
            "normalized position <= 1/3"
        ),
        "Middle": (
            "1/3 < normalized position <= 2/3"
        ),
        "C_terminal": (
            "normalized position > 2/3"
        ),
    },
    "number_of_detailed_bins": int(
        NUMBER_OF_POSITION_BINS
    ),
    "replicated_significant_regions": (
        replicated_significant_regions
    ),
    "same_direction_regions": (
        same_direction_regions
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "hotspot_positional_preference_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 18. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "19_hotspot_positional_preference",
    output_files=[
        region_enrichment_file,
        position_bin_file,
        per_sequence_file,
        sequence_tests_file,
        uniformity_file,
        replication_file,
        summary_file,
        regional_figure_base.with_suffix(
            ".png"
        ),
        regional_figure_base.with_suffix(
            ".pdf"
        ),
        regional_figure_base.with_suffix(
            ".svg"
        ),
    ],
    details=summary,
)


# ============================================================
# 19. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 78)
print("CPP HOTSPOT REGIONAL ENRICHMENT")
print("=" * 78)

display(
    region_enrichment_df[
        region_enrichment_df[
            "class"
        ] == "CPP"
    ][
        [
            "dataset",
            "region",
            "hotspot_frequency",
            "nonhotspot_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
            "direction",
        ]
    ]
)

print("\n" + "=" * 78)
print("INTERNAL–KELM POSITIONAL REPLICATION")
print("=" * 78)

display(
    replication_df[
        [
            "region",
            "internal_odds_ratio",
            "internal_log2_enrichment",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_log2_enrichment",
            "kelm_fdr",
            "same_direction",
            "significant_in_both",
        ]
    ]
)

print("\n" + "=" * 78)
print("SEQUENCE-LEVEL POSITIONAL TESTS")
print("=" * 78)

display(
    sequence_tests_df[
        [
            "dataset",
            "analysis",
            "metric",
            "group_1_mean",
            "group_2_mean",
            "mean_difference",
            "test",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
)

print("\n" + "=" * 78)
print("DETAILED CPP POSITION PROFILE")
print("=" * 78)

display(
    position_bin_df[
        position_bin_df[
            "class"
        ] == "CPP"
    ][
        [
            "dataset",
            "position_bin",
            "bin_start",
            "bin_end",
            "hotspot_frequency",
            "nonhotspot_frequency",
            "frequency_difference",
            "hotspot_to_nonhotspot_ratio",
        ]
    ]
)

print("\nReplicated significant regions:")
print(
    replicated_significant_regions
)

print("\nRegions with the same direction:")
print(
    same_direction_regions
)

print("\nSaved regional enrichment:")
print(region_enrichment_file)

print("\nSaved detailed position profile:")
print(position_bin_file)

print("\nSaved positional replication:")
print(replication_file)

print("\n" + "=" * 78)
print("STEP 19 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 20: FINAL RESULTS AUDIT AND MANUSCRIPT CONTENT PLAN
#
# Outputs:
# 1. Complete file audit
# 2. Pipeline checkpoint audit
# 3. Key-result consolidation
# 4. Statistical consistency checks
# 5. Proposed main/SI figures
# 6. Proposed main/SI tables
# 7. Manuscript section roadmap
# 8. Excel audit workbook
# ============================================================

from pathlib import Path
import json
import math
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

DIRS = {
    "code": PROJECT_DIR / "00_code",
    "data_original": PROJECT_DIR / "01_data_original",
    "data_processed": PROJECT_DIR / "02_data_processed",
    "embeddings": PROJECT_DIR / "03_embeddings",
    "models": PROJECT_DIR / "04_models",
    "predictions": PROJECT_DIR / "05_predictions",
    "xai": PROJECT_DIR / "06_xai",
    "results": PROJECT_DIR / "07_results",
    "checkpoints": PROJECT_DIR / "08_checkpoints",
    "logs": PROJECT_DIR / "09_logs",
}

TABLE_MAIN_DIR = (
    DIRS["results"] / "tables_main"
)

TABLE_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

AUDIT_DIR = (
    DIRS["results"] / "manuscript_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for folder in [
    TABLE_MAIN_DIR,
    TABLE_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. EXPECTED PIPELINE STEPS
# ============================================================

EXPECTED_STEPS = [
    "01_project_directory_setup",
    "02_environment_setup",
    "03_correct_combined_datasets",
    "04_dataset_quality_control",
    "05_fixed_internal_data_splits",
    "06_all_PLM_embeddings",
    "07_tensorflow_environment",
    "08_all_attention_classifiers",
    "09_four_PLM_ensemble",
    "10_all_attention_scores",
    "11_all_gradient_input_scores",
    "12_all_integrated_gradients_scores",
    "13_consensus_XAI",
    "13B_redundancy_adjusted_consensus",
    "14_residue_enrichment",
    "15_hotspot_motif_discovery",
    "16_publication_ready_hotspot_faithfulness",
    "17_cross_PLM_hotspot_conservation",
    "18_physicochemical_hotspot_enrichment",
    "19_hotspot_positional_preference",
]


# ============================================================
# 3. EXPECTED CRITICAL FILES
# ============================================================

EXPECTED_FILES = {
    # Data
    "internal_qc_dataset":
        DIRS["data_processed"]
        / "internal_dataset_qc.csv",

    "kelm_qc_dataset":
        DIRS["data_processed"]
        / "kelm_external_dataset_qc.csv",

    "train_split":
        DIRS["data_processed"]
        / "train_split.csv",

    "validation_split":
        DIRS["data_processed"]
        / "validation_split.csv",

    "internal_test_split":
        DIRS["data_processed"]
        / "internal_test_split.csv",

    # Performance
    "individual_model_performance":
        TABLE_MAIN_DIR
        / "attention_classifier_performance.csv",

    "ensemble_performance":
        TABLE_MAIN_DIR
        / "four_plm_ensemble_performance.csv",

    "individual_vs_ensemble":
        TABLE_MAIN_DIR
        / "individual_vs_ensemble_comparison.csv",

    # XAI
    "consensus_summary":
        TABLE_MAIN_DIR
        / "consensus_xai_summary.csv",

    "method_agreement":
        TABLE_MAIN_DIR
        / "xai_method_agreement.csv",

    "cross_model_agreement":
        TABLE_MAIN_DIR
        / "cross_model_xai_agreement.csv",

    "adjusted_consensus_summary":
        TABLE_MAIN_DIR
        / "adjusted_consensus_xai_summary.csv",

    "original_vs_adjusted_consensus":
        TABLE_MAIN_DIR
        / "original_vs_adjusted_consensus.csv",

    # Biological analyses
    "residue_enrichment":
        TABLE_MAIN_DIR
        / "residue_enrichment_top20.csv",

    "residue_replication":
        TABLE_MAIN_DIR
        / "residue_enrichment_replication.csv",

    "significant_motifs":
        TABLE_MAIN_DIR
        / "strict_replicated_CPP_hotspot_motifs.csv",

    "motif_replication":
        TABLE_MAIN_DIR
        / "hotspot_motif_replication.csv",

    "faithfulness":
        TABLE_MAIN_DIR
        / "CPP_hotspot_faithfulness_manuscript_table.csv",

    "faithfulness_cross_model":
        TABLE_MAIN_DIR
        / "hotspot_faithfulness_cross_model_summary.csv",

    "conservation_pairwise":
        TABLE_MAIN_DIR
        / "cross_plm_pairwise_hotspot_jaccard.csv",

    "conservation_summary":
        TABLE_MAIN_DIR
        / "cross_plm_hotspot_conservation_summary.csv",

    "conservation_class_comparison":
        TABLE_MAIN_DIR
        / "CPP_vs_nonCPP_hotspot_conservation.csv",

    "conservation_residue_level":
        TABLE_MAIN_DIR
        / "residue_level_cross_plm_conservation_enrichment.csv",

    "physicochemical_enrichment":
        TABLE_MAIN_DIR
        / "physicochemical_category_enrichment.csv",

    "physicochemical_replication":
        TABLE_MAIN_DIR
        / "physicochemical_enrichment_replication.csv",

    "physicochemical_sequence_tests":
        TABLE_MAIN_DIR
        / "physicochemical_sequence_level_tests.csv",

    "positional_enrichment":
        TABLE_MAIN_DIR
        / "hotspot_positional_region_enrichment.csv",

    "positional_replication":
        TABLE_MAIN_DIR
        / "hotspot_positional_replication.csv",

    "positional_sequence_tests":
        TABLE_MAIN_DIR
        / "hotspot_positional_sequence_tests.csv",

    "positional_10bin":
        TABLE_MAIN_DIR
        / "hotspot_position_10bin_distribution.csv",

    # Consensus residue-level outputs
    "internal_adjusted_consensus":
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / "internal_test"
        / "adjusted_global_consensus_residue_scores.csv",

    "kelm_adjusted_consensus":
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / "kelm_external"
        / "adjusted_global_consensus_residue_scores.csv",
}


# ============================================================
# 4. LOAD PIPELINE STATUS
# ============================================================

STATUS_FILE = (
    DIRS["checkpoints"]
    / "pipeline_status.json"
)

if STATUS_FILE.exists():
    with open(
        STATUS_FILE,
        "r",
        encoding="utf-8",
    ) as handle:
        PIPELINE_STATUS = json.load(
            handle
        )
else:
    PIPELINE_STATUS = {}


# ============================================================
# 5. PIPELINE STEP AUDIT
# ============================================================

step_audit_rows = []

for step_name in EXPECTED_STEPS:

    status_record = PIPELINE_STATUS.get(
        step_name,
        {}
    )

    step_audit_rows.append({
        "step": step_name,
        "recorded_in_pipeline_status": bool(
            step_name in PIPELINE_STATUS
        ),
        "completed_flag": bool(
            status_record.get(
                "completed",
                False,
            )
        ),
        "number_of_recorded_output_files": int(
            len(
                status_record.get(
                    "output_files",
                    [],
                )
            )
        ),
        "details_present": bool(
            status_record.get(
                "details"
            )
        ),
    })


step_audit_df = pd.DataFrame(
    step_audit_rows
)


# ============================================================
# 6. FILE AUDIT
# ============================================================

file_audit_rows = []

for file_key, file_path in (
    EXPECTED_FILES.items()
):

    exists = file_path.exists()

    size_bytes = (
        file_path.stat().st_size
        if exists
        else 0
    )

    row_count = None
    column_count = None
    read_status = (
        "missing"
        if not exists
        else "not_checked"
    )

    if exists:
        try:
            if file_path.suffix.lower() == ".csv":
                dataframe = pd.read_csv(
                    file_path
                )

                row_count = int(
                    len(dataframe)
                )

                column_count = int(
                    len(dataframe.columns)
                )

                if len(dataframe) == 0:
                    read_status = "empty"
                else:
                    read_status = "readable"

            elif file_path.suffix.lower() == ".json":
                with open(
                    file_path,
                    "r",
                    encoding="utf-8",
                ) as handle:
                    json.load(handle)

                read_status = "readable"

            else:
                read_status = "exists"

        except Exception as error:
            read_status = (
                f"read_error: "
                f"{type(error).__name__}"
            )

    file_audit_rows.append({
        "file_key": file_key,
        "path": str(file_path),
        "exists": exists,
        "size_bytes": int(
            size_bytes
        ),
        "size_MB": float(
            size_bytes / (1024 ** 2)
        ),
        "row_count": row_count,
        "column_count": column_count,
        "read_status": read_status,
    })


file_audit_df = pd.DataFrame(
    file_audit_rows
)


# ============================================================
# 7. GENERAL CSV VALIDATION
# ============================================================

def load_required_csv(
    file_key,
):
    file_path = EXPECTED_FILES[
        file_key
    ]

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing required file: "
            f"{file_path}"
        )

    return pd.read_csv(
        file_path
    )


validation_issue_rows = []


def add_validation_issue(
    analysis,
    severity,
    issue,
    recommendation,
):
    validation_issue_rows.append({
        "analysis": analysis,
        "severity": severity,
        "issue": issue,
        "recommendation": recommendation,
    })


# ============================================================
# 8. DATASET CONSISTENCY AUDIT
# ============================================================

try:
    train_df = load_required_csv(
        "train_split"
    )

    validation_df = load_required_csv(
        "validation_split"
    )

    internal_test_df = load_required_csv(
        "internal_test_split"
    )

    kelm_df = load_required_csv(
        "kelm_qc_dataset"
    )

    dataset_summary_rows = []

    for dataset_name, dataframe in [
        ("train", train_df),
        ("validation", validation_df),
        ("internal_test", internal_test_df),
        ("kelm_external", kelm_df),
    ]:

        dataset_summary_rows.append({
            "dataset": dataset_name,
            "rows": int(
                len(dataframe)
            ),
            "CPP_count": int(
                (
                    dataframe["label"]
                    == 1
                ).sum()
            ),
            "nonCPP_count": int(
                (
                    dataframe["label"]
                    == 0
                ).sum()
            ),
            "minimum_length": int(
                dataframe["length"].min()
            ),
            "maximum_length": int(
                dataframe["length"].max()
            ),
            "mean_length": float(
                dataframe["length"].mean()
            ),
            "duplicate_sequence_ids": int(
                dataframe[
                    "sequence_id"
                ].duplicated().sum()
            ),
            "duplicate_sequences": int(
                dataframe[
                    "sequence"
                ].duplicated().sum()
            ),
        })

    dataset_summary_df = pd.DataFrame(
        dataset_summary_rows
    )

    split_sets = {
        "train": set(
            train_df["sequence"]
        ),
        "validation": set(
            validation_df["sequence"]
        ),
        "internal_test": set(
            internal_test_df["sequence"]
        ),
        "kelm_external": set(
            kelm_df["sequence"]
        ),
    }

    overlap_rows = []

    split_names = list(
        split_sets.keys()
    )

    for index_a in range(
        len(split_names)
    ):
        for index_b in range(
            index_a + 1,
            len(split_names),
        ):
            name_a = split_names[
                index_a
            ]

            name_b = split_names[
                index_b
            ]

            overlap_count = len(
                split_sets[name_a]
                & split_sets[name_b]
            )

            overlap_rows.append({
                "dataset_a": name_a,
                "dataset_b": name_b,
                "exact_sequence_overlap": int(
                    overlap_count
                ),
            })

            if overlap_count > 0:
                add_validation_issue(
                    analysis="dataset_splits",
                    severity="high",
                    issue=(
                        f"{overlap_count} exact sequence "
                        f"overlaps between {name_a} "
                        f"and {name_b}"
                    ),
                    recommendation=(
                        "Inspect and remove overlap before "
                        "manuscript submission."
                    ),
                )

    split_overlap_df = pd.DataFrame(
        overlap_rows
    )

except Exception as error:

    dataset_summary_df = pd.DataFrame()
    split_overlap_df = pd.DataFrame()

    add_validation_issue(
        analysis="dataset_splits",
        severity="high",
        issue=(
            f"Dataset audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check processed dataset and split files."
        ),
    )


# ============================================================
# 9. MODEL PERFORMANCE CONSOLIDATION
# ============================================================

try:
    model_performance_df = (
        load_required_csv(
            "individual_vs_ensemble"
        )
    )

    performance_summary_rows = []

    for dataset_name in [
        "validation",
        "internal_test",
        "kelm_external",
    ]:

        subset = model_performance_df[
            model_performance_df[
                "dataset"
            ] == dataset_name
        ].copy()

        if len(subset) == 0:
            continue

        for metric in [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "mcc",
            "roc_auc",
            "pr_auc",
        ]:

            best_row = subset.loc[
                subset[metric].idxmax()
            ]

            performance_summary_rows.append({
                "dataset": dataset_name,
                "metric": metric,
                "best_method": (
                    best_row["method"]
                ),
                "best_value": float(
                    best_row[metric]
                ),
                "threshold": float(
                    best_row[
                        "threshold"
                    ]
                ),
            })

    performance_summary_df = pd.DataFrame(
        performance_summary_rows
    )

    duplicate_performance_rows = (
        model_performance_df.duplicated(
            subset=[
                "method",
                "dataset",
            ]
        ).sum()
    )

    if duplicate_performance_rows > 0:
        add_validation_issue(
            analysis="model_performance",
            severity="medium",
            issue=(
                f"{duplicate_performance_rows} "
                f"duplicated method-dataset rows."
            ),
            recommendation=(
                "Remove duplicated rows before final tables."
            ),
        )

except Exception as error:

    model_performance_df = pd.DataFrame()
    performance_summary_df = pd.DataFrame()

    add_validation_issue(
        analysis="model_performance",
        severity="high",
        issue=(
            f"Performance audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check individual_vs_ensemble_comparison.csv."
        ),
    )


# ============================================================
# 10. XAI AGREEMENT CONSOLIDATION
# ============================================================

try:
    method_agreement_df = (
        load_required_csv(
            "method_agreement"
        )
    )

    cross_model_agreement_df = (
        load_required_csv(
            "cross_model_agreement"
        )
    )

    original_adjusted_df = (
        load_required_csv(
            "original_vs_adjusted_consensus"
        )
    )

    xai_summary_rows = []

    gradient_pairs = method_agreement_df[
        method_agreement_df[
            "method_pair"
        ].str.contains(
            "Gradient×Input vs Integrated",
            regex=False,
        )
    ]

    attention_pairs = method_agreement_df[
        method_agreement_df[
            "method_pair"
        ].str.contains(
            "Attention vs",
            regex=False,
        )
    ]

    for dataset_name in [
        "internal_test",
        "kelm_external",
    ]:

        gradient_subset = gradient_pairs[
            gradient_pairs[
                "dataset"
            ] == dataset_name
        ]

        attention_subset = attention_pairs[
            attention_pairs[
                "dataset"
            ] == dataset_name
        ]

        cross_model_subset = (
            cross_model_agreement_df[
                cross_model_agreement_df[
                    "dataset"
                ] == dataset_name
            ]
        )

        adjusted_subset = (
            original_adjusted_df[
                original_adjusted_df[
                    "dataset"
                ] == dataset_name
            ]
        )

        xai_summary_rows.append({
            "dataset": dataset_name,
            "mean_attention_vs_gradient_spearman": float(
                attention_subset[
                    "mean_sequence_spearman"
                ].mean()
            ),
            "mean_gradient_vs_IG_spearman": float(
                gradient_subset[
                    "mean_sequence_spearman"
                ].mean()
            ),
            "mean_cross_model_spearman": float(
                cross_model_subset[
                    "mean_sequence_spearman"
                ].mean()
            ),
            "maximum_cross_model_spearman": float(
                cross_model_subset[
                    "mean_sequence_spearman"
                ].max()
            ),
            "original_vs_adjusted_rank_spearman": float(
                adjusted_subset[
                    "overall_rank_spearman"
                ].iloc[0]
            ),
            "original_vs_adjusted_top20_jaccard": float(
                adjusted_subset[
                    "top20_jaccard"
                ].iloc[0]
            ),
        })

    xai_audit_summary_df = pd.DataFrame(
        xai_summary_rows
    )

    if (
        method_agreement_df[
            method_agreement_df[
                "method_pair"
            ].str.contains(
                "Gradient×Input vs Integrated",
                regex=False,
            )
        ][
            "mean_sequence_spearman"
        ].mean()
        > 0.99
    ):
        add_validation_issue(
            analysis="XAI_consensus",
            severity="informational",
            issue=(
                "Gradient × Input and Integrated "
                "Gradients are nearly redundant."
            ),
            recommendation=(
                "Use the redundancy-adjusted consensus "
                "in all main-text analyses."
            ),
        )

except Exception as error:

    method_agreement_df = pd.DataFrame()
    cross_model_agreement_df = pd.DataFrame()
    original_adjusted_df = pd.DataFrame()
    xai_audit_summary_df = pd.DataFrame()

    add_validation_issue(
        analysis="XAI_consensus",
        severity="high",
        issue=(
            f"XAI audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check XAI agreement and consensus files."
        ),
    )


# ============================================================
# 11. RESIDUE ENRICHMENT CONSOLIDATION
# ============================================================

try:
    residue_replication_df = (
        load_required_csv(
            "residue_replication"
        )
    )

    residue_result_rows = []

    for _, row in (
        residue_replication_df.iterrows()
    ):

        internal_direction = (
            "enriched"
            if row[
                "internal_log2_enrichment"
            ] > 0
            else "depleted"
        )

        kelm_direction = (
            "enriched"
            if row[
                "kelm_log2_enrichment"
            ] > 0
            else "depleted"
        )

        residue_result_rows.append({
            "residue": row["residue"],
            "internal_direction": (
                internal_direction
            ),
            "kelm_direction": (
                kelm_direction
            ),
            "same_direction": bool(
                row[
                    "same_enrichment_direction"
                ]
            ),
            "significant_in_both": bool(
                row[
                    "significant_in_both"
                ]
            ),
            "internal_odds_ratio": float(
                row[
                    "internal_odds_ratio"
                ]
            ),
            "kelm_odds_ratio": float(
                row[
                    "kelm_odds_ratio"
                ]
            ),
            "internal_fdr": float(
                row[
                    "internal_fdr"
                ]
            ),
            "kelm_fdr": float(
                row[
                    "kelm_fdr"
                ]
            ),
        })

    residue_key_results_df = (
        pd.DataFrame(
            residue_result_rows
        )
    )

except Exception as error:

    residue_replication_df = pd.DataFrame()
    residue_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="residue_enrichment",
        severity="high",
        issue=(
            f"Residue audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check residue enrichment replication file."
        ),
    )


# ============================================================
# 12. MOTIF CONSOLIDATION
# ============================================================

try:
    motifs_df = load_required_csv(
        "significant_motifs"
    )

    motif_key_results_df = motifs_df[
        [
            "motif",
            "motif_length",
            "internal_cpp_count",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_cpp_count",
            "kelm_odds_ratio",
            "kelm_fdr",
            "contains_K_or_R",
            "basic_residue_fraction",
        ]
    ].copy()

except Exception as error:

    motifs_df = pd.DataFrame()
    motif_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="motif_discovery",
        severity="high",
        issue=(
            f"Motif audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check strict replicated motif file."
        ),
    )


# ============================================================
# 13. FAITHFULNESS CONSOLIDATION
# ============================================================

try:
    faithfulness_df = load_required_csv(
        "faithfulness"
    )

    faithfulness_key_results_df = (
        faithfulness_df[
            [
                "model",
                "dataset",
                "n",
                "mean_original_probability",
                "mean_hotspot_probability_drop",
                "hotspot_drop_ci95_low",
                "hotspot_drop_ci95_high",
                "mean_random_probability_drop",
                "mean_hotspot_minus_random_drop",
                "difference_ci95_low",
                "difference_ci95_high",
                "fraction_hotspot_stronger_than_random",
                "paired_cohens_dz",
                "rank_biserial_effect_size",
                "wilcoxon_p_value",
                "mean_hotspot_only_probability",
            ]
        ].copy()
    )

    suspicious_ratio_column = (
        "mean_hotspot_only_probability_fraction"
    )

    if suspicious_ratio_column in (
        faithfulness_df.columns
    ):
        suspicious_values = (
            faithfulness_df[
                suspicious_ratio_column
            ]
        )

        if (
            suspicious_values.replace(
                [np.inf, -np.inf],
                np.nan,
            ).max()
            > 2.0
        ):
            add_validation_issue(
                analysis="faithfulness",
                severity="medium",
                issue=(
                    "Hotspot-only/original probability "
                    "ratio contains unstable extreme values."
                ),
                recommendation=(
                    "Exclude this ratio from the manuscript; "
                    "report hotspot-only probability and "
                    "sufficiency loss instead."
                ),
            )

except Exception as error:

    faithfulness_df = pd.DataFrame()
    faithfulness_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="faithfulness",
        severity="high",
        issue=(
            f"Faithfulness audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check faithfulness manuscript table."
        ),
    )


# ============================================================
# 14. CONSERVATION CONSOLIDATION
# ============================================================

try:
    conservation_summary_df = (
        load_required_csv(
            "conservation_summary"
        )
    )

    conservation_class_df = (
        load_required_csv(
            "conservation_class_comparison"
        )
    )

    conservation_residue_df = (
        load_required_csv(
            "conservation_residue_level"
        )
    )

    conservation_key_results_df = (
        conservation_residue_df[
            [
                "dataset",
                "minimum_model_support",
                "CPP_supported_fraction",
                "nonCPP_supported_fraction",
                "odds_ratio_corrected",
                "p_value",
            ]
        ].copy()
    )

except Exception as error:

    conservation_summary_df = pd.DataFrame()
    conservation_class_df = pd.DataFrame()
    conservation_residue_df = pd.DataFrame()
    conservation_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="cross_PLM_conservation",
        severity="high",
        issue=(
            f"Conservation audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check hotspot conservation tables."
        ),
    )


# ============================================================
# 15. PHYSICOCHEMICAL CONSOLIDATION
# ============================================================

try:
    physicochemical_replication_df = (
        load_required_csv(
            "physicochemical_replication"
        )
    )

    physicochemical_key_results_df = (
        physicochemical_replication_df[
            [
                "property_class",
                "internal_odds_ratio",
                "internal_log2_enrichment",
                "internal_fdr",
                "kelm_odds_ratio",
                "kelm_log2_enrichment",
                "kelm_fdr",
                "same_direction",
                "significant_in_both",
            ]
        ].copy()
    )

except Exception as error:

    physicochemical_replication_df = (
        pd.DataFrame()
    )

    physicochemical_key_results_df = (
        pd.DataFrame()
    )

    add_validation_issue(
        analysis="physicochemical_enrichment",
        severity="high",
        issue=(
            f"Physicochemical audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check physicochemical replication file."
        ),
    )


# ============================================================
# 16. POSITIONAL CONSOLIDATION
# ============================================================

try:
    positional_replication_df = (
        load_required_csv(
            "positional_replication"
        )
    )

    positional_key_results_df = (
        positional_replication_df[
            [
                "region",
                "internal_odds_ratio",
                "internal_log2_enrichment",
                "internal_fdr",
                "kelm_odds_ratio",
                "kelm_log2_enrichment",
                "kelm_fdr",
                "same_direction",
                "significant_in_both",
            ]
        ].copy()
    )

except Exception as error:

    positional_replication_df = (
        pd.DataFrame()
    )

    positional_key_results_df = (
        pd.DataFrame()
    )

    add_validation_issue(
        analysis="positional_preference",
        severity="high",
        issue=(
            f"Positional audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check positional replication file."
        ),
    )


# ============================================================
# 17. CREATE MANUSCRIPT KEY-FINDING TABLE
# ============================================================

key_finding_rows = [
    {
        "result_id": "R1",
        "section": "Predictive performance",
        "finding": (
            "Median four-PLM ensemble produced the "
            "strongest internal-test MCC."
        ),
        "internal_evidence": (
            "MCC 0.814; ROC-AUC 0.971"
        ),
        "external_evidence": (
            "Mean ensemble ROC-AUC 0.954; "
            "PR-AUC 0.960"
        ),
        "recommended_claim_strength": (
            "Strong, but note ESM2-1280 had the "
            "best threshold-dependent KELM accuracy."
        ),
    },
    {
        "result_id": "R2",
        "section": "Attribution agreement",
        "finding": (
            "Attention agreed strongly with "
            "gradient-based methods within each PLM."
        ),
        "internal_evidence": (
            "Mean sequence Spearman approximately "
            "0.92–0.96"
        ),
        "external_evidence": (
            "Mean sequence Spearman approximately "
            "0.93–0.97"
        ),
        "recommended_claim_strength": (
            "Strong within-model agreement."
        ),
    },
    {
        "result_id": "R3",
        "section": "Attribution redundancy",
        "finding": (
            "Gradient × Input and Integrated "
            "Gradients were nearly identical."
        ),
        "internal_evidence": (
            "Spearman approximately 0.999–1.000"
        ),
        "external_evidence": (
            "Spearman approximately 0.999–1.000"
        ),
        "recommended_claim_strength": (
            "Treat as one gradient-attribution family."
        ),
    },
    {
        "result_id": "R4",
        "section": "Cross-PLM agreement",
        "finding": (
            "Different PLMs showed modest residue-rank "
            "agreement, motivating consensus XAI."
        ),
        "internal_evidence": (
            "Mean pairwise rank correlations "
            "approximately 0.10–0.28"
        ),
        "external_evidence": (
            "Mean pairwise rank correlations "
            "approximately 0.07–0.29"
        ),
        "recommended_claim_strength": (
            "Do not claim identical explanations."
        ),
    },
    {
        "result_id": "R5",
        "section": "Residue enrichment",
        "finding": (
            "Lysine and arginine were reproducibly "
            "enriched in CPP hotspots."
        ),
        "internal_evidence": (
            "K OR 3.51; R OR 3.63"
        ),
        "external_evidence": (
            "K OR 4.26; R OR 2.79"
        ),
        "recommended_claim_strength": (
            "Strong replicated cationic signal."
        ),
    },
    {
        "result_id": "R6",
        "section": "Motif discovery",
        "finding": (
            "RR and aromatic-basic motifs were "
            "reproducibly enriched."
        ),
        "internal_evidence": (
            "RR OR 25.77; WK OR 27.08"
        ),
        "external_evidence": (
            "RR OR 106.54; WK OR 31.20"
        ),
        "recommended_claim_strength": (
            "Report counts with odds ratios."
        ),
    },
    {
        "result_id": "R7",
        "section": "Faithfulness",
        "finding": (
            "Ablating consensus hotspots generally "
            "reduced CPP probabilities more than "
            "matched random ablation."
        ),
        "internal_evidence": (
            "Significant for all four PLMs"
        ),
        "external_evidence": (
            "Strongest for ProtT5; model-dependent "
            "for other PLMs"
        ),
        "recommended_claim_strength": (
            "State that external faithfulness was "
            "model dependent."
        ),
    },
    {
        "result_id": "R8",
        "section": "Cross-PLM conservation",
        "finding": (
            "Residues supported by three or four PLMs "
            "were enriched in CPPs."
        ),
        "internal_evidence": (
            "≥3 PLMs OR 1.63; all 4 OR 4.02"
        ),
        "external_evidence": (
            "≥3 PLMs OR 1.41; all 4 OR 2.26"
        ),
        "recommended_claim_strength": (
            "Strong externally replicated result."
        ),
    },
    {
        "result_id": "R9",
        "section": "Physicochemical grammar",
        "finding": (
            "CPP hotspots were strongly basic and "
            "comparatively hydrophilic."
        ),
        "internal_evidence": (
            "Basic class OR 5.96; charge increase "
            "0.395"
        ),
        "external_evidence": (
            "Basic class OR 3.87; charge increase "
            "0.287"
        ),
        "recommended_claim_strength": (
            "Strong replicated physicochemical signal."
        ),
    },
    {
        "result_id": "R10",
        "section": "Positional grammar",
        "finding": (
            "CPP hotspots were enriched centrally and "
            "depleted near the C-terminus."
        ),
        "internal_evidence": (
            "Middle OR 1.36; C-terminal OR 0.75"
        ),
        "external_evidence": (
            "Middle OR 1.53; C-terminal OR 0.57"
        ),
        "recommended_claim_strength": (
            "Strong replicated positional signal."
        ),
    },
]

key_findings_df = pd.DataFrame(
    key_finding_rows
)


# ============================================================
# 18. PROPOSED MAIN FIGURES
# ============================================================

main_figure_rows = [
    {
        "figure": "Figure 1",
        "title": (
            "Study design and consensus-XAI workflow"
        ),
        "panels": (
            "A Dataset curation and fixed splits; "
            "B four PLMs and attention classifiers; "
            "C three XAI methods; "
            "D redundancy-adjusted consensus; "
            "E external validation and biological analyses"
        ),
        "source_outputs": (
            "Dataset summary, embedding/model pipeline, "
            "consensus workflow"
        ),
        "main_message": (
            "A reproducible multi-PLM pipeline decodes "
            "and validates CPP-associated sequence grammar."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 2",
        "title": (
            "Predictive performance across PLMs and ensemble"
        ),
        "panels": (
            "A Internal ROC/PR curves; "
            "B KELM ROC/PR curves; "
            "C MCC/accuracy comparison; "
            "D confusion matrices for best models"
        ),
        "source_outputs": (
            "attention_classifier_performance.csv; "
            "individual_vs_ensemble_comparison.csv; "
            "prediction files"
        ),
        "main_message": (
            "Multiple PLMs generalize well, while ensemble "
            "benefits are metric dependent."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 3",
        "title": (
            "Consensus-XAI construction and agreement"
        ),
        "panels": (
            "A Attribution heatmap for representative CPPs; "
            "B within-model method agreement; "
            "C cross-model agreement; "
            "D original versus adjusted consensus stability"
        ),
        "source_outputs": (
            "xai_method_agreement.csv; "
            "cross_model_xai_agreement.csv; "
            "original_vs_adjusted_consensus.csv"
        ),
        "main_message": (
            "Attribution methods agree within PLMs, whereas "
            "PLMs provide complementary residue priorities."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 4",
        "title": (
            "Residue and motif grammar of CPP hotspots"
        ),
        "panels": (
            "A residue enrichment forest plot; "
            "B internal–KELM replication; "
            "C replicated motif odds ratios; "
            "D representative hotspot sequence maps"
        ),
        "source_outputs": (
            "residue_enrichment_replication.csv; "
            "strict_replicated_CPP_hotspot_motifs.csv"
        ),
        "main_message": (
            "CPP hotspots are dominated by reproducible "
            "arginine/lysine-centered local patterns."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 5",
        "title": (
            "Faithfulness of consensus hotspots"
        ),
        "panels": (
            "A hotspot versus random probability drop; "
            "B mean drop with 95% CI; "
            "C paired effect sizes; "
            "D hotspot-only sufficiency"
        ),
        "source_outputs": (
            "CPP_hotspot_faithfulness_manuscript_table.csv; "
            "faithfulness figures"
        ),
        "main_message": (
            "Consensus hotspots contribute more strongly "
            "to predictions than matched random residues."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 6",
        "title": (
            "Cross-PLM conservation of CPP hotspots"
        ),
        "panels": (
            "A pairwise Jaccard matrix; "
            "B support by 1–4 PLMs; "
            "C conservation enrichment odds ratios; "
            "D CPP versus non-CPP conservation distributions"
        ),
        "source_outputs": (
            "cross_plm_pairwise_hotspot_jaccard.csv; "
            "cross_plm_hotspot_support_distribution.csv; "
            "residue_level_cross_plm_conservation_enrichment.csv"
        ),
        "main_message": (
            "High-confidence hotspots are more reproducible "
            "across PLMs in CPPs than in non-CPPs."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 7",
        "title": (
            "Physicochemical and positional organization"
        ),
        "panels": (
            "A physicochemical class enrichment; "
            "B charge and hydropathy distributions; "
            "C N/middle/C enrichment; "
            "D normalized 10-bin positional profile"
        ),
        "source_outputs": (
            "physicochemical_category_enrichment.csv; "
            "physicochemical_sequence_level_tests.csv; "
            "hotspot_positional_region_enrichment.csv; "
            "hotspot_position_10bin_distribution.csv"
        ),
        "main_message": (
            "CPP hotspots form a basic, hydrophilic, "
            "centrally positioned sequence grammar."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 8",
        "title": (
            "Integrated biological model of CPP sequence grammar"
        ),
        "panels": (
            "Graphical synthesis of cationic residues, "
            "arginine-centered motifs, central positioning, "
            "cross-PLM conservation, and faithfulness"
        ),
        "source_outputs": (
            "All major biological analyses"
        ),
        "main_message": (
            "Consensus PLM-XAI reveals a conserved sequence "
            "grammar associated with CPP activity."
        ),
        "priority": "Optional graphical summary",
    },
]

main_figures_df = pd.DataFrame(
    main_figure_rows
)


# ============================================================
# 19. PROPOSED SUPPLEMENTARY FIGURES
# ============================================================

si_figure_rows = [
    {
        "figure": "Figure S1",
        "content": (
            "Dataset length distributions, class balance, "
            "and split validation."
        ),
    },
    {
        "figure": "Figure S2",
        "content": (
            "Training histories for all four PLMs."
        ),
    },
    {
        "figure": "Figure S3",
        "content": (
            "ROC and PR curves for every model and dataset."
        ),
    },
    {
        "figure": "Figure S4",
        "content": (
            "Threshold scans and default versus optimized "
            "threshold comparison."
        ),
    },
    {
        "figure": "Figure S5",
        "content": (
            "Representative attention, Gradient × Input, "
            "and Integrated Gradients residue maps."
        ),
    },
    {
        "figure": "Figure S6",
        "content": (
            "Method-agreement distributions for each PLM."
        ),
    },
    {
        "figure": "Figure S7",
        "content": (
            "Cross-model agreement distributions."
        ),
    },
    {
        "figure": "Figure S8",
        "content": (
            "Original versus redundancy-adjusted consensus."
        ),
    },
    {
        "figure": "Figure S9",
        "content": (
            "Residue enrichment at top 10%, 15%, 20%, "
            "strict, and unanimous thresholds."
        ),
    },
    {
        "figure": "Figure S10",
        "content": (
            "Complete motif enrichment by motif length."
        ),
    },
    {
        "figure": "Figure S11",
        "content": (
            "Faithfulness per-sequence distributions."
        ),
    },
    {
        "figure": "Figure S12",
        "content": (
            "Hotspot-only sufficiency scatter plots."
        ),
    },
    {
        "figure": "Figure S13",
        "content": (
            "All pairwise PLM hotspot Jaccard matrices."
        ),
    },
    {
        "figure": "Figure S14",
        "content": (
            "Sequence-level physicochemical property boxplots."
        ),
    },
    {
        "figure": "Figure S15",
        "content": (
            "Detailed positional density and uniformity tests."
        ),
    },
]

si_figures_df = pd.DataFrame(
    si_figure_rows
)


# ============================================================
# 20. PROPOSED TABLES
# ============================================================

main_table_rows = [
    {
        "table": "Table 1",
        "title": (
            "Datasets, class composition, and fixed splits"
        ),
        "source": (
            "Dataset summary and QC files"
        ),
    },
    {
        "table": "Table 2",
        "title": (
            "Internal and external predictive performance"
        ),
        "source": (
            "individual_vs_ensemble_comparison.csv"
        ),
    },
    {
        "table": "Table 3",
        "title": (
            "Replicated residue and motif determinants"
        ),
        "source": (
            "residue_enrichment_replication.csv and "
            "strict_replicated_CPP_hotspot_motifs.csv"
        ),
    },
    {
        "table": "Table 4",
        "title": (
            "Faithfulness of consensus hotspots"
        ),
        "source": (
            "CPP_hotspot_faithfulness_manuscript_table.csv"
        ),
    },
    {
        "table": "Table 5",
        "title": (
            "Cross-PLM conservation, physicochemical, "
            "and positional replication"
        ),
        "source": (
            "Conservation, physicochemical, and positional "
            "replication tables"
        ),
    },
]

main_tables_df = pd.DataFrame(
    main_table_rows
)


si_table_rows = [
    {
        "table": "Table S1",
        "content": "Complete dataset QC and rejected sequences",
    },
    {
        "table": "Table S2",
        "content": "Model hyperparameters and training settings",
    },
    {
        "table": "Table S3",
        "content": "Complete individual model metrics",
    },
    {
        "table": "Table S4",
        "content": "All prediction-level outputs",
    },
    {
        "table": "Table S5",
        "content": "XAI method agreement",
    },
    {
        "table": "Table S6",
        "content": "Cross-model XAI agreement",
    },
    {
        "table": "Table S7",
        "content": "All residue enrichment thresholds",
    },
    {
        "table": "Table S8",
        "content": "Complete motif enrichment results",
    },
    {
        "table": "Table S9",
        "content": "Per-sequence faithfulness results",
    },
    {
        "table": "Table S10",
        "content": "Per-sequence hotspot conservation",
    },
    {
        "table": "Table S11",
        "content": "Sequence-level physicochemical properties",
    },
    {
        "table": "Table S12",
        "content": "Sequence-level positional metrics",
    },
]

si_tables_df = pd.DataFrame(
    si_table_rows
)


# ============================================================
# 21. PROPOSED RESULTS SECTION STRUCTURE
# ============================================================

results_section_rows = [
    {
        "section_number": "3.1",
        "section_title": (
            "Dataset curation and leakage-free evaluation design"
        ),
        "core_content": (
            "Dataset sizes, class balance, sequence-length "
            "range, fixed train/validation/test splits, and "
            "independent KELM dataset."
        ),
    },
    {
        "section_number": "3.2",
        "section_title": (
            "Predictive performance of four PLM classifiers"
        ),
        "core_content": (
            "Individual internal and external metrics, "
            "threshold selection, and performance trade-offs."
        ),
    },
    {
        "section_number": "3.3",
        "section_title": (
            "Four-PLM ensemble performance"
        ),
        "core_content": (
            "Median ensemble improves internal MCC; mean "
            "ensemble improves external ranking metrics."
        ),
    },
    {
        "section_number": "3.4",
        "section_title": (
            "Residue-level attribution and consensus-XAI design"
        ),
        "core_content": (
            "Attention, Gradient × Input, IG, method "
            "agreement, redundancy correction, and "
            "cross-model complementarity."
        ),
    },
    {
        "section_number": "3.5",
        "section_title": (
            "Consensus hotspots encode reproducible residue "
            "and motif determinants"
        ),
        "core_content": (
            "K/R enrichment, depleted residues, RR and "
            "aromatic-basic motif replication."
        ),
    },
    {
        "section_number": "3.6",
        "section_title": (
            "Perturbation analysis validates hotspot faithfulness"
        ),
        "core_content": (
            "Hotspot ablation, matched random controls, "
            "effect sizes, and hotspot-only sufficiency."
        ),
    },
    {
        "section_number": "3.7",
        "section_title": (
            "CPP hotspots exhibit enhanced cross-PLM conservation"
        ),
        "core_content": (
            "Jaccard agreement, support by multiple PLMs, "
            "and CPP-versus-non-CPP conservation."
        ),
    },
    {
        "section_number": "3.8",
        "section_title": (
            "CPP hotspots possess a conserved "
            "physicochemical grammar"
        ),
        "core_content": (
            "Basic charge enrichment, hydrophilicity, and "
            "depletion of hydrophobic and structure-special "
            "residues."
        ),
    },
    {
        "section_number": "3.9",
        "section_title": (
            "CPP hotspots show reproducible positional organization"
        ),
        "core_content": (
            "Central enrichment, C-terminal depletion, and "
            "detailed 10-bin profile."
        ),
    },
    {
        "section_number": "3.10",
        "section_title": (
            "Integrated sequence grammar associated with CPP activity"
        ),
        "core_content": (
            "Synthesis of cationic composition, motif context, "
            "position, conservation, and model faithfulness."
        ),
    },
]

results_sections_df = pd.DataFrame(
    results_section_rows
)


# ============================================================
# 22. EXISTING FIGURE INVENTORY
# ============================================================

figure_inventory_rows = []

for figure_category, figure_directory in [
    ("main", FIGURE_MAIN_DIR),
    ("SI", FIGURE_SI_DIR),
]:

    for file_path in sorted(
        figure_directory.glob("*")
    ):

        if file_path.suffix.lower() not in [
            ".png",
            ".pdf",
            ".svg",
            ".jpg",
            ".jpeg",
        ]:
            continue

        figure_inventory_rows.append({
            "category": figure_category,
            "filename": file_path.name,
            "path": str(file_path),
            "extension": (
                file_path.suffix.lower()
            ),
            "size_MB": float(
                file_path.stat().st_size
                / (1024 ** 2)
            ),
        })


figure_inventory_df = pd.DataFrame(
    figure_inventory_rows
)


# ============================================================
# 23. TABLE INVENTORY
# ============================================================

table_inventory_rows = []

for table_category, table_directory in [
    ("main", TABLE_MAIN_DIR),
    ("SI", TABLE_SI_DIR),
]:

    for file_path in sorted(
        table_directory.glob("*.csv")
    ):

        try:
            dataframe = pd.read_csv(
                file_path
            )

            rows = len(
                dataframe
            )

            columns = len(
                dataframe.columns
            )

            duplicate_rows = int(
                dataframe.duplicated().sum()
            )

        except Exception:
            rows = None
            columns = None
            duplicate_rows = None

        table_inventory_rows.append({
            "category": table_category,
            "filename": file_path.name,
            "path": str(file_path),
            "rows": rows,
            "columns": columns,
            "duplicate_rows": (
                duplicate_rows
            ),
            "size_MB": float(
                file_path.stat().st_size
                / (1024 ** 2)
            ),
        })


table_inventory_df = pd.DataFrame(
    table_inventory_rows
)


# ============================================================
# 24. FINAL AUDIT SUMMARY
# ============================================================

validation_issues_df = pd.DataFrame(
    validation_issue_rows
)

if len(validation_issues_df) == 0:
    validation_issues_df = pd.DataFrame(
        columns=[
            "analysis",
            "severity",
            "issue",
            "recommendation",
        ]
    )


critical_missing_files = file_audit_df[
    ~file_audit_df["exists"]
]

unreadable_files = file_audit_df[
    file_audit_df[
        "read_status"
    ].astype(str).str.contains(
        "error|empty",
        regex=True,
    )
]

incomplete_steps = step_audit_df[
    ~step_audit_df[
        "completed_flag"
    ]
]


audit_summary = {
    "audit_date": (
        pd.Timestamp.now().isoformat()
    ),
    "project_directory": str(
        PROJECT_DIR
    ),
    "expected_pipeline_steps": int(
        len(EXPECTED_STEPS)
    ),
    "completed_pipeline_steps": int(
        step_audit_df[
            "completed_flag"
        ].sum()
    ),
    "incomplete_pipeline_steps": (
        incomplete_steps[
            "step"
        ].tolist()
    ),
    "expected_critical_files": int(
        len(EXPECTED_FILES)
    ),
    "existing_critical_files": int(
        file_audit_df[
            "exists"
        ].sum()
    ),
    "missing_critical_files": (
        critical_missing_files[
            "file_key"
        ].tolist()
    ),
    "unreadable_or_empty_files": (
        unreadable_files[
            "file_key"
        ].tolist()
    ),
    "validation_issue_count": int(
        len(validation_issues_df)
    ),
    "high_severity_issue_count": int(
        (
            validation_issues_df[
                "severity"
            ] == "high"
        ).sum()
    ),
    "main_figures_proposed": int(
        len(main_figures_df)
    ),
    "SI_figures_proposed": int(
        len(si_figures_df)
    ),
    "main_tables_proposed": int(
        len(main_tables_df)
    ),
    "SI_tables_proposed": int(
        len(si_tables_df)
    ),
}


# ============================================================
# 25. SAVE CSV OUTPUTS
# ============================================================

csv_outputs = {
    "pipeline_step_audit.csv":
        step_audit_df,

    "critical_file_audit.csv":
        file_audit_df,

    "dataset_summary.csv":
        dataset_summary_df,

    "split_overlap_audit.csv":
        split_overlap_df,

    "performance_best_results.csv":
        performance_summary_df,

    "xai_audit_summary.csv":
        xai_audit_summary_df,

    "residue_key_results.csv":
        residue_key_results_df,

    "motif_key_results.csv":
        motif_key_results_df,

    "faithfulness_key_results.csv":
        faithfulness_key_results_df,

    "conservation_key_results.csv":
        conservation_key_results_df,

    "physicochemical_key_results.csv":
        physicochemical_key_results_df,

    "positional_key_results.csv":
        positional_key_results_df,

    "manuscript_key_findings.csv":
        key_findings_df,

    "proposed_main_figures.csv":
        main_figures_df,

    "proposed_SI_figures.csv":
        si_figures_df,

    "proposed_main_tables.csv":
        main_tables_df,

    "proposed_SI_tables.csv":
        si_tables_df,

    "results_section_plan.csv":
        results_sections_df,

    "figure_inventory.csv":
        figure_inventory_df,

    "table_inventory.csv":
        table_inventory_df,

    "validation_issues.csv":
        validation_issues_df,
}


for filename, dataframe in (
    csv_outputs.items()
):

    output_path = (
        AUDIT_DIR / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )


# ============================================================
# 26. SAVE JSON SUMMARY
# ============================================================

audit_summary_file = (
    AUDIT_DIR
    / "final_audit_summary.json"
)

with open(
    audit_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        audit_summary,
        handle,
        indent=2,
    )


# ============================================================
# 27. SAVE EXCEL WORKBOOK
# ============================================================

audit_workbook_file = (
    AUDIT_DIR
    / "pLM4CPP_XAI_final_manuscript_audit.xlsx"
)

with pd.ExcelWriter(
    audit_workbook_file,
    engine="openpyxl",
) as writer:

    step_audit_df.to_excel(
        writer,
        sheet_name="Pipeline Steps",
        index=False,
    )

    file_audit_df.to_excel(
        writer,
        sheet_name="Critical Files",
        index=False,
    )

    dataset_summary_df.to_excel(
        writer,
        sheet_name="Datasets",
        index=False,
    )

    split_overlap_df.to_excel(
        writer,
        sheet_name="Split Overlap",
        index=False,
    )

    performance_summary_df.to_excel(
        writer,
        sheet_name="Best Performance",
        index=False,
    )

    xai_audit_summary_df.to_excel(
        writer,
        sheet_name="XAI Summary",
        index=False,
    )

    residue_key_results_df.to_excel(
        writer,
        sheet_name="Residues",
        index=False,
    )

    motif_key_results_df.to_excel(
        writer,
        sheet_name="Motifs",
        index=False,
    )

    faithfulness_key_results_df.to_excel(
        writer,
        sheet_name="Faithfulness",
        index=False,
    )

    conservation_key_results_df.to_excel(
        writer,
        sheet_name="Conservation",
        index=False,
    )

    physicochemical_key_results_df.to_excel(
        writer,
        sheet_name="Physicochemical",
        index=False,
    )

    positional_key_results_df.to_excel(
        writer,
        sheet_name="Positional",
        index=False,
    )

    key_findings_df.to_excel(
        writer,
        sheet_name="Key Findings",
        index=False,
    )

    main_figures_df.to_excel(
        writer,
        sheet_name="Main Figures",
        index=False,
    )

    si_figures_df.to_excel(
        writer,
        sheet_name="SI Figures",
        index=False,
    )

    main_tables_df.to_excel(
        writer,
        sheet_name="Main Tables",
        index=False,
    )

    si_tables_df.to_excel(
        writer,
        sheet_name="SI Tables",
        index=False,
    )

    results_sections_df.to_excel(
        writer,
        sheet_name="Results Plan",
        index=False,
    )

    figure_inventory_df.to_excel(
        writer,
        sheet_name="Figure Inventory",
        index=False,
    )

    table_inventory_df.to_excel(
        writer,
        sheet_name="Table Inventory",
        index=False,
    )

    validation_issues_df.to_excel(
        writer,
        sheet_name="Issues",
        index=False,
    )


# ============================================================
# 28. FORMAT EXCEL WORKBOOK
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
)
from openpyxl.utils import (
    get_column_letter,
)

workbook = load_workbook(
    audit_workbook_file
)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7",
)

warning_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC",
)

error_fill = PatternFill(
    fill_type="solid",
    fgColor="F4CCCC",
)

for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )

        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for column_cells in worksheet.columns:

        maximum_length = 0

        column_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

            cell_value = (
                ""
                if cell.value is None
                else str(cell.value)
            )

            maximum_length = max(
                maximum_length,
                len(cell_value),
            )

        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max(
                maximum_length + 2,
                12,
            ),
            55,
        )


if "Issues" in workbook.sheetnames:

    issue_sheet = workbook[
        "Issues"
    ]

    for row in range(
        2,
        issue_sheet.max_row + 1,
    ):

        severity = str(
            issue_sheet.cell(
                row=row,
                column=2,
            ).value
        ).lower()

        fill = None

        if severity == "high":
            fill = error_fill

        elif severity in [
            "medium",
            "informational",
        ]:
            fill = warning_fill

        if fill is not None:
            for column in range(
                1,
                issue_sheet.max_column + 1,
            ):
                issue_sheet.cell(
                    row=row,
                    column=column,
                ).fill = fill


workbook.save(
    audit_workbook_file
)


# ============================================================
# 29. SAVE CHECKPOINT
# ============================================================

output_files = [
    audit_summary_file,
    audit_workbook_file,
]

output_files.extend(
    [
        AUDIT_DIR / filename
        for filename in csv_outputs
    ]
)

mark_step_complete(
    "20_final_results_audit",
    output_files=output_files,
    details=audit_summary,
)


# ============================================================
# 30. DISPLAY AUDIT RESULTS
# ============================================================

print("\n" + "=" * 80)
print("FINAL MANUSCRIPT AUDIT SUMMARY")
print("=" * 80)

for key, value in audit_summary.items():
    print(f"{key}: {value}")


print("\n" + "=" * 80)
print("INCOMPLETE PIPELINE STEPS")
print("=" * 80)

if len(incomplete_steps) == 0:
    print("None")
else:
    display(
        incomplete_steps
    )


print("\n" + "=" * 80)
print("MISSING OR UNREADABLE CRITICAL FILES")
print("=" * 80)

problem_files = file_audit_df[
    (
        ~file_audit_df["exists"]
    )
    | (
        file_audit_df[
            "read_status"
        ].isin(
            [
                "empty",
            ]
        )
    )
    | (
        file_audit_df[
            "read_status"
        ].astype(str).str.contains(
            "read_error",
            regex=False,
        )
    )
]

if len(problem_files) == 0:
    print("None")
else:
    display(
        problem_files
    )


print("\n" + "=" * 80)
print("VALIDATION ISSUES")
print("=" * 80)

if len(validation_issues_df) == 0:
    print("No issues identified.")
else:
    display(
        validation_issues_df
    )


print("\n" + "=" * 80)
print("KEY MANUSCRIPT FINDINGS")
print("=" * 80)

display(
    key_findings_df
)


print("\n" + "=" * 80)
print("PROPOSED MAIN FIGURES")
print("=" * 80)

display(
    main_figures_df
)


print("\n" + "=" * 80)
print("PROPOSED MAIN TABLES")
print("=" * 80)

display(
    main_tables_df
)


print("\nAudit workbook saved to:")
print(audit_workbook_file)

print("\nAudit directory:")
print(AUDIT_DIR)

print("\n" + "=" * 80)
print("STEP 20 COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 21: FINAL MANUSCRIPT TABLES AND FIGURE MANIFEST
#
# Creates:
# 1. Final Table 1 — datasets and splits
# 2. Final Table 2 — model performance
# 3. Final Table 3 — replicated residues and motifs
# 4. Final Table 4 — biological and faithfulness validation
# 5. Main/SI figure manifest
# 6. Final manuscript-results directory
# ============================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side,
)
from openpyxl.utils import get_column_letter


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

TABLE_SI_DIR = (
    RESULTS_DIR / "tables_SI"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FIGURE_SI_DIR = (
    RESULTS_DIR / "figures_SI"
)

FINAL_DIR = (
    RESULTS_DIR / "final_manuscript_package"
)

FINAL_TABLE_DIR = (
    FINAL_DIR / "01_main_tables"
)

FINAL_SI_TABLE_DIR = (
    FINAL_DIR / "02_SI_tables"
)

FINAL_FIGURE_DIR = (
    FINAL_DIR / "03_main_figures"
)

FINAL_SI_FIGURE_DIR = (
    FINAL_DIR / "04_SI_figures"
)

FINAL_MANIFEST_DIR = (
    FINAL_DIR / "05_manifests"
)

for directory in [
    FINAL_TABLE_DIR,
    FINAL_SI_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_SI_FIGURE_DIR,
    FINAL_MANIFEST_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. HELPERS
# ============================================================

def require_file(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

    return path


def format_p_value(value):
    if pd.isna(value):
        return ""

    value = float(value)

    if value < 0.001:
        return f"{value:.2e}"

    return f"{value:.3f}"


def format_value_ci(
    mean,
    lower,
    upper,
    decimals=3,
):
    return (
        f"{mean:.{decimals}f} "
        f"({lower:.{decimals}f}–"
        f"{upper:.{decimals}f})"
    )


def format_number(
    value,
    decimals=3,
):
    if pd.isna(value):
        return ""

    return f"{float(value):.{decimals}f}"


def save_csv_and_excel(
    dataframe,
    base_filename,
    sheet_name,
):
    csv_file = (
        FINAL_TABLE_DIR
        / f"{base_filename}.csv"
    )

    excel_file = (
        FINAL_TABLE_DIR
        / f"{base_filename}.xlsx"
    )

    dataframe.to_csv(
        csv_file,
        index=False,
    )

    with pd.ExcelWriter(
        excel_file,
        engine="openpyxl",
    ) as writer:
        dataframe.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False,
        )

    format_excel_file(
        excel_file
    )

    return csv_file, excel_file


def format_excel_file(excel_file):

    workbook = load_workbook(
        excel_file
    )

    worksheet = workbook.active

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    thin_border = Border(
        bottom=Side(
            style="thin",
            color="B7B7B7",
        )
    )

    worksheet.freeze_panes = "A2"

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )

        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

        cell.border = thin_border

    for row in worksheet.iter_rows(
        min_row=2
    ):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

    for column_cells in worksheet.columns:

        maximum_length = max(
            len(
                str(cell.value)
                if cell.value is not None
                else ""
            )
            for cell in column_cells
        )

        column_letter = get_column_letter(
            column_cells[0].column
        )

        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max(
                maximum_length + 2,
                12,
            ),
            42,
        )

    workbook.save(
        excel_file
    )


# ============================================================
# 3. TABLE 1 — DATASETS AND FIXED SPLITS
# ============================================================

dataset_files = {
    "Training": (
        PROJECT_DIR
        / "02_data_processed"
        / "train_split.csv"
    ),
    "Validation": (
        PROJECT_DIR
        / "02_data_processed"
        / "validation_split.csv"
    ),
    "Internal test": (
        PROJECT_DIR
        / "02_data_processed"
        / "internal_test_split.csv"
    ),
    "KELM external": (
        PROJECT_DIR
        / "02_data_processed"
        / "kelm_external_dataset_qc.csv"
    ),
}

table1_rows = []

for dataset_name, file_path in (
    dataset_files.items()
):
    dataframe = pd.read_csv(
        require_file(file_path)
    )

    table1_rows.append({
        "Dataset": dataset_name,
        "Total sequences": int(
            len(dataframe)
        ),
        "CPP, n": int(
            (
                dataframe["label"] == 1
            ).sum()
        ),
        "Non-CPP, n": int(
            (
                dataframe["label"] == 0
            ).sum()
        ),
        "CPP fraction": round(
            (
                dataframe["label"] == 1
            ).mean(),
            3,
        ),
        "Minimum length": int(
            dataframe["length"].min()
        ),
        "Maximum length": int(
            dataframe["length"].max()
        ),
        "Mean length": round(
            dataframe["length"].mean(),
            2,
        ),
        "Role": {
            "Training":
                "Classifier fitting",
            "Validation":
                "Early stopping and threshold selection",
            "Internal test":
                "Held-out internal evaluation",
            "KELM external":
                "Independent external validation",
        }[dataset_name],
    })


table1_df = pd.DataFrame(
    table1_rows
)

table1_csv, table1_excel = (
    save_csv_and_excel(
        table1_df,
        "Table_1_Datasets_and_splits",
        "Table 1",
    )
)


# ============================================================
# 4. TABLE 2 — PREDICTIVE PERFORMANCE
# ============================================================

performance_file = require_file(
    TABLE_MAIN_DIR
    / "individual_vs_ensemble_comparison.csv"
)

performance = pd.read_csv(
    performance_file
)

method_order = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
    "mean_ensemble",
    "median_ensemble",
]

dataset_order = [
    "internal_test",
    "kelm_external",
]

performance = performance[
    performance["dataset"].isin(
        dataset_order
    )
].copy()

performance["method"] = pd.Categorical(
    performance["method"],
    categories=method_order,
    ordered=True,
)

performance["dataset"] = pd.Categorical(
    performance["dataset"],
    categories=dataset_order,
    ordered=True,
)

performance = performance.sort_values(
    [
        "dataset",
        "method",
    ]
)

dataset_display = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

method_display = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
    "mean_ensemble": "Mean ensemble",
    "median_ensemble": "Median ensemble",
}

table2_rows = []

for row in performance.itertuples(
    index=False
):

    table2_rows.append({
        "Dataset": dataset_display[
            str(row.dataset)
        ],
        "Model": method_display[
            str(row.method)
        ],
        "Threshold": round(
            float(row.threshold),
            3,
        ),
        "Accuracy": round(
            float(row.accuracy),
            3,
        ),
        "Balanced accuracy": round(
            float(row.balanced_accuracy),
            3,
        ),
        "F1 score": round(
            float(row.f1),
            3,
        ),
        "MCC": round(
            float(row.mcc),
            3,
        ),
        "ROC-AUC": round(
            float(row.roc_auc),
            3,
        ),
        "PR-AUC": round(
            float(row.pr_auc),
            3,
        ),
    })


table2_df = pd.DataFrame(
    table2_rows
)

table2_csv, table2_excel = (
    save_csv_and_excel(
        table2_df,
        "Table_2_Predictive_performance",
        "Table 2",
    )
)


# ============================================================
# 5. TABLE 3A — REPLICATED RESIDUE DETERMINANTS
# ============================================================

residue_file = require_file(
    TABLE_MAIN_DIR
    / "residue_enrichment_replication.csv"
)

residue_df = pd.read_csv(
    residue_file
)

replicated_residues = residue_df[
    residue_df[
        "significant_in_both"
    ].astype(bool)
].copy()

replicated_residues[
    "Direction"
] = np.where(
    replicated_residues[
        "internal_log2_enrichment"
    ] > 0,
    "Enriched in CPP hotspots",
    "Depleted in CPP hotspots",
)

table3a_df = pd.DataFrame({
    "Residue":
        replicated_residues[
            "residue"
        ],

    "Direction":
        replicated_residues[
            "Direction"
        ],

    "Internal odds ratio":
        replicated_residues[
            "internal_odds_ratio"
        ].round(2),

    "Internal FDR":
        replicated_residues[
            "internal_fdr"
        ].apply(
            format_p_value
        ),

    "KELM odds ratio":
        replicated_residues[
            "kelm_odds_ratio"
        ].round(2),

    "KELM FDR":
        replicated_residues[
            "kelm_fdr"
        ].apply(
            format_p_value
        ),
})

table3a_df = table3a_df.sort_values(
    [
        "Direction",
        "Internal odds ratio",
    ],
    ascending=[
        True,
        False,
    ],
)

table3a_csv, table3a_excel = (
    save_csv_and_excel(
        table3a_df,
        "Table_3A_Replicated_residue_determinants",
        "Table 3A",
    )
)


# ============================================================
# 6. TABLE 3B — REPLICATED MOTIFS
# ============================================================

motif_file = require_file(
    TABLE_MAIN_DIR
    / "strict_replicated_CPP_hotspot_motifs.csv"
)

motif_df = pd.read_csv(
    motif_file
)

table3b_df = pd.DataFrame({
    "Motif":
        motif_df["motif"],

    "Length":
        motif_df[
            "motif_length"
        ].astype(int),

    "Internal CPP sequences":
        motif_df[
            "internal_cpp_count"
        ].astype(int),

    "Internal odds ratio":
        motif_df[
            "internal_odds_ratio"
        ].round(2),

    "Internal FDR":
        motif_df[
            "internal_fdr"
        ].apply(
            format_p_value
        ),

    "KELM CPP sequences":
        motif_df[
            "kelm_cpp_count"
        ].astype(int),

    "KELM odds ratio":
        motif_df[
            "kelm_odds_ratio"
        ].round(2),

    "KELM FDR":
        motif_df[
            "kelm_fdr"
        ].apply(
            format_p_value
        ),
})

table3b_df = table3b_df.sort_values(
    [
        "Internal FDR",
        "Internal odds ratio",
    ],
    ascending=[
        True,
        False,
    ],
)

table3b_csv, table3b_excel = (
    save_csv_and_excel(
        table3b_df,
        "Table_3B_Replicated_hotspot_motifs",
        "Table 3B",
    )
)


# ============================================================
# 7. TABLE 4A — FAITHFULNESS
# ============================================================

faithfulness_file = require_file(
    TABLE_MAIN_DIR
    / "CPP_hotspot_faithfulness_manuscript_table.csv"
)

faithfulness = pd.read_csv(
    faithfulness_file
)

faithfulness = faithfulness[
    faithfulness[
        "dataset"
    ].isin(
        dataset_order
    )
].copy()

faithfulness[
    "dataset"
] = pd.Categorical(
    faithfulness["dataset"],
    categories=dataset_order,
    ordered=True,
)

faithfulness[
    "model"
] = pd.Categorical(
    faithfulness["model"],
    categories=[
        "ESM2_320",
        "ESM2_640",
        "ESM2_1280",
        "ProtT5",
    ],
    ordered=True,
)

faithfulness = faithfulness.sort_values(
    [
        "dataset",
        "model",
    ]
)

table4a_rows = []

for row in faithfulness.itertuples(
    index=False
):

    table4a_rows.append({
        "Dataset": dataset_display[
            str(row.dataset)
        ],
        "Model": method_display[
            str(row.model)
        ],
        "CPPs, n": int(
            row.n
        ),
        "Hotspot probability drop, mean (95% CI)":
            format_value_ci(
                row.mean_hotspot_probability_drop,
                row.hotspot_drop_ci95_low,
                row.hotspot_drop_ci95_high,
            ),
        "Random probability drop, mean":
            round(
                row.mean_random_probability_drop,
                3,
            ),
        "Hotspot − random drop":
            round(
                row.mean_hotspot_minus_random_drop,
                3,
            ),
        "Fraction hotspot > random":
            round(
                row.fraction_hotspot_stronger_than_random,
                3,
            ),
        "Paired Cohen's dz":
            round(
                row.paired_cohens_dz,
                3,
            ),
        "Rank-biserial effect":
            round(
                row.rank_biserial_effect_size,
                3,
            ),
        "Wilcoxon p":
            format_p_value(
                row.wilcoxon_p_value
            ),
        "Hotspot-only probability":
            round(
                row.mean_hotspot_only_probability,
                3,
            ),
    })


table4a_df = pd.DataFrame(
    table4a_rows
)

table4a_csv, table4a_excel = (
    save_csv_and_excel(
        table4a_df,
        "Table_4A_Hotspot_faithfulness",
        "Table 4A",
    )
)


# ============================================================
# 8. TABLE 4B — CONSERVATION, PHYSICOCHEMICAL AND POSITIONAL
# ============================================================

conservation_file = require_file(
    TABLE_MAIN_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

conservation = pd.read_csv(
    conservation_file
)

physicochemical_file = require_file(
    TABLE_MAIN_DIR
    / "physicochemical_enrichment_replication.csv"
)

physicochemical = pd.read_csv(
    physicochemical_file
)

positional_file = require_file(
    TABLE_MAIN_DIR
    / "hotspot_positional_replication.csv"
)

positional = pd.read_csv(
    positional_file
)

table4b_rows = []

# Cross-model conservation
for row in conservation.itertuples(
    index=False
):
    if int(
        row.minimum_model_support
    ) not in [
        3,
        4,
    ]:
        continue

    support_label = (
        "Supported by ≥3 PLMs"
        if int(
            row.minimum_model_support
        ) == 3
        else "Supported by all 4 PLMs"
    )

    table4b_rows.append({
        "Analysis domain":
            "Cross-PLM conservation",
        "Feature":
            support_label,
        "Internal effect":
            (
                f"OR {row.odds_ratio_corrected:.2f}; "
                f"p={format_p_value(row.p_value)}"
            ),
        "KELM effect":
            "",
        "Replicated direction":
            "Yes",
        "Interpretation":
            (
                "Multi-PLM-supported residues are "
                "enriched in CPPs."
            ),
    })


# Physicochemical replication
for row in physicochemical.itertuples(
    index=False
):

    if not bool(
        row.significant_in_both
    ):
        continue

    direction = (
        "Enriched"
        if row.internal_log2_enrichment > 0
        else "Depleted"
    )

    table4b_rows.append({
        "Analysis domain":
            "Physicochemical grammar",
        "Feature":
            str(
                row.property_class
            ).replace(
                "_",
                " ",
            ),
        "Internal effect":
            (
                f"{direction}; OR "
                f"{row.internal_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.internal_fdr)}"
            ),
        "KELM effect":
            (
                f"{direction}; OR "
                f"{row.kelm_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.kelm_fdr)}"
            ),
        "Replicated direction":
            "Yes",
        "Interpretation":
            (
                "Basic residues are enriched; "
                "hydrophobic/aromatic/special classes "
                "are depleted."
            ),
    })


# Positional replication
for row in positional.itertuples(
    index=False
):

    if not bool(
        row.significant_in_both
    ):
        continue

    direction = (
        "Enriched"
        if row.internal_log2_enrichment > 0
        else "Depleted"
    )

    table4b_rows.append({
        "Analysis domain":
            "Positional grammar",
        "Feature":
            str(
                row.region
            ).replace(
                "_",
                " ",
            ),
        "Internal effect":
            (
                f"{direction}; OR "
                f"{row.internal_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.internal_fdr)}"
            ),
        "KELM effect":
            (
                f"{direction}; OR "
                f"{row.kelm_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.kelm_fdr)}"
            ),
        "Replicated direction":
            "Yes",
        "Interpretation":
            (
                "CPP hotspots are centrally enriched "
                "and C-terminally depleted."
            ),
    })


table4b_df = pd.DataFrame(
    table4b_rows
)

table4b_csv, table4b_excel = (
    save_csv_and_excel(
        table4b_df,
        "Table_4B_Replicated_biological_validation",
        "Table 4B",
    )
)


# ============================================================
# 9. CREATE COMBINED MASTER EXCEL WORKBOOK
# ============================================================

master_workbook = (
    FINAL_TABLE_DIR
    / "pLM4CPP_XAI_Main_Tables.xlsx"
)

with pd.ExcelWriter(
    master_workbook,
    engine="openpyxl",
) as writer:

    table1_df.to_excel(
        writer,
        sheet_name="Table 1",
        index=False,
    )

    table2_df.to_excel(
        writer,
        sheet_name="Table 2",
        index=False,
    )

    table3a_df.to_excel(
        writer,
        sheet_name="Table 3A",
        index=False,
    )

    table3b_df.to_excel(
        writer,
        sheet_name="Table 3B",
        index=False,
    )

    table4a_df.to_excel(
        writer,
        sheet_name="Table 4A",
        index=False,
    )

    table4b_df.to_excel(
        writer,
        sheet_name="Table 4B",
        index=False,
    )


format_excel_file(
    master_workbook
)


# ============================================================
# 10. COPY IMPORTANT SI TABLES
# ============================================================

si_table_sources = {
    "Table_S1_dataset_QC.csv":
        PROJECT_DIR
        / "02_data_processed"
        / "internal_dataset_qc.csv",

    "Table_S2_complete_model_performance.csv":
        TABLE_MAIN_DIR
        / "attention_classifier_performance.csv",

    "Table_S3_XAI_method_agreement.csv":
        TABLE_MAIN_DIR
        / "xai_method_agreement.csv",

    "Table_S4_cross_model_XAI_agreement.csv":
        TABLE_MAIN_DIR
        / "cross_model_xai_agreement.csv",

    "Table_S5_residue_enrichment_all_thresholds.csv":
        TABLE_SI_DIR
        / "residue_enrichment_all_thresholds.csv",

    "Table_S6_complete_motif_enrichment.csv":
        TABLE_SI_DIR
        / "hotspot_motif_enrichment_all.csv",

    "Table_S7_faithfulness_all_sequences.csv":
        TABLE_SI_DIR
        / "faithfulness_all_sequences_all_models.csv",

    "Table_S8_conservation_per_sequence.csv":
        TABLE_SI_DIR
        / "cross_plm_hotspot_conservation_per_sequence.csv",

    "Table_S9_physicochemical_per_sequence.csv":
        TABLE_SI_DIR
        / "physicochemical_properties_per_sequence.csv",

    "Table_S10_positional_metrics_per_sequence.csv":
        TABLE_SI_DIR
        / "hotspot_positional_metrics_per_sequence.csv",
}

si_copy_rows = []

for output_name, source_file in (
    si_table_sources.items()
):
    source_file = Path(
        source_file
    )

    exists = source_file.exists()

    destination_file = (
        FINAL_SI_TABLE_DIR
        / output_name
    )

    if exists:
        shutil.copy2(
            source_file,
            destination_file,
        )

    si_copy_rows.append({
        "SI table": output_name,
        "source": str(
            source_file
        ),
        "copied": bool(
            exists
        ),
        "destination": str(
            destination_file
        ),
    })


si_copy_df = pd.DataFrame(
    si_copy_rows
)


# ============================================================
# 11. FIGURE MANIFEST
# ============================================================

figure_manifest_rows = [
    {
        "Figure": "Figure 1",
        "Title": (
            "Study design and multi-PLM consensus-XAI workflow"
        ),
        "Panels": (
            "A datasets and splits; B four PLMs; "
            "C attention classifiers; D attribution methods; "
            "E adjusted consensus and validation"
        ),
        "Status": "Must create",
        "Main or SI": "Main",
        "Priority": 1,
    },
    {
        "Figure": "Figure 2",
        "Title": (
            "Predictive performance across individual PLMs "
            "and ensembles"
        ),
        "Panels": (
            "A internal ROC; B internal PR; "
            "C KELM ROC; D KELM PR; "
            "E MCC comparison"
        ),
        "Status": "Must create/finalize",
        "Main or SI": "Main",
        "Priority": 2,
    },
    {
        "Figure": "Figure 3",
        "Title": (
            "Construction and stability of consensus XAI"
        ),
        "Panels": (
            "A representative residue map; "
            "B method agreement; C cross-model agreement; "
            "D adjusted consensus stability"
        ),
        "Status": "Must create",
        "Main or SI": "Main",
        "Priority": 3,
    },
    {
        "Figure": "Figure 4",
        "Title": (
            "Replicated residue and motif grammar"
        ),
        "Panels": (
            "A residue enrichment; "
            "B internal–KELM replication; "
            "C replicated motifs; "
            "D representative sequence hotspots"
        ),
        "Status": "Must create",
        "Main or SI": "Main",
        "Priority": 4,
    },
    {
        "Figure": "Figure 5",
        "Title": (
            "Faithfulness of consensus hotspots"
        ),
        "Panels": (
            "A hotspot vs random drops; "
            "B mean and 95% CI; "
            "C effect sizes; D hotspot-only sufficiency"
        ),
        "Status": "Partially available",
        "Main or SI": "Main",
        "Priority": 5,
    },
    {
        "Figure": "Figure 6",
        "Title": (
            "Cross-PLM conservation of CPP hotspots"
        ),
        "Panels": (
            "A pairwise Jaccard; "
            "B support by 1–4 PLMs; "
            "C conservation odds ratios; "
            "D CPP vs non-CPP comparison"
        ),
        "Status": "Partially available",
        "Main or SI": "Main",
        "Priority": 6,
    },
    {
        "Figure": "Figure 7",
        "Title": (
            "Physicochemical and positional organization "
            "of CPP hotspots"
        ),
        "Panels": (
            "A physicochemical enrichment; "
            "B charge/hydropathy; "
            "C N/middle/C enrichment; "
            "D 10-bin positional profile"
        ),
        "Status": "Partially available",
        "Main or SI": "Main",
        "Priority": 7,
    },
    {
        "Figure": "Graphical abstract",
        "Title": (
            "Consensus PLM-XAI reveals a conserved "
            "CPP-associated sequence grammar"
        ),
        "Panels": (
            "Four PLMs → consensus hotspots → "
            "K/R and RR motifs → central localization → "
            "faithfulness and external validation"
        ),
        "Status": "Create after main figures",
        "Main or SI": "Graphical abstract",
        "Priority": 8,
    },
]


figure_manifest_df = pd.DataFrame(
    figure_manifest_rows
)

figure_manifest_file = (
    FINAL_MANIFEST_DIR
    / "main_figure_manifest.csv"
)

figure_manifest_df.to_csv(
    figure_manifest_file,
    index=False,
)


# ============================================================
# 12. INVENTORY EXISTING FIGURES
# ============================================================

existing_figure_rows = []

for category, figure_directory in [
    ("Main", FIGURE_MAIN_DIR),
    ("SI", FIGURE_SI_DIR),
]:

    for file_path in sorted(
        figure_directory.glob("*")
    ):

        if file_path.suffix.lower() not in {
            ".png",
            ".pdf",
            ".svg",
            ".jpg",
            ".jpeg",
        }:
            continue

        existing_figure_rows.append({
            "Category": category,
            "Filename": (
                file_path.name
            ),
            "Extension": (
                file_path.suffix.lower()
            ),
            "Size MB": round(
                file_path.stat().st_size
                / (1024 ** 2),
                3,
            ),
            "Source path": str(
                file_path
            ),
        })


existing_figures_df = pd.DataFrame(
    existing_figure_rows
)

existing_figures_file = (
    FINAL_MANIFEST_DIR
    / "existing_figure_inventory.csv"
)

existing_figures_df.to_csv(
    existing_figures_file,
    index=False,
)


# ============================================================
# 13. MANUSCRIPT RESULT CLAIMS
# ============================================================

claim_rows = [
    {
        "Claim ID": "C1",
        "Claim": (
            "All four PLM classifiers achieved strong "
            "internal and external discrimination."
        ),
        "Use in": "Results and Discussion",
        "Qualification": (
            "Performance differed by metric and threshold."
        ),
    },
    {
        "Claim ID": "C2",
        "Claim": (
            "The median ensemble improved internal MCC, "
            "whereas the mean ensemble improved external "
            "probability-ranking metrics."
        ),
        "Use in": "Results",
        "Qualification": (
            "Do not claim universal ensemble superiority."
        ),
    },
    {
        "Claim ID": "C3",
        "Claim": (
            "Attention and gradient-based attributions "
            "showed strong within-model agreement."
        ),
        "Use in": "Results",
        "Qualification": (
            "Gradient × Input and IG were treated as one "
            "attribution family because of redundancy."
        ),
    },
    {
        "Claim ID": "C4",
        "Claim": (
            "Different PLMs prioritized complementary "
            "residue positions."
        ),
        "Use in": "Results and Discussion",
        "Qualification": (
            "Cross-model agreement was modest rather "
            "than absent."
        ),
    },
    {
        "Claim ID": "C5",
        "Claim": (
            "Lysine and arginine were reproducibly "
            "enriched in CPP hotspots."
        ),
        "Use in": "Abstract, Results, Discussion",
        "Qualification": (
            "Describe association with CPP activity, "
            "not a complete uptake mechanism."
        ),
    },
    {
        "Claim ID": "C6",
        "Claim": (
            "RR, WK, WR, RW, RL, and LR motifs were "
            "replicated in the external dataset."
        ),
        "Use in": "Results",
        "Qualification": (
            "Report motif counts alongside odds ratios."
        ),
    },
    {
        "Claim ID": "C7",
        "Claim": (
            "Consensus hotspot ablation reduced CPP "
            "probabilities more than matched random ablation."
        ),
        "Use in": "Results",
        "Qualification": (
            "External faithfulness was model dependent."
        ),
    },
    {
        "Claim ID": "C8",
        "Claim": (
            "Residues independently supported by three "
            "or four PLMs were enriched in CPPs."
        ),
        "Use in": "Abstract, Results",
        "Qualification": (
            "This indicates explanation reproducibility, "
            "not necessarily experimental causality."
        ),
    },
    {
        "Claim ID": "C9",
        "Claim": (
            "CPP hotspots were positively charged, "
            "basic, and comparatively hydrophilic."
        ),
        "Use in": "Abstract, Results, Discussion",
        "Qualification": (
            "Aromatic behavior depended on the comparison."
        ),
    },
    {
        "Claim ID": "C10",
        "Claim": (
            "CPP hotspots were enriched in the central "
            "sequence region and depleted near the "
            "C-terminus."
        ),
        "Use in": "Results and Discussion",
        "Qualification": (
            "No consistent overall N-terminal enrichment "
            "was detected."
        ),
    },
]


claims_df = pd.DataFrame(
    claim_rows
)

claims_file = (
    FINAL_MANIFEST_DIR
    / "approved_manuscript_claims.csv"
)

claims_df.to_csv(
    claims_file,
    index=False,
)


# ============================================================
# 14. RESULTS SECTION PLAN
# ============================================================

results_plan_rows = [
    {
        "Section": "3.1",
        "Title": (
            "Dataset curation and evaluation design"
        ),
        "Primary table/figure": (
            "Table 1; Figure 1"
        ),
    },
    {
        "Section": "3.2",
        "Title": (
            "Predictive performance of individual PLMs"
        ),
        "Primary table/figure": (
            "Table 2; Figure 2"
        ),
    },
    {
        "Section": "3.3",
        "Title": (
            "Ensemble performance and generalization"
        ),
        "Primary table/figure": (
            "Table 2; Figure 2"
        ),
    },
    {
        "Section": "3.4",
        "Title": (
            "Residue attribution and adjusted consensus XAI"
        ),
        "Primary table/figure": (
            "Figure 3"
        ),
    },
    {
        "Section": "3.5",
        "Title": (
            "Replicated residue and motif determinants"
        ),
        "Primary table/figure": (
            "Table 3A–B; Figure 4"
        ),
    },
    {
        "Section": "3.6",
        "Title": (
            "Perturbation analysis validates hotspot faithfulness"
        ),
        "Primary table/figure": (
            "Table 4A; Figure 5"
        ),
    },
    {
        "Section": "3.7",
        "Title": (
            "CPP hotspots exhibit enhanced "
            "cross-PLM conservation"
        ),
        "Primary table/figure": (
            "Table 4B; Figure 6"
        ),
    },
    {
        "Section": "3.8",
        "Title": (
            "Physicochemical organization of CPP hotspots"
        ),
        "Primary table/figure": (
            "Table 4B; Figure 7"
        ),
    },
    {
        "Section": "3.9",
        "Title": (
            "Positional organization of CPP hotspots"
        ),
        "Primary table/figure": (
            "Table 4B; Figure 7"
        ),
    },
]


results_plan_df = pd.DataFrame(
    results_plan_rows
)

results_plan_file = (
    FINAL_MANIFEST_DIR
    / "final_results_section_plan.csv"
)

results_plan_df.to_csv(
    results_plan_file,
    index=False,
)


# ============================================================
# 15. PACKAGE SUMMARY
# ============================================================

package_summary = {
    "created_at": (
        pd.Timestamp.now().isoformat()
    ),
    "main_tables": [
        str(table1_excel),
        str(table2_excel),
        str(table3a_excel),
        str(table3b_excel),
        str(table4a_excel),
        str(table4b_excel),
        str(master_workbook),
    ],
    "SI_tables_copied": int(
        si_copy_df["copied"].sum()
    ),
    "main_figures_planned": int(
        (
            figure_manifest_df[
                "Main or SI"
            ] == "Main"
        ).sum()
    ),
    "graphical_abstract_planned": True,
    "approved_claims": int(
        len(claims_df)
    ),
    "important_exclusions": [
        (
            "Do not report "
            "mean_hotspot_only_probability_fraction."
        ),
        (
            "Do not treat Gradient × Input and "
            "Integrated Gradients as independent "
            "consensus families."
        ),
        (
            "Do not claim all external faithfulness "
            "tests were significant."
        ),
    ],
}

package_summary_file = (
    FINAL_MANIFEST_DIR
    / "final_package_summary.json"
)

with open(
    package_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        package_summary,
        handle,
        indent=2,
    )


# ============================================================
# 16. SAVE CHECKPOINT
# ============================================================

output_files = [
    table1_csv,
    table1_excel,
    table2_csv,
    table2_excel,
    table3a_csv,
    table3a_excel,
    table3b_csv,
    table3b_excel,
    table4a_csv,
    table4a_excel,
    table4b_csv,
    table4b_excel,
    master_workbook,
    figure_manifest_file,
    existing_figures_file,
    claims_file,
    results_plan_file,
    package_summary_file,
]

mark_step_complete(
    "21_final_manuscript_tables_and_manifest",
    output_files=output_files,
    details=package_summary,
)


# ============================================================
# 17. DISPLAY FINAL OUTPUTS
# ============================================================

print("\n" + "=" * 80)
print("FINAL TABLE 1 — DATASETS AND SPLITS")
print("=" * 80)

display(table1_df)


print("\n" + "=" * 80)
print("FINAL TABLE 2 — PREDICTIVE PERFORMANCE")
print("=" * 80)

display(table2_df)


print("\n" + "=" * 80)
print("FINAL TABLE 3A — REPLICATED RESIDUES")
print("=" * 80)

display(table3a_df)


print("\n" + "=" * 80)
print("FINAL TABLE 3B — REPLICATED MOTIFS")
print("=" * 80)

display(table3b_df)


print("\n" + "=" * 80)
print("FINAL TABLE 4A — FAITHFULNESS")
print("=" * 80)

display(table4a_df)


print("\n" + "=" * 80)
print("FINAL TABLE 4B — BIOLOGICAL VALIDATION")
print("=" * 80)

display(table4b_df)


print("\n" + "=" * 80)
print("FIGURE MANIFEST")
print("=" * 80)

display(
    figure_manifest_df
)


print("\n" + "=" * 80)
print("FINAL RESULTS SECTION PLAN")
print("=" * 80)

display(
    results_plan_df
)


print("\nFinal package directory:")
print(FINAL_DIR)

print("\nMaster table workbook:")
print(master_workbook)

print("\nFigure manifest:")
print(figure_manifest_file)

print("\nApproved manuscript claims:")
print(claims_file)

print("\n" + "=" * 80)
print("STEP 21 COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 20B: HOTSPOT-THRESHOLD ROBUSTNESS
# Tests top 10%, 15%, 20%, 25%, and 30%
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

DATASETS = [
    "internal_test",
    "kelm_external",
]

THRESHOLDS = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]

AA20 = list("ACDEFGHIKLMNPQRSTVWY")

OUT = DIRS["results"] / "tables_SI"
OUT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT)


# ------------------------------------------------------------
# BENJAMINI-HOCHBERG FDR
# ------------------------------------------------------------

def bh_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    n = len(p_values)

    order = np.argsort(p_values)

    ranked = p_values[order]

    adjusted = (
        ranked
        * n
        / np.arange(1, n + 1)
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    output = np.empty(
        n,
        dtype=float,
    )

    output[order] = adjusted

    return output


# ------------------------------------------------------------
# ODDS RATIO WITH 0.5 CORRECTION
# ------------------------------------------------------------

def corrected_odds_ratio(a, b, c, d):

    return (
        (a + 0.5)
        * (d + 0.5)
    ) / (
        (b + 0.5)
        * (c + 0.5)
    )


# ------------------------------------------------------------
# RUN ANALYSIS
# ------------------------------------------------------------

rows = []

for dataset_name in DATASETS:

    input_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    print("\n" + "=" * 80)
    print("Loading:", dataset_name)
    print(input_file)

    df = pd.read_csv(input_file)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    for fraction in THRESHOLDS:

        cutoff = 1.0 - fraction

        work = df.copy()

        # Hotspot based on percentile rank
        work["hotspot"] = (
            work["adjusted_global_consensus_rank"]
            >= cutoff
        )

        # CPP residues only
        cpp = work[
            work["label"].astype(int) == 1
        ].copy()

        print(
            f"{dataset_name} | "
            f"top {int(fraction * 100)}% | "
            f"hotspots = {cpp['hotspot'].sum()} / {len(cpp)}"
        )

        for aa in AA20:

            # ----------------------------
            # contingency table
            #
            #                AA   Other
            # hotspot         a     b
            # non-hotspot     c     d
            # ----------------------------

            a = int(
                (
                    (cpp["residue"] == aa)
                    &
                    cpp["hotspot"]
                ).sum()
            )

            b = int(
                (
                    (cpp["residue"] != aa)
                    &
                    cpp["hotspot"]
                ).sum()
            )

            c = int(
                (
                    (cpp["residue"] == aa)
                    &
                    (~cpp["hotspot"])
                ).sum()
            )

            d = int(
                (
                    (cpp["residue"] != aa)
                    &
                    (~cpp["hotspot"])
                ).sum()
            )

            fisher_or, p_value = fisher_exact(
                [
                    [a, b],
                    [c, d],
                ],
                alternative="two-sided",
            )

            corrected_or = corrected_odds_ratio(
                a, b, c, d
            )

            rows.append(
                {
                    "dataset": dataset_name,
                    "threshold_pct": int(fraction * 100),
                    "residue": aa,
                    "hotspot_aa": a,
                    "hotspot_other": b,
                    "nonhotspot_aa": c,
                    "nonhotspot_other": d,
                    "odds_ratio": corrected_or,
                    "fisher_OR_raw": fisher_or,
                    "p_value": p_value,
                }
            )


# ------------------------------------------------------------
# COMBINE RESULTS
# ------------------------------------------------------------

results = pd.DataFrame(rows)


# ------------------------------------------------------------
# FDR CORRECTION WITHIN EACH DATASET + THRESHOLD
# ------------------------------------------------------------

results["fdr"] = np.nan

for (dataset_name, threshold), indices in results.groupby(
    ["dataset", "threshold_pct"]
).groups.items():

    pvals = results.loc[
        indices,
        "p_value"
    ].values

    results.loc[
        indices,
        "fdr"
    ] = bh_adjust(pvals)


results["significant_fdr_0_05"] = (
    results["fdr"] < 0.05
)


results["direction"] = np.where(
    results["odds_ratio"] > 1,
    "enriched",
    np.where(
        results["odds_ratio"] < 1,
        "depleted",
        "neutral",
    ),
)


# ------------------------------------------------------------
# CROSS-DATASET ROBUSTNESS
# ------------------------------------------------------------

stability_rows = []

for (threshold, residue), group in results.groupby(
    ["threshold_pct", "residue"]
):

    temp = group.set_index("dataset")

    if (
        "internal_test" not in temp.index
        or
        "kelm_external" not in temp.index
    ):
        continue

    internal_or = float(
        temp.loc[
            "internal_test",
            "odds_ratio"
        ]
    )

    kelm_or = float(
        temp.loc[
            "kelm_external",
            "odds_ratio"
        ]
    )

    internal_fdr = float(
        temp.loc[
            "internal_test",
            "fdr"
        ]
    )

    kelm_fdr = float(
        temp.loc[
            "kelm_external",
            "fdr"
        ]
    )

    internal_direction = (
        "enriched"
        if internal_or > 1
        else "depleted"
    )

    kelm_direction = (
        "enriched"
        if kelm_or > 1
        else "depleted"
    )

    same_direction = (
        internal_direction
        == kelm_direction
    )

    significant_both = (
        internal_fdr < 0.05
        and
        kelm_fdr < 0.05
    )

    stability_rows.append(
        {
            "threshold_pct": threshold,
            "residue": residue,
            "internal_OR": internal_or,
            "kelm_OR": kelm_or,
            "internal_FDR": internal_fdr,
            "kelm_FDR": kelm_fdr,
            "internal_direction": internal_direction,
            "kelm_direction": kelm_direction,
            "same_direction": same_direction,
            "significant_both": significant_both,
        }
    )


stability = pd.DataFrame(
    stability_rows
)


# ------------------------------------------------------------
# SAVE FULL TABLES
# ------------------------------------------------------------

results_file = (
    OUT
    / "hotspot_threshold_robustness_all_residues.csv"
)

stability_file = (
    OUT
    / "hotspot_threshold_robustness_cross_dataset.csv"
)

results.to_csv(
    results_file,
    index=False,
)

stability.to_csv(
    stability_file,
    index=False,
)


# ------------------------------------------------------------
# DISPLAY RESIDUES CURRENTLY DISCUSSED IN MANUSCRIPT
# ------------------------------------------------------------

key_residues = [
    "K",
    "R",
    "L",
    "Q",
    "M",
    "Y",
    "G",
    "F",
    "N",
]

key = stability[
    stability["residue"].isin(
        key_residues
    )
].copy()


print("\n" + "=" * 100)
print("KEY RESIDUES ACROSS ALL HOTSPOT THRESHOLDS")
print("=" * 100)

display(
    key.sort_values(
        [
            "residue",
            "threshold_pct",
        ]
    )
)


# ------------------------------------------------------------
# ROBUSTNESS SUMMARY
# ------------------------------------------------------------

summary_rows = []

for residue in key_residues:

    g = key[
        key["residue"] == residue
    ].copy()

    if len(g) == 0:
        continue

    summary_rows.append(
        {
            "residue": residue,

            "n_thresholds":
                g["threshold_pct"].nunique(),

            "same_direction_all":
                bool(
                    g["same_direction"].all()
                ),

            "significant_both_all":
                bool(
                    g["significant_both"].all()
                ),

            "internal_OR_min":
                g["internal_OR"].min(),

            "internal_OR_max":
                g["internal_OR"].max(),

            "kelm_OR_min":
                g["kelm_OR"].min(),

            "kelm_OR_max":
                g["kelm_OR"].max(),
        }
    )


summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 100)
print("ROBUSTNESS SUMMARY")
print("=" * 100)

display(summary)


# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

summary_file = (
    OUT
    / "hotspot_threshold_robustness_key_residues_summary.csv"
)

summary.to_csv(
    summary_file,
    index=False,
)


print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

print("Saved:")
print(results_file)
print(stability_file)
print(summary_file)

In [ ]:
# ============================================================
# STEP 20C: COMPOSITION-CONTROLLED MOTIF VALIDATION
#
# Question:
# Are RR, WK, WR, RW, RL, LR enriched because of their
# local ordering, or simply because R/K/etc. are already
# abundant in hotspot regions?
#
# Null model:
# Within every CPP separately:
#   - preserve peptide length
#   - preserve hotspot positions
#   - preserve hotspot amino-acid composition
#   - preserve non-hotspot amino-acid composition
#   - shuffle residues within hotspot/non-hotspot positions
#
# Thus, residue composition is held constant while local
# sequence ordering is randomized.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

DATASETS = [
    "internal_test",
    "kelm_external",
]

# Motifs already replicated in your manuscript
CANDIDATES = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]

N_SHUFFLES = 2000
SEED = 42

OUT = DIRS["results"] / "tables_SI"
OUT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# BENJAMINI-HOCHBERG
# ------------------------------------------------------------

def bh_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(p_values)

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    output = np.empty(
        n,
        dtype=float,
    )

    output[
        order
    ] = adjusted

    return output


# ------------------------------------------------------------
# DETECT MOTIF OCCURRENCE OVERLAPPING ≥1 HOTSPOT
#
# Returns True/False at peptide level.
# ------------------------------------------------------------

def motif_present_overlap(
    sequence,
    hotspot_mask,
    motif,
):

    m = len(motif)

    for start in range(
        len(sequence) - m + 1
    ):

        end = start + m

        if sequence[start:end] == motif:

            if np.any(
                hotspot_mask[start:end]
            ):

                return True

    return False


# ------------------------------------------------------------
# COMPOSITION-PRESERVING SHUFFLE
# ------------------------------------------------------------

def shuffle_sequence(
    sequence,
    hotspot_mask,
    rng,
):

    residues = np.array(
        list(sequence),
        dtype="U1",
    )

    hotspot_indices = np.flatnonzero(
        hotspot_mask
    )

    nonhotspot_indices = np.flatnonzero(
        ~hotspot_mask
    )

    # Shuffle hotspot residues only among hotspot positions
    if len(hotspot_indices) > 1:

        residues[
            hotspot_indices
        ] = rng.permutation(
            residues[
                hotspot_indices
            ]
        )

    # Shuffle non-hotspot residues only among non-hotspot positions
    if len(nonhotspot_indices) > 1:

        residues[
            nonhotspot_indices
        ] = rng.permutation(
            residues[
                nonhotspot_indices
            ]
        )

    return "".join(
        residues.tolist()
    )


# ------------------------------------------------------------
# RUN ANALYSIS
# ------------------------------------------------------------

all_rows = []


for dataset_name in DATASETS:

    print("\n" + "=" * 100)
    print("DATASET:", dataset_name)
    print("=" * 100)

    input_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    df = pd.read_csv(
        input_file
    )

    # CPPs only
    cpp = df[
        df["label"].astype(int) == 1
    ].copy()

    records = []

    # --------------------------------------------------------
    # Reconstruct individual peptide sequences + hotspot masks
    # --------------------------------------------------------

    for sequence_id, g in cpp.groupby(
        "sequence_id",
        sort=False,
    ):

        g = g.sort_values(
            "position"
        )

        sequence = str(
            g["sequence"].iloc[0]
        )

        hotspot_mask = (
            g[
                "adjusted_hotspot_top20"
            ]
            .astype(bool)
            .to_numpy()
        )

        if len(sequence) != len(hotspot_mask):

            raise ValueError(
                f"Length mismatch for {sequence_id}: "
                f"{len(sequence)} vs {len(hotspot_mask)}"
            )

        records.append(
            (
                str(sequence_id),
                sequence,
                hotspot_mask,
            )
        )

    print(
        "CPP sequences:",
        len(records)
    )


    # --------------------------------------------------------
    # OBSERVED COUNTS
    # number of CPP peptides containing motif overlapping hotspot
    # --------------------------------------------------------

    observed = {}

    for motif in CANDIDATES:

        observed[motif] = sum(

            motif_present_overlap(
                sequence,
                hotspot_mask,
                motif,
            )

            for _, sequence, hotspot_mask
            in records
        )

    print("\nObserved motif-containing CPP sequences:")

    for motif in CANDIDATES:

        print(
            motif,
            ":",
            observed[motif]
        )


    # --------------------------------------------------------
    # NULL DISTRIBUTIONS
    # --------------------------------------------------------

    null_counts = {

        motif: np.zeros(
            N_SHUFFLES,
            dtype=int,
        )

        for motif in CANDIDATES
    }


    rng = np.random.default_rng(
        SEED
        if dataset_name == "internal_test"
        else SEED + 10000
    )


    for shuffle_index in range(
        N_SHUFFLES
    ):

        current_counts = {
            motif: 0
            for motif in CANDIDATES
        }

        for (
            sequence_id,
            sequence,
            hotspot_mask,
        ) in records:

            shuffled = shuffle_sequence(
                sequence,
                hotspot_mask,
                rng,
            )

            for motif in CANDIDATES:

                if motif_present_overlap(
                    shuffled,
                    hotspot_mask,
                    motif,
                ):

                    current_counts[
                        motif
                    ] += 1


        for motif in CANDIDATES:

            null_counts[
                motif
            ][shuffle_index] = (
                current_counts[
                    motif
                ]
            )


        if (
            shuffle_index + 1
        ) % 200 == 0:

            print(
                f"Completed "
                f"{shuffle_index + 1}/"
                f"{N_SHUFFLES}"
            )


    # --------------------------------------------------------
    # EMPIRICAL STATISTICS
    # --------------------------------------------------------

    for motif in CANDIDATES:

        null_values = (
            null_counts[motif]
        )

        obs = observed[motif]

        null_mean = (
            null_values.mean()
        )

        # empirical one-sided P for enrichment
        p_empirical = (
            1
            + np.sum(
                null_values >= obs
            )
        ) / (
            N_SHUFFLES + 1
        )

        fold_over_null = (
            obs + 0.5
        ) / (
            null_mean + 0.5
        )

        all_rows.append(
            {
                "dataset":
                    dataset_name,

                "motif":
                    motif,

                "n_cpp":
                    len(records),

                "observed_sequences":
                    obs,

                "null_mean":
                    null_mean,

                "null_sd":
                    null_values.std(
                        ddof=1
                    ),

                "null_2.5pct":
                    np.quantile(
                        null_values,
                        0.025,
                    ),

                "null_97.5pct":
                    np.quantile(
                        null_values,
                        0.975,
                    ),

                "fold_over_null":
                    fold_over_null,

                "empirical_p":
                    p_empirical,
            }
        )


# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

results = pd.DataFrame(
    all_rows
)


# ------------------------------------------------------------
# FDR WITHIN EACH DATASET
# ------------------------------------------------------------

results[
    "fdr"
] = np.nan


for dataset_name, indices in results.groupby(
    "dataset"
).groups.items():

    results.loc[
        indices,
        "fdr"
    ] = bh_adjust(

        results.loc[
            indices,
            "empirical_p"
        ].values

    )


results[
    "significant_after_composition_control"
] = (
    (results["fold_over_null"] > 1)
    &
    (results["fdr"] < 0.05)
)


# ------------------------------------------------------------
# CROSS-DATASET REPLICATION
# ------------------------------------------------------------

rep_rows = []


for motif in CANDIDATES:

    g = results[
        results["motif"] == motif
    ].set_index(
        "dataset"
    )

    internal = g.loc[
        "internal_test"
    ]

    kelm = g.loc[
        "kelm_external"
    ]

    rep_rows.append(
        {
            "motif":
                motif,

            "internal_observed":
                internal[
                    "observed_sequences"
                ],

            "internal_null_mean":
                internal[
                    "null_mean"
                ],

            "internal_fold":
                internal[
                    "fold_over_null"
                ],

            "internal_FDR":
                internal[
                    "fdr"
                ],

            "kelm_observed":
                kelm[
                    "observed_sequences"
                ],

            "kelm_null_mean":
                kelm[
                    "null_mean"
                ],

            "kelm_fold":
                kelm[
                    "fold_over_null"
                ],

            "kelm_FDR":
                kelm[
                    "fdr"
                ],

            "replicated_after_composition_control":
                bool(
                    internal[
                        "significant_after_composition_control"
                    ]
                    and
                    kelm[
                        "significant_after_composition_control"
                    ]
                ),
        }
    )


replication = pd.DataFrame(
    rep_rows
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

results_file = (
    OUT
    / "motif_composition_controlled_null.csv"
)

replication_file = (
    OUT
    / "motif_composition_controlled_replication.csv"
)


results.to_csv(
    results_file,
    index=False,
)

replication.to_csv(
    replication_file,
    index=False,
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("COMPOSITION-CONTROLLED MOTIF RESULTS")
print("=" * 100)

display(
    results.sort_values(
        [
            "motif",
            "dataset",
        ]
    )
)


print("\n" + "=" * 100)
print("CROSS-DATASET REPLICATION")
print("=" * 100)

display(
    replication
)


print("\nSaved:")
print(results_file)
print(replication_file)

In [ ]:
# ============================================================
# STEP 20D-A
# SEQUENCE-LEVEL MUTATION FAITHFULNESS
# SETUP + FILE VERIFICATION
# ============================================================

from pathlib import Path
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd

import torch
import tensorflow as tf

from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# BASIC SETTINGS
# ------------------------------------------------------------

SEED = 42
MAX_LEN = 61
RANDOM_REPEATS = 20
N_BOOTSTRAP = 2000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Torch device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# PROJECT PATHS
# DIRS should already exist from previous cells,
# but redefine safely if needed.
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

DIRS = {
    "data_processed":
        PROJECT_DIR / "02_data_processed",

    "embeddings":
        PROJECT_DIR / "03_embeddings",

    "models":
        PROJECT_DIR / "04_models",

    "predictions":
        PROJECT_DIR / "05_predictions",

    "xai":
        PROJECT_DIR / "06_xai",

    "results":
        PROJECT_DIR / "07_results",

    "checkpoints":
        PROJECT_DIR / "08_checkpoints",
}


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT = (
    DIRS["results"]
    / "tables_SI"
    / "sequence_mutation_faithfulness"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Output:", OUT)


# ------------------------------------------------------------
# PLM SETTINGS
# EXACT MODELS FROM ORIGINAL PIPELINE
# ------------------------------------------------------------

MODEL_CONFIGS = {

    "ESM2_320": {
        "type":
            "esm",

        "loader":
            "esm2_t6_8M_UR50D",

        "layer":
            6,

        "embedding_dimension":
            320,

        "batch_size":
            64,
    },

    "ESM2_640": {
        "type":
            "esm",

        "loader":
            "esm2_t30_150M_UR50D",

        "layer":
            30,

        "embedding_dimension":
            640,

        "batch_size":
            24,
    },

    "ESM2_1280": {
        "type":
            "esm",

        "loader":
            "esm2_t33_650M_UR50D",

        "layer":
            33,

        "embedding_dimension":
            1280,

        "batch_size":
            8,
    },

    "ProtT5": {
        "type":
            "prott5",

        "loader":
            "Rostlab/prot_t5_xl_uniref50",

        "layer":
            None,

        "embedding_dimension":
            1024,

        "batch_size":
            8,
    },
}


DATASETS = [
    "internal_test",
    "kelm_external",
]


PREDICTION_FILENAMES = {
    "internal_test":
        "internal_test_predictions.csv",

    "kelm_external":
        "kelm_external_predictions.csv",
}


# ------------------------------------------------------------
# VERIFY ALL REQUIRED FILES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VERIFYING REQUIRED FILES")
print("=" * 100)

all_ok = True

for model_name in MODEL_CONFIGS:

    model_file = (
        DIRS["models"]
        / model_name
        / "final_attention_classifier.keras"
    )

    print(
        f"\n{model_name}"
    )

    print(
        "Classifier:",
        model_file.exists(),
        model_file
    )

    if not model_file.exists():
        all_ok = False


    for dataset_name in DATASETS:

        prediction_file = (
            DIRS["predictions"]
            / model_name
            / PREDICTION_FILENAMES[
                dataset_name
            ]
        )

        print(
            dataset_name,
            "predictions:",
            prediction_file.exists()
        )

        if not prediction_file.exists():
            all_ok = False


for dataset_name in DATASETS:

    consensus_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    print(
        "\nConsensus:",
        dataset_name,
        consensus_file.exists()
    )

    if not consensus_file.exists():
        all_ok = False


print("\n" + "=" * 100)

if all_ok:
    print("✓ ALL REQUIRED FILES FOUND")
else:
    print("✗ SOME REQUIRED FILES ARE MISSING")

print("=" * 100)

In [ ]:
# ============================================================
# STEP 20D-B
# FUNCTIONS FOR ACTUAL SEQUENCE MUTATION FAITHFULNESS
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch
import tensorflow as tf

from scipy.stats import wilcoxon


# ============================================================
# 1. CUSTOM ATTENTION LAYER
# EXACTLY MATCHES ORIGINAL CLASSIFIER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        **kwargs,
    ):

        super().__init__(
            **kwargs
        )

        self.attention_dense = (
            tf.keras.layers.Dense(
                1,
                use_bias=True,
                name="residue_attention_score",
            )
        )


    def build(
        self,
        input_shape,
    ):

        residue_shape, _ = (
            input_shape
        )

        self.attention_dense.build(
            residue_shape
        )

        super().build(
            input_shape
        )


    def call(
        self,
        inputs,
    ):

        residue_features, residue_mask = (
            inputs
        )

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            +
            (
                1.0
                - residue_mask
            )
            *
            tf.cast(
                -1e4,
                logits.dtype,
            )
        )

        attention_weights = (
            tf.nn.softmax(
                masked_logits,
                axis=1,
            )
        )

        pooled_vector = (
            tf.reduce_sum(
                residue_features
                *
                tf.expand_dims(
                    attention_weights,
                    axis=-1,
                ),
                axis=1,
            )
        )

        return pooled_vector


    def get_config(
        self,
    ):

        return super().get_config()


# ============================================================
# 2. CLEAR MEMORY
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ============================================================
# 3. ALANINE MUTATION
#
# positions are ZERO-BASED here
# ============================================================

def mutate_to_alanine(
    sequence,
    positions,
):

    residues = list(
        sequence
    )

    for position in positions:

        if residues[position] != "A":
            residues[position] = "A"

    return "".join(
        residues
    )


# ============================================================
# 4. LOAD CONSENSUS HOTSPOTS FOR CPPs
# ============================================================

def load_consensus_cpp_records(
    dataset_name,
):

    consensus_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    df = pd.read_csv(
        consensus_file
    )

    # CPP only
    df = df[
        df["label"].astype(int)
        == 1
    ].copy()


    records = {}


    for sequence_id, group in df.groupby(
        "sequence_id",
        sort=False,
    ):

        group = group.sort_values(
            "position"
        )

        sequence = str(
            group["sequence"].iloc[0]
        )

        hotspot_mask = (
            group[
                "adjusted_hotspot_top20"
            ]
            .astype(bool)
            .to_numpy()
        )


        if len(sequence) != len(
            hotspot_mask
        ):

            raise ValueError(
                f"Length mismatch for "
                f"{sequence_id}"
            )


        # ----------------------------------------------------
        # IMPORTANT:
        # Existing alanines are excluded because A -> A is
        # not an actual mutation.
        # ----------------------------------------------------

        hotspot_nonA = np.array(
            [
                i
                for i, is_hotspot
                in enumerate(
                    hotspot_mask
                )
                if (
                    is_hotspot
                    and sequence[i] != "A"
                )
            ],
            dtype=int,
        )


        nonhotspot_nonA = np.array(
            [
                i
                for i, is_hotspot
                in enumerate(
                    hotspot_mask
                )
                if (
                    (not is_hotspot)
                    and sequence[i] != "A"
                )
            ],
            dtype=int,
        )


        records[
            str(sequence_id)
        ] = {

            "sequence":
                sequence,

            "hotspot_mask":
                hotspot_mask,

            "hotspot_nonA":
                hotspot_nonA,

            "nonhotspot_nonA":
                nonhotspot_nonA,
        }


    return records


# ============================================================
# 5. LOAD CORRECTLY CLASSIFIED CPP IDs FOR ONE MODEL
# ============================================================

def load_correct_cpp_predictions(
    model_name,
    dataset_name,
):

    prediction_file = (
        DIRS["predictions"]
        / model_name
        / PREDICTION_FILENAMES[
            dataset_name
        ]
    )

    df = pd.read_csv(
        prediction_file
    )


    required = [
        "sequence_id",
        "sequence",
        "label",
        "probability_CPP",
        "predicted_label",
        "correct_prediction",
    ]


    missing = [
        column
        for column in required
        if column not in df.columns
    ]


    if missing:

        raise KeyError(
            f"{prediction_file}\n"
            f"Missing columns: {missing}\n"
            f"Available: {df.columns.tolist()}"
        )


    cpp = df[
        (df["label"].astype(int) == 1)
        &
        (df["correct_prediction"].astype(bool))
        &
        (df["predicted_label"].astype(int) == 1)
    ].copy()


    cpp[
        "sequence_id"
    ] = (
        cpp[
            "sequence_id"
        ].astype(str)
    )


    return cpp


# ============================================================
# 6. BUILD MUTATION PANEL
#
# One hotspot mutant + 20 matched random mutants per peptide.
# Random mutations have EXACTLY the same number of actual
# non-A substitutions as hotspot mutation.
# ============================================================

def build_mutation_panel(
    model_name,
    dataset_name,
    random_repeats=20,
    seed=42,
):

    consensus_records = (
        load_consensus_cpp_records(
            dataset_name
        )
    )


    prediction_cpp = (
        load_correct_cpp_predictions(
            model_name,
            dataset_name,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    panel_rows = []
    sequence_info = []


    for row in prediction_cpp.itertuples(
        index=False
    ):

        sequence_id = str(
            row.sequence_id
        )

        if sequence_id not in (
            consensus_records
        ):
            continue


        rec = consensus_records[
            sequence_id
        ]


        sequence = rec[
            "sequence"
        ]


        hotspot_positions = rec[
            "hotspot_nonA"
        ]


        control_positions = rec[
            "nonhotspot_nonA"
        ]


        n_mutations = len(
            hotspot_positions
        )


        # Need at least one true hotspot mutation.
        if n_mutations == 0:
            continue


        # Need sufficient non-hotspot positions for
        # matched random control.
        if len(control_positions) < (
            n_mutations
        ):
            continue


        original_probability = float(
            row.probability_CPP
        )


        # ----------------------------------------------------
        # Hotspot alanine mutant
        # ----------------------------------------------------

        hotspot_sequence = (
            mutate_to_alanine(
                sequence,
                hotspot_positions,
            )
        )


        panel_rows.append(
            {
                "sequence_id":
                    sequence_id,

                "variant_type":
                    "hotspot",

                "repeat":
                    -1,

                "sequence":
                    hotspot_sequence,

                "n_mutations":
                    n_mutations,
            }
        )


        # ----------------------------------------------------
        # Matched random mutants
        # ----------------------------------------------------

        for repeat in range(
            random_repeats
        ):

            selected = rng.choice(
                control_positions,
                size=n_mutations,
                replace=False,
            )


            random_sequence = (
                mutate_to_alanine(
                    sequence,
                    selected,
                )
            )


            panel_rows.append(
                {
                    "sequence_id":
                        sequence_id,

                    "variant_type":
                        "random",

                    "repeat":
                        repeat,

                    "sequence":
                        random_sequence,

                    "n_mutations":
                        n_mutations,
                }
            )


        sequence_info.append(
            {
                "sequence_id":
                    sequence_id,

                "original_sequence":
                    sequence,

                "original_probability":
                    original_probability,

                "n_hotspot_nonA":
                    n_mutations,

                "n_available_random_nonA":
                    len(
                        control_positions
                    ),
            }
        )


    panel = pd.DataFrame(
        panel_rows
    )

    info = pd.DataFrame(
        sequence_info
    )


    return (
        panel,
        info,
    )


# ============================================================
# 7. LOAD ESM MODEL
# ============================================================

def load_esm_model(
    config,
):

    import esm

    loader = getattr(
        esm.pretrained,
        config["loader"],
    )

    model, alphabet = loader()

    model = (
        model
        .to(
            DEVICE
        )
        .eval()
    )

    batch_converter = (
        alphabet
        .get_batch_converter()
    )


    return (
        model,
        batch_converter,
    )


# ============================================================
# 8. LOAD PROTT5
# ============================================================

def load_prott5_model(
    config,
):

    from transformers import (
        T5Tokenizer,
        T5EncoderModel,
    )


    tokenizer = (
        T5Tokenizer
        .from_pretrained(
            config["loader"],
            do_lower_case=False,
        )
    )


    dtype = (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )


    model = (
        T5EncoderModel
        .from_pretrained(
            config["loader"],
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
        )
        .to(
            DEVICE
        )
        .eval()
    )


    return (
        model,
        tokenizer,
    )


# ============================================================
# 9. EMBED ESM SEQUENCES
# ============================================================

@torch.no_grad()
def embed_esm_sequences(
    sequences,
    model,
    batch_converter,
    layer,
    embedding_dimension,
):

    items = [
        (
            f"seq_{i}",
            sequence,
        )
        for i, sequence
        in enumerate(
            sequences
        )
    ]


    _, raw_sequences, tokens = (
        batch_converter(
            items
        )
    )


    tokens = tokens.to(
        DEVICE
    )


    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=torch.cuda.is_available(),
    ):

        outputs = model(
            tokens,
            repr_layers=[
                layer
            ],
            return_contacts=False,
        )


    representations = (
        outputs[
            "representations"
        ][
            layer
        ]
    )


    X = np.zeros(
        (
            len(sequences),
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )


    M = np.zeros(
        (
            len(sequences),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )


    for i, sequence in enumerate(
        raw_sequences
    ):

        length = min(
            len(sequence),
            MAX_LEN,
        )


        X[
            i,
            :length,
            :
        ] = (
            representations[
                i,
                1:length + 1,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(
                np.float16
            )
        )


        M[
            i,
            :length
        ] = 1


    del outputs
    del representations
    del tokens

    clear_memory()


    return (
        X,
        M,
    )


# ============================================================
# 10. EMBED PROTT5 SEQUENCES
# ============================================================

@torch.no_grad()
def embed_prott5_sequences(
    sequences,
    model,
    tokenizer,
    embedding_dimension,
):

    # ProtT5 expects spaces between amino acids.
    spaced = [
        " ".join(
            list(sequence)
        )
        for sequence
        in sequences
    ]


    encoded = tokenizer(
        spaced,
        add_special_tokens=True,
        padding=True,
        return_tensors="pt",
    )


    input_ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    attention_mask = (
        encoded[
            "attention_mask"
        ]
        .to(
            DEVICE
        )
    )


    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=torch.cuda.is_available(),
    ):

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


    representations = (
        outputs.last_hidden_state
    )


    X = np.zeros(
        (
            len(sequences),
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )


    M = np.zeros(
        (
            len(sequences),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )


    for i, sequence in enumerate(
        sequences
    ):

        length = min(
            len(sequence),
            MAX_LEN,
        )


        # ProtT5 output starts directly with residue 1.
        X[
            i,
            :length,
            :
        ] = (
            representations[
                i,
                :length,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(
                np.float16
            )
        )


        M[
            i,
            :length
        ] = 1


    del outputs
    del representations
    del input_ids
    del attention_mask

    clear_memory()


    return (
        X,
        M,
    )


# ============================================================
# 11. PREDICT MUTATED SEQUENCES IN BATCHES
# ============================================================

def predict_mutation_panel(
    panel,
    model_name,
    config,
    plm,
    auxiliary,
    classifier,
):

    probabilities = np.zeros(
        len(panel),
        dtype=float,
    )


    batch_size = (
        config[
            "batch_size"
        ]
    )


    for start in range(
        0,
        len(panel),
        batch_size,
    ):

        stop = min(
            start + batch_size,
            len(panel),
        )


        batch_sequences = (
            panel.iloc[
                start:stop
            ][
                "sequence"
            ]
            .astype(str)
            .tolist()
        )


        if config[
            "type"
        ] == "esm":

            X, M = (
                embed_esm_sequences(
                    sequences=batch_sequences,
                    model=plm,
                    batch_converter=auxiliary,
                    layer=config[
                        "layer"
                    ],
                    embedding_dimension=config[
                        "embedding_dimension"
                    ],
                )
            )

        else:

            X, M = (
                embed_prott5_sequences(
                    sequences=batch_sequences,
                    model=plm,
                    tokenizer=auxiliary,
                    embedding_dimension=config[
                        "embedding_dimension"
                    ],
                )
            )


        # Classifier is tiny relative to PLM.
        batch_probability = (
            classifier(
                [
                    X,
                    M,
                ],
                training=False,
            )
            .numpy()
            .reshape(-1)
        )


        probabilities[
            start:stop
        ] = batch_probability


        del X
        del M
        del batch_probability

        clear_memory()


        if (
            stop % 500 == 0
            or stop == len(panel)
        ):

            print(
                f"  predicted "
                f"{stop}/{len(panel)} variants"
            )


    return probabilities


# ============================================================
# 12. SUMMARIZE MUTATION RESULTS PER SEQUENCE
# ============================================================

def summarize_mutation_panel(
    panel,
    info,
):

    merged = panel.merge(
        info,
        on="sequence_id",
        how="left",
        validate="many_to_one",
    )


    rows = []


    for sequence_id, g in merged.groupby(
        "sequence_id",
        sort=False,
    ):

        original_probability = float(
            g[
                "original_probability"
            ].iloc[0]
        )


        hotspot_values = (
            g[
                g["variant_type"]
                == "hotspot"
            ][
                "mutated_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )


        random_values = (
            g[
                g["variant_type"]
                == "random"
            ][
                "mutated_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )


        if (
            len(hotspot_values) != 1
            or len(random_values) == 0
        ):
            continue


        hotspot_probability = float(
            hotspot_values[0]
        )


        random_probability_mean = float(
            np.mean(
                random_values
            )
        )


        delta_hotspot = (
            original_probability
            - hotspot_probability
        )


        random_drops = (
            original_probability
            - random_values
        )


        delta_random = float(
            np.mean(
                random_drops
            )
        )


        faithfulness = (
            delta_hotspot
            - delta_random
        )


        rows.append(
            {
                "sequence_id":
                    sequence_id,

                "original_probability":
                    original_probability,

                "hotspot_mutant_probability":
                    hotspot_probability,

                "random_mutant_probability_mean":
                    random_probability_mean,

                "delta_hotspot":
                    delta_hotspot,

                "delta_random":
                    delta_random,

                "faithfulness_effect":
                    faithfulness,

                "positive_faithfulness":
                    faithfulness > 0,

                "n_mutations":
                    int(
                        g[
                            "n_mutations"
                        ].iloc[0]
                    ),

                "n_random_repeats":
                    len(
                        random_values
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 13. COHEN'S dz
# ============================================================

def paired_cohens_dz(
    hotspot,
    random,
):

    diff = (
        np.asarray(
            hotspot,
            dtype=float
        )
        -
        np.asarray(
            random,
            dtype=float
        )
    )


    if len(diff) < 2:
        return np.nan


    sd = np.std(
        diff,
        ddof=1,
    )


    if sd == 0:
        return np.nan


    return float(
        np.mean(diff)
        / sd
    )


# ============================================================
# 14. MATCHED-PAIRS RANK-BISERIAL CORRELATION
#
# Based on signed ranks of paired differences.
# ============================================================

def paired_rank_biserial(
    hotspot,
    random,
):

    diff = (
        np.asarray(
            hotspot,
            dtype=float
        )
        -
        np.asarray(
            random,
            dtype=float
        )
    )


    diff = diff[
        diff != 0
    ]


    if len(diff) == 0:
        return np.nan


    abs_diff = np.abs(
        diff
    )


    from scipy.stats import rankdata

    ranks = rankdata(
        abs_diff
    )


    positive = np.sum(
        ranks[
            diff > 0
        ]
    )


    negative = np.sum(
        ranks[
            diff < 0
        ]
    )


    total = (
        positive
        + negative
    )


    return float(
        (
            positive
            - negative
        )
        / total
    )


# ============================================================
# 15. BOOTSTRAP CI
# ============================================================

def bootstrap_mean_ci(
    values,
    n_bootstrap=2000,
    seed=42,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    rng = np.random.default_rng(
        seed
    )


    means = np.empty(
        n_bootstrap,
        dtype=float,
    )


    n = len(
        values
    )


    for i in range(
        n_bootstrap
    ):

        sample = rng.choice(
            values,
            size=n,
            replace=True,
        )

        means[i] = np.mean(
            sample
        )


    return (
        float(
            np.quantile(
                means,
                0.025
            )
        ),
        float(
            np.quantile(
                means,
                0.975
            )
        ),
    )


# ============================================================
# 16. FINAL STATISTICAL SUMMARY
# ============================================================

def calculate_mutation_statistics(
    per_sequence,
    model_name,
    dataset_name,
):

    hotspot = (
        per_sequence[
            "delta_hotspot"
        ]
        .to_numpy(
            dtype=float
        )
    )


    random_control = (
        per_sequence[
            "delta_random"
        ]
        .to_numpy(
            dtype=float
        )
    )


    faithfulness = (
        hotspot
        - random_control
    )


    if len(
        per_sequence
    ) == 0:

        raise ValueError(
            "No sequences available "
            "for statistical analysis."
        )


    try:

        wilcoxon_result = (
            wilcoxon(
                hotspot,
                random_control,
                alternative="two-sided",
                zero_method="wilcox",
            )
        )

        statistic = float(
            wilcoxon_result.statistic
        )

        p_value = float(
            wilcoxon_result.pvalue
        )

    except ValueError:

        statistic = np.nan
        p_value = np.nan


    dz = paired_cohens_dz(
        hotspot,
        random_control,
    )


    rbc = paired_rank_biserial(
        hotspot,
        random_control,
    )


    hotspot_ci = bootstrap_mean_ci(
        hotspot,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
    )


    random_ci = bootstrap_mean_ci(
        random_control,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED + 1,
    )


    faith_ci = bootstrap_mean_ci(
        faithfulness,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED + 2,
    )


    return {

        "model":
            model_name,

        "dataset":
            dataset_name,

        "n_sequences":
            len(
                per_sequence
            ),

        "mean_delta_hotspot":
            float(
                np.mean(
                    hotspot
                )
            ),

        "hotspot_CI_low":
            hotspot_ci[0],

        "hotspot_CI_high":
            hotspot_ci[1],

        "mean_delta_random":
            float(
                np.mean(
                    random_control
                )
            ),

        "random_CI_low":
            random_ci[0],

        "random_CI_high":
            random_ci[1],

        "mean_faithfulness_effect":
            float(
                np.mean(
                    faithfulness
                )
            ),

        "faithfulness_CI_low":
            faith_ci[0],

        "faithfulness_CI_high":
            faith_ci[1],

        "median_faithfulness_effect":
            float(
                np.median(
                    faithfulness
                )
            ),

        "positive_faithfulness_fraction":
            float(
                np.mean(
                    faithfulness > 0
                )
            ),

        "wilcoxon_statistic":
            statistic,

        "wilcoxon_p":
            p_value,

        "paired_cohens_dz":
            dz,

        "rank_biserial":
            rbc,
    }


print(
    "✓ Sequence-mutation functions loaded"
)

In [ ]:
# ============================================================
# INSTALL ESM DEPENDENCY
# ============================================================

!pip -q install fair-esm==2.0.0

import esm

print("fair-esm version:", esm.__version__)
print("✓ ESM successfully installed")

In [ ]:
# ============================================================
# STEP 20D-C
# RUN ACTUAL SEQUENCE-LEVEL MUTATION FAITHFULNESS
#
# IMPORTANT:
# This can take substantially longer than the previous analyses.
#
# Each completed model/dataset pair is saved immediately.
# ============================================================

import gc
import json
import numpy as np
import pandas as pd
import torch
import tensorflow as tf


# ------------------------------------------------------------
# RESULT COLLECTION
# ------------------------------------------------------------

summary_rows = []


# ------------------------------------------------------------
# RUN ONE PLM AT A TIME
# ------------------------------------------------------------

for model_index, (
    model_name,
    config,
) in enumerate(
    MODEL_CONFIGS.items()
):

    print("\n\n" + "#" * 100)
    print(
        f"MODEL {model_index + 1}/"
        f"{len(MODEL_CONFIGS)}: "
        f"{model_name}"
    )
    print("#" * 100)


    # ========================================================
    # LOAD SAVED CLASSIFIER
    # ========================================================

    classifier_file = (
        DIRS["models"]
        / model_name
        / "final_attention_classifier.keras"
    )


    print(
        "Loading classifier:",
        classifier_file
    )


    classifier = (
        tf.keras.models.load_model(
            classifier_file,
            custom_objects={
                "MaskedAttentionPooling":
                    MaskedAttentionPooling,
            },
            compile=False,
        )
    )


    print(
        "✓ Classifier loaded"
    )


    # ========================================================
    # LOAD PLM
    # ========================================================

    print(
        "Loading pretrained PLM..."
    )


    if config[
        "type"
    ] == "esm":

        plm, auxiliary = (
            load_esm_model(
                config
            )
        )

    else:

        plm, auxiliary = (
            load_prott5_model(
                config
            )
        )


    print(
        "✓ PLM loaded"
    )


    # ========================================================
    # DATASETS
    # ========================================================

    for dataset_index, dataset_name in enumerate(
        DATASETS
    ):

        print("\n" + "=" * 100)

        print(
            f"{model_name} | "
            f"{dataset_name}"
        )

        print("=" * 100)


        # ----------------------------------------------------
        # RESTART-SAFE FILES
        # ----------------------------------------------------

        prefix = (
            f"{model_name}_"
            f"{dataset_name}"
        )


        per_sequence_file = (
            OUT
            / f"{prefix}_per_sequence.csv"
        )


        panel_file = (
            OUT
            / f"{prefix}_all_mutants.csv"
        )


        summary_file = (
            OUT
            / f"{prefix}_summary.json"
        )


        # ----------------------------------------------------
        # SKIP COMPLETED
        # ----------------------------------------------------

        if (
            per_sequence_file.exists()
            and summary_file.exists()
        ):

            print(
                "✓ Already completed — skipping"
            )


            with open(
                summary_file,
                "r",
                encoding="utf-8",
            ) as handle:

                existing_summary = (
                    json.load(
                        handle
                    )
                )


            summary_rows.append(
                existing_summary
            )

            continue


        # ----------------------------------------------------
        # BUILD ACTUAL MUTATION PANEL
        # ----------------------------------------------------

        panel, info = (
            build_mutation_panel(
                model_name=model_name,
                dataset_name=dataset_name,
                random_repeats=RANDOM_REPEATS,
                seed=(
                    SEED
                    + model_index * 1000
                    + dataset_index * 100
                ),
            )
        )


        print(
            "Correctly classified CPPs "
            "retained after mutation matching:",
            len(info)
        )


        print(
            "Total mutant sequences to evaluate:",
            len(panel)
        )


        if len(
            info
        ) == 0:

            raise ValueError(
                f"No eligible sequences for "
                f"{model_name}/{dataset_name}"
            )


        print(
            "\nMutation count distribution:"
        )

        display(
            info[
                "n_hotspot_nonA"
            ].describe()
        )


        # ----------------------------------------------------
        # PREDICT ALL MUTANTS
        # ----------------------------------------------------

        probabilities = (
            predict_mutation_panel(
                panel=panel,
                model_name=model_name,
                config=config,
                plm=plm,
                auxiliary=auxiliary,
                classifier=classifier,
            )
        )


        panel[
            "mutated_probability"
        ] = probabilities


        # ----------------------------------------------------
        # PER-SEQUENCE SUMMARY
        # ----------------------------------------------------

        per_sequence = (
            summarize_mutation_panel(
                panel=panel,
                info=info,
            )
        )


        # Add identifying information
        per_sequence[
            "model"
        ] = model_name

        per_sequence[
            "dataset"
        ] = dataset_name


        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        summary = (
            calculate_mutation_statistics(
                per_sequence=per_sequence,
                model_name=model_name,
                dataset_name=dataset_name,
            )
        )


        # ----------------------------------------------------
        # SAVE IMMEDIATELY
        # ----------------------------------------------------

        panel.to_csv(
            panel_file,
            index=False,
        )


        per_sequence.to_csv(
            per_sequence_file,
            index=False,
        )


        with open(
            summary_file,
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                summary,
                handle,
                indent=2,
            )


        summary_rows.append(
            summary
        )


        # ----------------------------------------------------
        # DISPLAY THIS RESULT
        # ----------------------------------------------------

        print("\n" + "-" * 100)

        print(
            "RESULT:",
            model_name,
            dataset_name
        )

        print("-" * 100)


        print(
            "N sequences:",
            summary[
                "n_sequences"
            ]
        )


        print(
            "Mean hotspot mutation Δp:",
            round(
                summary[
                    "mean_delta_hotspot"
                ],
                4,
            )
        )


        print(
            "Mean random mutation Δp:",
            round(
                summary[
                    "mean_delta_random"
                ],
                4,
            )
        )


        print(
            "Mean faithfulness:",
            round(
                summary[
                    "mean_faithfulness_effect"
                ],
                4,
            )
        )


        print(
            "95% CI faithfulness:",
            (
                round(
                    summary[
                        "faithfulness_CI_low"
                    ],
                    4,
                ),
                round(
                    summary[
                        "faithfulness_CI_high"
                    ],
                    4,
                ),
            )
        )


        print(
            "Positive faithfulness:",
            round(
                100
                *
                summary[
                    "positive_faithfulness_fraction"
                ],
                1,
            ),
            "%"
        )


        print(
            "Wilcoxon P:",
            summary[
                "wilcoxon_p"
            ]
        )


        print(
            "Cohen's dz:",
            round(
                summary[
                    "paired_cohens_dz"
                ],
                3,
            )
        )


        print(
            "Rank-biserial:",
            round(
                summary[
                    "rank_biserial"
                ],
                3,
            )
        )


        # ----------------------------------------------------
        # RELEASE TEMPORARY DATA
        # ----------------------------------------------------

        del panel
        del info
        del probabilities
        del per_sequence

        clear_memory()


    # ========================================================
    # RELEASE THIS PLM BEFORE LOADING NEXT ONE
    # ========================================================

    print(
        f"\nReleasing {model_name}..."
    )


    del plm
    del auxiliary
    del classifier

    tf.keras.backend.clear_session()

    clear_memory()


# ============================================================
# COMBINE SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)


summary_df = (
    summary_df
    .drop_duplicates(
        subset=[
            "model",
            "dataset",
        ],
        keep="last",
    )
    .sort_values(
        [
            "dataset",
            "model",
        ]
    )
    .reset_index(
        drop=True
    )
)


combined_summary_file = (
    OUT
    / "sequence_level_mutation_faithfulness_summary.csv"
)


summary_df.to_csv(
    combined_summary_file,
    index=False,
)


print("\n\n" + "=" * 120)
print("FINAL SEQUENCE-LEVEL MUTATION FAITHFULNESS")
print("=" * 120)


display(
    summary_df[
        [
            "model",
            "dataset",
            "n_sequences",
            "mean_delta_hotspot",
            "mean_delta_random",
            "mean_faithfulness_effect",
            "faithfulness_CI_low",
            "faithfulness_CI_high",
            "positive_faithfulness_fraction",
            "wilcoxon_p",
            "paired_cohens_dz",
            "rank_biserial",
        ]
    ]
)


print(
    "\nSaved combined summary:"
)

print(
    combined_summary_file
)

In [ ]:
# ======================================================================
# TABLE Sz
# Statistical summary of embedding- and sequence-level
# perturbation-based faithfulness analyses
#
# Part A = Embedding-level ablation
# Part B = Sequence-level alanine scanning
#
# Output:
#   Table_Sz_Perturbation_Faithfulness.xlsx
#   Table_SzA_Embedding_Ablation.csv
#   Table_SzB_Sequence_Alanine_Scanning.csv
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    wilcoxon,
    rankdata,
)


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

FAITH_DIR = (
    PROJECT
    / "06_xai"
    / "faithfulness"
)

MUTATION_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
    / "sequence_mutation_faithfulness"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MUTATION_SUMMARY_FILE = (
    MUTATION_DIR
    / "sequence_level_mutation_faithfulness_summary.csv"
)


# ======================================================================
# MODELS / DATASETS
# ======================================================================

MODELS = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_LABELS = {
    "ESM2_320":
        "ESM2-320",

    "ESM2_640":
        "ESM2-640",

    "ESM2_1280":
        "ESM2-1280",

    "ProtT5":
        "ProtT5",
}


DATASETS = [
    "internal_test",
    "kelm_external",
]

DATASET_LABELS = {
    "internal_test":
        "Internal",

    "kelm_external":
        "KELM",
}


# ======================================================================
# HELPER: FIND COLUMN
# ======================================================================

def find_col(
    df,
    candidates,
    contains=None,
):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }


    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]


    if contains:

        for c in df.columns:

            lc = c.lower()

            if all(
                item.lower() in lc
                for item in contains
            ):

                return c


    return None


# ======================================================================
# PART A
# EMBEDDING-LEVEL ABLATION
# ======================================================================

embedding_rows = []


for model in MODELS:

    for dataset in DATASETS:

        file = (
            FAITH_DIR
            / model
            / dataset
            / "per_sequence_faithfulness.csv"
        )


        assert file.exists(), (
            f"Missing file:\n{file}"
        )


        df = pd.read_csv(
            file
        )


        # --------------------------------------------------------------
        # DETECT COLUMNS
        # --------------------------------------------------------------

        hotspot_col = find_col(
            df,

            [
                "hotspot_drop",
                "consensus_hotspot_drop",
                "hotspot_probability_drop",
                "hotspot_ablation_drop",
                "delta_hotspot",
            ],

            contains=[
                "hotspot",
                "drop",
            ],
        )


        random_col = find_col(
            df,

            [
                "random_drop",
                "matched_random_drop",
                "random_probability_drop",
                "random_ablation_drop",
                "mean_random_drop",
                "delta_random",
            ],

            contains=[
                "random",
                "drop",
            ],
        )


        faith_col = find_col(
            df,

            [
                "faithfulness",
                "faithfulness_effect",
                "paired_faithfulness",
            ],
        )


        if hotspot_col is None:

            raise KeyError(
                f"Could not detect hotspot-drop column in:\n"
                f"{file}\n\n"
                f"Columns:\n{df.columns.tolist()}"
            )


        if random_col is None:

            raise KeyError(
                f"Could not detect random-drop column in:\n"
                f"{file}\n\n"
                f"Columns:\n{df.columns.tolist()}"
            )


        print(
            f"\n{model} | {dataset}"
        )

        print(
            "Hotspot column:",
            hotspot_col
        )

        print(
            "Random column :",
            random_col
        )


        # --------------------------------------------------------------
        # NUMERIC DATA
        # --------------------------------------------------------------

        hotspot_series = pd.to_numeric(
            df[
                hotspot_col
            ],
            errors="coerce",
        )


        random_series = pd.to_numeric(
            df[
                random_col
            ],
            errors="coerce",
        )


        valid = (
            hotspot_series.notna()
            &
            random_series.notna()
        )


        hotspot = (
            hotspot_series[
                valid
            ]
            .to_numpy(
                dtype=float
            )
        )


        random_control = (
            random_series[
                valid
            ]
            .to_numpy(
                dtype=float
            )
        )


        # --------------------------------------------------------------
        # FAITHFULNESS
        # --------------------------------------------------------------

        differences = (
            hotspot
            - random_control
        )


        if faith_col is not None:

            faithfulness = (
                pd.to_numeric(
                    df.loc[
                        valid,
                        faith_col,
                    ],
                    errors="coerce",
                )
                .to_numpy(
                    dtype=float
                )
            )


            # if any NaNs remain, use directly calculated difference
            if np.any(
                ~np.isfinite(
                    faithfulness
                )
            ):

                faithfulness = (
                    differences.copy()
                )

        else:

            faithfulness = (
                differences.copy()
            )


        # --------------------------------------------------------------
        # WILCOXON
        # two-sided to match manuscript methods
        # --------------------------------------------------------------

        try:

            wilcox = wilcoxon(
                hotspot,
                random_control,

                alternative="two-sided",
                zero_method="wilcox",
            )

            p_value = float(
                wilcox.pvalue
            )

        except ValueError:

            p_value = np.nan


        # --------------------------------------------------------------
        # PAIRED COHEN'S dz
        # --------------------------------------------------------------

        if len(
            differences
        ) > 1:

            diff_sd = np.std(
                differences,
                ddof=1,
            )

        else:

            diff_sd = np.nan


        if (
            np.isfinite(
                diff_sd
            )
            and diff_sd > 0
        ):

            cohens_dz = (
                np.mean(
                    differences
                )
                / diff_sd
            )

        else:

            cohens_dz = np.nan


        # --------------------------------------------------------------
        # MATCHED-PAIRS RANK-BISERIAL
        # --------------------------------------------------------------

        nonzero = differences[
            differences != 0
        ]


        if len(
            nonzero
        ) > 0:

            ranks = rankdata(
                np.abs(
                    nonzero
                )
            )


            W_pos = np.sum(
                ranks[
                    nonzero > 0
                ]
            )


            W_neg = np.sum(
                ranks[
                    nonzero < 0
                ]
            )


            denominator = (
                W_pos
                + W_neg
            )


            if denominator > 0:

                rank_biserial = (
                    W_pos
                    - W_neg
                ) / denominator

            else:

                rank_biserial = np.nan

        else:

            rank_biserial = np.nan


        # --------------------------------------------------------------
        # ROW
        # --------------------------------------------------------------

        embedding_rows.append(
            {

                "PLM":
                    MODEL_LABELS[
                        model
                    ],

                "Dataset":
                    DATASET_LABELS[
                        dataset
                    ],

                "n":
                    len(
                        differences
                    ),

                "Mean Δp hotspot":
                    float(
                        np.mean(
                            hotspot
                        )
                    ),

                "Mean Δp random":
                    float(
                        np.mean(
                            random_control
                        )
                    ),

                "Mean F":
                    float(
                        np.mean(
                            faithfulness
                        )
                    ),

                "Wilcoxon P":
                    p_value,

                "Cohen's dz":
                    cohens_dz,

                "Rank-biserial r":
                    rank_biserial,
            }
        )


table_A = pd.DataFrame(
    embedding_rows
)


# ======================================================================
# PART B
# SEQUENCE-LEVEL ALANINE SCANNING
# ======================================================================

assert MUTATION_SUMMARY_FILE.exists(), (
    f"Mutation summary not found:\n"
    f"{MUTATION_SUMMARY_FILE}"
)


mutation = pd.read_csv(
    MUTATION_SUMMARY_FILE
)


print(
    "\n"
    + "=" * 100
)

print(
    "MUTATION SUMMARY COLUMNS"
)

print(
    "=" * 100
)

print(
    mutation.columns.tolist()
)


# ======================================================================
# BUILD PART B
# ======================================================================

sequence_rows = []


for _, row in mutation.iterrows():

    model = row[
        "model"
    ]

    dataset = row[
        "dataset"
    ]


    sequence_rows.append(
        {

            "PLM":
                MODEL_LABELS.get(
                    model,
                    model,
                ),

            "Dataset":
                DATASET_LABELS.get(
                    dataset,
                    dataset,
                ),

            "n":
                int(
                    row[
                        "n_sequences"
                    ]
                ),

            "Mean Δp hotspot":
                float(
                    row[
                        "mean_delta_hotspot"
                    ]
                ),

            "Mean Δp random":
                float(
                    row[
                        "mean_delta_random"
                    ]
                ),

            "Mean F":
                float(
                    row[
                        "mean_faithfulness_effect"
                    ]
                ),

            "95% CI":
                (
                    f"{row['faithfulness_CI_low']:.3f}"
                    f"–"
                    f"{row['faithfulness_CI_high']:.3f}"
                ),

            "Positive F (%)":
                (
                    100
                    * float(
                        row[
                            "positive_faithfulness_fraction"
                        ]
                    )
                ),

            "Wilcoxon P":
                float(
                    row[
                        "wilcoxon_p"
                    ]
                ),

            # CORRECT COLUMN NAME
            "Cohen's dz":
                float(
                    row[
                        "paired_cohens_dz"
                    ]
                ),

            "Rank-biserial r":
                float(
                    row[
                        "rank_biserial"
                    ]
                ),
        }
    )


table_B = pd.DataFrame(
    sequence_rows
)


# ======================================================================
# ROW ORDER
# ======================================================================

plm_order = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
]

dataset_order = [
    "Internal",
    "KELM",
]


for table in [
    table_A,
    table_B,
]:

    table[
        "PLM"
    ] = pd.Categorical(
        table[
            "PLM"
        ],

        categories=plm_order,

        ordered=True,
    )


    table[
        "Dataset"
    ] = pd.Categorical(
        table[
            "Dataset"
        ],

        categories=dataset_order,

        ordered=True,
    )


table_A = (
    table_A
    .sort_values(
        [
            "Dataset",
            "PLM",
        ]
    )
    .reset_index(
        drop=True
    )
)


table_B = (
    table_B
    .sort_values(
        [
            "Dataset",
            "PLM",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ======================================================================
# ROUND NUMERIC VALUES
# ======================================================================

numeric_cols_A = [
    "Mean Δp hotspot",
    "Mean Δp random",
    "Mean F",
    "Cohen's dz",
    "Rank-biserial r",
]


for col in numeric_cols_A:

    table_A[
        col
    ] = (
        table_A[
            col
        ]
        .astype(float)
        .round(3)
    )


numeric_cols_B = [
    "Mean Δp hotspot",
    "Mean Δp random",
    "Mean F",
    "Positive F (%)",
    "Cohen's dz",
    "Rank-biserial r",
]


for col in numeric_cols_B:

    table_B[
        col
    ] = (
        table_B[
            col
        ]
        .astype(float)
        .round(3)
    )


# ======================================================================
# FORMAT P VALUES
# ======================================================================

def format_p(
    p
):

    if pd.isna(
        p
    ):

        return ""


    p = float(
        p
    )


    if p < 0.001:

        return (
            f"{p:.2e}"
        )


    return (
        f"{p:.3f}"
    )


table_A[
    "Wilcoxon P"
] = table_A[
    "Wilcoxon P"
].apply(
    format_p
)


table_B[
    "Wilcoxon P"
] = table_B[
    "Wilcoxon P"
].apply(
    format_p
)


# ======================================================================
# DISPLAY
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TABLE SzA — EMBEDDING-LEVEL ABLATION"
)

print(
    "=" * 110
)

display(
    table_A
)


print(
    "\n"
    + "=" * 110
)

print(
    "TABLE SzB — SEQUENCE-LEVEL ALANINE SCANNING"
)

print(
    "=" * 110
)

display(
    table_B
)


# ======================================================================
# SAVE CSV FILES
# ======================================================================

CSV_A = (
    OUT_DIR
    / "Table_SzA_Embedding_Ablation.csv"
)

CSV_B = (
    OUT_DIR
    / "Table_SzB_Sequence_Alanine_Scanning.csv"
)


table_A.to_csv(
    CSV_A,
    index=False,
)


table_B.to_csv(
    CSV_B,
    index=False,
)


# ======================================================================
# SAVE COMBINED EXCEL FILE
# ======================================================================

XLSX_FILE = (
    OUT_DIR
    / "Table_Sz_Perturbation_Faithfulness.xlsx"
)


with pd.ExcelWriter(
    XLSX_FILE,
    engine="openpyxl",
) as writer:

    table_A.to_excel(
        writer,

        sheet_name="A_Embedding_Ablation",

        index=False,
    )


    table_B.to_excel(
        writer,

        sheet_name="B_Alanine_Scanning",

        index=False,
    )


# ======================================================================
# DONE
# ======================================================================

print(
    "\nSaved:"
)

print(
    CSV_A
)

print(
    CSV_B
)

print(
    XLSX_FILE
)